## Setup

Run these three cells first: the imports, the relative-error helpers, and the cell that runs every `helper`-tagged cell further down. After them any section below runs on its own.

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
import matplotlib.pyplot as plt
# from data_preparation.field import load_fields

processed_data_dir = Path("../processed_data")

# Palette and pixel size of the figures (each plot cell repeats the ones it uses).
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"
PLOT4_COLOR = "#2b5d8a"
px = 1/plt.rcParams['figure.dpi']  # pixel in inches


In [ ]:
# --- Relative rollout errors --------------------------------------------------
# Every rollout error plot shows the relative errors of CLAUDE.md, at each rollout
# step t, with sums over the nodes i:
#   RMAE_t = sum_i |T_pred - T_true| / sum_i |T_true|
#   RMSE_t = sum_i (T_pred - T_true)^2 / sum_i T_true^2
# The runs save the MAE per step in kelvin (and the notebook's own rollouts the
# root-mean-square error), so RMAE_t = MAE_t / mean_i |T_true| and
# RMSE_t = RMSE_t[K]^2 / mean_i T_true^2, with T_true the field step t is compared
# with: T_seq[start + history + t].
from data_preparation.mesh_cache import load_mesh_cached

PARAM_RAW_DIR  = Path("../raw_data/parametric")
PARAM_EXCLUDED = ["top", "bottom", "cbores"]


def T_step_stats(T_seq, start=0, history=1):
    """Per rollout step: mean_i |T_true| and mean_i T_true^2."""
    T = T_seq[start + history:].double()
    return T.abs().mean(1).numpy(), T.pow(2).mean(1).numpy()


_param_T_stats = {}


def param_T_stats(mesh):
    """T_step_stats of a parametric mesh, whose rollouts all start at step 0 with
    history = 1. The T sequence is the same with Cf and with point-mean boundary
    nodes (checked on the four test meshes), so the Cf cache serves every run."""
    if mesh not in _param_T_stats:
        _, T_seq = load_mesh_cached(str(PARAM_RAW_DIR / mesh), PARAM_EXCLUDED, "cf",
                                    processed_data_dir / "parametric_preproc_cache")
        _param_T_stats[mesh] = T_step_stats(T_seq)
    return _param_T_stats[mesh]


def rmae_curve(mae, absT):
    """Per-step RMAE from the per-step MAE [K] (last axis: steps); nan or inf
    where the rollout overflowed."""
    mae = np.asarray(mae, dtype=float)
    return mae / absT[:mae.shape[-1]]


def rmse_curve(rmse, sqT):
    """Per-step RMSE, a ratio of squared sums, from the per-step root-mean-square
    error [K] (last axis: steps)."""
    rmse = np.asarray(rmse, dtype=float)
    return rmse ** 2 / sqT[:rmse.shape[-1]]

In [ ]:
# --- Setup: run every helper cell ------------------------------------------------
# Runs, in notebook order, the code cells tagged "helper" (in their cell metadata):
# those that define something a later section uses, e.g. load_mesh,
# compute_reg_errors, seed_stats, or a section's runs, caches and line styles. After
# this cell any section can be run from its first cell, without the ones above it.
# The cells are read from the notebook file, so save it first. Each loads its cache
# from processed_data/ and computes it only if it is missing, as when run by hand.
# Their printed tables are hidden, unless the cell fails.
import contextlib
import io
import json
import linecache
import time

_helpers = [c for c in json.loads(Path("data_analysis.ipynb").read_text())["cells"]
            if "helper" in c["metadata"].get("tags", [])]
for _k, _cell in enumerate(_helpers, 1):
    _src = "".join(_cell["source"])
    _name = f"<helper {_k}: cell {_cell['id']}>"
    linecache.cache[_name] = (len(_src), None, _src.splitlines(True), _name)   # tracebacks
    print(f"[{_k:2d}/{len(_helpers)}] {_src.splitlines()[0].strip('# -')}", end="", flush=True)
    _t0, _out = time.time(), io.StringIO()
    try:
        with contextlib.redirect_stdout(_out):
            exec(compile(_src, _name, "exec"), globals())
    except Exception:
        print("\n" + _out.getvalue())
        raise
    print(f"  ({time.time() - _t0:.0f} s)")
plt.close("all")


## Flange case

In [ ]:
processed_data_dir = Path("../processed_data")
exp_name = "history1_fv"
case_name = "flange"
case_name = f"{case_name}_{exp_name}"
case_dir = processed_data_dir / case_name
checkpoint_dir = case_dir / "checkpoints"
pred_dir = case_dir / "predictions"
error_dir = case_dir / "errors"

In [ ]:
train_loss = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
val_loss = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
rollout_mae = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, h = 1, FV", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{case_name}.png", dpi=300, transparent = True)

In [ ]:
train_loss, val_loss, rollout_mae = {},{},{}
processed_data_dir = Path("../processed_data")
for case_name in ["flange_history1_fv", "flange_history1_nofv"]:
    case_dir = processed_data_dir / case_name
    checkpoint_dir = case_dir / "checkpoints"
    pred_dir = case_dir / "predictions"
    error_dir = case_dir / "errors"

    train_loss[case_name] = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
    val_loss[case_name] = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
    rollout_mae[case_name] = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

# The flange T, for the RMAE of the rollout plot below. These runs predate
# git_commit.json and record neither their history nor their split; their
# rollout_mae.npy is 210 steps long, and of the temporal splits only history = 1
# with train_frac = 0.5 and val_frac = 0.15 (those train.py still uses) gives a
# 210-sample test set. It starts at sample 390, so step t is compared with
# T[391 + t]. Inferred, not recorded: the excluded patches are those of train.py.
from data_preparation.field import load_fields

flange_T = load_fields("../raw_data/flange", "T", excluded_patches=["patch1", "patch3"])
n_flange = len(flange_T) - 1
flange_start = round(0.5 * n_flange) + round(0.15 * n_flange)
assert all(len(m) == n_flange - flange_start for m in rollout_mae.values()), \
    "the rollout length does not match the inferred split"
flange_absT, _ = T_step_stats(flange_T, start=flange_start)

In [ ]:
fig, axs = plt.subplots(1,1)
axs.set_title("Training Loss", fontsize=20)
axs.set_yscale("log")
axs.set_xlabel("Epoch", fontsize=18)
axs.set_ylabel("MSE Loss", fontsize=18)
for i,case_name in enumerate(train_loss):
    plt.plot(train_loss[case_name], label=f"{case_name}: Train Loss")
    plt.plot(val_loss[case_name], label=f"{case_name}: Validation Loss")

axs.legend(fontsize=14)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(1,1, figsize=(960*px, 540*px))


# Transparent background
loss_graph.patch.set_alpha(0)
axs.patch.set_alpha(0)

axs.title.set_color(AX_COLOR)
axs.xaxis.label.set_color(AX_COLOR)
axs.yaxis.label.set_color(AX_COLOR)
axs.tick_params(colors=AX_COLOR)
for spine in axs.spines.values():
    spine.set_edgecolor(AX_COLOR)


for i,case_name in enumerate(train_loss):
    if case_name.endswith("_fv"):
        plt.plot(rmae_curve(rollout_mae[case_name], flange_absT), label="FV", color = PLOT1_COLOR, linewidth=2.5)
    elif case_name.endswith("_nofv"):
        plt.plot(rmae_curve(rollout_mae[case_name], flange_absT), label="No FV", color = PLOT2_COLOR, linewidth=2.5, linestyle='--')

legend = axs.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

axs.set_title("Rollout Error Accumulation, h = 1", fontsize=24)
axs.set_xlabel("Time step", fontsize=18)
axs.set_ylabel("RMAE", fontsize=18)
axs.tick_params(axis='both', labelsize=16)
axs.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig(f"../outputs/plots/{case_name[:15]}_rollout.png", dpi=300, transparent = True)


## Parametric case

Training/validation-loss and rollout-error plots for the parametric runs at `history = 1`.
The rollout panels mirror the flange **FV vs no-FV** comparison
(`parametric_history1_mesh_correct_edge_attr` vs `parametric_history1_mesh_nofv`),
one panel per mesh held out from those checkpoints: `model_003` and `model_007`
(the original 10-mesh study's test set) plus `model_010` and `model_011`
(the 2-hole / 3-hole geometries added later, which postdate both checkpoints).

In [ ]:
# --- Parametric case (history = 1, FV vs no-FV) ------------------------
# Each parametric run is tested on the held-out meshes model_003 and
# model_007, so there is one rollout-MAE curve per test mesh.
param_cases = {
    "fv":   "parametric_history1_mesh_correct_edge_attr",
    "nofv": "parametric_history1_mesh_nofv",
}
param_test_meshes = ["model_003", "model_007", "model_010", "model_011"]

# The FV run drives the single-case loss plot below.
param_case = param_cases["fv"]
param_ckpt = processed_data_dir / param_case / "checkpoints"
param_train_loss = np.load(os.path.join(param_ckpt, "train_losses.npy"))
param_val_loss = np.load(os.path.join(param_ckpt, "val_losses.npy"))

# rollout MAE per {fv/nofv} run and per test mesh.
param_rollout = {}
for key, cname in param_cases.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    param_rollout[key] = {
        mesh: np.load(os.path.join(ckpt, f"rollout_mae_{mesh}.npy"))
        for mesh in param_test_meshes
    }


In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(param_train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(param_val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, Parametric FV, h = 1", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{param_case}.png", dpi=300, transparent = True)


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import StrMethodFormatter

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
# One color per test mesh (validated categorical order, fixed per mesh);
# line style carries FV (solid) vs no-FV (dashed).
MESH_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# All held-out test meshes on a single axes: FV vs no-FV rollout for each.
fig, ax = plt.subplots(figsize=(1200*px, 640*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

all_curves = []
for mesh, color in zip(param_test_meshes, MESH_COLORS):
    fv_curve = rmae_curve(param_rollout["fv"][mesh], param_T_stats(mesh)[0])
    nofv_curve = rmae_curve(param_rollout["nofv"][mesh], param_T_stats(mesh)[0])
    ax.plot(fv_curve, color=color, linewidth=2.5)
    ax.plot(nofv_curve, color=color, linewidth=2.5, linestyle='--')
    all_curves += [fv_curve, nofv_curve]

# Log-scale whenever the curves span more than ~1.5 decades, otherwise the FV
# curves are squashed onto the axis by the diverging no-FV ones.
both = np.concatenate(all_curves)
if both.max() / max(both.min(), 1e-30) > 30:
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.1e}'))

# Two-part legend: mesh -> color, model -> line style.
handles = [Line2D([], [], color=c, linewidth=2.5, label=m)
           for m, c in zip(param_test_meshes, MESH_COLORS)]
handles += [Line2D([], [], color=AX_COLOR, linewidth=2.5, label="FV"),
            Line2D([], [], color=AX_COLOR, linewidth=2.5, linestyle='--', label="No FV")]
legend = ax.legend(handles=handles, fontsize=18,
                   loc="upper left", bbox_to_anchor=(1.01, 1))
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

ax.set_title("Rollout Error Accumulation, test meshes, h = 1", fontsize=24)
ax.set_xlabel("Time step", fontsize=18)
ax.set_ylabel("RMAE", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_mesh_rollout_exp.pdf", dpi=300,
            transparent=True, bbox_inches="tight", pad_inches=0)

## Direct vs. residual prediction

The same parametric FV model (`history = 1`, one MP layer), producing $T^{n+1}$ two ways:

- **Direct** (`parametric_history1_mesh_correct_edge_attr`): the network outputs the
  normalized $T^{n+1}$.
- **Residual** (`parametric_history1_mesh_correct_edge_attr_residual`): the network
  outputs the increment and `FVSurrogate` adds it back,
  $T^{n+1} = T^n + s\,f(\cdot)$ with $s = \mathrm{rms}(\Delta T)/\sigma_T$ fitted on the
  training meshes.

Both train on the same six meshes, validate on `model_000` / `model_009` and share the
normalizer, and in both the loss is the MSE on the normalized $T^{n+1}$, so the loss
curves compare directly. The direct run trained for 500 epochs and the residual one for
200, so the loss panels stop at epoch 200; the rollout panels use each run's final
checkpoint on the four held-out meshes.

In [ ]:
# --- Direct vs residual prediction (parametric, FV, history = 1) ----------
# Both runs share the train/val meshes and the normalizer, and both losses are
# the MSE on the normalized T^{n+1}, so their curves compare directly.
import json

resid_cases = {
    "direct":   "parametric_history1_mesh_correct_edge_attr",
    "residual": "parametric_history1_mesh_correct_edge_attr_residual",
}
resid_test_meshes = ["model_003", "model_007", "model_010", "model_011"]
# The residual run trained for 200 epochs and the direct one for 500: compare
# the losses over the window both have.
N_EPOCHS_CMP = 200

resid_train_loss, resid_val_loss, resid_rollout, resid_split = {}, {}, {}, {}
for key, cname in resid_cases.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    resid_train_loss[key] = np.load(ckpt / "train_losses.npy")[:N_EPOCHS_CMP]
    resid_val_loss[key] = np.load(ckpt / "val_losses.npy")[:N_EPOCHS_CMP]
    resid_rollout[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                          for mesh in resid_test_meshes}
    with open(ckpt / "mesh_split.json") as fh:
        resid_split[key] = json.load(fh)

# The comparison only means something if both runs saw the same meshes.
for part in ("train", "val"):
    assert sorted(resid_split["direct"][part]) == sorted(resid_split["residual"][part]), \
        f"{part} meshes differ: {resid_split['direct'][part]} vs {resid_split['residual'][part]}"

print(f"{'':10s}  {'rollout MAE, mean [K]':>22s}  {'rollout MAE, last step [K]':>27s}")
print(f"{'mesh':10s}  {'direct':>10s} {'residual':>11s}  {'direct':>15s} {'residual':>11s}")
for mesh in resid_test_meshes:
    d, r = resid_rollout["direct"][mesh], resid_rollout["residual"][mesh]
    print(f"{mesh:10s}  {d.mean():10.3f} {r.mean():11.3f}  {d[-1]:15.3f} {r[-1]:11.3f}")
print(f"\nbest validation loss in the first {N_EPOCHS_CMP} epochs: " + ", ".join(
    f"{key} {v.min():.3e} (epoch {v.argmin()})" for key, v in resid_val_loss.items()))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Residual is the variant under test (solid, PLOT1) and direct the baseline
# (dashed, PLOT2), the roles FV / no-FV take in the rollout panels above.
# Shared y-axis, so the training and validation panels read on one scale.
fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Direct vs Residual Loss, Parametric FV, h = 1 (first {N_EPOCHS_CMP} epochs)",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", resid_train_loss),
                                     ("Validation Loss", resid_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    ax.plot(losses["residual"], label="Residual", color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(losses["direct"], label="Direct", color=PLOT2_COLOR, linewidth=2.5, linestyle='--')

    # Value where each curve ends: the higher one labelled above its line and
    # the lower one below, so the two labels cannot collide.
    ends = sorted(((losses[k][-1], k) for k in ("residual", "direct")), reverse=True)
    for (y, key), dy, va in zip(ends, (8, -8), ("bottom", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=14, color=AX_COLOR)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_direct_vs_residual_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out mesh, each run at its final checkpoint; same roles and
# line styles as the loss panels.
fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, Direct vs Residual, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, mesh in zip(axs.ravel(), resid_test_meshes):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    ax.plot(rmae_curve(resid_rollout["residual"][mesh], param_T_stats(mesh)[0]), label="Residual",
            color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(rmae_curve(resid_rollout["direct"][mesh], param_T_stats(mesh)[0]), label="Direct",
            color=PLOT2_COLOR, linewidth=2.5, linestyle='--')
    ax.set_ylim(bottom=0)

    legend = ax.legend(fontsize=15)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel("Time step", fontsize=15)
    ax.set_ylabel("RMAE", fontsize=15)
    ax.tick_params(axis='both', labelsize=13)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_direct_vs_residual_rollout.png", dpi=300, transparent = True)

## Message vs. finite-volume flux

`FiniteVolumeGraphNet` sums its messages at the receiving node the way a finite-volume
scheme sums face fluxes over a cell, so the obvious question is whether the learned
$m_{ij}$ *is* the flux `laplacianFoam` assembles for `laplacian(DT,T)` with
`Gauss linear corrected`:

$$F_{j\rightarrow i} \;=\; D_T\,|S_f|\left[\;\frac{T_j - T_i}{|d|\,\max(\cos,\,0.05)} \;+\; k_f\cdot(\nabla T)_f\;\right]$$

Everything below runs on the **parametric** study — the same two `history = 1`
checkpoints the rollout panels compare (`parametric_history1_mesh_correct_edge_attr`
vs `parametric_history1_mesh_nofv`), probed on the same four held-out meshes:
`model_003` and `model_007` from the recorded split, plus the later `model_010` /
`model_011`. Nothing in the split is temporal, so every timestep of those meshes is
test data and the probe is free to sample the whole sequence.

`propagate` throws $m_{ij}$ away before `forward` returns, so it is pulled back out with
`models.message_probe.MessageProbe`, which hangs a PyG `register_message_forward_hook` on
each MP layer. Nothing in `fvgnn.py` or the `state_dict` changes, so every checkpoint
already on disk is probed as-is:

```python
probe = MessageProbe(model)
with probe, torch.no_grad():
    for i in range(n): model(data_i)

probe.messages[0]            # (E, 128)    last pass, layer 0
probe.stacked_messages(0)    # (n, E, 128) whole rollout
probe.stacked_aggregated(0)  # (n, N, 128) sum_j m_ij
```

`analyze_messages.py` runs the same comparison from the command line.

### Orientation — the thing that silently breaks this

PyG aggregates at `edge_index[1]`, so inside `message()` **`x_i` is the receiver and
`x_j` the sender**, and `build_static_graph` already orients each edge's stored $S_f$
*into* the receiver. `fv_flux` follows the same convention: a positive flux heats the
receiver, and summing over the in-edges of node $i$ gives $V_i\,\mathrm{d}T_i/\mathrm{d}t$.
Get this backwards and every correlation below flips sign while still looking plausible.

In [ ]:
import sys
print(sys.executable)
print(torch.__file__)

In [ ]:
# --- Message vs FV flux: extract, and cache -------------------------------
# Runs both trained parametric models over N_STEPS timesteps of every held-out
# mesh with a MessageProbe attached, and stores everything the figures below
# need. A parametric mesh carries up to ~270k edges, so a full (steps, E, 128)
# message stack is several GB: only a random subset of whole faces is held in
# memory, and only the reduced statistics reach the cache.

import torch

from data_preparation.field import load_fields
from data_preparation.mesh_dataset import SingleMeshDataset
from data_preparation.normalization import FeatureNormalizer
from data_preparation.static_graph import build_static_graph
from mesh2graph.utils import filter_of_time_directories
from models.fvgnn import FVSurrogate, widths_from_state_dict
from models.message_probe import (MessageProbe, fv_flux, fv_flux_corrected,
                                  load_cell_gradient, paired_edge_index)

FLUX_EXPS   = {"FV": "parametric_history1_mesh_correct_edge_attr",
               "No FV": "parametric_history1_mesh_nofv"}
# The same held-out meshes the rollout panels above compare.
FLUX_MESHES = ["model_003", "model_007", "model_010", "model_011"]
EXCLUDED    = ["top", "bottom", "cbores"]
# FLUX_EXPS use point-mean boundary nodes.
BOUNDARY_POS = "point_mean"
DT_PARAM    = 4e-5          # constant/transportProperties
HISTORY     = 1
# The mesh split is over GEOMETRY, not time, so every timestep of a held-out
# mesh is test data. These 12 are spread over the whole sequence rather than
# taken consecutively, so the transient does not dominate the statistics.
N_STEPS     = 12
KEEP_FACES  = 30_000        # faces (= 2 opposite edges) kept in memory per mesh
N_SCATTER   = 40_000        # points kept for the density panels
REF_STEPS   = list(range(1, 41))   # window the reference flux is validated on

raw_param_dir = Path("../raw_data/parametric")
flux_cache = processed_data_dir / "parametric_message_flux.npz"


def _r2(X, y):
    """R^2 of the least-squares fit y ~ [X, 1]."""
    X, y = X.double(), y.double()
    # Columns standardised first: R^2 does not depend on their scale, but lstsq
    # does. A message whose channels span five decades (the l1 runs) puts [X, 1]
    # at a condition number of ~3e10, where the default CPU driver cuts the rank
    # (30 of 65 on one checkpoint) and can return a fit worse than the mean.
    sd = X.std(0)
    X = (X - X.mean(0)) / torch.where(sd > 0, sd, torch.ones_like(sd))
    X = torch.cat([X, torch.ones(X.shape[0], 1, dtype=X.dtype)], dim=1)
    beta = torch.linalg.lstsq(X, y[:, None]).solution
    ss_res = ((y[:, None] - X @ beta) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return float(1.0 - ss_res / ss_tot.clamp(min=1e-30))


def load_mesh(name):
    """(static_graph, T_sequence) for one parametric case, cached to disk.

    Parsing an OpenFOAM case is by far the slowest step here, and the raw
    geometry is the same for both checkpoints, so it is parsed once per mesh.
    """
    # point_mean files keep the old, untagged name.
    suffix = "" if BOUNDARY_POS == "point_mean" else f"__{BOUNDARY_POS}"
    cache = (processed_data_dir / "parametric_preproc_cache"
             / f"{name}__T__{'-'.join(EXCLUDED)}{suffix}.pt")
    if cache.exists():
        blob = torch.load(cache, weights_only=False)
        return blob["graph"], blob["T"]
    g = build_static_graph(str(raw_param_dir / name), EXCLUDED, boundary_pos=BOUNDARY_POS)
    T = load_fields(str(raw_param_dir / name), "T", excluded_patches=EXCLUDED)
    cache.parent.mkdir(exist_ok=True)
    torch.save({"graph": g, "T": T}, cache)
    return g, T


def compute_flux_stats():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in FLUX_MESHES:
        graph_full, T_seq = load_mesh(mesh)
        case_dir = str(raw_param_dir / mesh)
        times = filter_of_time_directories(case_dir)

        ei, ea = graph_full.edge_index, graph_full.edge_attr   # RAW geometry
        N, E = graph_full.num_nodes, ei.shape[1]
        n_int = int((graph_full.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)
        cos = ea[:, 9].double()

        # -- how trustworthy is the reference? Fit V_i per cell in
        #    V_i dT_i/dt = sum_j F_ij and count the impossible (V_i < 0) fits,
        #    for the orthogonal flux and for the non-orth-corrected one.
        dt = float(times[2]) - float(times[1])
        gradT_ref = load_cell_gradient(case_dir, [times[i] for i in REF_STEPS])
        dTdt = (T_seq[[i + 1 for i in REF_STEPS]].double()
                - T_seq[[i - 1 for i in REF_STEPS]].double()) / (2 * dt)
        worst = torch.full((n_int,), 2.0, dtype=torch.double)   # worst face cos per cell
        into_cell = ei[1] < n_int
        worst.scatter_reduce_(0, ei[1][into_cell], cos[into_cell], reduce="amin")
        for tag, F_ref in (
                ("orth", fv_flux(ei, ea, T_seq[REF_STEPS].double(), DT=DT_PARAM)),
                ("corr", fv_flux_corrected(ei, ea, T_seq[REF_STEPS].double(),
                                           gradT_ref, DT=DT_PARAM,
                                           n_internal_nodes=n_int))):
            Fn = torch.zeros(len(REF_STEPS), N, dtype=torch.double)
            Fn.index_add_(1, ei[1], F_ref)
            Fn, d = Fn[:, :n_int], dTdt[:, :n_int]
            V = (d * Fn).sum(0) / (d * d).sum(0).clamp(min=1e-300)
            out[f"ref__{mesh}__{tag}_r2"] = float(
                1 - ((Fn - V[None] * d) ** 2).sum() / (Fn - Fn.mean(0)).pow(2).sum())
            out[f"ref__{mesh}__{tag}_negV"] = float((V < 0).double().mean())
            out[f"ref__{mesh}__{tag}_bins"] = np.array(
                [[int(sel.sum()), float((V[sel] < 0).double().mean())]
                 for sel in ((worst > lo) & (worst <= hi)
                             for lo, hi in ((0.999, 2.0), (0.99, 0.999),
                                            (0.95, 0.99), (-1.0, 0.95)))])

        # -- edges held in memory: whole faces, both halves internal, so that
        #    the antisymmetry test below has both m_ij and m_ji.
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        rev_local = torch.cat([torch.arange(n_faces) + n_faces,
                               torch.arange(n_faces)])
        near = cos[kept] > 0.99
        out[f"{mesh}__n_edges"] = E
        out[f"{mesh}__n_internal"] = int(internal.sum())
        out[f"{mesh}__n_kept"] = int(kept.numel())
        out[f"{mesh}__n_kept_near"] = int(near.sum())

        # -- the reference flux over the probed timesteps
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()
        gradT = load_cell_gradient(case_dir, [times[i] for i in steps])
        T_at = T_seq[steps].double()
        F = fv_flux_corrected(ei, ea, T_at, gradT, DT=DT_PARAM,
                              n_internal_nodes=n_int)            # (S, E)
        F_node = torch.zeros(N_STEPS, N, dtype=torch.double)
        F_node.index_add_(1, ei[1], F)                           # (S, N)
        # F is w*dT plus the correction, so regressing on each factor separately
        # says WHICH one the message picked up.
        tgts = {"F": F[:, kept].reshape(-1),
                "dT": (T_at[:, ei[0]] - T_at[:, ei[1]])[:, kept].reshape(-1),
                "w": (DT_PARAM * ea[:, 7].double()
                      / (ea[:, 3].double() * cos.clamp(min=0.05)))[kept].repeat(N_STEPS)}

        for label, exp in FLUX_EXPS.items():
            graph = graph_full.clone()
            if exp.endswith("_nofv"):
                graph.edge_attr = graph.edge_attr[:, :4]
            # The normalizer was fitted on the TRAINING meshes and saved with the
            # checkpoint; refitting it here would leak the held-out geometry.
            norm = FeatureNormalizer()
            norm.load(processed_data_dir / exp / "checkpoints" / "normalizer.pt")
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)

            state = torch.load(processed_data_dir / exp / "checkpoints" / "model.pt",
                               map_location=device)
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                **widths_from_state_dict(state), out_dim=1,
            ).to(device)
            model.load_state_dict(state)
            model.eval()

            # history=1, so sample i is built from T[i] — the field the flux
            # must be evaluated on. Reduce every step immediately: the full
            # (E, 128) message is dropped as soon as `kept` is sliced out of it.
            msg_kept, aggr_all = [], []
            probe = MessageProbe(model)
            with probe, torch.no_grad():
                for i in steps:
                    model(ds[i].to(device))
                    msg_kept.append(probe.messages[0][kept])
                    aggr_all.append(probe.aggregated[0])
                    probe.clear()
            msg = torch.stack(msg_kept)                 # (S, kept, C)
            aggr = torch.stack(aggr_all)                # (S, N, C)

            # antisymmetric energy fraction of m_ij across the two halves of a face
            S_, A_ = 0.5 * (msg + msg[:, rev_local]), 0.5 * (msg - msg[:, rev_local])
            sa = A_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
            ss = S_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
            out[f"{exp}__{mesh}__anti"] = (sa / (sa + ss)).numpy()      # (C,)

            # linear read-out of each target from the full 128-channel message
            m_flat = msg.reshape(-1, msg.shape[-1]).double()
            out[f"{exp}__{mesh}__r2"] = np.array(
                [_r2(m_flat, t) for t in tgts.values()])
            # control: the near-orthogonal edges, where even the uncorrected
            # flux is exact and the reference cannot be blamed.
            nf = near.repeat(N_STEPS)
            out[f"{exp}__{mesh}__r2_near"] = np.array(
                [_r2(m_flat[nf], t[nf]) for t in tgts.values()])
            # node level: sum_j m_ij vs the net flux that drives dT_i/dt
            out[f"{exp}__{mesh}__r2_node"] = _r2(
                aggr[:, :n_int].reshape(-1, msg.shape[-1]).double(),
                F_node[:, :n_int].reshape(-1))

            # best-correlated channel, and a subsample for the density panels
            mc = m_flat - m_flat.mean(0, keepdim=True)
            Fc = tgts["F"] - tgts["F"].mean()
            r = (mc * Fc[:, None]).sum(0) / (mc.norm(dim=0) * Fc.norm()).clamp(min=1e-30)
            best = int(r.abs().argmax())
            pick = torch.randperm(m_flat.shape[0],
                                  generator=torch.Generator().manual_seed(0))[:N_SCATTER]
            out[f"{exp}__{mesh}__best_ch"] = best
            out[f"{exp}__{mesh}__best_r"] = float(r[best])
            out[f"{exp}__{mesh}__sc_m"] = m_flat[pick, best].float().numpy()
            out[f"{exp}__{mesh}__sc_F"] = tgts["F"][pick].float().numpy()
            out[f"{exp}__{mesh}__sc_dT"] = tgts["dT"][pick].float().numpy()
            del msg, aggr, m_flat, probe, model
    return out


if flux_cache.exists():
    flux = dict(np.load(flux_cache, allow_pickle=False))
else:
    flux = compute_flux_stats()
    np.savez_compressed(flux_cache, **flux)

print(f"{'mesh':<11}{'model':<7}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    print(f"{mesh}  reference: V_i fit R^2 "
          f"{float(flux[f'ref__{mesh}__orth_r2']):.3f} (orthogonal) -> "
          f"{float(flux[f'ref__{mesh}__corr_r2']):.3f} (corrected), V_i < 0 on "
          f"{100*float(flux[f'ref__{mesh}__orth_negV']):.0f}% -> "
          f"{100*float(flux[f'ref__{mesh}__corr_negV']):.0f}% of cells")
    for label, exp in FLUX_EXPS.items():
        print(f"{mesh:<11}{label:<7}{int(flux[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux[f"{exp}__{mesh}__r2"])
              + f"{float(flux[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux[f"{exp}__{mesh}__r2_near"]))

### Is the reference trustworthy?

Checked against the actual `laplacianFoam` solution before reading anything into the
comparison. The geometry is exact: every edge has its reverse partner,
$S_{f,ij} = -S_{f,ji}$ to machine zero, and $\cos(S_f, d) > 0$ everywhere — internal
minimum 0.63 on `model_003` and 0.74 on `model_011`, down to 0.11 on the boundary edges,
whose column `build_static_graph` recomputes instead of leaving it at OpenFOAM's
hardcoded 1.0.

**The orthogonal flux alone is not good enough on these meshes.** Fitting $V_i$ per cell
in $V_i\,\mathrm{d}T_i/\mathrm{d}t = \sum_j F_{ij}$ over $t = 0.025\ldots1.0$ and counting
the physically impossible negative fits, pooled over the four held-out meshes:

| worst face $\cos$ of cell | cells | $V_i < 0$, orthogonal | $V_i < 0$, corrected |
|---|---|---|---|
| $> 0.999$ | 5803 | 4.2 % | 4.1 % |
| $0.99 - 0.999$ | 22544 | 5.4 % | 2.7 % |
| $0.95 - 0.99$ | 29157 | 21.3 % | 1.7 % |
| $< 0.95$ | 52546 | 25.7 % | 2.3 % |

These are `snappyHexMesh` geometries: only about a quarter of the cells have all their
faces above $\cos = 0.99$, so the flange's fix — restricting to near-orthogonal edges —
would throw most of the mesh away here. Instead the explicit non-orthogonal correction
$k_f\cdot(\nabla T)_f$ is added back with OpenFOAM's **own** cell gradient, which
`laplacianFoam` writes at every write time (`gradTx/y/z`), via
`message_probe.fv_flux_corrected`. That closes the balance to $R^2 = 0.995 - 0.996$ per
mesh, against $0.79 - 0.84$ uncorrected, and drops the impossible fits from ~20 % of
cells to 0.1 - 4.7 %.

So everything below runs on **all internal edges**, not just the orthogonal ones. Two
approximations survive in the correction — the face gradient is a 0.5/0.5 midpoint rather
than OpenFOAM's `linear` weights, and the correction is applied on internal faces only —
and they are what the residual few percent of bad cells are. The near-orthogonal control
($\cos > 0.99$, about 43k of the 60k edges kept per mesh) is computed alongside and
raises every $R^2$ below by 0.01 - 0.10. It changes no conclusion — the only FV vs no-FV
comparison it flips is the near-tied $\Delta T$ on `model_007` — so nothing here rests on
the correction being perfect.

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)

bins = np.linspace(0, 1, 31)
for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for (label, exp), color, ls in zip(FLUX_EXPS.items(),
                                       (PLOT1_COLOR, PLOT2_COLOR), ("-", "--")):
        ax.hist(flux[f"{exp}__{mesh}__anti"], bins=bins, histtype="step",
                linewidth=2.5, color=color, linestyle=ls, label=label)

    # The FV flux sits exactly at 1.0 — that is what makes the scheme conservative.
    ax.axvline(1.0, color=AX_COLOR, linewidth=2.5)
    ax.annotate("FV flux = 1.0\n(conservative)", xy=(1.0, 0.97),
                xycoords=("data", "axes fraction"), xytext=(-10, 0),
                textcoords="offset points", ha="right", va="top",
                fontsize=13, color=AX_COLOR)

    legend = ax.legend(fontsize=15, loc="upper center")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_xlim(0, 1.05)
    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel(r"Antisymmetric energy fraction  $\|A\|^2/(\|S\|^2+\|A\|^2)$",
                  fontsize=15)
    ax.set_ylabel("Message channels", fontsize=15)
    ax.grid(True, which="both", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)

fig.suptitle("Is the message antisymmetric across a face?",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_antisymmetry.png", dpi=300, transparent=True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# F is w*dT plus the non-orthogonal correction, so regressing on each factor
# separately says WHICH one the message picked up.
targets = [r"$F$", r"$\Delta T = T_j - T_i$", r"$w = D_T|S_f|/(|d|\cos)$"]
x = np.arange(len(targets))
width = 0.28

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, ((label, exp), color) in enumerate(zip(FLUX_EXPS.items(),
                                                  (PLOT1_COLOR, PLOT2_COLOR))):
        vals = flux[f"{exp}__{mesh}__r2"]
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    legend = ax.legend(fontsize=15)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 0.82)
    ax.set_title(f"{mesh}  —  {int(flux[f'{mesh}__n_kept'])//1000}k of "
                 f"{int(flux[f'{mesh}__n_internal'])//1000}k internal edges",
                 fontsize=17)
    ax.set_ylabel(r"$R^2$  (all 128 channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"What the message linearly encodes  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2.png", dpi=300, transparent=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# Columns: held-out meshes. Rows: the flux, and the bare temperature difference
# that is one of its two factors.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = FLUX_EXPS["FV"]
for col, mesh in enumerate(FLUX_MESHES):
    ch = int(flux[f"{exp}__{mesh}__best_ch"])
    m = flux[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        if row == 0:
            ax.set_title(f"{mesh} · channel {ch}", fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$", xy=(0.03, 0.95), xycoords="axes fraction",
                    va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message value", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("Most flux-correlated channel of the FV model",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density.png", dpi=300, transparent=True)

### What comes out

**The message is not a conservative flux.** A true FV flux is exactly antisymmetric
across a face, $F_{ij} = -F_{ji}$ — that is what makes the scheme conservative — so its
antisymmetric energy fraction is 1.0. The learned message averages **0.17 - 0.28** across
channels on all four held-out meshes: roughly three quarters of its energy is
*symmetric*, the opposite of a flux. The most antisymmetric single channel reaches only
0.50 - 0.62 (FV) and 0.76 - 0.81 (no-FV). The no-FV message is consistently the more
antisymmetric of the two — 9 to 20 channels above 0.5, against 0 to 5 for the FV model —
which buys it nothing anywhere else below.

**What linear structure it carries tracks $\Delta T$ more than $F$.** Regressing each
target on all 128 channels, the FV model gives $R^2 = 0.34 - 0.42$ for the flux against
$0.48 - 0.52$ for the bare temperature difference. The network leans on the driving
difference more than on the FV weighting that turns it into a flux, though it carries a
substantial amount of both.

**Here the FV features do buy something.** The geometric weight
$w = D_T|S_f|/(|d|\cos)$, which only the FV model's edge features encode, reads out at
$R^2 = 0.45 - 0.64$ from the FV message against $0.08 - 0.20$ from the no-FV one — and
that lead carries through to the flux itself on **every** mesh (0.34 - 0.42 vs
0.29 - 0.32) and to the node level (0.59 - 0.69 vs 0.50 - 0.56, a gain of 0.06 - 0.15).
The no-FV model compensates where it can: with no geometry to lean on, it matches or
beats the FV model on the bare $\Delta T$ on `model_003` and, narrowly, `model_007` — the
one thing left for it to encode. This is the opposite of what the same test gives on the
flange (cached in `flange_message_flux.npz`), where the FV model led only on $w$ and lost
on both $F$ and $\Delta T$.

**The aggregate is far more flux-like than any single message.** At node level, where it
actually matters, $\sum_j m_{ij}$ explains $R^2 = 0.59 - 0.69$ of the net flux
$\sum_j F_{ij}$ that drives $\mathrm{d}T_i/\mathrm{d}t$ — 1.5 to 1.9 times the per-edge
number. The network gets the cell balance broadly right while splitting it over the faces
in a way that is not, face by face, a flux. All of this holds on `model_010` and
`model_011`, whose 2-hole / 3-hole geometries postdate both checkpoints, so it is not an
artefact of the recorded test split.

The density panels show what the correlations leave out: a dense vertical spine at
$F \approx 0$, where the channel spans most of its range while the flux does not move at
all, and a cloud that widens with $|\Delta T|$ rather than following a line.

### Two things this does *not* show

- These are **linear** read-outs of a 128-dimensional message that a **nonlinear** MLP
  consumes. A low $R^2$ bounds how *directly* flux-like the message is; it does not prove
  the flux is unrecoverable from it.
- Both checkpoints are `n_mp_layers=1`, so a single message has to do all the work. A
  deeper stack has no reason to put the flux in layer 0.

### The residual model

The same read-outs for the residual checkpoint
(`parametric_history1_mesh_correct_edge_attr_residual`), on the same four held-out meshes,
the same 12 timesteps and the same seeded subset of faces as above, next to the direct FV
model's cached numbers (`FV` above, `Direct` here). Only the per-model part is recomputed,
and it is cached in `parametric_message_flux_residual.npz`. Every read-out is a
correlation, an $R^2$ or an energy fraction, so none depends on the scale the residual
network's messages take.

In [ ]:
# --- Message vs FV flux: the residual model --------------------------------
# The per-model part of compute_flux_stats above, for one checkpoint. The face
# subset and the timesteps are seeded exactly as there, so every number lines
# up row for row with the direct FV model's cached ones. Helpers (load_mesh,
# _r2, the probe and the flux) come from the flux cell above.
RESID_EXP = "parametric_history1_mesh_correct_edge_attr_residual"
flux_resid_cache = processed_data_dir / "parametric_message_flux_residual.npz"


def compute_model_flux_stats(exp, load=load_mesh, layer=-1, n_edge_feats=None, raw_dir=None):
    # layer: the MP layer whose message is read out, the last by default (the one
    # aggregated into the node update that outputs T); with one layer, layer 0.
    # Another layer's entries are keyed <exp>__L<layer>__..., so the default keys
    # mean the same thing for every depth.
    name = exp if layer == -1 else f"{exp}__L{layer}"
    # load: mesh -> (graph, T_seq), built as this checkpoint's graphs were;
    # load_mesh's point-mean ones by default
    # n_edge_feats: the model is fed the first n_edge_feats edge columns (4 for a
    # _nofv run), all by default; the reference flux is computed from all of them.
    # raw_dir: the case directories the time directories and grad(T) are read from,
    # raw_param_dir by default; parametric_old for the runs on the dt = 0.025 data.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = processed_data_dir / exp / "checkpoints"
    state = torch.load(ckpt / "model.pt", map_location=device)
    # The normalizer was fitted on the TRAINING meshes and saved with the
    # checkpoint; refitting it here would leak the held-out geometry.
    norm = FeatureNormalizer()
    norm.load(ckpt / "normalizer.pt")

    out = {}
    for mesh in FLUX_MESHES:
        graph, T_seq = load(mesh)
        case_dir = str((raw_param_dir if raw_dir is None else Path(raw_dir)) / mesh)
        times = filter_of_time_directories(case_dir)

        ei, ea = graph.edge_index, graph.edge_attr   # RAW geometry
        N, E = graph.num_nodes, ei.shape[1]
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)
        cos = ea[:, 9].double()

        # the same whole faces and timesteps as compute_flux_stats
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        rev_local = torch.cat([torch.arange(n_faces) + n_faces,
                               torch.arange(n_faces)])
        near = (cos[kept] > 0.99).repeat(N_STEPS)
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()

        gradT = load_cell_gradient(case_dir, [times[i] for i in steps])
        T_at = T_seq[steps].double()
        F = fv_flux_corrected(ei, ea, T_at, gradT, DT=DT_PARAM,
                              n_internal_nodes=n_int)            # (S, E)
        F_node = torch.zeros(N_STEPS, N, dtype=torch.double)
        F_node.index_add_(1, ei[1], F)                           # (S, N)
        tgts = {"F": F[:, kept].reshape(-1),
                "dT": (T_at[:, ei[0]] - T_at[:, ei[1]])[:, kept].reshape(-1),
                "w": (DT_PARAM * ea[:, 7].double()
                      / (ea[:, 3].double() * cos.clamp(min=0.05)))[kept].repeat(N_STEPS)}

        if n_edge_feats is not None:
            graph = graph.clone()
            graph.edge_attr = graph.edge_attr[:, :n_edge_feats]
        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            # residual checkpoints carry their delta_scale buffer, LayerNorm
            # ones their node_encoder_norm weights
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()

        msg_kept, aggr_all = [], []
        probe = MessageProbe(model)
        with probe, torch.no_grad():
            for i in steps:
                model(ds[i].to(device))
                msg_kept.append(probe.messages[layer][kept])
                aggr_all.append(probe.aggregated[layer])
                probe.clear()
        msg = torch.stack(msg_kept)                 # (S, kept, C)
        aggr = torch.stack(aggr_all)                # (S, N, C)

        # antisymmetric energy fraction of m_ij across the two halves of a face
        S_, A_ = 0.5 * (msg + msg[:, rev_local]), 0.5 * (msg - msg[:, rev_local])
        sa = A_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
        ss = S_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
        out[f"{name}__{mesh}__anti"] = (sa / (sa + ss)).numpy()      # (C,)

        m_flat = msg.reshape(-1, msg.shape[-1]).double()
        out[f"{name}__{mesh}__r2"] = np.array([_r2(m_flat, t) for t in tgts.values()])
        out[f"{name}__{mesh}__r2_near"] = np.array(
            [_r2(m_flat[near], t[near]) for t in tgts.values()])
        out[f"{name}__{mesh}__r2_node"] = _r2(
            aggr[:, :n_int].reshape(-1, msg.shape[-1]).double(),
            F_node[:, :n_int].reshape(-1))

        mc = m_flat - m_flat.mean(0, keepdim=True)
        Fc = tgts["F"] - tgts["F"].mean()
        r = (mc * Fc[:, None]).sum(0) / (mc.norm(dim=0) * Fc.norm()).clamp(min=1e-30)
        best = int(r.abs().argmax())
        # subsample for the density panels: same seed and stack size as
        # compute_flux_stats, so the same edge-samples as the direct model's
        pick = torch.randperm(m_flat.shape[0],
                              generator=torch.Generator().manual_seed(0))[:N_SCATTER]
        out[f"{name}__{mesh}__best_ch"] = best
        out[f"{name}__{mesh}__best_r"] = float(r[best])
        out[f"{name}__{mesh}__sc_m"] = m_flat[pick, best].float().numpy()
        out[f"{name}__{mesh}__sc_F"] = tgts["F"][pick].float().numpy()
        out[f"{name}__{mesh}__sc_dT"] = tgts["dT"][pick].float().numpy()
        del msg, aggr, m_flat, probe, model
    return out


if flux_resid_cache.exists():
    flux_resid = dict(np.load(flux_resid_cache, allow_pickle=False))
else:
    flux_resid = compute_model_flux_stats(RESID_EXP)
    np.savez_compressed(flux_resid_cache, **flux_resid)

# Direct FV (cached by the flux cell above) vs residual, same columns as there.
MSG_CMP = {"Direct": FLUX_EXPS["FV"], "Residual": RESID_EXP}
flux_cmp = {**flux, **flux_resid}
print(f"{'mesh':<11}{'model':<10}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    for label, exp in MSG_CMP.items():
        print(f"{mesh:<11}{label:<10}{int(flux_cmp[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux_cmp[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux_cmp[f"{exp}__{mesh}__r2"])
              + f"{float(flux_cmp[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux_cmp[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_cmp[f"{exp}__{mesh}__r2_near"]))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# The three edge-level read-outs of the panels above plus the node-level one,
# direct vs residual. Residual is the variant under test (PLOT1), direct the
# baseline (PLOT2), as in the direct vs residual section.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.28
models = {"Residual": (RESID_EXP, PLOT1_COLOR), "Direct": (FLUX_EXPS["FV"], PLOT2_COLOR)}

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (label, (exp, color)) in enumerate(models.items()):
        vals = np.append(flux_cmp[f"{exp}__{mesh}__r2"], flux_cmp[f"{exp}__{mesh}__r2_node"])
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    # R^2 stops at 1; the space above it holds the legend clear of the bars.
    legend = ax.legend(fontsize=14, loc="upper left", ncol=len(models))
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    # Mean antisymmetric energy fraction over channels (1.0 for a true flux).
    anti = ", ".join(f"{label.lower()} {flux_cmp[f'{exp}__{mesh}__anti'].mean():.2f}"
                     for label, (exp, _) in models.items())

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.2)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_title(f"{mesh}\nantisymmetric energy: {anti}", fontsize=15)
    ax.set_ylabel(r"$R^2$  (all 128 channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"Message vs FV flux, direct vs residual  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2_residual.png", dpi=300, transparent=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# The density panels above, for the residual model: its most flux-correlated
# channel against the flux and the bare temperature difference, on the same
# edge-samples as the direct model's.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = RESID_EXP
for col, mesh in enumerate(FLUX_MESHES):
    ch = int(flux_resid[f"{exp}__{mesh}__best_ch"])
    m = flux_resid[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux_resid[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux_resid[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        if row == 0:
            ax.set_title(f"{mesh} · channel {ch}", fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$", xy=(0.03, 0.95), xycoords="axes fraction",
                    va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message value", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("Most flux-correlated channel of the residual model",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density_residual.png", dpi=300, transparent=True)

## One-channel message (`msg_dim = 1`)

The tightest possible message bottleneck: the residual model with $m_{ij}$ cut from 128
channels to **one** (`parametric_history1_msg_dim1_residual`), against the 128-channel
residual run above (`parametric_history1_mesh_correct_edge_attr_residual`). Both train on
the same six meshes for 200 epochs and are tested on the same four held-out meshes, and the
first cell below asserts that the mesh split, the normalizer and the increment scale
$s = \mathrm{rms}(\Delta T)/\sigma_T$ are identical — so the message width is what the
comparison isolates.

One channel is exactly the width of a face flux. If the network still does the job through
a scalar $m_{ij}$, the physically natural thing for that scalar to be is
$F_{j\rightarrow i}$ up to scale and sign, and the read-outs further down test exactly that.

- **Loss decay and rollout error.** `train_parametric.py` saves the rollout MAE but not the
  RMSE, so both checkpoints are rolled out again on the four held-out meshes (cached in
  `parametric_rollout_msg_dim1.npz`), and the recomputed MAE is checked against the curves
  saved at training time.
- **Message vs flux.** `compute_model_flux_stats` from the residual cell, on the same 12
  timesteps and the same seeded faces, cached in `parametric_message_flux_msg_dim1.npz`.
  With one channel every $R^2$ is a squared correlation, so the rank correlation
  $\rho_s$ is reported next to it: a message that is a monotone but *nonlinear* function
  of $F$ keeps $\rho_s$ high while $r$ drops.

In [ ]:
# --- msg_dim = 1 vs 128: loss decay and rollout errors ------------------------
# Two residual runs that differ in the message width. train_parametric.py saves
# the rollout MAE but not the RMSE, so both checkpoints are rolled out again on
# the held-out meshes, and the recomputed MAE has to reproduce the saved curves.
# Helpers (load_mesh, HISTORY, RESID_EXP) come from the flux cells above.
import json

from models.autoregressive_training import rollout

MSG1_EXP  = "parametric_history1_msg_dim1_residual"
MD_CASES  = {"msg_dim = 1": MSG1_EXP, "msg_dim = 128": RESID_EXP}
MD_MESHES = ["model_003", "model_007", "model_010", "model_011"]
md_rollout_cache = processed_data_dir / "parametric_rollout_msg_dim1.npz"

md_train_loss, md_val_loss, md_saved_mae, md_setup = {}, {}, {}, {}
for key, cname in MD_CASES.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    md_train_loss[key] = np.load(ckpt / "train_losses.npy")
    md_val_loss[key] = np.load(ckpt / "val_losses.npy")
    md_saved_mae[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                         for mesh in MD_MESHES}
    with open(ckpt / "mesh_split.json") as fh:
        split = json.load(fh)
    md_setup[key] = {
        "split": split,
        "norm": torch.load(ckpt / "normalizer.pt"),
        "delta_scale": float(torch.load(ckpt / "model.pt", map_location="cpu")["delta_scale"]),
    }

# The comparison isolates the message width only if nothing else moved.
s1, s128 = md_setup["msg_dim = 1"], md_setup["msg_dim = 128"]
assert s1["split"] == s128["split"], \
    f"mesh splits differ: {s1['split']} vs {s128['split']}"
assert all(torch.equal(s1["norm"][k], s128["norm"][k]) for k in s1["norm"]), \
    "normalizers differ"
assert s1["delta_scale"] == s128["delta_scale"], \
    f"delta_scale differs: {s1['delta_scale']} vs {s128['delta_scale']}"


def compute_rollout_errors():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in MD_MESHES:
        graph, T_seq = load_mesh(mesh)
        for cname in MD_CASES.values():
            ckpt = processed_data_dir / cname / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                **widths_from_state_dict(state), out_dim=1,
                residual="delta_scale" in state, history=HISTORY,
            ).to(device)
            model.load_state_dict(state)

            # every node, boundary included, as rollout_mae_*.npy was computed
            T_pred, T_true = rollout(model, SingleMeshDataset(T_seq, graph, norm, HISTORY),
                                     device=device)
            err = (T_pred - T_true).double()                     # (n_steps, N) [K]
            out[f"{cname}__{mesh}__mae"] = err.abs().mean(1).numpy()
            out[f"{cname}__{mesh}__rmse"] = err.pow(2).mean(1).sqrt().numpy()
            del model
    return out


if md_rollout_cache.exists():
    md_rollout = dict(np.load(md_rollout_cache, allow_pickle=False))
else:
    md_rollout = compute_rollout_errors()
    np.savez_compressed(md_rollout_cache, **md_rollout)

# If the MAE did not reproduce, the RMSE next to it would describe a different
# rollout from the one saved at training time.
mae_dev = max(float(np.abs(md_rollout[f"{cname}__{mesh}__mae"] - md_saved_mae[key][mesh]).max())
              for key, cname in MD_CASES.items() for mesh in MD_MESHES)
print(f"recomputed vs saved rollout MAE: max |difference| = {mae_dev:.1e} K\n")

print(f"{'':18s}" + "".join(f"{key:>17s}" for key in MD_CASES))
print(f"{'train loss, last':18s}" + "".join(f"{md_train_loss[k][-1]:>17.3e}" for k in MD_CASES))
print(f"{'val loss, last':18s}" + "".join(f"{md_val_loss[k][-1]:>17.3e}" for k in MD_CASES))
print(f"{'val loss, best':18s}" + "".join(
    f"{f'{md_val_loss[k].min():.3e} @{md_val_loss[k].argmin()}':>17s}" for k in MD_CASES))

c1, c128 = MD_CASES["msg_dim = 1"], MD_CASES["msg_dim = 128"]
print(f"\n{'':10s}{'rollout MAE [K]':^36s}{'rollout RMSE [K]':^36s}")
print(f"{'':10s}" + f"{'mean':^18s}{'last step':^18s}" * 2)
print(f"{'msg_dim':10s}" + f"{'1':>9s}{'128':>9s}" * 4)
for mesh in MD_MESHES:
    row = f"{mesh:10s}"
    for metric in ("mae", "rmse"):
        e1, e128 = md_rollout[f"{c1}__{mesh}__{metric}"], md_rollout[f"{c128}__{mesh}__{metric}"]
        row += f"{e1.mean():9.3f}{e128.mean():9.3f}{e1[-1]:9.3f}{e128[-1]:9.3f}"
    print(row)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# msg_dim = 1 is the variant under test (solid, PLOT1) and the 128-channel
# residual run the baseline (dashed, PLOT2), the roles residual / direct take
# above. Shared y-axis, so the training and validation panels read on one scale.
md_styles = {"msg_dim = 1": (PLOT1_COLOR, "-"), "msg_dim = 128": (PLOT2_COLOR, "--")}

fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Loss Decay, msg_dim = 1 vs 128, Residual FV, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", md_train_loss),
                                     ("Validation Loss", md_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, (color, ls) in md_styles.items():
        ax.plot(losses[key], label=key, color=color, linewidth=2.5, linestyle=ls)

    # Value where each curve ends: the higher one labelled above its line and
    # the lower one below, so the two labels cannot collide.
    ends = sorted(((losses[k][-1], k) for k in md_styles), reverse=True)
    for (y, key), dy, va in zip(ends, (8, -8), ("bottom", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=14, color=AX_COLOR)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_msg_dim1_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

md_styles = {"msg_dim = 1": (PLOT1_COLOR, "-"), "msg_dim = 128": (PLOT2_COLOR, "--")}

# Rows: rollout RMAE and RMSE (relative, CLAUDE.md); columns: the held-out
# meshes, each run at its final checkpoint. md_rel turns the kelvin curves of
# md_rollout into the relative ones.
md_rel = {"mae": lambda curve, mesh: rmae_curve(curve, param_T_stats(mesh)[0]),
          "rmse": lambda curve, mesh: rmse_curve(curve, param_T_stats(mesh)[1])}
fig, axs = plt.subplots(2, len(MD_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, msg_dim = 1 vs 128, h = 1",
             fontsize=24, color=AX_COLOR)

for row, (metric, ylabel) in enumerate((("mae", "RMAE"), ("rmse", "RMSE"))):
    for col, mesh in enumerate(MD_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, (color, ls) in md_styles.items():
            ax.plot(md_rel[metric](md_rollout[f"{MD_CASES[key]}__{mesh}__{metric}"], mesh), label=key,
                    color=color, linewidth=2.5, linestyle=ls)
        ax.set_ylim(bottom=0)

        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=15)
        ax.tick_params(axis='both', labelsize=13)
        ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

legend = axs[0, 0].legend(fontsize=14, loc="upper left")
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

# plt.savefig("../outputs/plots/parametric_history1_msg_dim1_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- Message vs FV flux: the one-channel message -------------------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces, so every row lines up with the 128-channel residual model's.
# With one channel there is nothing to pick (the best channel is channel 0) and
# every R^2 is a squared correlation, so the Spearman rank correlation with F is
# printed next to r: a monotone but nonlinear m = f(F) keeps rho high while r
# drops. rho is taken on the density-panel subsample; for the 128-channel model
# both r and rho describe its most flux-correlated channel.
from scipy.stats import spearmanr

flux_md1_cache = processed_data_dir / "parametric_message_flux_msg_dim1.npz"

if flux_md1_cache.exists():
    flux_md1 = dict(np.load(flux_md1_cache, allow_pickle=False))
else:
    flux_md1 = compute_model_flux_stats(MSG1_EXP)
    np.savez_compressed(flux_md1_cache, **flux_md1)

MD_MSG = {"msg_dim = 128": RESID_EXP, "msg_dim = 1": MSG1_EXP}
flux_md = {**flux_resid, **flux_md1}
print(f"{'mesh':<11}{'model':<15}{'ch':>4}{'r':>8}{'rho':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    for label, exp in MD_MSG.items():
        rho = spearmanr(flux_md[f"{exp}__{mesh}__sc_m"], flux_md[f"{exp}__{mesh}__sc_F"])[0]
        print(f"{mesh:<11}{label:<15}{int(flux_md[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux_md[f'{exp}__{mesh}__best_r']):>+8.3f}{rho:>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux_md[f"{exp}__{mesh}__r2"])
              + f"{float(flux_md[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux_md[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_md[f"{exp}__{mesh}__r2_near"]))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# The three edge-level read-outs plus the node-level one, one channel vs 128.
# For msg_dim = 1 each bar is a squared correlation; for 128 it is the best
# linear combination of all channels, so the wide message has the easier test.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.28
models = {"msg_dim = 1": (MSG1_EXP, PLOT1_COLOR), "msg_dim = 128": (RESID_EXP, PLOT2_COLOR)}

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (label, (exp, color)) in enumerate(models.items()):
        vals = np.append(flux_md[f"{exp}__{mesh}__r2"], flux_md[f"{exp}__{mesh}__r2_node"])
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    # R^2 stops at 1; the space above it holds the legend clear of the bars.
    legend = ax.legend(fontsize=14, loc="upper left", ncol=len(models))
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    # Mean antisymmetric energy fraction over channels (1.0 for a true flux).
    anti = ", ".join(f"{flux_md[f'{exp}__{mesh}__anti'].mean():.2f} ({label.split()[-1]} ch)"
                     for label, (exp, _) in models.items())

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.2)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_title(f"{mesh}\nantisymmetric energy: {anti}", fontsize=15)
    ax.set_ylabel(r"$R^2$  (all channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"Message vs FV flux, msg_dim = 1 vs 128  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2_msg_dim1.png", dpi=300, transparent=True)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# The density panels above, for the one-channel model. No channel is selected
# here: this is the whole message, on the same edge-samples as the direct and
# residual models' panels.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = MSG1_EXP
for col, mesh in enumerate(FLUX_MESHES):
    m = flux_md1[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux_md1[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux_md1[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        rho = spearmanr(xv, m)[0]
        if row == 0:
            ax.set_title(mesh, fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$\n$\\rho_s = {rho:+.2f}$", xy=(0.03, 0.95),
                    xycoords="axes fraction", va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message $m_{ij}$", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("The one-channel message against the flux",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density_msg_dim1.png", dpi=300, transparent=True)

In [ ]:
# --- Is the one-channel message used, and what does it carry? -----------------
# A message this unlike the flux could simply be ignored by node_fnc. Two
# ablations on the same 12 timesteps answer that, for both widths: replace every
# m_ij by the message's mean over those steps (no information left), or shuffle
# the messages across edges (right distribution, wrong faces). The one-step MSE
# is in normalized T, the training loss's units, next to the persistence
# baseline T^{n+1} = T^n. Alongside, what the one-channel message tracks
# linearly instead: the temperatures on the two sides of its face, their sum and
# their difference, on the same seeded faces as the flux read-outs.
msg_usage_cache = processed_data_dir / "parametric_message_usage_msg_dim1.npz"


def compute_message_usage():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in FLUX_MESHES:
        graph, T_seq = load_mesh(mesh)
        ei = graph.edge_index
        E = ei.shape[1]
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)

        # the same whole faces and timesteps as compute_flux_stats
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()

        for exp in MD_CASES.values():
            ckpt = processed_data_dir / exp / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            width = state["mp_layers.0.msg_fnc.6.weight"].shape[0]
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                **widths_from_state_dict(state), out_dim=1,
                residual="delta_scale" in state, history=HISTORY,
            ).to(device)
            model.load_state_dict(state)
            model.eval()
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
            batches = [ds[i].to(device) for i in steps]

            # true messages: their mean feeds the ablation, and for the one-channel
            # model the kept faces feed the read-outs below
            mse = {"true": [], "mean": [], "shuffle": []}
            msg_sum, msg_n, msg_kept = 0.0, 0, []
            probe = MessageProbe(model)
            with probe, torch.no_grad():
                for d in batches:
                    mse["true"].append(float(((model(d) - d.y) ** 2).mean()))
                    m = probe.messages[0]
                    msg_sum, msg_n = msg_sum + m.double().sum(0), msg_n + m.shape[0]
                    if width == 1:
                        msg_kept.append(m[kept, 0])
                    probe.clear()

            # a message forward hook that returns a tensor replaces the message
            msg_mean = (msg_sum / msg_n).float().to(device)
            gen = torch.Generator().manual_seed(0)
            replace = {
                "mean": lambda out: msg_mean.expand_as(out),
                "shuffle": lambda out: out[torch.randperm(out.shape[0], generator=gen)
                                           .to(out.device)],
            }
            for mode, fn in replace.items():
                handle = model.mp_layers[0].register_message_forward_hook(
                    lambda module, inputs, out, fn=fn: fn(out))
                with torch.no_grad():
                    for d in batches:
                        mse[mode].append(float(((model(d) - d.y) ** 2).mean()))
                handle.remove()
            for mode, vals in mse.items():
                out[f"{exp}__{mesh}__mse_{mode}"] = float(np.mean(vals))
            out[f"{mesh}__mse_persistence"] = float(np.mean(
                [float(((d.x[:, HISTORY - 1] - d.y) ** 2).mean()) for d in batches]))

            if width == 1:
                m_flat = torch.stack(msg_kept).reshape(-1, 1).double()
                T_at = T_seq[steps].double()
                Ti, Tj = T_at[:, ei[1]][:, kept], T_at[:, ei[0]][:, kept]   # receiver, sender
                out[f"{exp}__{mesh}__r2_levels"] = np.array(
                    [_r2(m_flat, t.reshape(-1)) for t in (Ti, Tj, Ti + Tj, Tj - Ti)])
            del model, batches
    return out


if msg_usage_cache.exists():
    msg_usage = dict(np.load(msg_usage_cache, allow_pickle=False))
else:
    msg_usage = compute_message_usage()
    np.savez_compressed(msg_usage_cache, **msg_usage)

print(f"one-step MSE (normalized T) on the {N_STEPS} probed timesteps")
print(f"{'mesh':<11}{'model':<15}{'true m':>10}{'mean m':>10}{'shuffled m':>12}{'persistence':>13}")
for mesh in FLUX_MESHES:
    for label, exp in MD_CASES.items():
        print(f"{mesh:<11}{label:<15}"
              + "".join(f"{msg_usage[f'{exp}__{mesh}__mse_{mode}']:>{w}.2e}"
                        for mode, w in (("true", 10), ("mean", 10), ("shuffle", 12)))
              + f"{msg_usage[f'{mesh}__mse_persistence']:>13.2e}")

print(f"\nR^2 of the one-channel message on")
print(f"{'mesh':<11}{'T_i':>7}{'T_j':>7}{'T_i+T_j':>9}{'T_j-T_i':>9}")
for mesh in FLUX_MESHES:
    print(f"{mesh:<11}" + "".join(f"{v:>{w}.3f}" for v, w in
                                  zip(msg_usage[f"{MSG1_EXP}__{mesh}__r2_levels"], (7, 7, 9, 9))))

### What comes out

**Loss.** One channel trains to a validation MSE of $5.0\times10^{-5}$, against
$2.8\times10^{-5}$ with 128 (best $4.4\times10^{-5}$ at epoch 60 vs $2.5\times10^{-5}$ at
epoch 79): about 1.8 times higher, with both runs levelling off after epoch ~90. The
bottleneck costs one-step accuracy, but not an order of magnitude of it.

**Rollout.** That gap does not carry through uniformly. On `model_003` and `model_007` the
one-channel model accumulates *less* error than the 128-channel one — mean MAE 5.98 vs
6.06 K and 3.12 vs 5.23 K, last-step RMSE 9.7 vs 16.4 K and 8.0 vs 16.7 K — flattening out
where the wide model keeps climbing. On the 2-hole / 3-hole geometries `model_010` and
`model_011` it is worse: mean MAE 6.84 vs 6.19 K and 6.68 vs 5.06 K, last-step RMSE 17.9
vs 13.3 K and 12.1 vs 9.4 K. For both models the RMSE runs 1.4 – 2.3 times the MAE, so the
error sits on a minority of cells. Each width is a single training run, so the per-mesh
ordering cannot be separated from seed-to-seed spread here.

**The message is not the flux.** With a single channel every read-out is one squared
correlation, and none of them moves: $R^2 = 0.02 - 0.04$ for $F$, $0.02 - 0.05$ for
$\Delta T$, $0.00 - 0.09$ for $w$, and $0.00 - 0.01$ between $\sum_j m_{ij}$ and the net
flux. The 128-channel model's full linear read-out reaches $0.76 - 0.82$ for $F$, and even
its single most flux-correlated channel ($r = +0.23 \ldots +0.39$) beats the one-channel
message ($r = -0.15 \ldots -0.20$). The rank correlation is no better
($\rho_s = -0.06 \ldots -0.14$), so no monotone nonlinear relation is hiding behind the low
$r$ either. The antisymmetric energy fraction is **0.04** on every mesh: the message is
almost exactly *symmetric* across a face, $m_{ij} \approx m_{ji}$ — the opposite of a
conservative flux, and further from one than the 128-channel message (0.25 – 0.28).

**But the network does use it.** Replacing every $m_{ij}$ by the message's mean raises the
one-step MSE 7 – 22 fold, to just below the persistence baseline $T^{n+1} = T^n$, and
shuffling the messages across edges is worse than persistence. The 128-channel model
behaves the same way. The one channel carries essentially all the neighbour information the
update receives; it is simply not organised as a flux.

**What it tracks instead is the temperature level.** The message follows the temperatures
on the two sides of its face — $R^2 = 0.16 - 0.55$ on $T_i$, $0.18 - 0.57$ on $T_j$,
$0.17 - 0.56$ on $T_i + T_j$ — against $0.02 - 0.05$ on their difference. A symmetric,
level-like message is enough because `node_fnc` is nonlinear and also receives $T_i$: the
differencing a flux does face by face can instead happen after aggregation, against the
cell's own temperature. The flux is one solution to the one-channel problem, not the only
one, and the width constraint alone does not select it.

**What would select it.** Build the antisymmetry in,
$m_{ij} = \tfrac12\left[f(x_i, x_j, e_{ij}) - f(x_j, x_i, e_{ji})\right]$. A level-like
message then cancels across every face by construction, and $\sum_j m_{ij}$ is conservative.

## PCA spectrum of the message

`msg_dim = 1` costs a factor ~1.8 in validation loss, so one channel is too few, but that
does not say how many channels the 128-channel message actually uses. Its PCA spectrum
does, straight from the trained checkpoint: the eigenvalues
$\lambda_1 \ge \dots \ge \lambda_{128}$ of the channel covariance of the `msg_dim = 128`
residual model (`parametric_history1_mesh_correct_edge_attr_residual`), on the same four
held-out meshes and 12 timesteps as above. Unlike the flux read-outs nothing is subsampled:
the covariance is accumulated over every edge and every node, and cached in
`parametric_message_pca_residual.npz`.

The spectrum does not care which channels carry the message. `node_fnc` opens with a
`Linear` layer, which absorbs any rotation of its input, so a message confined to a
$k$-dimensional subspace is a $k$-channel message in whatever basis it is written.

Two levels:

- **edge**, $m_{ij}$ — what a width bottleneck on `msg_dim` constrains;
- **node**, $\sum_j m_{ij}$ — what `node_fnc` actually receives. A direction that sums to
  zero around every cell adds variance to the edge spectrum but not to the node one.

Each spectrum is summarised by its participation ratio
$d_{PR} = \left(\sum_k \lambda_k\right)^2 / \sum_k \lambda_k^2$ — exactly $k$ for $k$ equal
eigenvalues and zeros elsewhere — and by the number of components that reaches 90, 99 and
99.9 % of the variance.

In [ ]:
# --- PCA spectrum of the 128-channel message ------------------------------------
# Channel covariance of m_ij over EVERY edge and of sum_j m_ij over every node of
# a held-out mesh, pooled over the same N_STEPS timesteps as the flux read-outs.
# Only running first and second moments are kept, never the (steps, E, 128)
# stack, so unlike the read-outs above nothing is subsampled. The cache holds the
# covariances, not just their eigenvalues, so other projections of the message
# need no new pass. Helpers (load_mesh, HISTORY, RESID_EXP, the probe) come from
# the flux cells above.
pca_cache = processed_data_dir / "parametric_message_pca_residual.npz"
PCA_THRESH = (0.90, 0.99, 0.999)


def compute_message_covariance(exp):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = processed_data_dir / exp / "checkpoints"
    state = torch.load(ckpt / "model.pt", map_location=device)
    # The normalizer was fitted on the TRAINING meshes and saved with the
    # checkpoint; refitting it here would leak the held-out geometry.
    norm = FeatureNormalizer()
    norm.load(ckpt / "normalizer.pt")

    out = {}
    for mesh in FLUX_MESHES:
        graph, T_seq = load_mesh(mesh)
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()
        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()

        moments = {"edge": [0, 0, 0], "node": [0, 0, 0]}   # sum x, sum x x^T, count
        probe = MessageProbe(model, to_cpu=False)
        with probe, torch.no_grad():
            for i in steps:
                model(ds[i].to(device))
                for level, x in (("edge", probe.messages[0]), ("node", probe.aggregated[0])):
                    x = x.double()
                    mom = moments[level]
                    mom[0], mom[1], mom[2] = mom[0] + x.sum(0), mom[1] + x.T @ x, mom[2] + x.shape[0]
                probe.clear()

        for level, (s, ss, n) in moments.items():
            mean = s / n
            out[f"{exp}__{mesh}__{level}_cov"] = (ss / n - torch.outer(mean, mean)).cpu().numpy()
        del probe, model
    return out


def message_spectrum(exp, mesh, level):
    """Eigenvalues of the cached channel covariance, largest first."""
    return np.linalg.eigvalsh(msg_pca[f"{exp}__{mesh}__{level}_cov"])[::-1].clip(min=0)


def spectrum_summary(lam):
    """Participation ratio (sum lam)^2 / sum lam^2, and the number of components
    that reaches each fraction of the variance in PCA_THRESH."""
    frac = np.cumsum(lam) / lam.sum()
    return (lam.sum() ** 2 / (lam ** 2).sum(),
            [int(np.searchsorted(frac, t)) + 1 for t in PCA_THRESH])


if pca_cache.exists():
    msg_pca = dict(np.load(pca_cache, allow_pickle=False))
else:
    msg_pca = compute_message_covariance(RESID_EXP)
    np.savez_compressed(pca_cache, **msg_pca)

print(f"PCA of the msg_dim = 128 residual message, {N_STEPS} probed timesteps")
print(f"{'':25s}{'variance share':^21s}{'components for':^24s}")
print(f"{'mesh':<11}{'level':<7}{'d_PR':>7}{'PC1':>7}{'PC1-2':>7}{'PC1-3':>7}"
      + "".join(f"{f'{100*t:g}%':>8}" for t in PCA_THRESH))
for mesh in FLUX_MESHES:
    for level in ("edge", "node"):
        lam = message_spectrum(RESID_EXP, mesh, level)
        d_pr, n_comp = spectrum_summary(lam)
        share = np.cumsum(lam[:3]) / lam.sum()
        print(f"{mesh:<11}{level:<7}{d_pr:>7.2f}" + "".join(f"{v:>7.2f}" for v in share)
              + "".join(f"{k:>8d}" for k in n_comp))

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The edge message m_ij is what a width bottleneck constrains (solid, PLOT1) and
# its sum at the receiver what node_fnc actually receives (dashed, PLOT2).
# Rows: the spectrum itself, and the variance the first k components leave
# unexplained, on which 90 / 99 / 99.9 % are the 1e-1 / 1e-2 / 1e-3 lines.
pca_styles = {"edge": (r"edge  $m_{ij}$", PLOT1_COLOR, "-"),
              "node": (r"node  $\Sigma_j\, m_{ij}$", PLOT2_COLOR, "--")}
K_TICKS = [1, 2, 4, 8, 16, 32, 64, 128]
# The last few eigenvalues fall to round-off (~1e-15 of the total); showing them
# would spend most of each axis on digits that carry nothing.
Y_FLOOR = 1e-7

fig, axs = plt.subplots(2, len(FLUX_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, sharey="row", constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("PCA Spectrum of the Message, msg_dim = 128, Residual FV, h = 1",
             fontsize=24, color=AX_COLOR)

for col, mesh in enumerate(FLUX_MESHES):
    for ax in axs[:, col]:
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)
        # ax.set_yscale("log")
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

    d_pr = []
    for level, (label, color, ls) in pca_styles.items():
        lam = message_spectrum(RESID_EXP, mesh, level)
        k = np.arange(1, lam.size + 1)
        d_pr.append(f"{level} {spectrum_summary(lam)[0]:.1f}")
        axs[0, col].plot(k, lam / lam.sum(), label=label,
                         color=color, linewidth=2.5, linestyle=ls)
        axs[1, col].plot(k, 1 - np.cumsum(lam) / lam.sum(), label=label,
                         color=color, linewidth=2.5, linestyle=ls)

    for t in PCA_THRESH:
        axs[1, col].axhline(1 - t, color=AX_COLOR, linewidth=1.5, linestyle=":")
        # labelled below the line at the left, where the curves are still above all three
        axs[1, col].annotate(f"{100*t:g} %", xy=(0.0, 1 - t), xycoords=("axes fraction", "data"),
                             xytext=(4, -3), textcoords="offset points", ha="left", va="top",
                             fontsize=12, color=AX_COLOR)

    axs[0, col].set_title(mesh, fontsize=19)
    axs[0, col].annotate(r"$d_{PR}$: " + ", ".join(d_pr), xy=(0.97, 0.95),
                         xycoords="axes fraction", ha="right", va="top",
                         fontsize=14, color=AX_COLOR)
    axs[1, col].set_xlabel("Principal component $k$", fontsize=15)

# shared axes: setting the scale and ticks once applies to every panel
axs[1, 0].set_xscale("log", base=2)
axs[1, 0].set_xticks(K_TICKS, labels=[str(k) for k in K_TICKS])
axs[1, 0].xaxis.set_minor_locator(NullLocator())
axs[1, 0].set_xlim(1, K_TICKS[-1])
for ax in axs[:, 0]:
    ax.set_ylim(Y_FLOOR, 1.5)

axs[0, 0].set_ylabel(r"$\lambda_k \,/\, \Sigma\, \lambda$", fontsize=15)
axs[1, 0].set_ylabel(r"$1 - \Sigma_{i \leq k}\, \lambda_i \,/\, \Sigma\, \lambda$", fontsize=15)

legend = axs[0, 0].legend(fontsize=14, loc="lower left")
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

# plt.savefig("../outputs/plots/parametric_message_pca_residual.png", dpi=300, transparent=True)

### What comes out

**No clean cutoff per edge.** The spectrum of $m_{ij}$ decays smoothly, with no gap after a
handful of components that would name a width. The first component carries only 26 – 36 %
of the variance and the participation ratio is $d_{PR} = 6.0 - 8.2$; 90 % of the variance
takes 11 – 14 components, 99 % takes 37 – 42 and 99.9 % takes 66 – 70. The last four
eigenvalues are zero to round-off (below $10^{-12}$ of the total, off the bottom of the axes),
so the message spans at most 124 of its 128 dimensions, and another 11 – 12 sit below
$10^{-7}$.

**The aggregate is far more concentrated.** At node level $d_{PR} = 2.1 - 2.9$: the first
component of $\sum_j m_{ij}$ carries 54 – 67 % of its variance and the first three 82 – 87 %,
and 90 % takes 5 – 7 components against 11 – 14 per edge. From $k = 3$ down to the round-off
tail the node spectrum lies below the edge one on every mesh, so summing over a cell's faces
piles the variance onto a few leading directions. Not all of that is temperature
information: every node receives its number of faces times the mean message, which puts
variance along the mean direction wherever the face count changes from node to node.

**What this says about the width.** The spectrum counts the directions the message spreads
its variance over, not the ones `node_fnc` needs: a low-variance direction still matters if
the first `Linear` of `node_fnc` weighs it heavily, and a high-variance one can be ignored.
So it gives no width to hard-code, only a range — one channel is below even the node-level
$d_{PR}$, about ten keep 90 % of the edge variance, and past ~40 only a 1 % tail is left.
Whether that tail matters is a question for the loss (a sweep over `msg_dim`, or a
bottleneck that lets training choose), not for the spectrum.

## LayerNorm (`layer_norm = True`)

The 128-channel residual model with LayerNorm added
(`parametric_history1_msg_dim128_layernorm_residual`), against the same model without it
(`parametric_history1_mesh_correct_edge_attr_residual`, the `msg_dim = 128` run above). A
LayerNorm follows the node and edge encoders and every message $m_{ij}$, before the sum at the
receiver; the node MLP of the single MP layer decodes $T$ and is left plain. Both runs train on
the same six meshes for 200 epochs, and the first cell below asserts that the mesh split, the
normalizer and the increment scale $s$ are identical, so LayerNorm is what the comparison
isolates.

- **Loss decay.** Training and validation MSE as saved by `train_parametric.py`.
- **RMAE and RMSE** exactly as `test_parametric.py` prints them, in kelvin, summed over every
  node and sample of a held-out mesh:
  $\mathrm{RMAE} = \sum|T_{pred}-T_{true}| \,/\, \sum|T_{true}|$ and
  $\mathrm{RMSE} = \sum(T_{pred}-T_{true})^2 \,/\, \sum T_{true}^2$ — a relative *squared*
  error, with no square root. Reported for the one-step predictions, as there, and for the
  autoregressive rollout, whose MAE is checked against the curves saved at training time.
  Cached in `parametric_errors_layernorm.npz`.
- **Message vs flux, $R^2$ only.** `compute_model_flux_stats` from the residual cell, on the
  same 12 timesteps and seeded faces, cached in `parametric_message_flux_layernorm.npz`. The
  probe records the message *after* the LayerNorm, since that is what gets summed: before
  LayerNorm's learned scale and shift, every $m_{ij}$ has zero mean and unit spread over its
  128 channels, so the flux can only be carried by the message's direction, not its size.

In [ ]:
# --- LayerNorm vs plain, 128-channel residual: loss, RMAE and RMSE -------------
# RMAE and RMSE as test_parametric.py prints them: sums over every node and every
# sample of a held-out mesh, in kelvin, where "RMSE" is the relative SQUARED error
# (no square root). Computed for the one-step predictions, as there, and over the
# rollout. Helpers (load_mesh, HISTORY, RESID_EXP) come from the flux cells above.
import json

from models.autoregressive_training import rollout

LN_EXP    = "parametric_history1_msg_dim128_layernorm_residual"
LN_CASES  = {"no LayerNorm": RESID_EXP, "LayerNorm": LN_EXP}
LN_MESHES = ["model_003", "model_007", "model_010", "model_011"]
ln_errors_cache = processed_data_dir / "parametric_errors_layernorm.npz"

ln_train_loss, ln_val_loss, ln_saved_mae, ln_setup = {}, {}, {}, {}
for key, cname in LN_CASES.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    ln_train_loss[key] = np.load(ckpt / "train_losses.npy")
    ln_val_loss[key] = np.load(ckpt / "val_losses.npy")
    ln_saved_mae[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                         for mesh in LN_MESHES}
    with open(ckpt / "mesh_split.json") as fh:
        split = json.load(fh)
    state = torch.load(ckpt / "model.pt", map_location="cpu")
    ln_setup[key] = {
        "split": split,
        "norm": torch.load(ckpt / "normalizer.pt"),
        "delta_scale": float(state["delta_scale"]),
        "msg_dim": state["mp_layers.0.msg_fnc.6.weight"].shape[0],
        "layer_norm": "node_encoder_norm.weight" in state,
    }

# The comparison isolates LayerNorm only if nothing else moved.
s0, s1 = ln_setup["no LayerNorm"], ln_setup["LayerNorm"]
assert s0["split"] == s1["split"], \
    f"mesh splits differ: {s0['split']} vs {s1['split']}"
assert all(torch.equal(s0["norm"][k], s1["norm"][k]) for k in s0["norm"]), \
    "normalizers differ"
assert s0["delta_scale"] == s1["delta_scale"], \
    f"delta_scale differs: {s0['delta_scale']} vs {s1['delta_scale']}"
assert s0["msg_dim"] == s1["msg_dim"] == 128, \
    f"msg_dim differs: {s0['msg_dim']} vs {s1['msg_dim']}"
assert (s0["layer_norm"], s1["layer_norm"]) == (False, True), \
    "the two checkpoints are not LayerNorm off / on"


def compute_ln_errors():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in LN_MESHES:
        graph, T_seq = load_mesh(mesh)
        for cname in LN_CASES.values():
            ckpt = processed_data_dir / cname / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                **widths_from_state_dict(state), out_dim=1,
                residual="delta_scale" in state, history=HISTORY,
                layer_norm="node_encoder_norm.weight" in state,
            ).to(device)
            model.load_state_dict(state)
            model.eval()
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)

            # one-step: every sample sees the true T^n. The four sums are
            # accumulated as it goes instead of stacking (n_samples, N) twice.
            sums = torch.zeros(4, dtype=torch.float64)   # |e|, |T|, e^2, T^2
            with torch.no_grad():
                for i in range(len(ds)):
                    d = ds[i].to(device)
                    T_p = norm.inverse_transform_T(model(d).cpu()).double()
                    T_t = norm.inverse_transform_T(d.y.cpu()).double()
                    e = T_p - T_t
                    sums += torch.stack([e.abs().sum(), T_t.abs().sum(),
                                         e.pow(2).sum(), T_t.pow(2).sum()])
            out[f"{cname}__{mesh}__rmae_step"] = float(sums[0] / sums[1])
            out[f"{cname}__{mesh}__rmse_step"] = float(sums[2] / sums[3])

            T_pred, T_true = rollout(model, ds, device=device)
            e, T_true = (T_pred - T_true).double(), T_true.double()   # (n_steps, N) [K]
            out[f"{cname}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_true.abs().sum())
            out[f"{cname}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_true.pow(2).sum())
            out[f"{cname}__{mesh}__mae_roll"] = e.abs().mean(1).numpy()
            del model
    return out


if ln_errors_cache.exists():
    ln_errors = dict(np.load(ln_errors_cache, allow_pickle=False))
else:
    ln_errors = compute_ln_errors()
    np.savez_compressed(ln_errors_cache, **ln_errors)

# If the rollout MAE did not reproduce, the rollout RMAE / RMSE would describe a
# different rollout from the one saved at training time.
mae_dev = max(float(np.abs(ln_errors[f"{cname}__{mesh}__mae_roll"] - ln_saved_mae[key][mesh]).max())
              for key, cname in LN_CASES.items() for mesh in LN_MESHES)
print(f"recomputed vs saved rollout MAE: max |difference| = {mae_dev:.1e} K\n")

print(f"{'':18s}" + "".join(f"{key:>17s}" for key in LN_CASES))
print(f"{'train loss, last':18s}" + "".join(f"{ln_train_loss[k][-1]:>17.3e}" for k in LN_CASES))
print(f"{'val loss, last':18s}" + "".join(f"{ln_val_loss[k][-1]:>17.3e}" for k in LN_CASES))
print(f"{'val loss, best':18s}" + "".join(
    f"{f'{ln_val_loss[k].min():.3e} @{ln_val_loss[k].argmin()}':>17s}" for k in LN_CASES))

# Columns: (one-step, rollout) x (RMAE, RMSE), each without / with LayerNorm.
cols = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
pair = (LN_CASES["no LayerNorm"], LN_CASES["LayerNorm"])
print(f"\n{'':10s}{'one-step':^44s}{'rollout':^44s}")
print(f"{'':10s}" + "".join(f"{metric.upper():^22s}" for _, metric in cols))
print(f"{'mesh':10s}" + f"{'no LN':>11s}{'LN':>11s}" * len(cols))
for mesh in LN_MESHES:
    print(f"{mesh:10s}" + "".join(f"{float(ln_errors[f'{c}__{mesh}__{metric}_{mode}']):>11.3e}"
                                  for mode, metric in cols for c in pair))
print(f"{'mean':10s}" + "".join(
    f"{np.mean([float(ln_errors[f'{c}__{mesh}__{metric}_{mode}']) for mesh in LN_MESHES]):>11.3e}"
    for mode, metric in cols for c in pair))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# LayerNorm is the variant under test (solid, PLOT1) and the plain 128-channel
# residual run the baseline (dashed, PLOT2), the roles msg_dim = 1 / 128 take
# above. Shared y-axis, so the training and validation panels read on one scale.
ln_styles = {"LayerNorm": (PLOT1_COLOR, "-"), "no LayerNorm": (PLOT2_COLOR, "--")}

fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Loss Decay, LayerNorm vs none, Residual FV, msg_dim = 128, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", ln_train_loss),
                                     ("Validation Loss", ln_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, (color, ls) in ln_styles.items():
        ax.plot(losses[key], label=key, color=color, linewidth=2.5, linestyle=ls)

    # Value where each curve ends: the higher one labelled above its line and
    # the lower one below, so the two labels cannot collide.
    ends = sorted(((losses[k][-1], k) for k in ln_styles), reverse=True)
    for (y, key), dy, va in zip(ends, (8, -8), ("bottom", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=14, color=AX_COLOR)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_layernorm_loss.png", dpi=300, transparent = True)

In [ ]:
# --- Message vs FV flux: R^2 only, LayerNorm vs plain --------------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces, so every row lines up with the plain 128-channel model's
# (flux_resid). Only the R^2 read-outs are printed: F, dT and w each regressed on
# all 128 channels, sum_j m_ij against the net flux at node level, and the same
# three edge read-outs restricted to the near-orthogonal faces.
flux_ln_cache = processed_data_dir / "parametric_message_flux_layernorm.npz"

if flux_ln_cache.exists():
    flux_ln = dict(np.load(flux_ln_cache, allow_pickle=False))
else:
    flux_ln = compute_model_flux_stats(LN_EXP)
    np.savez_compressed(flux_ln_cache, **flux_ln)

flux_lnc = {**flux_resid, **flux_ln}
print(f"R^2 of the linear read-out from the message, {N_STEPS} probed timesteps")
print(f"{'mesh':<11}{'model':<14}{'F':>7}{'dT':>7}{'w':>7}{'node':>7}   F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    for label, exp in LN_CASES.items():
        print(f"{mesh:<11}{label:<14}"
              + "".join(f"{v:>7.3f}" for v in flux_lnc[f"{exp}__{mesh}__r2"])
              + f"{float(flux_lnc[f'{exp}__{mesh}__r2_node']):>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_lnc[f"{exp}__{mesh}__r2_near"]))

## Regularization (`l1`)

The message-sparsity machinery of `models/msg_regularization.py` on the 64-channel residual
model, at two penalty weights, against the run it was added to. All three are `history = 1`
with no LayerNorm, trained on the same six meshes for 200 epochs, and the first cell below
asserts that the mesh split, the normalizer, the increment scale $s$ and the message width are
identical:

- **baseline** (`parametric_history1_msg_dim64_residual`): no `MessageRegularizer`.
- **`l1`, $\alpha = 0$** (`..._l1_a0`): the control of the sparsity sweep. `kind = "l1"` with
  `alpha = 0`, so the penalty $\langle|m_{ij}|\rangle$ is measured and logged every epoch but
  never enters the loss — `lam_eff` is 0 for all 200 epochs, which the cell below checks. What
  the control *does* switch on is the **per-channel scale fix**: after every optimizer step,
  `after_step` divides column $c$ of `node_fnc[0]`'s message block by $r_c$ and multiplies row
  $c$ of `msg_fnc[-1]` (weight and bias) by the same $r_c$, until every message column has norm
  $c_0$ — their mean at initialisation. That is exactly the rescaling symmetry
  $m_c \to s\,m_c$, $W_{:,c} \to W_{:,c}/s$ of the two `Linear`s either side of the message, so
  it leaves every prediction unchanged. It only decides *where* each channel's scale sits, so
  that $\sigma_c$ measures how much `node_fnc` uses channel $c$ rather than how large `msg_fnc`
  happens to write it.
- **`l1`, $\alpha = 0.01$** (`..._l1_a0.01`): the same, with the penalty actually on. The loss
  becomes $\mathrm{mse} + \alpha\,\mathrm{ramp}(e)\,\overline{\mathrm{mse}}\,
  \langle|m_{ij}|\rangle / \mathrm{ref}$, where $\overline{\mathrm{mse}}$ is a moving average of
  the batch MSE and $\mathrm{ref}$ the mean $|m_{ij}|$ over the last warmup epoch. Tying the
  penalty to $\overline{\mathrm{mse}}$ fixes its exchange rate with the data term however far
  the MSE falls: $\alpha = 0.01$ buys a 1 % MSE increase per unit of $|m|$ removed, measured
  against where $|m|$ stood at the end of warmup. The ramp is 0 for 20 epochs, then rises
  linearly to 1 over 10, so the penalty is at full strength from epoch 29.

**A three-point ladder, not a controlled experiment.** `train_parametric.py` never calls
`torch.manual_seed`, so the three runs differ in weight initialisation and batch order as well
as in the regularizer — visible in the two $c_0$ printed below, which are each run's own mean
column norm at init. The baseline additionally predates commit `249dd36`, in which
`ReduceLROnPlateau(patience = 10)` stepped on the bare validation MSE from epoch 0, where the
two regularized runs step it only once the ramp completes. All three end at $3\times10^{-9}$
by different paths. So this section reports what these three checkpoints do; it does not
isolate $\alpha$.

- **Loss decay.** Training and validation MSE as saved by `train_parametric.py`.
- **RMAE and RMSE** exactly as `test_parametric.py` prints them, in kelvin, summed over every
  node and sample of a held-out mesh:
  $\mathrm{RMAE} = \sum|T_{pred}-T_{true}| \,/\, \sum|T_{true}|$ and
  $\mathrm{RMSE} = \sum(T_{pred}-T_{true})^2 \,/\, \sum T_{true}^2$ — a relative *squared*
  error, with no square root. Reported for the one-step predictions, as there, and for the
  autoregressive rollout, whose MAE is checked against the curves saved at training time.
  Cached per run in `parametric_errors_msg_reg.npz`.
- **Message vs flux, $R^2$ only.** `compute_model_flux_stats` from the residual cell, on the
  same 12 timesteps and seeded faces as every read-out above, cached per run in
  `parametric_message_flux_msg_reg.npz`. No checkpoint here carries a message LayerNorm, so the
  probe records $m_{ij}$ as `msg_fnc` writes it; the scale fix moves that scale between
  `msg_fnc[-1]` and `node_fnc[0]`, and every read-out is an $R^2$ or a correlation, so none of
  them depends on it.
- **How many channels the message uses**, two ways. The participation ratio
  $\mathrm{PR} = (\sum_c \sigma_c)^2 / \sum_c \sigma_c^2$ over the channels, logged every
  epoch by `MessageRegularizer` — the quantity `hoyer` minimises directly and `l1` shrinks
  indirectly — and the same functional applied to the *eigenvalues* of the message's channel
  covariance, $d_{PR}$, as in the PCA section above. The second is the one that counts
  directions rather than channels, since `node_fnc` opens with a `Linear` that absorbs any
  rotation of its input. Covariances cached per run in
  `parametric_message_pca_msg_reg.npz`, over every edge and node of the held-out meshes at
  the same 12 timesteps.

In [ ]:
# --- l1 at two alphas vs baseline, 64-channel residual: loss, RMAE and RMSE ----
# RMAE and RMSE as test_parametric.py prints them: sums over every node and every
# sample of a held-out mesh, in kelvin, where "RMSE" is the relative SQUARED error
# (no square root). Computed for the one-step predictions, as there, and over the
# rollout. Helpers (load_mesh, HISTORY) come from the flux cells above.
import json

from models.autoregressive_training import rollout

MSG64_EXP   = "parametric_history1_msg_dim64_residual"
REG_A0_EXP  = "parametric_history1_msg_dim64_residual_l1_a0"
REG_A01_EXP = "parametric_history1_msg_dim64_residual_l1_a0.01"
# ordered by how much is switched on: nothing, the scale fix, the penalty too
REG_CASES   = {"baseline": MSG64_EXP, "l1, α = 0": REG_A0_EXP, "l1, α = 0.01": REG_A01_EXP}
REG_ALPHAS  = {"l1, α = 0": 0.0, "l1, α = 0.01": 0.01}   # the penalised runs only
REG_MESHES  = ["model_003", "model_007", "model_010", "model_011"]
reg_errors_cache = processed_data_dir / "parametric_errors_msg_reg.npz"

reg_train_loss, reg_val_loss, reg_saved_mae, reg_setup = {}, {}, {}, {}
for key, cname in REG_CASES.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    reg_train_loss[key] = np.load(ckpt / "train_losses.npy")
    reg_val_loss[key] = np.load(ckpt / "val_losses.npy")
    reg_saved_mae[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                          for mesh in REG_MESHES}
    with open(ckpt / "mesh_split.json") as fh:
        split = json.load(fh)
    state = torch.load(ckpt / "model.pt", map_location="cpu")
    reg_setup[key] = {
        "split": split,
        "norm": torch.load(ckpt / "normalizer.pt"),
        "delta_scale": float(state["delta_scale"]),
        "msg_dim": state["mp_layers.0.msg_fnc.6.weight"].shape[0],
        "layer_norm": "node_encoder_norm.weight" in state,
    }

# Nothing about the model itself may differ, or the comparison is not about the
# regularizer at all.
base = reg_setup["baseline"]
for key, s in reg_setup.items():
    assert s["split"] == base["split"], f"{key}: mesh split differs from the baseline's"
    assert all(torch.equal(s["norm"][k], base["norm"][k]) for k in base["norm"]), \
        f"{key}: normalizer differs from the baseline's"
    assert s["delta_scale"] == base["delta_scale"], \
        f"{key}: delta_scale {s['delta_scale']} vs {base['delta_scale']}"
    assert s["msg_dim"] == 64, f"{key}: msg_dim {s['msg_dim']}, not 64"
    assert not s["layer_norm"], \
        f"{key}: the scale fix needs msg_fnc[-1] to write the message, so layer_norm must be False"

# ... and each regularized run really ran at the alpha its name claims.
reg_logs = {key: dict(np.load(processed_data_dir / REG_CASES[key] / "checkpoints"
                              / "msg_reg_log.npz"))
            for key in REG_ALPHAS}
for key, alpha in REG_ALPHAS.items():
    log = reg_logs[key]
    assert str(log["kind"]) == "l1" and float(log["alpha"]) == alpha, \
        f"{key}: log says kind={log['kind']}, alpha={log['alpha']}"
    # the penalty is off through warmup either way, and stays off only at alpha = 0
    assert np.all(log["lam_eff"][:int(log["warmup"])] == 0.0), f"{key}: penalty on during warmup"
    assert (float(log["lam_eff"][-1]) > 0.0) == (alpha > 0.0), \
        f"{key}: lam_eff ends at {log['lam_eff'][-1]}, which does not match alpha = {alpha}"


def compute_reg_errors(exps, load=load_mesh):
    # load: as in compute_model_flux_stats
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in REG_MESHES:
        graph, T_seq = load(mesh)
        # the span of the field, to read an error in kelvin against
        out[f"{mesh}__T_span"] = float(T_seq.double().max() - T_seq.double().min())
        for cname in exps:
            ckpt = processed_data_dir / cname / "checkpoints"
            state = torch.load(ckpt / "model.pt", map_location=device)
            norm = FeatureNormalizer()
            norm.load(ckpt / "normalizer.pt")
            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                **widths_from_state_dict(state), out_dim=1,
                residual="delta_scale" in state, history=HISTORY,
                layer_norm="node_encoder_norm.weight" in state,
            ).to(device)
            model.load_state_dict(state)
            model.eval()
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)

            # one-step: every sample sees the true T^n. The four sums are
            # accumulated as it goes instead of stacking (n_samples, N) twice.
            sums = torch.zeros(4, dtype=torch.float64)   # |e|, |T|, e^2, T^2
            with torch.no_grad():
                for i in range(len(ds)):
                    d = ds[i].to(device)
                    T_p = norm.inverse_transform_T(model(d).cpu()).double()
                    T_t = norm.inverse_transform_T(d.y.cpu()).double()
                    e = T_p - T_t
                    sums += torch.stack([e.abs().sum(), T_t.abs().sum(),
                                         e.pow(2).sum(), T_t.pow(2).sum()])
            out[f"{cname}__{mesh}__rmae_step"] = float(sums[0] / sums[1])
            out[f"{cname}__{mesh}__rmse_step"] = float(sums[2] / sums[3])

            T_pred, T_true = rollout(model, ds, device=device)
            e, T_true = (T_pred - T_true).double(), T_true.double()   # (n_steps, N) [K]
            out[f"{cname}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_true.abs().sum())
            out[f"{cname}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_true.pow(2).sum())
            # per-step curves, in kelvin: MAE to check against the saved one, and
            # the RMSE train_parametric.py does not save
            out[f"{cname}__{mesh}__mae_roll"] = e.abs().mean(1).numpy()
            out[f"{cname}__{mesh}__rmse_roll_curve"] = e.pow(2).mean(1).sqrt().numpy()
            del model
    return out


# Filled per run, so that adding an alpha to REG_CASES costs only that run's
# rollouts instead of recomputing the ones already cached.
reg_errors = (dict(np.load(reg_errors_cache, allow_pickle=False))
              if reg_errors_cache.exists() else {})
missing = [c for c in REG_CASES.values()
           if f"{c}__{REG_MESHES[0]}__rmae_step" not in reg_errors]
if missing:
    reg_errors.update(compute_reg_errors(missing))
    np.savez_compressed(reg_errors_cache, **reg_errors)

# If the rollout MAE did not reproduce, the rollout RMAE / RMSE would describe a
# different rollout from the one saved at training time. Relative, not absolute:
# the baseline's curve reaches ~1e15 K, where an absolute difference says nothing.
mae_dev = max(float(np.abs(reg_errors[f"{cname}__{mesh}__mae_roll"] / reg_saved_mae[key][mesh] - 1).max())
              for key, cname in REG_CASES.items() for mesh in REG_MESHES)
print(f"recomputed vs saved rollout MAE: max |relative difference| = {mae_dev:.1e}\n")

print(f"{'':18s}" + "".join(f"{key:>17s}" for key in REG_CASES))
print(f"{'train loss, last':18s}" + "".join(f"{reg_train_loss[k][-1]:>17.3e}" for k in REG_CASES))
print(f"{'val loss, last':18s}" + "".join(f"{reg_val_loss[k][-1]:>17.3e}" for k in REG_CASES))
print(f"{'val loss, best':18s}" + "".join(
    f"{f'{reg_val_loss[k].min():.3e} @{reg_val_loss[k].argmin()}':>17s}" for k in REG_CASES))

# The learning-rate path is one of the things that differ between the runs, and
# only the two regularized ones saved it.
print()
reg_lrs = {}
for key in REG_ALPHAS:
    reg_lrs[key] = lrs = np.load(processed_data_dir / REG_CASES[key] / "checkpoints" / "lrs.npy")
    print(f"{key:12s}: LR {lrs[0]:.0e} -> {lrs[-1]:.0e}, cut at epochs "
          + ", ".join(str(int(e)) for e in np.nonzero(np.diff(lrs))[0] + 1))
print(f"{'baseline':12s}: LR not saved (the run predates commit 249dd36)")

# col_norm pinned at c0 is the scale fix doing its job; PR and mean |m| are what
# the penalty moves. c0 is each run's own mean column norm at initialisation, so
# two different values are two different initialisations.
print()
for key, log in reg_logs.items():
    c0 = float(log["c0"])
    print(f"{key:12s}: node_fnc[0] message columns at c0 = {c0:.4f} "
          f"(max deviation {np.abs(log['col_norm'] / c0 - 1).max():.1e}), "
          f"PR {log['pr'][0]:.1f} -> {log['pr'][-1]:.1f} of {log['sigma'].shape[1]}, "
          f"mean |m| {log['mean_abs'][0]:.3f} -> {log['mean_abs'][-1]:.3f}")

# One row per (mesh, run): with three runs the four metrics read better down the
# columns than side by side.
cols = [("step", "rmae"), ("step", "rmse"), ("roll", "rmae"), ("roll", "rmse")]
print(f"\n{'mesh':<11}{'run':<14}{'one-step RMAE':>15}{'one-step RMSE':>15}"
      f"{'rollout RMAE':>15}{'rollout RMSE':>15}")
for mesh in REG_MESHES:
    for key, cname in REG_CASES.items():
        print(f"{mesh:<11}{key:<14}"
              + "".join(f"{float(reg_errors[f'{cname}__{mesh}__{metric}_{mode}']):>15.3e}"
                        for mode, metric in cols))
print()
for key, cname in REG_CASES.items():
    print(f"{'mean':<11}{key:<14}"
          + "".join(f"{np.mean([float(reg_errors[f'{cname}__{mesh}__{metric}_{mode}']) for mesh in REG_MESHES]):>15.3e}"
                    for mode, metric in cols))

# Line styles / curves of this section, also used by the sections below.
# Relative per-step errors (CLAUDE.md) from the kelvin curves of reg_errors.
reg_rel = {"mae_roll": lambda curve, mesh: rmae_curve(curve, param_T_stats(mesh)[0]),
           "rmse_roll_curve": lambda curve, mesh: rmse_curve(curve, param_T_stats(mesh)[1])}


In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"   # third run: warm, so it separates from both above

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The ladder runs baseline -> scale fix -> scale fix + penalty, so the penalised
# run is the variant under test (solid, PLOT1), the alpha = 0 control sits between
# (dash-dot, PLOT3) and the unregularized run is the baseline (dashed, PLOT2).
# Shared y-axis, so the training and validation panels read on one scale.
reg_styles = {"l1, α = 0.01": (PLOT1_COLOR, "-"),
              "l1, α = 0":    (PLOT3_COLOR, "-."),
              "baseline":     (PLOT2_COLOR, "--")}

fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Loss Decay, l1 Message Penalty, Residual FV, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

for ax, (title, losses) in zip(axs, (("Training Loss", reg_train_loss),
                                     ("Validation Loss", reg_val_loss))):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, (color, ls) in reg_styles.items():
        ax.plot(losses[key], label=key, color=color, linewidth=2.5, linestyle=ls)

    # Value where each curve ends, stacked top to bottom in that order, so three
    # labels at nearly the same height cannot collide.
    ends = sorted(((losses[k][-1], k) for k in reg_styles), reverse=True)
    for (y, key), dy, va in zip(ends, (16, 0, -16), ("bottom", "center", "top")):
        ax.annotate(f"{y:.1e}", xy=(len(losses[key]) - 1, y), xytext=(0, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=13, color=reg_styles[key][0])

    # The end of the penalty ramp: from here the alpha = 0.01 run is training on
    # the full penalty, and the LR scheduler is allowed to step.
    ramp_end = int(reg_logs["l1, α = 0.01"]["warmup"]) + int(reg_logs["l1, α = 0.01"]["ramp_len"]) - 1
    ax.axvline(ramp_end, color=AX_COLOR, linewidth=1.3, alpha=0.85,
               linestyle=(0, (4, 3)))
    ax.annotate("ramp complete", xy=(ramp_end, 0.02), xycoords=("data", "axes fraction"),
                xytext=(4, 0), textcoords="offset points", ha="left", va="bottom",
                fontsize=12, color=AX_COLOR, alpha=0.8)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=16)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_msg_reg_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

reg_styles = {"l1, α = 0.01": (PLOT1_COLOR, "-"),
              "l1, α = 0":    (PLOT3_COLOR, "-."),
              "baseline":     (PLOT2_COLOR, "--")}

# Rows: rollout RMAE and RMSE; columns: the held-out meshes, each run at
# its final checkpoint. Log y on every panel: the baseline's rollout leaves the
# physical range entirely, and on a linear axis the other two curves would lie
# flat on the bottom of the panel.
fig, axs = plt.subplots(2, len(REG_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, l1 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

for row, (metric, ylabel) in enumerate((("mae_roll", "RMAE"),
                                        ("rmse_roll_curve", "RMSE"))):
    for col, mesh in enumerate(REG_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, (color, ls) in reg_styles.items():
            ax.plot(reg_rel[metric](reg_errors[f"{REG_CASES[key]}__{mesh}__{metric}"], mesh), label=key,
                    color=color, linewidth=2.5, linestyle=ls)
        ax.set_yscale("log")

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        # (an error as large as the span at every step, in the same relative units)
        span = float(reg_errors[f"{mesh}__T_span"])
        span_rel = reg_rel[metric](np.full(len(param_T_stats(mesh)[0]), span), mesh)
        ax.plot(span_rel, color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({span:.0f} K)", xy=(0, span_rel[0]),
                        xytext=(4, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=12, color=AX_COLOR)

        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=15)

        legend = ax.legend(fontsize=12, loc="center right")
        legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
        legend.get_frame().set_edgecolor(AX_COLOR)        # border color
        legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
        for text in legend.get_texts():
            text.set_color(AX_COLOR)                      # label text color

        ax.tick_params(axis='both', labelsize=13)
        # major only: the panels that span few decades would otherwise be ruled
        # by minor gridlines the twenty-decade ones do not show.
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_msg_reg_rollout.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The printed table as a figure. Rows: one-step and rollout; columns: RMAE and
# RMSE (the relative SQUARED error, so it sits far below the RMAE). Bars run in
# the ladder's order, baseline first.
reg_bars = {"baseline":     (MSG64_EXP, PLOT2_COLOR),
            "l1, α = 0":    (REG_A0_EXP, PLOT3_COLOR),
            "l1, α = 0.01": (REG_A01_EXP, PLOT1_COLOR)}
x = np.arange(len(REG_MESHES))
width = 0.26

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE, l1 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

panels = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
for ax, (mode, metric) in zip(axs.ravel(), panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    vals = {label: np.array([float(reg_errors[f"{exp}__{mesh}__{metric}_{mode}"])
                             for mesh in REG_MESHES])
            for label, (exp, _) in reg_bars.items()}
    hi = max(v.max() for v in vals.values())
    # The one-step errors sit inside half a decade and read better from 0; the
    # rollout spans thirty, where only a log axis shows every run at once.
    if mode == "roll":
        # bars on a log axis grow from the bottom of the axis, not from 0, so
        # the floor has to be set before they are drawn
        floor = min(v.min() for v in vals.values()) / 10
        ax.set_yscale("log")
        # top chosen so the tallest bar reaches 80% of the panel whatever the
        # span, leaving the same room for its label in a 16- and a 38-decade one
        ax.set_ylim(floor, 10 ** (np.log10(floor)
                                  + (np.log10(hi) - np.log10(floor)) / 0.80))
    else:
        floor = 0.0
        ax.set_ylim(0, hi * 1.35)

    # A label is wider than its bar, so a group's three are stacked above the
    # group's TALLEST bar rather than each above its own: at these spans a short
    # bar's label would otherwise land on top of its taller neighbour's.
    tops = np.maximum.reduce(list(vals.values()))
    for i, (label, (exp, color)) in enumerate(reg_bars.items()):
        off = (i - 1) * (width + 0.02)        # 2% surface gap between neighbours
        ax.bar(x + off, vals[label] - floor, width, color=color,
               label=label, bottom=floor)
        for xi, v, top in zip(x + off, vals[label], tops):
            ax.annotate(f"{v:.1e}", (xi, top),
                        xytext=(0, 4 + 13 * (2 - i)), textcoords="offset points",
                        ha="center", fontsize=10, color=color)

    ax.set_title(f"{'one-step' if mode == 'step' else 'rollout'}  ·  {metric.upper()}",
                 fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(REG_MESHES, fontsize=13)
    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure, drawn inside the first panel: four boxes would
# each cover a group's labels, and a figure-level one lands on the suptitle.
legend = axs[0, 0].legend(loc="upper right", fontsize=12)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_msg_reg_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- Message vs FV flux: the l1 runs vs the baseline ---------------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces, so every row lines up with the read-outs above. All three runs
# are msg_dim = 64, so none has the wider message's easier test. The scale fix
# moves each channel's scale between msg_fnc[-1] and node_fnc[0]; every column
# here is an R^2, a correlation or an energy fraction, so none of them sees it.
flux_reg_cache = processed_data_dir / "parametric_message_flux_msg_reg.npz"

flux_reg = (dict(np.load(flux_reg_cache, allow_pickle=False))
            if flux_reg_cache.exists() else {})
missing = [c for c in REG_CASES.values() if f"{c}__{REG_MESHES[0]}__r2" not in flux_reg]
if missing:
    for exp in missing:
        flux_reg.update(compute_model_flux_stats(exp))
    np.savez_compressed(flux_reg_cache, **flux_reg)

print(f"R^2 of the linear read-out from the message, {N_STEPS} probed timesteps")
print(f"{'mesh':<11}{'run':<14}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in REG_MESHES:
    for label, exp in REG_CASES.items():
        print(f"{mesh:<11}{label:<14}{int(flux_reg[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux_reg[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux_reg[f"{exp}__{mesh}__r2"])
              + f"{float(flux_reg[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux_reg[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_reg[f"{exp}__{mesh}__r2_near"]))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs FV Flux, l1 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

# The three edge-level read-outs of the panels above plus the node-level one.
# All three runs are 64 channels wide, so each bar is the best linear combination
# of the same number of channels.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.26
models = {"baseline":     (MSG64_EXP, PLOT2_COLOR),
          "l1, α = 0":    (REG_A0_EXP, PLOT3_COLOR),
          "l1, α = 0.01": (REG_A01_EXP, PLOT1_COLOR)}

for ax, mesh in zip(axs.ravel(), REG_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (label, (exp, color)) in enumerate(models.items()):
        vals = np.append(flux_reg[f"{exp}__{mesh}__r2"], flux_reg[f"{exp}__{mesh}__r2_node"])
        off = (i - 1) * (width + 0.02)        # 2% surface gap between neighbours
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=AX_COLOR)

    legend = ax.legend(fontsize=13, ncols=3, loc="upper center")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_title(mesh, fontsize=19)
    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.25)   # room for the legend above the bars
    ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
    ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# plt.savefig("../outputs/plots/parametric_history1_msg_reg_flux_r2.png", dpi=300, transparent = True)

In [ ]:
# --- PCA spectrum of the message, and the PR the penalty actually sees ---------
# Channel covariance of m_ij over EVERY edge and of sum_j m_ij over every node of
# a held-out mesh, pooled over the same N_STEPS timesteps as the flux read-outs,
# for each of the three runs. compute_message_covariance and the two spectrum
# helpers come from the PCA cell above; only the cache is this section's.
#
# Two different counts of "how many channels" appear below and they answer
# different questions:
#   PR    from msg_reg_log.npz: (sum_c sigma_c)^2 / sum_c sigma_c^2 over the
#         CHANNELS, which is what the hoyer penalty minimises and what l1 shrinks
#         indirectly. It is basis-dependent.
#   d_PR  the same functional applied to the EIGENVALUES of the channel
#         covariance. node_fnc opens with a Linear, which absorbs any rotation of
#         its input, so this is the count that says how many directions the
#         message really uses.
pca_reg_cache = processed_data_dir / "parametric_message_pca_msg_reg.npz"

msg_pca_reg = (dict(np.load(pca_reg_cache, allow_pickle=False))
               if pca_reg_cache.exists() else {})
missing = [c for c in REG_CASES.values()
           if f"{c}__{REG_MESHES[0]}__edge_cov" not in msg_pca_reg]
if missing:
    for exp in missing:
        msg_pca_reg.update(compute_message_covariance(exp))
    np.savez_compressed(pca_reg_cache, **msg_pca_reg)


def reg_spectrum(exp, mesh, level):
    """Eigenvalues of the cached channel covariance, largest first."""
    return np.linalg.eigvalsh(msg_pca_reg[f"{exp}__{mesh}__{level}_cov"])[::-1].clip(min=0)


print(f"PCA of the msg_dim = 64 message, {N_STEPS} probed timesteps")
print(f"{'':30s}{'variance share':^14s}{'components for':^21s}")
print(f"{'mesh':<11}{'run':<14}{'level':<6}{'d_PR':>7}{'PC1':>7}{'PC1-3':>7}"
      + "".join(f"{f'{100*t:g}%':>7}" for t in PCA_THRESH))
for mesh in REG_MESHES:
    for key, exp in REG_CASES.items():
        for level in ("edge", "node"):
            lam = reg_spectrum(exp, mesh, level)
            d_pr, n_comp = spectrum_summary(lam)
            share = np.cumsum(lam[:3]) / lam.sum()
            print(f"{mesh:<11}{key:<14}{level:<6}{d_pr:>7.2f}"
                  + f"{share[0]:>7.2f}{share[2]:>7.2f}"
                  + "".join(f"{k:>7d}" for k in n_comp))

# The two counts side by side: the penalty moves the channel-basis PR it is
# written against, and d_PR is what that buys in directions.
print(f"\n{'run':<14}{'PR (channels, training)':>26}{'d_PR edge':>22}{'d_PR node':>22}")
for key, exp in REG_CASES.items():
    pr = (f"{reg_logs[key]['pr'][0]:.1f} -> {reg_logs[key]['pr'][-1]:.1f} of "
          f"{reg_logs[key]['sigma'].shape[1]}") if key in reg_logs else "not logged"
    rng = {lvl: [spectrum_summary(reg_spectrum(exp, m, lvl))[0] for m in REG_MESHES]
           for lvl in ("edge", "node")}
    print(f"{key:<14}{pr:>26}"
          + "".join(f"{f'{min(rng[l]):.1f} - {max(rng[l]):.1f}':>22}" for l in ("edge", "node")))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# What the penalty does while it trains, straight from msg_reg_log.npz: the
# channel participation ratio it is meant to cut, and the mean |m| it is written
# on. Only the two regularized runs have a log — the baseline never built a
# MessageRegularizer, so there is no third curve here.
pr_styles = {"l1, α = 0.01": (PLOT1_COLOR, "-"), "l1, α = 0": (PLOT3_COLOR, "-.")}

fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Message Statistics During Training, l1 Message Penalty, msg_dim = 64",
             fontsize=24, color=AX_COLOR)

panels = [("Participation ratio of $\\sigma_c$", "pr",
           "PR  [channels of 64]"),
          ("Mean $|m_{ij}|$", "mean_abs", r"$\langle |m_{ij}| \rangle$")]
for ax, (title, field, ylabel) in zip(axs, panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, (color, ls) in pr_styles.items():
        y = reg_logs[key][field]
        ax.plot(y, label=key, color=color, linewidth=2.5, linestyle=ls)
        ax.annotate(f"{y[-1]:.3g}", xy=(len(y) - 1, y[-1]), xytext=(-4, 0),
                    textcoords="offset points", ha="right", va="bottom",
                    fontsize=14, color=color)

    # Warmup ends where l1 fixes its reference; the ramp ends where the penalty
    # reaches full strength. Nothing before the second line is being penalised.
    warm = int(reg_logs["l1, α = 0.01"]["warmup"])
    ramp_end = warm + int(reg_logs["l1, α = 0.01"]["ramp_len"]) - 1
    for e, tag in ((warm - 1, "warmup ends"), (ramp_end, "ramp complete")):
        ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.annotate("warmup / ramp", xy=(warm - 1, 0.02), xycoords=("data", "axes fraction"),
                xytext=(-4, 0), textcoords="offset points", ha="right", va="bottom",
                fontsize=12, color=AX_COLOR, alpha=0.8)

    legend = ax.legend(fontsize=16, loc="center right")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

axs[0].set_ylim(0, 64)

# plt.savefig("../outputs/plots/parametric_history1_msg_reg_pr.png", dpi=300, transparent = True)

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Rows: the edge message m_ij, which a width bottleneck constrains, and its sum
# at the receiver, which is what node_fnc gets. Columns: the held-out meshes.
# Each spectrum is normalized by its own total variance, so the three runs are
# comparable however large each writes its message.
spec_styles = {"l1, α = 0.01": (PLOT1_COLOR, "-"),
               "l1, α = 0":    (PLOT3_COLOR, "-."),
               "baseline":     (PLOT2_COLOR, "--")}
K_TICKS = [1, 2, 4, 8, 16, 32, 64]

fig, axs = plt.subplots(2, len(REG_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, sharey="row", constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("PCA Spectrum of the Message, l1 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

for col, mesh in enumerate(REG_MESHES):
    for row, level in enumerate(("edge", "node")):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        d_pr = []
        for key, (color, ls) in spec_styles.items():
            lam = reg_spectrum(REG_CASES[key], mesh, level)
            ax.plot(np.arange(1, lam.size + 1), lam / lam.sum(), label=key,
                    color=color, linewidth=2.5, linestyle=ls)
            d_pr.append(f"{spectrum_summary(lam)[0]:.1f}")
        # d_PR in the legend's order, so each number reads off its own curve
        ax.annotate(r"$d_{PR}$: " + " / ".join(d_pr), xy=(0.97, 0.95),
                    xycoords="axes fraction", ha="right", va="top",
                    fontsize=13, color=AX_COLOR)

        ax.set_yscale("log")
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Principal component $k$", fontsize=15)
        if col == 0:
            ax.set_ylabel(f"{level}: $\\lambda_k / \\sum\\lambda$", fontsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# shared axes: setting the scale and ticks once applies to every panel
axs[1, 0].set_xscale("log", base=2)
axs[1, 0].set_xticks(K_TICKS, labels=[str(k) for k in K_TICKS])
axs[1, 0].xaxis.set_minor_locator(NullLocator())
axs[1, 0].set_xlim(1, K_TICKS[-1])
for ax in axs[:, 0]:
    ax.set_ylim(3e-6, 1.5)

# Drawn inside the first panel: every curve falls from top-left to bottom-right,
# so its lower-left corner is empty, and a figure-level legend lands on the suptitle.
legend = axs[0, 0].legend(loc="lower left", fontsize=12)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_msg_reg_pca.png", dpi=300, transparent = True)

### What comes out

**One-step accuracy improves monotonically along the ladder.** Best validation MSE goes
$2.79\times10^{-5}$ (baseline, epoch 41) → $2.69\times10^{-5}$ ($\alpha = 0$, epoch 74) →
$2.54\times10^{-5}$ ($\alpha = 0.01$, epoch 84), and last training loss
$3.33 \to 2.35 \to 1.98\times10^{-5}$. The one-step errors on the held-out meshes follow:
mean RMSE $8.8 \to 7.3 \to 6.5\times10^{-6}$, with mean RMAE essentially tied between the two
regularized runs ($4.81\times10^{-4}$ each) against $5.6\times10^{-4}$ for the baseline. Adding
the penalty costs nothing one step at a time — if anything it helps slightly, which is what a
regularizer is supposed to do.

**The rollout does not follow it.** Over 120 autoregressive steps the unregularized baseline
leaves the physical range on every held-out mesh — final MAE $6\times10^{15}$ K on `model_003`,
$2\times10^{19}$ K on `model_007`, $9\times10^{5}$ K on `model_010`, $8\times10^{9}$ K on
`model_011`, against a field that spans 300 K. The $\alpha = 0$ control is bounded everywhere,
14 – 26 K. The penalised run sits **between the two, closer to the control**: 10 K on
`model_010` (its best of the three) but 408 K on `model_003` and $1$–$3\times10^{4}$ K on
`model_011` and `model_007`. As rollout RMAE: 0.020 – 0.034 at $\alpha = 0$, 0.019 – 4.3 at
$\alpha = 0.01$, and $82$ to $1.3\times10^{15}$ for the baseline. So the penalty does not bring
back the instability the baseline shows, but it gives up part of the margin the scale fix alone
had. With one seed per run and three different initialisations, that ordering between the two
regularized runs is not something this pair can separate from seed-to-seed spread; the gap to
the baseline, at ten or more decades, is not.

Note also that these are the same checkpoints ranked in opposite orders by the two measures:
$\alpha = 0.01$ has the best one-step loss of the three and the second-best rollout. One-step
validation MSE says very little about rollout stability here — the recomputed MAE reproduces
the curves saved at training time to $8\times10^{-6}$ relative, divergence included, so none of
this is an artefact of re-running the rollouts.

**The penalty moves its own metric, and almost nothing else.** At $\alpha = 0.01$ the channel
participation ratio falls from 59.1 to 49.2 of 64 and mean $|m_{ij}|$ ends at 0.70, against 57.6
and 1.32 for the $\alpha = 0$ control: about eight effective channels removed and the message
halved in size. But no channel goes quiet — the weakest $\sigma_c$ is still 10 % of the
strongest (24 % at $\alpha = 0$) — so what L1 does at this weight is skew the channel spectrum,
top $\sigma$ up from 2.45 to 3.58 and median down from 1.18 to 0.80, not sparsify it. The PR
panel also shows the run never finished: PR bottoms out at 41.3 at epoch 51, then *climbs back*
to 49.2 and freezes from epoch 96, which is where the scheduler has cut the learning rate to
$3\times10^{-6}$. What 200 epochs measures here is as much the LR schedule as the penalty.

**And the PCA says that skew buys no dimensions at all.** The basis-free count, $d_{PR}$ of the
covariance eigenvalues, is $5.4 - 6.7$ per edge for the baseline, $3.1 - 5.4$ at $\alpha = 0$
and $4.5 - 6.2$ at $\alpha = 0.01$ — the penalised run is *above* the control on three of the
four meshes, and it takes **more** components to reach 90 % of the variance, 14 – 16 against
12 – 15 for the control and 11 – 12 for the baseline. At node level all three are tighter
($d_{PR} = 2.1 - 3.7$) and again the penalty does not lead. This is exactly the gap the PCA
section above warns about: `node_fnc` opens with a `Linear`, which absorbs any rotation of its
input, so a message confined to $k$ directions is a $k$-channel message in *some* basis — and
PR over the channels, which is what `l1` shrinks and what `hoyer` minimises directly, is
measured in the one basis that has no particular claim on it. All three messages already live
in 3 – 7 of their 64 dimensions before any penalty is applied; $\alpha = 0.01$ does not reduce
that, it only redistributes the variance inside it.

**What the penalty does buy is a more flux-like message.** $\alpha = 0.01$ has the best flux
read-out of the three on every target but the geometric weight: $R^2 = 0.67-0.73$ for $F$
(against $0.63-0.70$ baseline, $0.60-0.69$ at $\alpha = 0$), $0.74-0.77$ for $\Delta T$,
$0.66-0.76$ at node level, and its single most flux-correlated channel reaches
$|r| = 0.39-0.49$. Only $w = D_T|S_f|/(|d|\cos)$ slips, $0.82-0.88$ against $0.83-0.90$. The
antisymmetric energy fraction rises too, $0.28-0.30$ against $0.20-0.21$ for the other two —
still far from the 1.0 a conservative flux would give, but moving toward it for the first time
in this study. Shrinking the message pushes what is left onto the physically meaningful
directions, even though it does not reduce how many of them there are.

## Regularization (`hoyer`)

The participation-ratio penalty of `models/msg_regularization.py`, against the `l1`, $\alpha = 0$
control of the section above. Both are the 64-channel residual model at `history = 1` with no
LayerNorm, trained on the same six meshes for 200 epochs, and the first cell below asserts that
the mesh split, the normalizer, the increment scale $s$ and the message width are identical:

- **`l1`, $\alpha = 0$** (`..._l1_a0`): the per-channel scale fix on, no penalty in the loss.
  The control, exactly as above.
- **`hoyer`, $\alpha = 0.01$** (`..._hoyer_a0.01`): the same scale fix, plus a penalty on the
  participation ratio of the channel standard deviations,
  $$\mathrm{PR}(\sigma) = \frac{\left(\sum_c \sigma_c\right)^2}{\sum_c \sigma_c^2},
  \qquad \sigma_c = \mathrm{std}_{ij}\, m_{ij,c},$$
  with $\sigma_c$ taken over the edges of each training batch. PR is an effective number of
  channels: $k$ equally active channels and the rest silent give exactly $k$. The loss is
  $\mathrm{mse} + \alpha\,\mathrm{ramp}(e)\,\overline{\mathrm{mse}}\,\mathrm{PR}$ with
  $\mathrm{ref} = 1$, so $\alpha = 0.01$ accepts a 1 % MSE increase per effective channel
  removed. Same 20-epoch warmup and 10-epoch ramp as `l1`, full strength from epoch 29.

Two properties set it apart from `l1`. **PR is blind to a scale common to all channels**:
shrinking the whole message does not lower it, only concentrating the variance on fewer
channels does. Geometrically $\mathrm{PR} = 64\cos^2\theta$, with $\theta$ the angle between
$\sigma$ and the diagonal $(1, \dots, 1)$, so the penalty turns $\sigma$ toward the axes; its
gradient is orthogonal to $\sigma$, pushing the channels above the $\sigma$-weighted mean
$\sum\sigma_c^2 / \sum\sigma_c$ up and the ones below it down. And **a channel it switches off
is constant, not zero**: $\sigma_c = 0$ leaves the mean $\mu_c$ free, so such a channel is
removed by mean-ablation, not by zeroing it. It still needs the scale fix: PR ignores a common
scale but not per-channel ones, and without the fix the network could point $\sigma$ along a
single axis for free by rescaling each channel against its `node_fnc[0]` column.

**Controlled as far as a pair of single runs allows.** Unlike the baseline of the `l1` ladder
above, both runs step `ReduceLROnPlateau` by the same rule: on
`MessageRegularizer.plateau_metric`, and only once the ramp is complete (for the control that
metric is the bare validation MSE). `train_parametric.py` still never calls
`torch.manual_seed`, so they differ in weight initialisation and batch order as well — the two
$c_0$ printed below — and each side is one seed.

- **Loss decay and PR**, as saved by `train_parametric.py` and logged every epoch by
  `MessageRegularizer`, with the channel spectrum $\sigma_c$ at the last epoch.
- **RMAE and RMSE**, one-step and rollout, exactly as in the section above and added to the
  same cache, `parametric_errors_msg_reg.npz`.
- **Message vs flux, $R^2$ only**, as above, cached in `parametric_message_flux_msg_reg.npz`.
- **How many directions the message uses.** $d_{PR}$ of the covariance eigenvalues as above,
  cached in `parametric_message_pca_msg_reg.npz`, and one more count that separates the two
  ways PR can fall. `node_fnc` opens with a `Linear`, so rotating the channel basis leaves every
  prediction unchanged while changing the channel PR. The lowest channel PR any rotation can
  reach is $\left(\sum_k\sqrt{\lambda_k}\right)^2 / \sum_k\lambda_k$, the channel PR of the
  principal basis (Schur–Horn: the diagonal of a rotated covariance is majorized by its
  eigenvalues, and $\left(\sum_c\sqrt{d_c}\right)^2$ is Schur-concave). A channel PR at that
  floor means the channels are lined up with the directions the message uses; a floor that
  falls means the message uses fewer directions.

In [ ]:
# --- hoyer vs the l1 alpha = 0 control, 64-channel residual: loss, PR, RMAE, RMSE
# Both runs have the per-channel scale fix on and only the Hoyer one trains on a
# penalty, so the penalty is the one thing meant to differ. compute_reg_errors
# and the errors cache come from the Regularization cell above: the Hoyer run is
# added to the same cache, and the control's entries are read back from it.
HOY_EXP   = "parametric_history1_msg_dim64_residual_hoyer_a0.01"
# control first, then the run under test
HOY_CASES = {"l1, α = 0": REG_A0_EXP, "hoyer, α = 0.01": HOY_EXP}
HOY_KINDS = {"l1, α = 0": ("l1", 0.0), "hoyer, α = 0.01": ("hoyer", 0.01)}

hoy_train_loss, hoy_val_loss, hoy_lrs, hoy_saved_mae, hoy_setup, hoy_logs = {}, {}, {}, {}, {}, {}
for key, cname in HOY_CASES.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    hoy_train_loss[key] = np.load(ckpt / "train_losses.npy")
    hoy_val_loss[key] = np.load(ckpt / "val_losses.npy")
    hoy_lrs[key] = np.load(ckpt / "lrs.npy")
    hoy_logs[key] = dict(np.load(ckpt / "msg_reg_log.npz"))
    hoy_saved_mae[key] = {mesh: np.load(ckpt / f"rollout_mae_{mesh}.npy")
                          for mesh in REG_MESHES}
    with open(ckpt / "mesh_split.json") as fh:
        split = json.load(fh)
    state = torch.load(ckpt / "model.pt", map_location="cpu")
    hoy_setup[key] = {
        "split": split,
        "norm": torch.load(ckpt / "normalizer.pt"),
        "delta_scale": float(state["delta_scale"]),
        "msg_dim": state["mp_layers.0.msg_fnc.6.weight"].shape[0],
        "layer_norm": "node_encoder_norm.weight" in state,
    }

# Nothing about the model itself may differ, or the comparison is not about the
# penalty at all.
base = hoy_setup["l1, α = 0"]
for key, s in hoy_setup.items():
    assert s["split"] == base["split"], f"{key}: mesh split differs from the control's"
    assert all(torch.equal(s["norm"][k], base["norm"][k]) for k in base["norm"]), \
        f"{key}: normalizer differs from the control's"
    assert s["delta_scale"] == base["delta_scale"], \
        f"{key}: delta_scale {s['delta_scale']} vs {base['delta_scale']}"
    assert s["msg_dim"] == 64, f"{key}: msg_dim {s['msg_dim']}, not 64"
    assert not s["layer_norm"], \
        f"{key}: the scale fix needs msg_fnc[-1] to write the message, so layer_norm must be False"

# ... each run is the kind and alpha its name claims, with the penalty off through
# warmup and on at the end only where alpha > 0 ...
for key, (kind, alpha) in HOY_KINDS.items():
    log = hoy_logs[key]
    assert str(log["kind"]) == kind and float(log["alpha"]) == alpha, \
        f"{key}: log says kind={log['kind']}, alpha={log['alpha']}"
    assert np.all(log["lam_eff"][:int(log["warmup"])] == 0.0), f"{key}: penalty on during warmup"
    assert (float(log["lam_eff"][-1]) > 0.0) == (alpha > 0.0), \
        f"{key}: lam_eff ends at {log['lam_eff'][-1]}, which does not match alpha = {alpha}"
# ... and on one schedule, so a warmup or ramp line drawn below marks the same
# epoch in both runs.
for field in ("warmup", "ramp_len"):
    assert len({int(log[field]) for log in hoy_logs.values()}) == 1, f"{field} differs between the runs"
HOY_WARM = int(hoy_logs["hoyer, α = 0.01"]["warmup"])
HOY_RAMP_END = HOY_WARM + int(hoy_logs["hoyer, α = 0.01"]["ramp_len"]) - 1

reg_errors = (dict(np.load(reg_errors_cache, allow_pickle=False))
              if reg_errors_cache.exists() else {})
missing = [c for c in HOY_CASES.values()
           if f"{c}__{REG_MESHES[0]}__rmae_step" not in reg_errors]
if missing:
    reg_errors.update(compute_reg_errors(missing))
    np.savez_compressed(reg_errors_cache, **reg_errors)

# If the rollout MAE did not reproduce, the rollout RMAE / RMSE would describe a
# different rollout from the one saved at training time.
mae_dev = max(float(np.abs(reg_errors[f"{cname}__{mesh}__mae_roll"] / hoy_saved_mae[key][mesh] - 1).max())
              for key, cname in HOY_CASES.items() for mesh in REG_MESHES)
print(f"recomputed vs saved rollout MAE: max |relative difference| = {mae_dev:.1e}\n")

print(f"{'':18s}" + "".join(f"{key:>19s}" for key in HOY_CASES))
print(f"{'train loss, last':18s}" + "".join(f"{hoy_train_loss[k][-1]:>19.3e}" for k in HOY_CASES))
print(f"{'val loss, last':18s}" + "".join(f"{hoy_val_loss[k][-1]:>19.3e}" for k in HOY_CASES))
print(f"{'val loss, best':18s}" + "".join(
    f"{f'{hoy_val_loss[k].min():.3e} @{hoy_val_loss[k].argmin()}':>19s}" for k in HOY_CASES))

print()
for key, lrs in hoy_lrs.items():
    print(f"{key:16s}: LR {lrs[0]:.0e} -> {lrs[-1]:.0e}, cut at epochs "
          + ", ".join(str(int(e)) for e in np.nonzero(np.diff(lrs))[0] + 1))

# col_norm pinned at c0 is the scale fix holding. PR at the last warmup epoch,
# at the end of the ramp and at the end; then how the final sigma is spread: the
# share of sum(sigma^2) in the two largest channels, and the largest, median and
# smallest sigma_c. c0 is each run's own mean column norm at initialisation.
print()
for key, log in hoy_logs.items():
    c0, pr, sig = float(log["c0"]), log["pr"], np.sort(log["sigma"][-1])[::-1]
    print(f"{key:16s}: node_fnc[0] message columns at c0 = {c0:.4f} "
          f"(max deviation {np.abs(log['col_norm'] / c0 - 1).max():.1e})")
    print(f"{'':18s}PR {pr[0]:.1f} (epoch 0) -> {pr[HOY_WARM - 1]:.1f} (warmup ends) -> "
          f"{pr[HOY_RAMP_END]:.1f} (ramp complete) -> {pr[-1]:.2f} (epoch {len(pr) - 1}), "
          f"min {pr.min():.2f} @{pr.argmin()}")
    print(f"{'':18s}sigma: top-2 share {(sig[:2] ** 2).sum() / (sig ** 2).sum():.4f}, "
          f"max {sig[0]:.3g}, median {np.median(sig):.3g}, min {sig[-1]:.3g}, "
          f"mean |m| {log['mean_abs'][-1]:.3g}")

cols = [("step", "rmae"), ("step", "rmse"), ("roll", "rmae"), ("roll", "rmse")]
print(f"\n{'mesh':<11}{'run':<17}{'one-step RMAE':>15}{'one-step RMSE':>15}"
      f"{'rollout RMAE':>15}{'rollout RMSE':>15}")
for mesh in REG_MESHES:
    for key, cname in HOY_CASES.items():
        print(f"{mesh:<11}{key:<17}"
              + "".join(f"{float(reg_errors[f'{cname}__{mesh}__{metric}_{mode}']):>15.3e}"
                        for mode, metric in cols))
print()
for key, cname in HOY_CASES.items():
    print(f"{'mean':<11}{key:<17}"
          + "".join(f"{np.mean([float(reg_errors[f'{cname}__{mesh}__{metric}_{mode}']) for mesh in REG_MESHES]):>15.3e}"
                    for mode, metric in cols))
print()
print(f"{'final rollout MAE [K]':<28}" + "".join(f"{mesh:>11}" for mesh in REG_MESHES))
for key, cname in HOY_CASES.items():
    print(f"{'':<11}{key:<17}"
          + "".join(f"{float(reg_errors[f'{cname}__{mesh}__mae_roll'][-1]):>11.3g}"
                    for mesh in REG_MESHES))

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The Hoyer run is the variant under test (solid, PLOT1); the l1 alpha = 0 control
# keeps the colour and dash-dot it has in the section above.
hoy_styles = {"hoyer, α = 0.01": (PLOT1_COLOR, "-"), "l1, α = 0": (PLOT3_COLOR, "-.")}
PR_TICKS = [1, 2, 4, 8, 16, 32, 64]

# Top: the MSE as saved by train_parametric.py, the two panels on one y scale.
# Bottom left: the channel participation ratio, logged every epoch over all the
# epoch's training edges. Bottom right: where it ends, as the last epoch's sigma_c
# sorted and divided by the largest, since the two runs write their message at
# very different sizes.
fig, axs = plt.subplots(2, 2, figsize=(1400*px, 1000*px), constrained_layout=True)
axs[0, 1].sharey(axs[0, 0])

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Training, Hoyer vs l1 α = 0 Message Penalty, Residual FV, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

for ax in axs.ravel():
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=15)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# The last field puts both end labels below their curves: on the PR panel the
# upper curve ends against the top of the axis.
epoch_panels = [(axs[0, 0], "Training Loss", hoy_train_loss, "MSE Loss", False),
                (axs[0, 1], "Validation Loss", hoy_val_loss, None, False),
                (axs[1, 0], r"Participation Ratio of $\sigma_c$",
                 {k: log["pr"] for k, log in hoy_logs.items()}, "PR  [channels of 64]", True)]
for ax, title, curves, ylabel, both_below in epoch_panels:
    for key, (color, ls) in hoy_styles.items():
        ax.plot(curves[key], label=key, color=color, linewidth=2.5, linestyle=ls)

    # Value where each curve ends, the higher one above its end and the lower one
    # below, so two ends at nearly the same height cannot collide.
    ends = sorted(((curves[k][-1], k) for k in hoy_styles), reverse=True)
    offsets = [(-6, "top")] * 2 if both_below else [(6, "bottom"), (-6, "top")]
    for (y, key), (dy, va) in zip(ends, offsets):
        ax.annotate(f"{y:.3g}", xy=(len(curves[key]) - 1, y), xytext=(-4, dy),
                    textcoords="offset points", ha="right", va=va,
                    fontsize=13, color=hoy_styles[key][0])

    # Warmup ends where the penalty starts to ramp in; the ramp ends where it is at
    # full strength and the LR scheduler is allowed to step.
    for e in (HOY_WARM - 1, HOY_RAMP_END):
        ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.annotate("warmup / ramp", xy=(HOY_WARM - 1, 0.02), xycoords=("data", "axes fraction"),
                xytext=(-4, 0), textcoords="offset points", ha="right", va="bottom",
                fontsize=12, color=AX_COLOR, alpha=0.8)

    ax.set_yscale("log")
    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=17)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=17)

# PR spans 1 to 64: a base-2 axis gives every halving of the channel count the
# same height, so the collapse and the slow tail after it both read.
ax = axs[1, 0]
ax.set_yscale("log", base=2)
ax.set_yticks(PR_TICKS, labels=[str(k) for k in PR_TICKS])
ax.yaxis.set_minor_locator(NullLocator())
ax.set_ylim(1, 64)

ax = axs[1, 1]
for key, (color, ls) in hoy_styles.items():
    sig = np.sort(hoy_logs[key]["sigma"][-1])[::-1]
    ax.plot(np.arange(1, sig.size + 1), sig / sig[0], label=key, color=color,
            linewidth=2.5, linestyle=ls, marker="o", markersize=3.5)
ax.set_yscale("log")
ax.set_xlim(0, 65)
ax.set_title(r"Channel Stds $\sigma_c$ at the Last Epoch", fontsize=20)
ax.set_xlabel("Channel, sorted by $\\sigma_c$", fontsize=17)
ax.set_ylabel(r"$\sigma_c$ / largest $\sigma_c$", fontsize=17)

for ax, loc in zip(axs.ravel(), ("upper right", "upper right", "center right", "lower left")):
    legend = ax.legend(fontsize=14, loc=loc)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

# plt.savefig("../outputs/plots/parametric_history1_msg_hoyer_training.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

hoy_styles = {"hoyer, α = 0.01": (PLOT1_COLOR, "-"), "l1, α = 0": (PLOT3_COLOR, "-.")}

# Rows: rollout RMAE and RMSE, through reg_rel of the l1 plot above; columns:
# the held-out meshes, each run at its final checkpoint. Log y, as in the section above, so a run that leaves the
# physical range does not flatten the other against the bottom of the panel.
fig, axs = plt.subplots(2, len(REG_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, Hoyer vs l1 α = 0 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

for row, (metric, ylabel) in enumerate((("mae_roll", "RMAE"),
                                        ("rmse_roll_curve", "RMSE"))):
    for col, mesh in enumerate(REG_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, (color, ls) in hoy_styles.items():
            ax.plot(reg_rel[metric](reg_errors[f"{HOY_CASES[key]}__{mesh}__{metric}"], mesh), label=key,
                    color=color, linewidth=2.5, linestyle=ls)
        ax.set_yscale("log")

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        # (an error as large as the span at every step, in the same relative units)
        span = float(reg_errors[f"{mesh}__T_span"])
        span_rel = reg_rel[metric](np.full(len(param_T_stats(mesh)[0]), span), mesh)
        ax.plot(span_rel, color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({span:.0f} K)", xy=(0, span_rel[0]),
                        xytext=(4, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=12, color=AX_COLOR)

        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=15)

        legend = ax.legend(fontsize=12, loc="upper left")
        legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
        legend.get_frame().set_edgecolor(AX_COLOR)        # border color
        legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
        for text in legend.get_texts():
            text.set_color(AX_COLOR)                      # label text color

        ax.tick_params(axis='both', labelsize=13)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_msg_hoyer_rollout.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The printed table as a figure. Rows: one-step and rollout; columns: RMAE and
# RMSE (the relative SQUARED error, so it sits far below the RMAE). Control first.
hoy_bars = {"l1, α = 0":       (REG_A0_EXP, PLOT3_COLOR),
            "hoyer, α = 0.01": (HOY_EXP, PLOT1_COLOR)}
x = np.arange(len(REG_MESHES))
width = 0.42

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE, Hoyer vs l1 α = 0 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

panels = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
for ax, (mode, metric) in zip(axs.ravel(), panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    vals = {label: np.array([float(reg_errors[f"{exp}__{mesh}__{metric}_{mode}"])
                             for mesh in REG_MESHES])
            for label, (exp, _) in hoy_bars.items()}
    hi = max(v.max() for v in vals.values())
    lo = min(v.min() for v in vals.values())
    # From 0 when both runs sit within a decade of each other, on a log axis
    # when they do not: a diverging rollout would otherwise leave the other run's
    # bars too short to see.
    if hi / lo > 10:
        # bars on a log axis grow from the bottom of the axis, not from 0, so
        # the floor has to be set before they are drawn
        floor = lo / 10
        ax.set_yscale("log")
        # top chosen so the tallest bar reaches 80% of the panel whatever the span
        ax.set_ylim(floor, 10 ** (np.log10(floor)
                                  + (np.log10(hi) - np.log10(floor)) / 0.80))
    else:
        floor = 0.0
        ax.set_ylim(0, hi * 1.3)

    # With two bars a group, each label fits over its own bar; on a log panel a
    # label stacked over the taller one would float decades above the shorter.
    for i, (label, (exp, color)) in enumerate(hoy_bars.items()):
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        ax.bar(x + off, vals[label] - floor, width, color=color,
               label=label, bottom=floor)
        for xi, v in zip(x + off, vals[label]):
            ax.annotate(f"{v:.2e}", (xi, v), xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=9.5, color=color)

    ax.set_title(f"{'one-step' if mode == 'step' else 'rollout'}  ·  {metric.upper()}",
                 fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(REG_MESHES, fontsize=13)
    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure, drawn inside the first panel: four boxes would
# each cover a group's labels, and a figure-level one lands on the suptitle.
legend = axs[0, 0].legend(loc="upper right", fontsize=12)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_msg_hoyer_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- Message vs FV flux: hoyer vs the l1 alpha = 0 control ----------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces as every read-out above; the Hoyer run is added to the cache of
# the l1 section. Every column is an R^2, a correlation or an energy fraction, so
# none depends on how large a run writes its message.
flux_reg = (dict(np.load(flux_reg_cache, allow_pickle=False))
            if flux_reg_cache.exists() else {})
missing = [c for c in HOY_CASES.values() if f"{c}__{REG_MESHES[0]}__r2" not in flux_reg]
if missing:
    for exp in missing:
        flux_reg.update(compute_model_flux_stats(exp))
    np.savez_compressed(flux_reg_cache, **flux_reg)

print(f"R^2 of the linear read-out from the message, {N_STEPS} probed timesteps")
print(f"{'mesh':<11}{'run':<17}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in REG_MESHES:
    for label, exp in HOY_CASES.items():
        print(f"{mesh:<11}{label:<17}{int(flux_reg[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux_reg[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux_reg[f"{exp}__{mesh}__r2"])
              + f"{float(flux_reg[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux_reg[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux_reg[f"{exp}__{mesh}__r2_near"]))

# The "anti" column is the plain mean over the 64 channels, and in the Hoyer run
# almost all of them carry next to nothing. The two channels that carry the
# message (largest sigma_c in the training log's last epoch) are read out on their
# own, as the antisymmetric energy fraction of each on the same faces.
print(f"\n{'antisymmetric fraction':<41}" + "".join(f"{mesh:>11}" for mesh in REG_MESHES))
for label, exp in HOY_CASES.items():
    top2 = np.argsort(hoy_logs[label]["sigma"][-1])[::-1][:2]
    rows = [("mean of 64 channels", None)] + [(f"channel {c} (sigma rank {r + 1})", c)
                                             for r, c in enumerate(top2)]
    for tag, c in rows:
        vals = [flux_reg[f"{exp}__{mesh}__anti"] for mesh in REG_MESHES]
        print(f"{label:<17}{tag:<24}"
              + "".join(f"{(v.mean() if c is None else v[c]):>11.3f}" for v in vals))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs FV Flux, Hoyer vs l1 α = 0 Message Penalty, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

# The three edge-level read-outs plus the node-level one. Both runs are 64
# channels wide, so each bar is the best linear combination of the same number
# of channels, however few of them a run actually uses.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.36
models = {"l1, α = 0":       (REG_A0_EXP, PLOT3_COLOR),
          "hoyer, α = 0.01": (HOY_EXP, PLOT1_COLOR)}

for ax, mesh in zip(axs.ravel(), REG_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (label, (exp, color)) in enumerate(models.items()):
        vals = np.append(flux_reg[f"{exp}__{mesh}__r2"], flux_reg[f"{exp}__{mesh}__r2_node"])
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=12, color=AX_COLOR)

    legend = ax.legend(fontsize=13, ncols=2, loc="upper center")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_title(mesh, fontsize=19)
    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.25)   # room for the legend above the bars
    ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
    ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# plt.savefig("../outputs/plots/parametric_history1_msg_hoyer_flux_r2.png", dpi=300, transparent = True)

In [ ]:
# --- PCA of the message: fewer directions, or the same ones lined up? ----------
# Channel covariance of m_ij over every edge and of sum_j m_ij over every node of
# a held-out mesh, at the same N_STEPS timesteps; compute_message_covariance and
# spectrum_summary come from the PCA cell above, reg_spectrum and the cache from
# the l1 section, to which the Hoyer run is added.
#
# Three counts, all from the same held-out covariance C:
#   d_PR     (sum lam)^2 / sum lam^2 over the eigenvalues, as everywhere above:
#            how many directions the message uses, in any basis.
#   PR ch    (sum sqrt d_c)^2 / sum d_c over the diagonal d_c = sigma_c^2: the
#            channel participation ratio the penalty sees, here on held-out data.
#   PR min   (sum sqrt lam)^2 / sum lam: the lowest PR ch any rotation of the
#            channels reaches (the diagonal of Q C Q^T is majorized by lam, and
#            (sum sqrt d)^2 is Schur-concave), attained in the principal basis.
# PR ch / PR min = 1 means the channels are lined up with the principal directions;
# a smaller PR min means fewer directions. node_fnc opens with a Linear, so the
# rotation between the two is free: it changes no prediction.
msg_pca_reg = (dict(np.load(pca_reg_cache, allow_pickle=False))
               if pca_reg_cache.exists() else {})
missing = [c for c in HOY_CASES.values()
           if f"{c}__{REG_MESHES[0]}__edge_cov" not in msg_pca_reg]
if missing:
    for exp in missing:
        msg_pca_reg.update(compute_message_covariance(exp))
    np.savez_compressed(pca_reg_cache, **msg_pca_reg)


def channel_prs(exp, mesh, level):
    """PR over the channels, and its floor over rotations, of the cached covariance."""
    d = np.diag(msg_pca_reg[f"{exp}__{mesh}__{level}_cov"]).clip(min=0)
    lam = reg_spectrum(exp, mesh, level)
    return np.sqrt(d).sum() ** 2 / d.sum(), np.sqrt(lam).sum() ** 2 / lam.sum()


print(f"PCA of the msg_dim = 64 message, {N_STEPS} probed timesteps")
print(f"{'':34s}{'variance share':^14s}{'components for':^21s}")
print(f"{'mesh':<11}{'run':<17}{'level':<6}{'d_PR':>7}{'PC1':>7}{'PC1-2':>7}"
      + "".join(f"{f'{100*t:g}%':>7}" for t in PCA_THRESH)
      + f"{'PR ch':>9}{'PR min':>8}")
for mesh in REG_MESHES:
    for key, exp in HOY_CASES.items():
        for level in ("edge", "node"):
            lam = reg_spectrum(exp, mesh, level)
            d_pr, n_comp = spectrum_summary(lam)
            share = np.cumsum(lam[:2]) / lam.sum()
            pr_ch, pr_min = channel_prs(exp, mesh, level)
            print(f"{mesh:<11}{key:<17}{level:<6}{d_pr:>7.2f}"
                  + f"{share[0]:>7.3f}{share[1]:>7.3f}"
                  + "".join(f"{k:>7d}" for k in n_comp)
                  + f"{pr_ch:>9.2f}{pr_min:>8.2f}")

# Ranges over the four meshes, next to the PR logged in training.
print(f"\n{'run':<17}{'PR (training)':>15}" + "".join(
    f"{f'{name} {lvl}':>16}" for lvl in ("edge", "node") for name in ("d_PR", "PR ch", "PR min")))
for key, exp in HOY_CASES.items():
    cells = []
    for lvl in ("edge", "node"):
        per_mesh = [(spectrum_summary(reg_spectrum(exp, m, lvl))[0], *channel_prs(exp, m, lvl))
                    for m in REG_MESHES]
        for j in range(3):
            v = [row[j] for row in per_mesh]
            cells.append(f"{min(v):.2f} - {max(v):.2f}")
    print(f"{key:<17}{hoy_logs[key]['pr'][-1]:>15.2f}" + "".join(f"{c:>16}" for c in cells))

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Rows: the edge message m_ij and its sum at the receiver; columns: the held-out
# meshes. Lines are the PCA spectra, eigenvalues of the channel covariance; dots
# are the same covariance's DIAGONAL, the channel variances sigma_c^2, sorted. The
# two coincide exactly when the channels are the principal directions, so the gap
# between a run's dots and its line is how far its channel basis is from them.
# Each is divided by the total variance, so the two runs compare however large
# each writes its message.
spec_styles = {"hoyer, α = 0.01": (PLOT1_COLOR, "-"), "l1, α = 0": (PLOT3_COLOR, "-.")}
K_TICKS = [1, 2, 4, 8, 16, 32, 64]

fig, axs = plt.subplots(2, len(REG_MESHES), figsize=(1600*px, 800*px),
                        sharex=True, sharey="row", constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("PCA Spectrum vs Channel Variances, Hoyer vs l1 α = 0, msg_dim = 64, h = 1",
             fontsize=24, color=AX_COLOR)

for col, mesh in enumerate(REG_MESHES):
    for row, level in enumerate(("edge", "node")):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        d_pr = []
        for key, (color, ls) in spec_styles.items():
            exp = HOY_CASES[key]
            lam = reg_spectrum(exp, mesh, level)
            var = np.sort(np.diag(msg_pca_reg[f"{exp}__{mesh}__{level}_cov"]).clip(min=0))[::-1]
            k = np.arange(1, lam.size + 1)
            ax.plot(k, lam / lam.sum(), label=f"{key}, eigenvalues",
                    color=color, linewidth=2.5, linestyle=ls)
            ax.plot(k, var / var.sum(), label=f"{key}, channel variances",
                    color=color, linestyle="none", marker="o", markersize=3.5, alpha=0.8)
            d_pr.append(f"{spectrum_summary(lam)[0]:.1f}")
        # d_PR in the legend's order, so each number reads off its own curve
        ax.annotate(r"$d_{PR}$: " + " / ".join(d_pr), xy=(0.97, 0.95),
                    xycoords="axes fraction", ha="right", va="top",
                    fontsize=13, color=AX_COLOR)

        ax.set_yscale("log")
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Component / channel rank $k$", fontsize=15)
        if col == 0:
            ax.set_ylabel(f"{level}: share of variance", fontsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# shared axes: setting the scale and ticks once applies to every panel
axs[1, 0].set_xscale("log", base=2)
axs[1, 0].set_xticks(K_TICKS, labels=[str(k) for k in K_TICKS])
axs[1, 0].xaxis.set_minor_locator(NullLocator())
axs[1, 0].set_xlim(1, K_TICKS[-1])

# Four entries and no corner of a panel left empty, so one legend under the grid.
legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                    loc="outside lower center", ncols=4, fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_msg_hoyer_pca.png", dpi=300, transparent = True)

### What comes out

**The penalty does what it is written to do, fast.** PR is 57.0 at the end of warmup, 4.2 when
the ramp completes at epoch 29, and 2.36 from epoch 80 on, against 56 – 61 throughout for the
control. Two channels, 6 and 45, take the whole message: their $\sigma_c$ grows from about 1.1
at the end of warmup to about 42 at the end of the ramp and 250 by epoch 80, while the other 62
stay at 0.2 – 1.8, and together the two hold 99.97 % of $\sum_c\sigma_c^2$. Little moves after
epoch 80, because little can: the scheduler first cuts the LR at epoch 62 and has it at
$3\times10^{-5}$ by epoch 75 and $3\times10^{-8}$ by 108.

**One-step accuracy pays a moderate price.** Best validation MSE $3.16\times10^{-5}$ (epoch 50)
against $2.69\times10^{-5}$ (epoch 74), +17 %, and last training loss +40 %
($3.29$ vs $2.35\times10^{-5}$). On the held-out meshes mean one-step RMAE rises 12 %
($5.40$ vs $4.82\times10^{-4}$) and RMSE 4 % ($7.66$ vs $7.34\times10^{-6}$), with `model_003`
slightly better than the control on RMSE.

**The rollout diverges on every held-out mesh.** Final MAE is $6\times10^{5}$ K on `model_010`
and $10^{11}$ – $5\times10^{14}$ K on the other three, against 14 – 26 K for the control. The
error leaves the 300 K span of the field at step 29 (`model_007`), 38 (`model_003`), 43
(`model_011`) and 87 (`model_010`), and from there it grows by a constant factor of 1.27 – 1.37
per step — a straight line on the log axis, which is a step map amplifying a perturbation, not
error piling up. The recomputed MAE matches the curves saved at training time to
$5\times10^{-6}$ relative. So the stability the scale fix alone showed in the section above does
not survive this penalty at $\alpha = 0.01$, and the one-step losses give no warning of it.
What drives the divergence is not isolated here; one candidate is the size of what the penalty
keeps. PR is blind to overall scale and nothing else bounds it, so the two surviving channels
grew about 200-fold during training, and `node_fnc` is fed a very large one-dimensional signal.
One seed per side, as before.

**Unlike `l1`, it removes directions, not just channels.** $d_{PR}$ is 1.00 – 1.01 on every
mesh at both edge and node level, against 3.1 – 5.4 (edge) and 2.2 – 2.5 (node) for the
control and 4.5 – 6.2 (edge) for `l1` at $\alpha = 0.01$. The first principal component holds
more than 99.9 % of the edge variance and 99.7 % of the node variance; the control needs 12 – 15
components for 90 %. This is compression, not a rotation of the same information into fewer
channels.

**But the one direction sits in two channels.** Channels 6 and 45 correlate at
$r > 0.9999$ on every held-out mesh: the same signal, written twice. That is why PR stops at
2.36 rather than 1, and why the channel PR on held-out data, 2.3 – 2.6, stays above the
rotation floor of 1.15 – 1.31. Two equal channels are a balance the penalty leaves only slowly:
its gradient in $\sigma_c$ is proportional to $1 - \sigma_c\sum\sigma/\sum\sigma^2$, the same
for two channels of equal $\sigma$, so the push that would separate them grows only with their
difference, here 254 against 234. With the LR gone by epoch 75, the split froze.

**The direction it keeps is not the flux.** The two dominant channels are mostly symmetric
under $i \leftrightarrow j$: antisymmetric fraction 0.07 – 0.12, where a conservative flux gives
1.0 and the control's channels average 0.17 – 0.20. The 64-channel mean, 0.21 – 0.24, is
higher than the control's, but that comes from the 62 near-silent channels, not from the part
of the message that carries the variance. The
linear read-out from the full message drops on every edge target and every mesh: $R^2$ for $F$
0.54 – 0.60 against 0.60 – 0.69, $\Delta T$ 0.60 – 0.71 against 0.67 – 0.75, $w$ 0.76 – 0.86
against 0.84 – 0.92. Only the node level rises slightly (0.64 – 0.68 against 0.61 – 0.67). The
best single channel against $F$ is channel 54, one of the silent ones, at $|r| = 0.39 - 0.49$,
a channel holding about a millionth of the message variance (0.00003 – 0.00012 %): what
flux information is left sits in the channels the penalty silenced.

Note that $d_{PR}$ counts variance, not use. With the scale fix, a silent channel's push into
`node_fnc` is still $c_0\,\sigma_c \approx 0.1 - 1$. Whether the node update reads them — and so
whether the message is really one-dimensional to the network — is what a mean-ablation of the
62 small channels would answer.

## Multi-seed sweep without LR scheduler (`nosched`)

The two regularization sections above compare single runs, so every difference in them is
confounded with the seed. This sweep repeats the comparison with **5 seeds per configuration**
(`FVGNN_SEED = 0 … 4`, `launcher.sh`), each seed collected by `aggregate_seeds.py` into one
`seeds.npz` per configuration. All six are the 64-channel residual model at `history = 1` with
no LayerNorm, 200 epochs, SOAP at a **constant** learning rate of $3\times10^{-3}$
(`lr_scheduler = False`), the same 20-epoch warmup and 10-epoch ramp as above:

- **baseline** (`parametric_history1_msg_dim64_residual_nosched`): no `MessageRegularizer`.
- **control, $\alpha = 0$** (`..._nosched_hoyer_a0`): the per-channel scale fix on, the penalty
  off. It is the same control as `l1`, $\alpha = 0$ above: with $\alpha = 0$ the kind only
  decides what is logged, and with no scheduler there is no plateau metric for it to change.
- **`hoyer`, $\alpha = 0.01$ and $0.1$** (`..._nosched_hoyer_a0.01`, `..._a0.1`).
- **`l1`, $\alpha = 0.01$ and $0.1$** (`..._nosched_l1_a0.01`, `..._a0.1`).

**Now a controlled experiment.** Within a seed, every configuration starts from the same
weights: the mean message-column norm at initialisation, $c_0$, is identical across all six
for each seed, which the first cell below asserts. The regularizer is the only thing that
differs between two runs of the same seed. With no scheduler the learning-rate path is also the
same everywhere, which removes the confound the `l1` ladder had.

- **Loss decay and PR**: training and validation MSE and the channel participation ratio,
  mean ± std over the seeds (sample std, $n - 1$).
- **Rollout MAE**: the per-step curves saved at training time, plotted as the per-step RMAE
  $\mathrm{MAE}_t \,/\, \overline{|T_{true}|}_t$. Across seeds they span up to
  thirty decades, so they are averaged in $\log_{10}$: geometric mean and ×/÷ one geometric std.
  An arithmetic mean would just be the worst seed.
- **RMAE and RMSE**, defined as in the sections above. The **rollout RMAE** needs no checkpoint:
  over $N$ nodes $\sum_i|e_i| = N\,\mathrm{MAE}_t$ at every step, so
  $\mathrm{RMAE} = \sum_t \mathrm{MAE}_t \,/\, \sum_t \overline{|T_{true}|}_t$. On the
  checkpoints cached above this reproduces `compute_reg_errors` to $5\times10^{-6}$ relative.
  The one-step RMAE/RMSE and the rollout RMSE do need the weights.
- **Message vs flux, $R^2$**: `compute_model_flux_stats` per seed, as above.

**What `seeds.npz` does not hold.** `aggregate_seeds.py` reads `model.pt` only to count
parameters, so the weights stay in `processed_data/<exp>/seed_<n>/checkpoints/`, which has not
been copied here. The one-step errors, the rollout RMSE and the flux read-outs are therefore
written to run on whatever seed checkpoints exist, cached in `parametric_errors_nosched.npz`
and `parametric_message_flux_nosched.npz`, and until then print which checkpoints are missing.

In [ ]:
# --- nosched sweep: load the seeds, check them, summarise ---------------------
# One seeds.npz per configuration (aggregate_seeds.py), every array with a leading
# seed axis. Statistics over seeds use the sample std (ddof = 1). Losses are
# averaged as they are; anything that spans decades across seeds (rollout MAE,
# RMAE, RMSE) is averaged in log10, i.e. a geometric mean and a geometric std,
# since one diverging seed would otherwise be the whole arithmetic mean.
# Helpers (load_mesh, HISTORY) come from the flux cells above.
NS_BASE = "parametric_history1_msg_dim64_residual_nosched"
# baseline, the alpha = 0 control (scale fix on, penalty off), then the penalties
NS_CASES = {"baseline":        NS_BASE,
            "control, α = 0":  f"{NS_BASE}_hoyer_a0",
            "hoyer, α = 0.01": f"{NS_BASE}_hoyer_a0.01",
            "hoyer, α = 0.1":  f"{NS_BASE}_hoyer_a0.1",
            "l1, α = 0.01":    f"{NS_BASE}_l1_a0.01",
            "l1, α = 0.1":     f"{NS_BASE}_l1_a0.1"}
NS_KINDS = {"baseline": ("none", 0.0), "control, α = 0": ("hoyer", 0.0),
            "hoyer, α = 0.01": ("hoyer", 0.01), "hoyer, α = 0.1": ("hoyer", 0.1),
            "l1, α = 0.01": ("l1", 0.01), "l1, α = 0.1": ("l1", 0.1)}
NS_MESHES = ["model_003", "model_007", "model_010", "model_011"]
NS_SPAN = 300.0   # K, span of T on every held-out mesh (reg_errors' T_span)

ns = {key: dict(np.load(processed_data_dir / exp / "seeds.npz"))
      for key, exp in NS_CASES.items()}
ref = ns["baseline"]
NS_SEEDS = ref["seeds"]

# Same seeds, meshes and model everywhere; and since c0 is the mean message
# column norm at initialisation, equal c0 per seed means equal initial weights:
# within a seed the configurations differ only in the regularizer.
for key, z in ns.items():
    kind, alpha = NS_KINDS[key]
    assert np.array_equal(z["seeds"], NS_SEEDS), f"{key}: seeds {z['seeds']}"
    assert list(z["test_meshes"]) == NS_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert int(z["n_params"]) == int(ref["n_params"]), f"{key}: {z['n_params']} parameters"
    assert np.array_equal(z["msg_c0"], ref["msg_c0"]), f"{key}: initial weights differ"
    assert str(z["msg_kind"]) == kind and float(z["msg_alpha"]) == alpha, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    assert np.all(z["lrs"] == z["lrs"][0, 0]), f"{key}: the LR moved, not a nosched run"
    assert np.all(z["msg_lam_eff"][:, :int(z["msg_warmup"])] == 0), f"{key}: penalty on in warmup"
    assert np.all((z["msg_lam_eff"][:, -1] > 0) == (alpha > 0)), f"{key}: lam_eff vs alpha"
    if kind != "none":   # the scale fix holds every message column at c0
        dev = np.abs(z["msg_col_norm"] / z["msg_c0"][:, None, None] - 1).max()
        assert dev < 1e-4, f"{key}: column norms off c0 by {dev:.1e}"
for field in ("msg_warmup", "msg_ramp_len"):
    assert len({int(z[field]) for z in ns.values()}) == 1, f"{field} differs between configs"
NS_WARM = int(ref["msg_warmup"])
NS_RAMP_END = NS_WARM + int(ref["msg_ramp_len"]) - 1


def seed_stats(a, log=False):
    """Mean and std over the seed axis (0); in log10 if log, over finite seeds."""
    a = np.asarray(a, dtype=float)
    if not log:
        return a.mean(0), a.std(0, ddof=1)
    la = np.where(np.isfinite(a) & (a > 0), np.log10(np.where(a > 0, a, 1.0)), np.nan)
    n = np.isfinite(la).sum(0)
    mu = np.nanmean(la, 0)
    sd = np.sqrt(np.nansum((la - mu) ** 2, 0) / np.maximum(n - 1, 1))
    return mu, sd


# Rollout RMAE from the saved per-step MAE, no checkpoint needed: over N nodes,
# sum|e| = N * MAE_t at every step, so RMAE = sum_t MAE_t / sum_t mean_i |T_true|.
# T_true at rollout step t is T[HISTORY + t]. This reproduces the rollouts RMAE of
# compute_reg_errors to < 5e-6 relative on the checkpoints cached above. An
# overflowed rollout (MAE nan/inf) gets an infinite RMAE.
ns_absT = {}
for mesh in NS_MESHES:
    _, T_seq = load_mesh(mesh)
    ns_absT[mesh] = T_seq[HISTORY:].double().abs().mean(1).numpy()
ns_rmae_roll = {}   # key -> (S, M)
for key, z in ns.items():
    cols = []
    for mesh in NS_MESHES:
        mae = z[f"rollout_mae_{mesh}"].astype(float)
        r = mae.sum(1) / ns_absT[mesh][:mae.shape[1]].sum()
        cols.append(np.where(np.isfinite(r), r, np.inf))
    ns_rmae_roll[key] = np.stack(cols, 1)

print(f"{len(NS_SEEDS)} seeds {NS_SEEDS.tolist()}, {int(ref['n_params'])} parameters, "
      f"LR {ref['lrs'][0, 0]:.0e} constant, warmup {NS_WARM}, ramp complete at epoch {NS_RAMP_END}\n")
print(f"{'':16s}{'train, last':>22}{'val, last':>22}{'val, best':>22}{'best @':>10}"
      f"{'PR, last':>14}{'mean |m|, last':>18}")
for key, z in ns.items():
    cells = [seed_stats(z["train_losses"][:, -1]), seed_stats(z["val_losses"][:, -1]),
             seed_stats(z["val_losses"].min(1))]
    pr, ma = seed_stats(z["msg_pr"][:, -1]), seed_stats(z["msg_mean_abs"][:, -1])
    ep = z["val_losses"].argmin(1)
    print(f"{key:16s}" + "".join(f"{f'{m:.2e} ± {s:.1e}':>22}" for m, s in cells)
          + f"{f'{ep.mean():.0f} ± {ep.std(ddof=1):.0f}':>10}"
          + f"{f'{pr[0]:.1f} ± {pr[1]:.1f}':>14}{f'{ma[0]:.3g} ± {ma[1]:.2g}':>18}")

# Rollout: how many seeds stay within the span of the field, and the RMAE as a
# geometric mean with its spread as a factor (over the seeds that did not
# overflow; "(4)" when only four of them count).
print(f"\nrollout, 120 steps: seeds whose final MAE stays below {NS_SPAN:.0f} K (the span of T)")
print(f"{'':16s}" + "".join(f"{mesh:>11}" for mesh in NS_MESHES))
for key, z in ns.items():
    print(f"{key:16s}" + "".join(
        f"{int((np.isfinite(z[f'rollout_mae_{m}'][:, -1]) & (z[f'rollout_mae_{m}'][:, -1] < NS_SPAN)).sum())}"
        f"/{len(NS_SEEDS)}".rjust(11) for m in NS_MESHES))
print(f"\nrollout RMAE, geometric mean ×/÷ geometric std over seeds")
print(f"{'':16s}" + "".join(f"{mesh:>22}" for mesh in NS_MESHES))
for key in NS_CASES:
    row = f"{key:16s}"
    for j in range(len(NS_MESHES)):
        r = ns_rmae_roll[key][:, j]
        mu, sd = seed_stats(r, log=True)
        n = int(np.isfinite(r).sum())
        row += f"{f'{10**mu:.1e} ×/÷ {10**sd:.1g}' + (f' ({n})' if n < len(r) else ''):>22}"
    print(row)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Baseline and control keep the colour and line style they have in the single-run
# sections above; each penalty gets one hue, lighter at the smaller alpha.
NS_STYLES = {"baseline":        (PLOT2_COLOR, "--"),
             "control, α = 0":  (PLOT3_COLOR, "-."),
             "hoyer, α = 0.01": ("#a58cc6", "-"),
             "hoyer, α = 0.1":  (PLOT1_COLOR, "-"),
             "l1, α = 0.01":    ("#7fa9cf", "-"),
             "l1, α = 0.1":     ("#1f4e79", "-")}
# One row per penalty, each against the same two references, so no panel carries
# more than four bands.
NS_FAMILIES = {"hoyer": ["baseline", "control, α = 0", "hoyer, α = 0.01", "hoyer, α = 0.1"],
               "l1":    ["baseline", "control, α = 0", "l1, α = 0.01", "l1, α = 0.1"]}

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 1000*px), sharex=True, sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Loss Decay, No LR Scheduler, msg_dim = 64, h = 1 "
             f"(mean ± std over {len(NS_SEEDS)} seeds)", fontsize=24, color=AX_COLOR)

epochs = np.arange(ref["train_losses"].shape[1])
for row, (family, keys) in enumerate(NS_FAMILIES.items()):
    for col, (title, field) in enumerate((("Training Loss", "train_losses"),
                                          ("Validation Loss", "val_losses"))):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key in keys:
            color, ls = NS_STYLES[key]
            mu, sd = seed_stats(ns[key][field])
            ax.plot(epochs, mu, label=key, color=color, linewidth=2.2, linestyle=ls)
            # the band's lower edge is clipped: on a log axis mean - std <= 0 has
            # no place to go
            ax.fill_between(epochs, np.clip(mu - sd, mu / 10, None), mu + sd,
                            color=color, alpha=0.15, linewidth=0)

        # Warmup ends where the penalty starts to ramp in, the ramp where it is at
        # full strength. With no scheduler nothing else happens at these epochs.
        for e in (NS_WARM - 1, NS_RAMP_END):
            ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
        ax.annotate("warmup / ramp", xy=(NS_WARM - 1, 0.02), xycoords=("data", "axes fraction"),
                    xytext=(-4, 0), textcoords="offset points", ha="right", va="bottom",
                    fontsize=12, color=AX_COLOR, alpha=0.8)

        ax.set_yscale("log")

        legend = ax.legend(fontsize=13, loc="upper right")
        legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
        legend.get_frame().set_edgecolor(AX_COLOR)        # border color
        legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
        for text in legend.get_texts():
            text.set_color(AX_COLOR)                      # label text color

        ax.set_title(f"{title}, {family}", fontsize=20)
        if row == 1:
            ax.set_xlabel("Epoch", fontsize=18)
        if col == 0:
            ax.set_ylabel("MSE Loss", fontsize=18)
        ax.tick_params(axis='both', labelsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_nosched_loss.png", dpi=300, transparent = True)

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

PR_TICKS = [1, 2, 4, 8, 16, 32, 64]

# The channel participation ratio of sigma_c, logged every epoch by
# MessageRegularizer: what hoyer minimises directly and l1 shrinks indirectly.
# The baseline logs it too, but without the scale fix its sigma_c also measures
# how large msg_fnc writes each channel, not only how much node_fnc uses it.
fig, axs = plt.subplots(1, 2, figsize=(1400*px, 540*px), sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Participation Ratio During Training, No LR Scheduler "
             f"(mean ± std over {len(NS_SEEDS)} seeds)", fontsize=24, color=AX_COLOR)

for ax, (family, keys) in zip(axs, NS_FAMILIES.items()):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key in keys:
        color, ls = NS_STYLES[key]
        mu, sd = seed_stats(ns[key]["msg_pr"])
        ax.plot(epochs, mu, label=key, color=color, linewidth=2.2, linestyle=ls)
        ax.fill_between(epochs, np.clip(mu - sd, 1, None), mu + sd,
                        color=color, alpha=0.15, linewidth=0)

    for e in (NS_WARM - 1, NS_RAMP_END):
        ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.annotate("warmup / ramp", xy=(NS_WARM - 1, 0.02), xycoords=("data", "axes fraction"),
                xytext=(-4, 0), textcoords="offset points", ha="right", va="bottom",
                fontsize=12, color=AX_COLOR, alpha=0.8)

    legend = ax.legend(fontsize=13, loc="center right")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_title(family, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if ax is axs[0]:
        ax.set_ylabel("PR  [channels of 64]", fontsize=18)
    ax.tick_params(axis='both', labelsize=15)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# PR spans 1 to 64: a base-2 axis gives every halving of the channel count the
# same height (shared y, so once sets both panels).
axs[0].set_yscale("log", base=2)
axs[0].set_yticks(PR_TICKS, labels=[str(k) for k in PR_TICKS])
axs[0].yaxis.set_minor_locator(NullLocator())
axs[0].set_ylim(1, 64)

# plt.savefig("../outputs/plots/parametric_history1_nosched_pr.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Rows: the two penalty families, each against baseline and control; columns: the
# held-out meshes. The RMAE is computed from the MAE saved at
# training time, per seed. Across
# seeds it spans up to thirty decades, so the line is the geometric mean and the
# band ×/÷ one geometric std; the thin lines are the individual seeds. Once a
# seed's rollout overflows (nan) the geometric mean is infinite, so the line and
# band stop there rather than drop to the mean of the seeds that are left.
fig, axs = plt.subplots(2, len(NS_MESHES), figsize=(1600*px, 850*px),
                        sharex=True, sharey=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Rollout RMAE, No LR Scheduler, msg_dim = 64, h = 1 "
             f"(geometric mean ×/÷ std over {len(NS_SEEDS)} seeds)",
             fontsize=24, color=AX_COLOR)

for row, (family, keys) in enumerate(NS_FAMILIES.items()):
    for col, mesh in enumerate(NS_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key in keys:
            color, ls = NS_STYLES[key]
            rel = rmae_curve(ns[key][f"rollout_mae_{mesh}"], param_T_stats(mesh)[0])
            steps = np.arange(1, rel.shape[1] + 1)
            for m in rel:
                ax.plot(steps, m, color=color, linewidth=0.6, alpha=0.35)
            mu, sd = seed_stats(rel, log=True)
            mu[~np.isfinite(rel).all(0)] = np.nan
            ax.plot(steps, 10 ** mu, label=key, color=color, linewidth=2.4, linestyle=ls)
            ax.fill_between(steps, 10 ** (mu - sd), 10 ** (mu + sd),
                            color=color, alpha=0.15, linewidth=0)
        ax.set_yscale("log")

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        ax.plot(steps, rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0]),
                color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, NS_SPAN / param_T_stats(mesh)[0][0]),
                        xytext=(4, 4), textcoords="offset points",
                        ha="left", va="bottom", fontsize=12, color=AX_COLOR)

        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        if col == 0:
            ax.set_ylabel(f"{family}: RMAE", fontsize=15)

        ax.tick_params(axis='both', labelsize=13)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

    # one legend per row: the four bands are the same in every panel of it
    legend = axs[row, 0].legend(fontsize=12, loc="upper left")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nosched_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- nosched sweep: one-step RMAE / RMSE and rollout RMSE, per seed -----------
# These need the trained weights: compute_reg_errors (Regularization cell above)
# on each seed's checkpoint, processed_data/<exp>/seed_<n>/checkpoints/, which
# aggregate_seeds.py reads but does not copy into seeds.npz. Seeds whose
# checkpoint is not on this machine are listed and left out; the rollout RMAE of
# every seed comes from the saved MAE (cell above) either way.
ns_errors_cache = processed_data_dir / "parametric_errors_nosched.npz"


def ns_seed_run(exp, seed):
    """The run path compute_reg_errors / compute_model_flux_stats take, and whether
    its weights are here."""
    run = f"{exp}/seed_{seed}"
    ck = processed_data_dir / run / "checkpoints"
    return run, (ck / "model.pt").exists() and (ck / "normalizer.pt").exists()


ns_runs = {key: {int(s): ns_seed_run(exp, int(s)) for s in NS_SEEDS}
           for key, exp in NS_CASES.items()}
ns_absent = [f"{NS_CASES[k]}/seed_{s}" for k, runs in ns_runs.items()
             for s, (_, ok) in runs.items() if not ok]
if ns_absent:
    print(f"{len(ns_absent)} of {len(NS_CASES) * len(NS_SEEDS)} seed checkpoints are not in "
          f"{processed_data_dir}, left out of one-step RMAE/RMSE and rollout RMSE:")
    for key, runs in ns_runs.items():
        gone = [s for s, (_, ok) in runs.items() if not ok]
        if gone:
            print(f"    {NS_CASES[key]}/seed_{{{','.join(map(str, gone))}}}/checkpoints/model.pt")
    print()

ns_errors = (dict(np.load(ns_errors_cache, allow_pickle=False))
             if ns_errors_cache.exists() else {})
missing = [run for runs in ns_runs.values() for run, ok in runs.values()
           if ok and f"{run}__{NS_MESHES[0]}__rmae_step" not in ns_errors]
if missing:
    ns_errors.update(compute_reg_errors(missing))
    np.savez_compressed(ns_errors_cache, **ns_errors)


def ns_metric(key, metric, mode):
    """(S, M) array of one metric, nan where the seed's checkpoint is missing.
    Rollout RMAE is always the one from the saved MAE."""
    if (metric, mode) == ("rmae", "roll"):
        return ns_rmae_roll[key]
    out = np.full((len(NS_SEEDS), len(NS_MESHES)), np.nan)
    for i, s in enumerate(NS_SEEDS):
        run, ok = ns_runs[key][int(s)]
        if ok:   # an overflowed rollout is nan here too: diverged, not missing
            v = np.array([float(ns_errors[f"{run}__{mesh}__{metric}_{mode}"]) for mesh in NS_MESHES])
            out[i] = np.where(np.isfinite(v), v, np.inf)
    return out


# The rollouts recomputed from the checkpoints have to be the ones whose MAE was
# saved at training time, or their RMSE would describe a different rollout.
# nan-safe: an overflowed rollout is nan in both.
devs = [np.nanmax(np.abs(ns_errors[f"{run}__{mesh}__mae_roll"]
                         / ns[key][f"rollout_mae_{mesh}"][i] - 1))
        for key in NS_CASES for i, s in enumerate(NS_SEEDS)
        for run, ok in [ns_runs[key][int(s)]] if ok for mesh in NS_MESHES]
if devs:
    print(f"recomputed vs saved rollout MAE: max |relative difference| = {max(devs):.1e}\n")

# Geometric mean ×/÷ geometric std over the seeds with a finite value; (n/m): n
# finite of the m seeds that have the metric, the rest overflowed.
cols = [("step", "rmae"), ("step", "rmse"), ("roll", "rmae"), ("roll", "rmse")]
print(f"{'mesh':<11}{'run':<17}" + "".join(
    f"{f'{m} {x.upper()}':>28}" for m, x in (("one-step", "rmae"), ("one-step", "rmse"),
                                           ("rollout", "rmae"), ("rollout", "rmse"))))
for j, mesh in enumerate(NS_MESHES):
    for key in NS_CASES:
        row = f"{mesh:<11}{key:<17}"
        for mode, metric in cols:
            v = ns_metric(key, metric, mode)[:, j]
            n, n_have = int(np.isfinite(v).sum()), int((~np.isnan(v)).sum())
            if n == 0:
                row += f"{'—' if n_have == 0 else f'inf ({n}/{n_have})':>28}"
                continue
            mu, sd = seed_stats(v, log=True)
            row += f"{f'{10**mu:.2e} ×/÷ {10**sd:.2g} ({n}/{n_have})':>28}"
        print(row)

In [ ]:
from matplotlib.patches import Patch

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The table above as a figure. Rows: one-step and rollout; columns: RMAE and RMSE
# (the relative SQUARED error, so it sits far below the RMAE). Each bar is the
# geometric mean over the seeds, the whisker ×/÷ one geometric std, the dots the
# seeds themselves; an overflowed seed is drawn as a triangle at the top of its
# panel. Log y everywhere: the rollouts span tens of decades across seeds.
x = np.arange(len(NS_MESHES))
width = 0.13

fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE, No LR Scheduler, msg_dim = 64, h = 1 (geometric mean ×/÷ std)",
             fontsize=22, color=AX_COLOR)

panels = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
for ax, (mode, metric) in zip(axs.ravel(), panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_title(f"{'one-step' if mode == 'step' else 'rollout'}  ·  {metric.upper()}",
                 fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(NS_MESHES, fontsize=13)

    vals = {key: ns_metric(key, metric, mode) for key in NS_CASES}
    finite = np.concatenate([v[np.isfinite(v)] for v in vals.values()])
    if finite.size == 0:
        ax.set_yticks([])
        ax.annotate("needs the seed checkpoints\n(<exp>/seed_<n>/checkpoints/model.pt)",
                    xy=(0.5, 0.5), xycoords="axes fraction", ha="center", va="center",
                    fontsize=15, color=AX_COLOR, alpha=0.8)
        continue

    # bars on a log axis grow from the bottom of the axis, not from 0, so the
    # floor has to be set before they are drawn
    floor, hi = finite.min() / 10, finite.max() * 10
    ax.set_yscale("log")
    ax.set_ylim(floor, hi)
    for i, key in enumerate(NS_CASES):
        color = NS_STYLES[key][0]
        off = (i - (len(NS_CASES) - 1) / 2) * (width + 0.01)
        mu, sd = seed_stats(vals[key], log=True)            # (M,) each
        has = np.isfinite(mu)
        ax.bar((x + off)[has], (10 ** mu - floor)[has], width, bottom=floor,
               color=color, label=key)
        ax.errorbar((x + off)[has], 10 ** mu[has],
                    yerr=[10 ** mu[has] - 10 ** (mu - sd)[has], 10 ** (mu + sd)[has] - 10 ** mu[has]],
                    fmt="none", ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5)
        for j in range(len(NS_MESHES)):
            v = vals[key][:, j]
            ok = np.isfinite(v)
            ax.plot(np.full(ok.sum(), x[j] + off), v[ok], "o", color=AX_COLOR,
                    markersize=2.8, alpha=0.7)
            if np.isinf(v).any():
                ax.plot(x[j] + off, hi, "v", color=color, markersize=7,
                        markeredgecolor=AX_COLOR, clip_on=False)

    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure, below the grid: six entries would cover a
# group of bars in any corner. Built from the styles, not from the bars, since a
# run with no checkpoint here has no bars to take a handle from.
legend = fig.legend([Patch(color=NS_STYLES[k][0]) for k in NS_CASES], list(NS_CASES),
                    loc="outside lower center", ncols=6, fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nosched_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- nosched sweep: message vs FV flux, per seed --------------------------------
# compute_model_flux_stats from the residual cell, on the same meshes, timesteps
# and seeded faces as every read-out above, once per seed checkpoint (ns_runs, from
# the errors cell). Every column is an R^2, a correlation or an energy fraction,
# so none depends on how large a run writes its message. Seeds without a
# checkpoint here are left out, as in the errors cell.
ns_flux_cache = processed_data_dir / "parametric_message_flux_nosched.npz"
NS_FLUX_COLS = ("R2 F", "R2 dT", "R2 w", "R2 node", "|r| best", "anti")

ns_flux = (dict(np.load(ns_flux_cache, allow_pickle=False))
           if ns_flux_cache.exists() else {})
missing = [run for runs in ns_runs.values() for run, ok in runs.values()
           if ok and f"{run}__{NS_MESHES[0]}__r2" not in ns_flux]
if missing:
    for run in missing:
        ns_flux.update(compute_model_flux_stats(run))
    np.savez_compressed(ns_flux_cache, **ns_flux)


def ns_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(NS_SEEDS), len(NS_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, s in enumerate(NS_SEEDS):
        run, ok = ns_runs[key][int(s)]
        if not ok:
            continue
        for j, mesh in enumerate(NS_MESHES):
            out[i, j] = [*ns_flux[f"{run}__{mesh}__r2"], float(ns_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(ns_flux[f"{run}__{mesh}__best_r"])),
                         ns_flux[f"{run}__{mesh}__anti"].mean()]
    return out


ns_flux_tab = {key: ns_flux_table(key) for key in NS_CASES}
ns_flux_n = {key: int((~np.isnan(t[:, 0, 0])).sum()) for key, t in ns_flux_tab.items()}

if not any(ns_flux_n.values()):
    print("No seed checkpoint here: the flux read-outs need "
          "<exp>/seed_<n>/checkpoints/model.pt (see the list in the errors cell).")
else:
    # Arithmetic mean ± std over the seeds that have a checkpoint: these are all
    # bounded in [0, 1], unlike the errors.
    print(f"Linear read-out from the message, {N_STEPS} probed timesteps, "
          f"mean ± std over seeds (n = seeds with a checkpoint)")
    print(f"{'mesh':<11}{'run':<17}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
    for j, mesh in enumerate(NS_MESHES):
        for key in NS_CASES:
            n = ns_flux_n[key]
            if n == 0:
                continue
            t = ns_flux_tab[key][:, j]
            mu, sd = np.nanmean(t, 0), (np.nanstd(t, 0, ddof=1) if n > 1 else np.zeros(t.shape[1]))
            print(f"{mesh:<11}{key:<17}{n:>3}"
                  + "".join(f"{f'{m:.3f} ± {s:.3f}':>15}" for m, s in zip(mu, sd)))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The three edge-level read-outs plus the node-level one, per held-out mesh; bar
# = mean over the seeds with a checkpoint, whisker = ± one std, dots = the seeds.
# All six runs are 64 channels wide, so each bar is the best linear combination
# of the same number of channels.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.13
flux_keys = [k for k in NS_CASES if ns_flux_n[k] > 0]

if not flux_keys:
    print("Nothing to plot: no seed checkpoint here (see the errors cell).")
else:
    fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
    fig.patch.set_alpha(0)
    fig.suptitle("Message vs FV Flux, No LR Scheduler, msg_dim = 64, h = 1 "
                 "(mean ± std over seeds)", fontsize=24, color=AX_COLOR)

    for j, (ax, mesh) in enumerate(zip(axs.ravel(), NS_MESHES)):
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=14)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for i, key in enumerate(flux_keys):
            color = NS_STYLES[key][0]
            t = ns_flux_tab[key][:, j, :4]                       # (S, 4) R^2
            t = t[~np.isnan(t[:, 0])]
            off = (i - (len(flux_keys) - 1) / 2) * (width + 0.01)
            sd = t.std(0, ddof=1) if len(t) > 1 else None
            ax.bar(x + off, t.mean(0), width, color=color, label=key,
                   yerr=sd, error_kw=dict(ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5))
            for row in t:
                ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.8, alpha=0.7)

        ax.set_title(mesh, fontsize=19)
        ax.set_xticks(x)
        ax.set_xticklabels(targets, fontsize=15)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
        ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
        ax.set_axisbelow(True)

    # one legend below the grid: with six bars a group no corner is free
    legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                        loc="outside lower center", ncols=len(flux_keys), fontsize=13)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nosched_flux_r2.png", dpi=300, transparent = True)

### What comes out

All 30 seed checkpoints are now here, so every number below is over the full 5 seeds. The
one-step errors, the rollout RMSE and the flux read-outs come from `model.pt`, which holds the
**last-epoch** weights. The losses, the PR and the rollout MAE come from `seeds.npz`.

**Best validation loss: the baseline is best, and the penalties add a little.** Best validation MSE is
$2.53 \pm 0.26$ for the baseline, $2.70 \pm 0.23$ for the control, $2.85 - 2.95$ at
$\alpha = 0.01$ and $3.09 - 3.20$ at $\alpha = 0.1$ (units of $10^{-5}$). The stronger penalty
is worse for both kinds. The seed spread is about $2\times10^{-6}$, so the baseline–control gap is
under one std, and the step from the control to $\alpha = 0.1$ is 2 – 2.5 std. With no scheduler
the last epoch is a noisy snapshot, and the mean over the last 50 epochs reads more reliably:
$5.4\times10^{-5}$ for the baseline, $7.5$ for the control, $7.5 - 7.9$ for `l1` and $8.1 - 9.2$
for `hoyer`.

**At the last-epoch weights, the scale fix costs one-step accuracy and the penalties do not.**
The geometric mean over seeds and meshes of the one-step RMAE is $6.1\times10^{-4}$ for the
baseline and $9.7\times10^{-4}$ for the control, and the RMSE is $8.1\times10^{-6}$ against
$2.2\times10^{-5}$. Seed by seed, the control is worse than the baseline on 19 of 20 seed–mesh
pairs, by a factor of 1.6 in RMAE and 2.7 in RMSE. Every scale-fixed run is 1.3 – 2.3× the
baseline in RMAE. Against the control, which starts from the same weights, the penalties are
neutral or slightly better. Paired RMAE ratios are 0.81 for `hoyer` at $\alpha = 0.1$ (lower on
14 of 20 pairs) and 0.89 – 0.92 for `l1` (12 of 20). The exception is `hoyer` at
$\alpha = 0.01$, at 1.45, but that comes from one seed. Seed 1's validation loss jumps from
$1.4\times10^{-4}$ to $3.2\times10^{-3}$ in the last epoch, which is the epoch saved, and its
one-step RMAE is $9.7\times10^{-3}$, ten times that of the other four. So the best-validation
ordering among the penalties does not carry over to the saved weights. At a constant
$3\times10^{-3}$, which epoch is saved matters as much as the penalty. The one gap that holds on
both readings is baseline vs scale fix.

The training loss separates the runs more than the validation loss does. Over the last 50
epochs it is $9.0\times10^{-5}$ for the baseline and $3.0 - 3.5\times10^{-4}$ for all five
scale-fixed runs, the control included. That is 2 – 7 times their own validation loss, against
1.4 – 1.9 for the baseline. The training loss leaves out the penalty, and the control has none, so
the gap comes from the scale fix itself. One candidate: the fix rescales `msg_fnc[-1]` and
`node_fnc[0]` after every optimizer step without telling SOAP, so its preconditioner then works on
parameters that are no longer the ones it saw. This is not isolated here.

**The penalties do what they are written to do, reproducibly.** The final channel PR is
$58.4 \pm 1.0$ for the baseline and $51.1 \pm 3.3$ for the control. `hoyer` brings it to
$2.2 \pm 0.4$ and $1.6 \pm 0.4$ by the end of the ramp, and $1.6 \pm 0.4$ and $1.1 \pm 0.1$ at the
end of training, at $\alpha = 0.01$ and $0.1$. At the stronger penalty about one channel is left.
`l1` moves the PR only to $45.4 \pm 3.4$ and $44.2 \pm 1.8$. The ×10 in $\alpha$ shows in
$\langle|m|\rangle$ ($1.9$ → $0.37$) but hardly in the PR: `l1` skews the channels rather than
sparsifying them, as in the single runs.

**Rollout: almost everything diverges, and one configuration mostly does not.** The count is of
seed–mesh pairs whose 120-step MAE ends below the 300 K span of the field: baseline 2/20,
control 4/20, `hoyer` 3/20 and 8/20, `l1` at $\alpha = 0.01$ 0/20, and **`l1` at
$\alpha = 0.1$ 14/20**. The rollout RMAE and RMSE show the same thing. For `l1` at
$\alpha = 0.1$ the geometric-mean RMAE is 0.25 – 0.90 per mesh and the rollout RMSE 0.6 – 7.4.
Every other configuration has an RMAE of $5\times10^{2}$ – $5\times10^{13}$ and an RMSE of
$10^{7}$ – $10^{31}$, and one control seed (seed 0) overflows to nan on three meshes. The rollouts
recomputed from the checkpoints are the ones saved at training time. The one large deviation the
check prints, 1.0, is control seed 0 on `model_003`: from step 116 the saved float32 MAE is
already `inf` while the float64 recomputation is still finite ($1.2\times10^{34}$). Apart from that,
the largest difference is $5\times10^{-3}$ relative, in `l1` at $\alpha = 0.01$, seed 1, a
rollout that has already diverged to $10^{5}$ – $10^{7}$ K and amplifies round-off. Two caveats
keep this from being a success:

- "Below the span" does not mean accurate. The `l1` $\alpha = 0.1$ rollouts that stay bounded end
  at 20 – 284 K MAE. Only seed 1 (20 – 50 K) comes close to the 14 – 26 K that the scheduled
  single-run control reached. Holding the learning rate at $3\times10^{-3}$ costs a great deal
  of rollout stability everywhere, so part of what the scheduled runs above showed was the LR
  decay, not the regularizer.
- The seed matters as much as the configuration. Seed 1 is the good seed for the baseline (2 of
  its 4 meshes bounded), the control (4/4) and `l1` at $\alpha = 0.1$ (4/4). Seed 2 is the bad
  one for `l1` at $\alpha = 0.1$ (0/4) but the only good one for `hoyer` at $\alpha = 0.01$
  (3/4). The error leaves the span anywhere between step 9 and step 104, most often within the
  first 15 – 40 steps, and then grows geometrically, a straight line on the log axis, as in the
  single runs.

**One-step error says nothing about the rollout.** Over all 120 seed–mesh pairs, the Spearman
correlation between one-step and rollout RMAE is $\rho = 0.09$ ($p = 0.35$). Over the 30 runs,
best validation MSE against mean log rollout RMAE gives $\rho = -0.35$ ($p = 0.06$). If
anything, a lower validation loss goes with a worse rollout. `l1` at $\alpha = 0.1$ has the
worst best-validation MSE of the six and the most stable rollout. Neither the validation loss nor
the one-step RMAE can be used to select for stability here.

**The penalties make the edge message less flux-like, but not the node sum.** The $R^2$ of the
linear read-out of the FV flux $F$ from the message falls steadily as the regularization gets
stronger. Over the four meshes it is 0.64 – 0.69 for the baseline, 0.54 – 0.59 for the control,
0.51 – 0.57 and 0.44 – 0.46 for `hoyer`, and 0.32 – 0.33 and 0.21 – 0.22 for `l1`. $\Delta T$
follows (0.68 – 0.73 down to 0.30 – 0.39), and so does $w$, with a large seed spread under the
penalties (std up to 0.24). The node-level read-out of $\sum_j F_{ij}$ barely moves: 0.69 – 0.76 for
the baseline, 0.62 – 0.69 for the control and 0.64 – 0.78 for all four penalties. So what the node
update receives is as flux-readable as before. The penalties change how the message on a single
edge encodes it, and `l1` at $\alpha = 0.1$, the most stable configuration, has the least
flux-like edge message. The antisymmetric energy fraction rises with `l1` (0.33 – 0.35 at
$\alpha = 0.01$, 0.38 – 0.40 at $0.1$, against 0.23 – 0.26 for the baseline and control), as it
did in the single run. It is still far from the 1.0 of a conservative flux. For `hoyer` the
channel mean (0.18 – 0.25) is dominated by the dead channels and says little.

**What the paired design allows.** From the same initial weights, `l1` at $\alpha = 0.1$ keeps
the rollout bounded on 14 of 20 seed–mesh pairs, against 4 for the unpenalised control. It pays
nothing in one-step error at the saved weights (paired RMAE ratio 0.92) and about
$5\times10^{-6}$ in best validation MSE. `hoyer` at $\alpha = 0.1$ is second at 8/20, and the
weaker penalties do not beat the control. What the multi-seed run takes back from the single runs
is the rest of the story: the scale fix on its own costs one-step accuracy against the
baseline, and without LR decay no configuration is accurate over 120 steps.

## Node positions vs none (`nopos`)

Every model above takes the absolute position $(x, y, z)$ of each node as an input, next to its
type. A finite-volume update does not use them: the change of $T$ in a cell depends on its
neighbours and on the local geometry (face area vectors, distances, the cell itself), which the
edge features already carry. This section takes the positions out and asks what that changes, with
**5 seeds per configuration** (`FVGNN_SEED = 0 … 4`), each configuration collected by
`aggregate_seeds.py`:

- **positions** (`parametric_history1_msg_dim64_residual_cf`, commit `cf45013`), `seed_<n>`
  runs in `seeds.npz`: node features $[\,\text{internal}, \text{boundary}, x, y, z\,]$.
- **no positions** (`parametric_history1_msg_dim64_residual_cf_nopos`, commit `cf45013` +
  uncommitted changes), the `seed_<n>_01` reruns in `seeds_01.npz`: node features
  $[\,\text{internal}, \text{boundary}\,]$ (`use_pos_features = False`).

Everything else is shared: the 64-channel residual model at `history = 1` with no LayerNorm and no
message penalty, Cf boundary nodes, and all 10 edge features. So $\mathbf{d}_{ij}$,
$|\mathbf{d}_{ij}|$, $\mathbf{S}_f$, $|\mathbf{S}_f|$, skewness and non-orthogonality are in
both, and only the absolute position goes. Training is SOAP from $3\times10^{-3}$ with
`ReduceLROnPlateau` (`lr_scheduler = True`), at `batch_size = 2`. The batch size is not saved with
a run, so nothing below can check it.

**Not the same training length.** The positions runs train for 200 epochs, the no-position runs
for **500**: the uncommitted change in their script is `epochs = 200` → `500` (and the numbered
run directories). A first 200-epoch no-position sweep (`seed_<n>`, `seeds.npz` of `..._cf_nopos`,
same commit) is not used here: at epoch 200 three of its five seeds still had a learning rate of
$3\times10^{-5}$ or more, so it had not converged, whereas every positions seed had been cut to
$3\times10^{-9}$ by epoch 111 – 128. The 500-epoch seeds reach $3\times10^{-9}$ at epoch 226 – 331
and do not change after that: without positions the scheduler takes about twice as long to run out. The comparison is therefore of two converged trainings, not of two equal budgets: the
learning rate panel below shows where each seed stopped learning. The earlier cf seeds, trained
at `batch_size = 8` (commit `fd42720` + uncommitted changes), are not comparable with these and
must be out of the `..._cf` directory before it is read here.

**Not a paired design.** Both configurations use the same seeds, but the no-position node encoder
has 3 inputs instead of 6. It draws fewer random numbers at initialisation, so every layer after
it and the shuffle order of every epoch come out different: seed $n$ of one configuration is not
seed $n$ of the other. The two are compared as two samples of 5. Next to the means, the tables
give the two-sided Mann–Whitney $U$ test over the $5 \times 5$ seed pairs. With 5 against 5 its
smallest possible $p$ is $0.008$, reached only when every seed of one configuration beats every
seed of the other.

- **Loss decay and learning rate**: training and validation MSE, mean ± std over the seeds, and
  each seed's LR path, which the scheduler sets per seed.
- **Rollout MAE** and the **rollout RMAE** computed from it, as in the `nosched` sweep (plotted
  per step as RMAE): geometric
  mean ×/÷ geometric std over the seeds.
- **One-step RMAE / RMSE and rollout RMSE**, from the seed checkpoints.
- **Message vs flux, $R^2$**: `compute_model_flux_stats` per seed checkpoint.

The graphs are rebuilt as the runs' own were: Cf boundary nodes, read from the mesh cache that
`train_parametric.py` fills (`load_mesh` above builds point-mean ones), and the node type alone
for `_nopos`. The cells reuse `seed_stats`, `NS_SPAN`, `ns_seed_run` and `NS_FLUX_COLS` from the
`nosched` sweep, and `compute_reg_errors` and `compute_model_flux_stats` from the sections above.

In [ ]:
# --- Positions vs none: load the seeds, check them, summarise -------------------
# Laid out as the nosched sweep, whose seed_stats, NS_SPAN and ns_seed_run it
# reuses. The seeds of the two configurations are NOT paired (see above), so
# every comparison is between the two samples: Mann-Whitney U, two-sided.
from scipy.stats import mannwhitneyu

from data_preparation.mesh_cache import load_mesh_cached
from data_preparation.static_graph import load_graph_config, node_feature_columns
from git_commit import load_git_commit

POS_CASES  = {"positions":    "parametric_history1_msg_dim64_residual_cf",
              "no positions": "parametric_history1_msg_dim64_residual_cf_nopos"}
POS_MESHES = ["model_003", "model_007", "model_010", "model_011"]
# node type (2), then x, y, z unless _nopos
POS_NODE_FEATS = {"positions": 5, "no positions": 2}
# Rerun index of each configuration's seeds (aggregate_seeds.py): 0 is seed_<n>
# in seeds.npz, i is seed_<n>_<ii> in seeds_<ii>.npz. The no-position runs are the
# 500-epoch reruns; their 200-epoch seed_<n> had not converged (see above).
POS_RERUN = {"positions": 0, "no positions": 1}


def pos_npz(key):
    i = POS_RERUN[key]
    return processed_data_dir / POS_CASES[key] / ("seeds.npz" if i == 0 else f"seeds_{i:02d}.npz")


pos_no_npz = [str(pos_npz(key)) for key in POS_CASES if not pos_npz(key).exists()]
if pos_no_npz:
    raise FileNotFoundError(
        f"no {', '.join(pos_no_npz)} yet: train with FVGNN_SEED = 0 ... 4, then "
        f"python aggregate_seeds.py <exp_name>:<rerun index>")

pos = {key: dict(np.load(pos_npz(key))) for key in POS_CASES}
POS_SEEDS = pos["positions"]["seeds"]
POS_EPOCHS = {key: z["train_losses"].shape[1] for key, z in pos.items()}


def pos_load(n_node_feats):
    """load_mesh for these runs: Cf graphs from the mesh cache train_parametric.py
    fills, cut to the node features the run was trained with. These runs predate
    the cell-volume column, so it is dropped."""
    def load(mesh):
        g, T = load_mesh_cached(str(raw_param_dir / mesh), EXCLUDED, "cf",
                                processed_data_dir / "parametric_preproc_cache")
        g.node_attr = g.node_attr[:, node_feature_columns(False, n_node_feats == 5)]
        return g, T
    return load


POS_LOAD = {key: pos_load(n) for key, n in POS_NODE_FEATS.items()}
# ns_seed_run's run path, seed_<n>, with the rerun's _<ii> appended
pos_runs = {key: {int(s): ns_seed_run(exp, f"{int(s)}_{POS_RERUN[key]:02d}" if POS_RERUN[key]
                                      else int(s)) for s in POS_SEEDS}
            for key, exp in POS_CASES.items()}

# The same seeds, meshes and model but for the node encoder's weights on x, y, z
# (3 x hidden_dim = 3 x 64), no penalty, and the scheduler on in both. Not the
# same number of epochs (200 vs 500, see above), but the same warmup and ramp,
# so the scheduler starts at the same epoch.
for key, z in pos.items():
    assert np.array_equal(z["seeds"], POS_SEEDS), f"{key}: seeds {z['seeds']}"
    assert list(z["test_meshes"]) == POS_MESHES, f"{key}: test meshes {z['test_meshes']}"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(pos["positions"][field]), f"{key}: {field} differs"
    assert str(z["msg_kind"]) == "none" and float(z["msg_alpha"]) == 0.0, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    assert (z["lrs"] != z["lrs"][:, :1]).any(), f"{key}: the LR never moved, not a scheduled run"
n_drop = int(pos["positions"]["n_params"]) - int(pos["no positions"]["n_params"])
assert n_drop == 3 * 64, f"the models differ by {n_drop} parameters, not the 3 x 64 of x, y, z"

# Per seed checkpoint on this machine: Cf graphs, the node features its normalizer
# was fitted on, and the commit it was trained with (for the markdown above).
for key, runs in pos_runs.items():
    commits = {}
    for s, (run, ok) in runs.items():
        ck = processed_data_dir / run / "checkpoints"
        if not ck.exists():
            continue
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg.get("excluded_patches") == EXCLUDED, \
            f"{run}: graphs built with {cfg}"
        if ok:
            n = torch.load(ck / "normalizer.pt")["node_mean"].numel()
            assert n == POS_NODE_FEATS[key], f"{run}: normalizer fitted on {n} node features"
        c = load_git_commit(ck)
        tag = ("not recorded" if c is None or c["commit"] is None
               else c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else ""))
        commits.setdefault(tag, []).append(s)
    print(f"{POS_CASES[key]}: " + ("; ".join(f"commit {tag} (seeds {', '.join(map(str, ss))})"
                                            for tag, ss in commits.items())
                                   or "no seed checkpoint here, commit unknown"))

# ReduceLROnPlateau steps only once the message ramp is complete (train_parametric.py)
POS_SCHED_START = int(pos["positions"]["msg_warmup"]) + int(pos["positions"]["msg_ramp_len"]) - 1


def pos_p(a, b):
    """Two-sided Mann-Whitney U p of two seed samples. A missing seed (nan) is left
    out; an overflowed one (inf) ranks above every finite value."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if min(len(a), len(b)) < 2:
        return np.nan
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue)


def first_cut(lrs):
    """Epoch of each seed's first LR cut, nan if it never came."""
    cut = np.diff(lrs, axis=1) != 0
    return np.where(cut.any(1), cut.argmax(1) + 1, np.nan)


# Rollout RMAE from the saved per-step MAE, no checkpoint needed, as in the
# nosched sweep.
pos_absT = {}
for mesh in POS_MESHES:
    _, T_seq = POS_LOAD["positions"](mesh)
    pos_absT[mesh] = T_seq[HISTORY:].double().abs().mean(1).numpy()
pos_rmae_roll = {}   # key -> (S, M)
for key, z in pos.items():
    cols = []
    for mesh in POS_MESHES:
        mae = z[f"rollout_mae_{mesh}"].astype(float)
        r = mae.sum(1) / pos_absT[mesh][:mae.shape[1]].sum()
        cols.append(np.where(np.isfinite(r), r, np.inf))
    pos_rmae_roll[key] = np.stack(cols, 1)

za, zb = pos["positions"], pos["no positions"]
print(f"\n{len(POS_SEEDS)} seeds {POS_SEEDS.tolist()}, "
      f"{POS_EPOCHS['positions']} vs {POS_EPOCHS['no positions']} epochs, "
      f"{int(za['n_params'])} vs {int(zb['n_params'])} parameters, "
      f"scheduler on from epoch {POS_SCHED_START}\n")
pos_losses = {"train, last": lambda z: z["train_losses"][:, -1],
              "val, last":   lambda z: z["val_losses"][:, -1],
              "val, best":   lambda z: z["val_losses"].min(1),
              "best @":      lambda z: z["val_losses"].argmin(1),
              "1st LR cut":  lambda z: first_cut(z["lrs"])}
print(f"{'':14s}" + "".join(f"{h:>22}" for h in list(pos_losses)[:3])
      + "".join(f"{h:>12}" for h in list(pos_losses)[3:]))
for key, z in pos.items():
    vals = {h: f(z) for h, f in pos_losses.items()}
    print(f"{key:14s}" + "".join(f"{f'{m:.2e} ± {s:.1e}':>22}"
                                 for m, s in (seed_stats(vals[h]) for h in list(pos_losses)[:3]))
          + "".join(f"{f'{np.nanmean(v):.0f} ± {np.nanstd(v, ddof=1):.0f}':>12}"
                    for v in list(vals.values())[3:]))
print(f"{'p (MWU)':14s}" + "".join(
    f"{pos_p(f(za), f(zb)):>{22 if i < 3 else 12}.3f}" for i, f in enumerate(pos_losses.values())))

# Rollout: how many seeds stay within the span of the field, and the RMAE as a
# geometric mean with its spread as a factor ("(4)": only four seeds finite).
print(f"\nrollout, {za[f'rollout_mae_{POS_MESHES[0]}'].shape[1]} steps: "
      f"seeds whose final MAE stays below {NS_SPAN:.0f} K (the span of T)")
print(f"{'':14s}" + "".join(f"{mesh:>11}" for mesh in POS_MESHES))
for key, z in pos.items():
    final = [z[f"rollout_mae_{mesh}"][:, -1] for mesh in POS_MESHES]
    print(f"{key:14s}" + "".join(
        f"{int((np.isfinite(f) & (f < NS_SPAN)).sum())}/{len(POS_SEEDS)}".rjust(11) for f in final))
print(f"\nrollout RMAE, geometric mean ×/÷ geometric std over seeds")
print(f"{'':14s}" + "".join(f"{mesh:>22}" for mesh in POS_MESHES))
for key in POS_CASES:
    row = f"{key:14s}"
    for j in range(len(POS_MESHES)):
        r = pos_rmae_roll[key][:, j]
        mu, sd = seed_stats(r, log=True)
        n = int(np.isfinite(r).sum())
        row += f"{f'{10**mu:.1e} ×/÷ {10**sd:.1g}' + (f' ({n})' if n < len(r) else ''):>22}"
    print(row)
print(f"{'p (MWU)':14s}" + "".join(
    f"{pos_p(pos_rmae_roll['positions'][:, j], pos_rmae_roll['no positions'][:, j]):>22.3f}"
    for j in range(len(POS_MESHES))))

# Line styles / curves of this section, also used by the sections below.
# With positions is the reference, in the baseline's colour and line style of the
# nosched sweep; without them is the new run.
POS_STYLES = {"positions": (PLOT2_COLOR, "--"), "no positions": (PLOT1_COLOR, "-")}


In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(1, 3, figsize=(1800*px, 560*px), constrained_layout=True)
axs[1].sharey(axs[0])

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Loss Decay, Positions vs None, msg_dim = 64, h = 1 "
             f"(mean ± std over {len(POS_SEEDS)} seeds)", fontsize=24, color=AX_COLOR)

panels = (("Training Loss", "train_losses"), ("Validation Loss", "val_losses"),
          ("Learning Rate", "lrs"))
for ax, (title, field) in zip(axs, panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, z in pos.items():
        color, ls = POS_STYLES[key]
        epochs = np.arange(POS_EPOCHS[key])   # 200 with positions, 500 without
        if field == "lrs":
            # the scheduler cuts each seed's LR at its own epochs: one line per
            # seed, at the LR each epoch trained at
            for i, lr in enumerate(z["lrs"]):
                ax.step(epochs, lr, where="post", color=color, linewidth=1.4,
                        linestyle=ls, alpha=0.7, label=key if i == 0 else None)
        else:
            mu, sd = seed_stats(z[field])
            ax.plot(epochs, mu, label=key, color=color, linewidth=2.2, linestyle=ls)
            # the band's lower edge is clipped: on a log axis mean - std <= 0 has
            # no place to go
            ax.fill_between(epochs, np.clip(mu - sd, mu / 10, None), mu + sd,
                            color=color, alpha=0.15, linewidth=0)

    # No LR can move before the scheduler starts stepping.
    ax.axvline(POS_SCHED_START, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.annotate("scheduler on", xy=(POS_SCHED_START, 0.02), xycoords=("data", "axes fraction"),
                xytext=(4, 0), textcoords="offset points", ha="left", va="bottom",
                fontsize=12, color=AX_COLOR, alpha=0.8)

    ax.set_yscale("log")

    legend = ax.legend(fontsize=13, loc="upper right")
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if field != "val_losses":   # the validation panel shares the training one's y
        ax.set_ylabel("LR" if field == "lrs" else "MSE Loss", fontsize=18)
    ax.tick_params(axis='both', labelsize=15)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_nopos_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out mesh. The RMAE is computed from the MAE saved at
# training time; thin lines are the seeds, the thick line and band the geometric mean ×/÷ one
# geometric std, which stop where a seed's rollout overflows (nan), as in the
# nosched sweep.
fig, axs = plt.subplots(1, len(POS_MESHES), figsize=(1600*px, 500*px),
                        sharex=True, sharey=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Rollout RMAE, Positions vs None, msg_dim = 64, h = 1 "
             f"(geometric mean ×/÷ std over {len(POS_SEEDS)} seeds)",
             fontsize=24, color=AX_COLOR)

for col, (ax, mesh) in enumerate(zip(axs, POS_MESHES)):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, z in pos.items():
        color, ls = POS_STYLES[key]
        rel = rmae_curve(z[f"rollout_mae_{mesh}"], param_T_stats(mesh)[0])
        steps = np.arange(1, rel.shape[1] + 1)
        for m in rel:
            ax.plot(steps, m, color=color, linewidth=0.6, alpha=0.35)
        mu, sd = seed_stats(rel, log=True)
        mu[~np.isfinite(rel).all(0)] = np.nan
        ax.plot(steps, 10 ** mu, label=key, color=color, linewidth=2.4, linestyle=ls)
        ax.fill_between(steps, 10 ** (mu - sd), 10 ** (mu + sd),
                        color=color, alpha=0.15, linewidth=0)
    ax.set_yscale("log")

    # The span of the field itself: an error above this line is no longer a
    # prediction of anything.
    ax.plot(steps, rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0]),
            color=AX_COLOR, linewidth=1.5, linestyle=":")
    if col == 0:
        ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, NS_SPAN / param_T_stats(mesh)[0][0]),
                    xytext=(4, 4), textcoords="offset points",
                    ha="left", va="bottom", fontsize=12, color=AX_COLOR)
        ax.set_ylabel("RMAE", fontsize=15)

    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel("Time step", fontsize=15)
    ax.tick_params(axis='both', labelsize=13)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

legend = axs[-1].legend(fontsize=12, loc="lower right")
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nopos_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- Positions vs none: one-step RMAE / RMSE and rollout RMSE, per seed ----------
# compute_reg_errors (Regularization cell above) on each seed checkpoint here,
# with this section's graphs. Seeds whose checkpoint is not on this machine are
# listed and left out; the rollout RMAE of every seed comes from the saved MAE
# either way. The cache is keyed by run path, and the positions runs reuse the
# path of the earlier batch-8 cf seeds, so every entry also records the model.pt
# it came from (its mtime) and is recomputed when that changes.
pos_errors_cache = processed_data_dir / "parametric_errors_nopos.npz"


def pos_stamp(run):
    return np.int64((processed_data_dir / run / "checkpoints" / "model.pt").stat().st_mtime_ns)


pos_absent = [run for runs in pos_runs.values() for run, ok in runs.values() if not ok]
if pos_absent:
    print(f"{len(pos_absent)} of {len(POS_CASES) * len(POS_SEEDS)} seed checkpoints are not in "
          f"{processed_data_dir}, left out of one-step RMAE/RMSE and rollout RMSE:")
    for run in pos_absent:
        print(f"    {run}/checkpoints/model.pt")
    print()

pos_errors = (dict(np.load(pos_errors_cache, allow_pickle=False))
              if pos_errors_cache.exists() else {})
for key, runs in pos_runs.items():
    stale = [run for run, ok in runs.values()
             if ok and pos_errors.get(f"{run}__stamp") != pos_stamp(run)]
    if stale:
        pos_errors.update(compute_reg_errors(stale, load=POS_LOAD[key]))
        pos_errors.update({f"{run}__stamp": pos_stamp(run) for run in stale})
        np.savez_compressed(pos_errors_cache, **pos_errors)


def pos_metric(key, metric, mode):
    """(S, M) array of one metric, nan where the seed's checkpoint is missing.
    Rollout RMAE is always the one from the saved MAE."""
    if (metric, mode) == ("rmae", "roll"):
        return pos_rmae_roll[key]
    out = np.full((len(POS_SEEDS), len(POS_MESHES)), np.nan)
    for i, s in enumerate(POS_SEEDS):
        run, ok = pos_runs[key][int(s)]
        if ok:   # an overflowed rollout is nan here too: diverged, not missing
            v = np.array([float(pos_errors[f"{run}__{mesh}__{metric}_{mode}"]) for mesh in POS_MESHES])
            out[i] = np.where(np.isfinite(v), v, np.inf)
    return out


# The rollouts recomputed from the checkpoints have to be the ones whose MAE was
# saved at training time. A large difference can also mean that seeds.npz is
# older than the checkpoints (aggregate_seeds.py not rerun). nan-safe: an
# overflowed rollout is nan in both.
devs = [np.nanmax(np.abs(pos_errors[f"{run}__{mesh}__mae_roll"]
                         / pos[key][f"rollout_mae_{mesh}"][i] - 1))
        for key in POS_CASES for i, s in enumerate(POS_SEEDS)
        for run, ok in [pos_runs[key][int(s)]] if ok for mesh in POS_MESHES]
if devs:
    print(f"recomputed vs saved rollout MAE: max |relative difference| = {max(devs):.1e}\n")

# Geometric mean ×/÷ geometric std over the seeds with a finite value; (n/m): n
# finite of the m seeds that have the metric, the rest overflowed. The p row
# compares the two configurations on that metric and mesh.
cols = [("step", "rmae"), ("step", "rmse"), ("roll", "rmae"), ("roll", "rmse")]
print(f"{'mesh':<11}{'run':<15}" + "".join(
    f"{f'{m} {x.upper()}':>28}" for m, x in (("one-step", "rmae"), ("one-step", "rmse"),
                                           ("rollout", "rmae"), ("rollout", "rmse"))))
for j, mesh in enumerate(POS_MESHES):
    for key in POS_CASES:
        row = f"{mesh:<11}{key:<15}"
        for mode, metric in cols:
            v = pos_metric(key, metric, mode)[:, j]
            n, n_have = int(np.isfinite(v).sum()), int((~np.isnan(v)).sum())
            if n == 0:
                row += f"{'—' if n_have == 0 else f'inf ({n}/{n_have})':>28}"
                continue
            mu, sd = seed_stats(v, log=True)
            row += f"{f'{10**mu:.2e} ×/÷ {10**sd:.2g} ({n}/{n_have})':>28}"
        print(row)
    print(f"{mesh:<11}{'p (MWU)':<15}" + "".join(
        f"{pos_p(pos_metric('positions', metric, mode)[:, j], pos_metric('no positions', metric, mode)[:, j]):>28.3f}"
        for mode, metric in cols))

In [ ]:
from matplotlib.patches import Patch

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The table above as a figure, laid out as in the nosched sweep. Rows: one-step
# and rollout; columns: RMAE and RMSE. Each bar is the geometric mean over the
# seeds, the whisker ×/÷ one geometric std, the dots the seeds themselves; an
# overflowed seed is drawn as a triangle at the top of its panel.
x = np.arange(len(POS_MESHES))
width = 0.3

fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE, Positions vs None, msg_dim = 64, h = 1 (geometric mean ×/÷ std)",
             fontsize=22, color=AX_COLOR)

panels = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
for ax, (mode, metric) in zip(axs.ravel(), panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_title(f"{'one-step' if mode == 'step' else 'rollout'}  ·  {metric.upper()}",
                 fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(POS_MESHES, fontsize=13)

    vals = {key: pos_metric(key, metric, mode) for key in POS_CASES}
    finite = np.concatenate([v[np.isfinite(v)] for v in vals.values()])
    if finite.size == 0:
        ax.set_yticks([])
        ax.annotate("needs the seed checkpoints\n(<exp>/seed_<n>/checkpoints/model.pt)",
                    xy=(0.5, 0.5), xycoords="axes fraction", ha="center", va="center",
                    fontsize=15, color=AX_COLOR, alpha=0.8)
        continue

    # bars on a log axis grow from the bottom of the axis, not from 0, so the
    # floor has to be set before they are drawn
    floor, hi = finite.min() / 10, finite.max() * 10
    ax.set_yscale("log")
    ax.set_ylim(floor, hi)
    for i, key in enumerate(POS_CASES):
        color = POS_STYLES[key][0]
        off = (i - (len(POS_CASES) - 1) / 2) * (width + 0.02)
        mu, sd = seed_stats(vals[key], log=True)            # (M,) each
        has = np.isfinite(mu)
        ax.bar((x + off)[has], (10 ** mu - floor)[has], width, bottom=floor,
               color=color, label=key)
        ax.errorbar((x + off)[has], 10 ** mu[has],
                    yerr=[10 ** mu[has] - 10 ** (mu - sd)[has], 10 ** (mu + sd)[has] - 10 ** mu[has]],
                    fmt="none", ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5)
        for j in range(len(POS_MESHES)):
            v = vals[key][:, j]
            ok = np.isfinite(v)
            ax.plot(np.full(ok.sum(), x[j] + off), v[ok], "o", color=AX_COLOR,
                    markersize=2.8, alpha=0.7)
            if np.isinf(v).any():
                ax.plot(x[j] + off, hi, "v", color=color, markersize=7,
                        markeredgecolor=AX_COLOR, clip_on=False)

    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure, built from the styles, since a configuration
# with no checkpoint here has no bars to take a handle from.
legend = fig.legend([Patch(color=POS_STYLES[k][0]) for k in POS_CASES], list(POS_CASES),
                    loc="outside lower center", ncols=len(POS_CASES), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nopos_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- Positions vs none: message vs FV flux, per seed ------------------------------
# compute_model_flux_stats (residual cell above) on each seed checkpoint here, with
# this section's graphs: the same meshes, timesteps and seeded faces as every
# read-out above. Cached and stamped as the errors are. Every column is an R^2, a
# correlation or an energy fraction, so none depends on how large a run writes
# its message.
pos_flux_cache = processed_data_dir / "parametric_message_flux_nopos.npz"

pos_flux = (dict(np.load(pos_flux_cache, allow_pickle=False))
            if pos_flux_cache.exists() else {})
for key, runs in pos_runs.items():
    stale = [run for run, ok in runs.values()
             if ok and pos_flux.get(f"{run}__stamp") != pos_stamp(run)]
    for run in stale:
        pos_flux.update(compute_model_flux_stats(run, load=POS_LOAD[key]))
        pos_flux[f"{run}__stamp"] = pos_stamp(run)
    if stale:
        np.savez_compressed(pos_flux_cache, **pos_flux)


def pos_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(POS_SEEDS), len(POS_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, s in enumerate(POS_SEEDS):
        run, ok = pos_runs[key][int(s)]
        if not ok:
            continue
        for j, mesh in enumerate(POS_MESHES):
            out[i, j] = [*pos_flux[f"{run}__{mesh}__r2"], float(pos_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(pos_flux[f"{run}__{mesh}__best_r"])),
                         pos_flux[f"{run}__{mesh}__anti"].mean()]
    return out


pos_flux_tab = {key: pos_flux_table(key) for key in POS_CASES}
pos_flux_n = {key: int((~np.isnan(t[:, 0, 0])).sum()) for key, t in pos_flux_tab.items()}

if not any(pos_flux_n.values()):
    print("No seed checkpoint here: the flux read-outs need "
          "<exp>/seed_<n>/checkpoints/model.pt (see the list in the errors cell).")
else:
    # Arithmetic mean ± std over the seeds that have a checkpoint: these are all
    # bounded in [0, 1], unlike the errors.
    print(f"Linear read-out from the message, {N_STEPS} probed timesteps, "
          f"mean ± std over seeds (n = seeds with a checkpoint)")
    print(f"{'mesh':<11}{'run':<15}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
    for j, mesh in enumerate(POS_MESHES):
        for key in POS_CASES:
            n = pos_flux_n[key]
            if n == 0:
                continue
            t = pos_flux_tab[key][:, j]
            mu, sd = np.nanmean(t, 0), (np.nanstd(t, 0, ddof=1) if n > 1 else np.zeros(t.shape[1]))
            print(f"{mesh:<11}{key:<15}{n:>3}"
                  + "".join(f"{f'{m:.3f} ± {s:.3f}':>15}" for m, s in zip(mu, sd)))
        print(f"{mesh:<11}{'p (MWU)':<15}{'':>3}" + "".join(
            f"{pos_p(pos_flux_tab['positions'][:, j, c], pos_flux_tab['no positions'][:, j, c]):>15.3f}"
            for c in range(len(NS_FLUX_COLS))))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The three edge-level read-outs plus the node-level one, per held-out mesh; bar
# = mean over the seeds with a checkpoint, whisker = ± one std, dots = the seeds.
# Both models are 64 channels wide, so each bar is the best linear combination of
# the same number of channels.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.3
flux_keys = [k for k in POS_CASES if pos_flux_n[k] > 0]

if not flux_keys:
    print("Nothing to plot: no seed checkpoint here (see the errors cell).")
else:
    fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
    fig.patch.set_alpha(0)
    fig.suptitle("Message vs FV Flux, Positions vs None, msg_dim = 64, h = 1 "
                 "(mean ± std over seeds)", fontsize=24, color=AX_COLOR)

    for j, (ax, mesh) in enumerate(zip(axs.ravel(), POS_MESHES)):
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=14)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for i, key in enumerate(flux_keys):
            color = POS_STYLES[key][0]
            t = pos_flux_tab[key][:, j, :4]                      # (S, 4) R^2
            t = t[~np.isnan(t[:, 0])]
            off = (i - (len(flux_keys) - 1) / 2) * (width + 0.02)
            sd = t.std(0, ddof=1) if len(t) > 1 else None
            ax.bar(x + off, t.mean(0), width, color=color, label=key,
                   yerr=sd, error_kw=dict(ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5))
            for row in t:
                ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.8, alpha=0.7)

        ax.set_title(mesh, fontsize=19)
        ax.set_xticks(x)
        ax.set_xticklabels(targets, fontsize=15)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
        ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
        ax.set_axisbelow(True)

    legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                        loc="outside lower center", ncols=len(flux_keys), fontsize=13)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    # plt.savefig("../outputs/plots/parametric_history1_nopos_flux_r2.png", dpi=300, transparent = True)

## Node positions vs none, with `l1` (α = 0.1, no LR scheduler)

The `nosched` sweep left one recipe whose rollouts mostly stay bounded: `l1` at $\alpha = 0.1$
with the learning rate held at $3\times10^{-3}$ (14 of 20 seed–mesh pairs below the span of $T$).
The section above took the node positions out of the scheduled model with no penalty. This one
takes them out of that recipe, with **5 seeds per configuration** (`FVGNN_SEED = 0 … 4`). Both
sweeps were run together by `launcher.sh` (`logs/20261012_104923`, alternating the two
configurations seed by seed) and collected by `aggregate_seeds.py`:

- **positions** (`parametric_history1_msg_dim64_residual_nosched_l1_a0.1_cf`): node features
  $[\,\text{internal}, \text{boundary}, x, y, z\,]$.
- **no positions** (`parametric_history1_msg_dim64_residual_nosched_l1_a0.1_cf_nopos`): node
  features $[\,\text{internal}, \text{boundary}\,]$.

Both are the `seed_<n>` runs in their `seeds.npz`, trained at commit `cf45013` + uncommitted
changes. The launcher's copy of `train_parametric.py` differs from `cf45013` in three places:
`FVGNN_POS_FEATURES` picks the node features, `lr_scheduler = False`, and a run directory that is
already taken gets a numbered sibling instead of being overwritten. Everything else is shared: the
64-channel residual model at `history = 1` with no LayerNorm, Cf boundary nodes and all 10 edge
features, SOAP at a constant $3\times10^{-3}$ for **200 epochs**, `batch_size = 2`, and `l1` at
$\alpha = 0.1$ with the 20-epoch warmup, the 10-epoch ramp and the per-channel scale fix. The batch
size comes from the launcher's copy of the script, since the run does not save it.

**The same budget this time.** With no scheduler, both configurations train at the same learning
rate for the same number of epochs. The asymmetry of the section above, 200 epochs against 500 so
that both could converge, does not arise.

**Still not a paired design**, for the reason given above: the no-position node encoder draws
fewer random numbers at initialisation, and the first cell shows that its $c_0$ differs from the
positions run's on every seed. The two are compared as two samples of 5, with the two-sided
Mann–Whitney $U$ test (smallest possible $p = 0.008$).

**Against the earlier `l1`, $\alpha = 0.1$ sweep.** Seed $n$ of the positions runs starts from
the same weights as seed $n$ of `..._nosched_l1_a0.1` in the `nosched` sweep: $c_0$ is identical,
which the first cell asserts. That sweep has point-mean boundary nodes, though, and was trained on
the other machine at `batch_size = 8`. The run does not record it, but every committed script with
`msg_dim = 64` and the scheduler off sets 8, from `741b454` ("parameters for hpc") on. The graphs
and the batch order both differ, so it is not a control. It appears in the tables as an
unpaired reference, `earlier (pm)`, and in no test.

- **Loss decay and PR**: training and validation MSE and the channel participation ratio, mean ±
  std over the seeds. At a constant learning rate the last epoch is a noisy snapshot, so the table
  also gives the validation MSE averaged over the last 50 epochs.
- **Rollout MAE** and the **rollout RMAE** computed from it, as in the `nosched` sweep, with the
  step at which each seed's MAE first passes the span of $T$; plotted per step as RMAE.
- **One-step RMAE / RMSE and rollout RMSE**, from the seed checkpoints, which hold the last-epoch
  weights.
- **Message vs flux, $R^2$**: `compute_model_flux_stats` per seed checkpoint.

The cells reuse `seed_stats`, `NS_SPAN`, `ns_seed_run`, `NS_FLUX_COLS` and the earlier sweep's
tables (`ns`, `ns_rmae_roll`, `ns_metric`, `ns_flux_tab`) from the `nosched` sweep; `POS_LOAD`,
`POS_STYLES`, `pos_absT`, `pos_p` and `pos_stamp` from the section above; and `compute_reg_errors`
and `compute_model_flux_stats` from the sections before.

In [ ]:
# --- Positions vs none with l1, alpha = 0.1: load the seeds, check them, summarise
# Laid out as the positions section above, whose POS_LOAD, pos_p and pos_absT it
# reuses, with seed_stats, NS_SPAN and ns_seed_run from the nosched sweep. The
# seeds of the two configurations are NOT paired (see above), so every comparison
# is between the two samples: Mann-Whitney U, two-sided.
L1P_BASE   = "parametric_history1_msg_dim64_residual_nosched_l1_a0.1"
L1P_CASES  = {"positions":    f"{L1P_BASE}_cf",
              "no positions": f"{L1P_BASE}_cf_nopos"}
L1P_MESHES = ["model_003", "model_007", "model_010", "model_011"]
# The nosched sweep's l1, alpha = 0.1, with point-mean boundary nodes and
# batch_size = 8 (not recorded with the run, see above): a reference row in the
# tables, in no test.
L1P_REF = "l1, α = 0.1"
assert NS_CASES[L1P_REF] == L1P_BASE

l1p_no_npz = [exp for exp in L1P_CASES.values()
              if not (processed_data_dir / exp / "seeds.npz").exists()]
if l1p_no_npz:
    raise FileNotFoundError(
        f"no seeds.npz in {', '.join(l1p_no_npz)} yet: train with FVGNN_SEED = 0 ... 4, "
        f"then python aggregate_seeds.py <exp_name>")

l1p = {key: dict(np.load(processed_data_dir / exp / "seeds.npz")) for key, exp in L1P_CASES.items()}
L1P_SEEDS = l1p["positions"]["seeds"]
L1P_EPOCHS = l1p["positions"]["train_losses"].shape[1]
z_ref = ns[L1P_REF]

# The same seeds, meshes, epochs, penalty and constant LR in both, and the same
# warmup and ramp as the nosched sweep, so its NS_WARM and NS_RAMP_END hold here.
for key, z in l1p.items():
    assert np.array_equal(z["seeds"], L1P_SEEDS), f"{key}: seeds {z['seeds']}"
    assert list(z["test_meshes"]) == L1P_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert z["train_losses"].shape[1] == L1P_EPOCHS, f"{key}: {z['train_losses'].shape[1]} epochs"
    assert str(z["msg_kind"]) == "l1" and float(z["msg_alpha"]) == 0.1, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(z_ref[field]), f"{key}: {field} differs from the nosched sweep"
    assert np.all(z["lrs"] == z_ref["lrs"][0, 0]), f"{key}: the LR moved, not a nosched run"
    assert np.all(z["msg_lam_eff"][:, :int(z["msg_warmup"])] == 0), f"{key}: penalty on in warmup"
    assert np.all(z["msg_lam_eff"][:, -1] > 0), f"{key}: penalty off at the end"
    dev = np.abs(z["msg_col_norm"] / z["msg_c0"][:, None, None] - 1).max()
    assert dev < 1e-4, f"{key}: column norms off c0 by {dev:.1e}"
n_drop = int(l1p["positions"]["n_params"]) - int(l1p["no positions"]["n_params"])
assert n_drop == 3 * 64, f"the models differ by {n_drop} parameters, not the 3 x 64 of x, y, z"
assert int(l1p["positions"]["n_params"]) == int(z_ref["n_params"]), "not the nosched sweep's model"

# Equal c0 per seed means equal initial weights: positions seed n starts where
# the nosched sweep's seed n did, and no seed without positions starts where
# any seed with them does.
assert np.array_equal(z_ref["seeds"], L1P_SEEDS)
assert np.array_equal(l1p["positions"]["msg_c0"], z_ref["msg_c0"]), \
    "positions seeds do not start from the nosched sweep's weights"
assert not np.isin(l1p["no positions"]["msg_c0"], l1p["positions"]["msg_c0"]).any(), \
    "a no-position seed shares its initial weights with a positions one"

# Per seed checkpoint on this machine: Cf graphs, the node features its normalizer
# was fitted on, and the commit it was trained with (for the markdown above).
l1p_runs = {key: {int(s): ns_seed_run(exp, int(s)) for s in L1P_SEEDS}
            for key, exp in L1P_CASES.items()}
for key, runs in l1p_runs.items():
    commits = {}
    for s, (run, ok) in runs.items():
        ck = processed_data_dir / run / "checkpoints"
        if not ck.exists():
            continue
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg.get("excluded_patches") == EXCLUDED, \
            f"{run}: graphs built with {cfg}"
        if ok:
            n = torch.load(ck / "normalizer.pt")["node_mean"].numel()
            assert n == POS_NODE_FEATS[key], f"{run}: normalizer fitted on {n} node features"
        c = load_git_commit(ck)
        tag = ("not recorded" if c is None or c["commit"] is None
               else c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else ""))
        commits.setdefault(tag, []).append(s)
    print(f"{L1P_CASES[key]}: " + ("; ".join(f"commit {tag} (seeds {', '.join(map(str, ss))})"
                                             for tag, ss in commits.items())
                                    or "no seed checkpoint here, commit unknown"))

# Rollout RMAE from the saved per-step MAE, no checkpoint needed, as in the
# nosched sweep; pos_absT holds the same Cf fields these runs were tested on.
l1p_rmae_roll = {}   # key -> (S, M)
for key, z in l1p.items():
    cols = []
    for mesh in L1P_MESHES:
        mae = z[f"rollout_mae_{mesh}"].astype(float)
        r = mae.sum(1) / pos_absT[mesh][:mae.shape[1]].sum()
        cols.append(np.where(np.isfinite(r), r, np.inf))
    l1p_rmae_roll[key] = np.stack(cols, 1)

za, zb = l1p["positions"], l1p["no positions"]
print(f"\n{len(L1P_SEEDS)} seeds {L1P_SEEDS.tolist()}, {L1P_EPOCHS} epochs, "
      f"{int(za['n_params'])} vs {int(zb['n_params'])} parameters, "
      f"LR {za['lrs'][0, 0]:.0e} constant, warmup {NS_WARM}, ramp complete at epoch {NS_RAMP_END}")
print("c0 per seed:  positions " + " ".join(f"{c:.4f}" for c in za["msg_c0"])
      + "   no positions " + " ".join(f"{c:.4f}" for c in zb["msg_c0"]) + "\n")

# With a constant LR the last epoch is a noisy snapshot: the mean over the last
# 50 epochs reads more reliably, as in the nosched sweep.
l1p_stats = {"train, last":    (lambda z: z["train_losses"][:, -1],         "{:.2e} ± {:.1e}", 20),
             "val, last":      (lambda z: z["val_losses"][:, -1],           "{:.2e} ± {:.1e}", 20),
             "val, last 50":   (lambda z: z["val_losses"][:, -50:].mean(1), "{:.2e} ± {:.1e}", 20),
             "val, best":      (lambda z: z["val_losses"].min(1),           "{:.2e} ± {:.1e}", 20),
             "best @":         (lambda z: z["val_losses"].argmin(1),        "{:.0f} ± {:.0f}", 10),
             "PR, last":       (lambda z: z["msg_pr"][:, -1],               "{:.1f} ± {:.1f}", 13),
             "mean |m|, last": (lambda z: z["msg_mean_abs"][:, -1],         "{:.3g} ± {:.2g}", 17)}
print(f"{'':14s}" + "".join(f"{h:>{w}}" for h, (_, _, w) in l1p_stats.items()))
for key, z in l1p.items():
    print(f"{key:14s}" + "".join(f"{fmt.format(*seed_stats(f(z))):>{w}}"
                                 for f, fmt, w in l1p_stats.values()))
print(f"{'p (MWU)':14s}" + "".join(f"{pos_p(f(za), f(zb)):>{w}.3f}" for f, _, w in l1p_stats.values()))
print(f"{'earlier (pm)':14s}" + "".join(f"{fmt.format(*seed_stats(f(z_ref))):>{w}}"
                                        for f, fmt, w in l1p_stats.values()))

# Rollout: how many seeds stay within the span of the field, at which step each
# seed leaves it, and the RMAE as a geometric mean with its spread as a factor
# ("(4)": only four seeds finite).
l1p_all = {**l1p, "earlier (pm)": z_ref}
print(f"\nrollout, {za[f'rollout_mae_{L1P_MESHES[0]}'].shape[1]} steps: "
      f"seeds whose final MAE stays below {NS_SPAN:.0f} K (the span of T)")
print(f"{'':14s}" + "".join(f"{mesh:>11}" for mesh in L1P_MESHES) + f"{'all':>9}")
for key, z in l1p_all.items():
    final = np.stack([z[f"rollout_mae_{mesh}"][:, -1] for mesh in L1P_MESHES], 1)
    ok = np.isfinite(final) & (final < NS_SPAN)
    print(f"{key:14s}" + "".join(f"{int(o.sum())}/{len(o)}".rjust(11) for o in ok.T)
          + f"{int(ok.sum())}/{ok.size}".rjust(9))

# The first step whose MAE is above the span (nan counts as above), per seed;
# "-" when the rollout never gets there.
print(f"\nrollout, first step with MAE above {NS_SPAN:.0f} K, per seed (- never)")
print(f"{'':14s}{'seed':>5}" + "".join(f"{mesh:>11}" for mesh in L1P_MESHES))
for key, z in l1p_all.items():
    for i, s in enumerate(z["seeds"]):
        row = f"{key if i == 0 else '':14s}{int(s):>5}"
        for mesh in L1P_MESHES:
            out = ~(z[f"rollout_mae_{mesh}"][i] < NS_SPAN)
            row += f"{int(out.argmax()) + 1 if out.any() else '-':>11}"
        print(row)

print(f"\nrollout RMAE, geometric mean ×/÷ geometric std over seeds")
print(f"{'':14s}" + "".join(f"{mesh:>22}" for mesh in L1P_MESHES))
for key, r_all in [*l1p_rmae_roll.items(), ("earlier (pm)", ns_rmae_roll[L1P_REF])]:
    row = f"{key:14s}"
    for j in range(len(L1P_MESHES)):
        r = r_all[:, j]
        mu, sd = seed_stats(r, log=True)
        n = int(np.isfinite(r).sum())
        row += f"{f'{10**mu:.1e} ×/÷ {10**sd:.1g}' + (f' ({n})' if n < len(r) else ''):>22}"
    print(row)
    if key == "no positions":
        print(f"{'p (MWU)':14s}" + "".join(
            f"{pos_p(l1p_rmae_roll['positions'][:, j], l1p_rmae_roll['no positions'][:, j]):>22.3f}"
            for j in range(len(L1P_MESHES))))

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

PR_TICKS = [1, 2, 4, 8, 16, 32, 64]

# Same colours and line styles as the positions section above (POS_STYLES). The
# LR is constant, so the third panel is the channel participation ratio the
# penalty acts on instead of the learning rate.
fig, axs = plt.subplots(1, 3, figsize=(1800*px, 560*px), constrained_layout=True)
axs[1].sharey(axs[0])

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Loss Decay, Positions vs None, l1 α = 0.1, No LR Scheduler "
             f"(mean ± std over {len(L1P_SEEDS)} seeds)", fontsize=24, color=AX_COLOR)

epochs = np.arange(L1P_EPOCHS)
panels = (("Training Loss", "train_losses"), ("Validation Loss", "val_losses"),
          ("Participation Ratio", "msg_pr"))
for ax, (title, field) in zip(axs, panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, z in l1p.items():
        color, ls = POS_STYLES[key]
        mu, sd = seed_stats(z[field])
        ax.plot(epochs, mu, label=key, color=color, linewidth=2.2, linestyle=ls)
        # the band's lower edge is clipped: on a log axis mean - std <= 0 has
        # no place to go (and a PR below one channel has no meaning)
        lo = np.clip(mu - sd, 1, None) if field == "msg_pr" else np.clip(mu - sd, mu / 10, None)
        ax.fill_between(epochs, lo, mu + sd, color=color, alpha=0.15, linewidth=0)

    # Warmup ends where the penalty starts to ramp in, the ramp where it is at
    # full strength. With no scheduler nothing else happens at these epochs.
    for e in (NS_WARM - 1, NS_RAMP_END):
        ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.annotate("warmup / ramp", xy=(NS_WARM - 1, 0.02), xycoords=("data", "axes fraction"),
                xytext=(-4, 0), textcoords="offset points", ha="right", va="bottom",
                fontsize=12, color=AX_COLOR, alpha=0.8)

    if field == "msg_pr":
        # PR spans 1 to 64: a base-2 axis gives every halving of the channel
        # count the same height
        ax.set_yscale("log", base=2)
        ax.set_yticks(PR_TICKS, labels=[str(k) for k in PR_TICKS])
        ax.yaxis.set_minor_locator(NullLocator())
        ax.set_ylim(1, 64)
    else:
        ax.set_yscale("log")

    legend = ax.legend(fontsize=13, loc="upper right")
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(title, fontsize=20)
    ax.set_xlabel("Epoch", fontsize=18)
    if field == "train_losses":   # the validation panel shares the training one's y
        ax.set_ylabel("MSE Loss", fontsize=18)
    elif field == "msg_pr":
        ax.set_ylabel("PR  [channels of 64]", fontsize=18)
    ax.tick_params(axis='both', labelsize=15)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_nopos_l1_loss.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out mesh. The RMAE is computed from the MAE saved at
# training time; thin lines are the seeds, the thick line and band the geometric mean ×/÷ one
# geometric std, which stop where a seed's rollout overflows (nan), as in the
# nosched sweep.
fig, axs = plt.subplots(1, len(L1P_MESHES), figsize=(1600*px, 500*px),
                        sharex=True, sharey=True, constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle(f"Rollout RMAE, Positions vs None, l1 α = 0.1, No LR Scheduler "
             f"(geometric mean ×/÷ std over {len(L1P_SEEDS)} seeds)",
             fontsize=24, color=AX_COLOR)

for col, (ax, mesh) in enumerate(zip(axs, L1P_MESHES)):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for key, z in l1p.items():
        color, ls = POS_STYLES[key]
        rel = rmae_curve(z[f"rollout_mae_{mesh}"], param_T_stats(mesh)[0])
        steps = np.arange(1, rel.shape[1] + 1)
        for m in rel:
            ax.plot(steps, m, color=color, linewidth=0.6, alpha=0.35)
        mu, sd = seed_stats(rel, log=True)
        mu[~np.isfinite(rel).all(0)] = np.nan
        ax.plot(steps, 10 ** mu, label=key, color=color, linewidth=2.4, linestyle=ls)
        ax.fill_between(steps, 10 ** (mu - sd), 10 ** (mu + sd),
                        color=color, alpha=0.15, linewidth=0)
    ax.set_yscale("log")

    # The span of the field itself: an error above this line is no longer a
    # prediction of anything.
    ax.plot(steps, rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0]),
            color=AX_COLOR, linewidth=1.5, linestyle=":")
    if col == 0:
        ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, NS_SPAN / param_T_stats(mesh)[0][0]),
                    xytext=(4, 4), textcoords="offset points",
                    ha="left", va="bottom", fontsize=12, color=AX_COLOR)
        ax.set_ylabel("RMAE", fontsize=15)

    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel("Time step", fontsize=15)
    ax.tick_params(axis='both', labelsize=13)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

legend = axs[-1].legend(fontsize=12, loc="lower right")
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nopos_l1_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- Positions vs none with l1: one-step RMAE / RMSE and rollout RMSE, per seed ---
# compute_reg_errors (Regularization cell above) on each seed checkpoint here,
# with the Cf graphs of POS_LOAD, cached and stamped with the model.pt each entry
# came from (pos_stamp), as in the positions section. Seeds whose checkpoint is
# not on this machine are listed and left out; the rollout RMAE of every seed
# comes from the saved MAE either way. The earlier (pm) rows are the nosched
# sweep's (ns_metric), for reference.
l1p_errors_cache = processed_data_dir / "parametric_errors_nopos_l1.npz"

l1p_absent = [run for runs in l1p_runs.values() for run, ok in runs.values() if not ok]
if l1p_absent:
    print(f"{len(l1p_absent)} of {len(L1P_CASES) * len(L1P_SEEDS)} seed checkpoints are not in "
          f"{processed_data_dir}, left out of one-step RMAE/RMSE and rollout RMSE:")
    for run in l1p_absent:
        print(f"    {run}/checkpoints/model.pt")
    print()

l1p_errors = (dict(np.load(l1p_errors_cache, allow_pickle=False))
              if l1p_errors_cache.exists() else {})
for key, runs in l1p_runs.items():
    stale = [run for run, ok in runs.values()
             if ok and l1p_errors.get(f"{run}__stamp") != pos_stamp(run)]
    if stale:
        l1p_errors.update(compute_reg_errors(stale, load=POS_LOAD[key]))
        l1p_errors.update({f"{run}__stamp": pos_stamp(run) for run in stale})
        np.savez_compressed(l1p_errors_cache, **l1p_errors)


def l1p_metric(key, metric, mode):
    """(S, M) array of one metric, nan where the seed's checkpoint is missing.
    Rollout RMAE is always the one from the saved MAE."""
    if (metric, mode) == ("rmae", "roll"):
        return l1p_rmae_roll[key]
    out = np.full((len(L1P_SEEDS), len(L1P_MESHES)), np.nan)
    for i, s in enumerate(L1P_SEEDS):
        run, ok = l1p_runs[key][int(s)]
        if ok:   # an overflowed rollout is nan here too: diverged, not missing
            v = np.array([float(l1p_errors[f"{run}__{mesh}__{metric}_{mode}"]) for mesh in L1P_MESHES])
            out[i] = np.where(np.isfinite(v), v, np.inf)
    return out


# The rollouts recomputed from the checkpoints have to be the ones whose MAE was
# saved at training time. Compared only over the steps where both are below the
# span of T: past it the rollout amplifies round-off, and the nosched sweep
# already showed float32 inf against a finite float64 there.
devs = []
for key in L1P_CASES:
    for i, s in enumerate(L1P_SEEDS):
        run, ok = l1p_runs[key][int(s)]
        if not ok:
            continue
        for mesh in L1P_MESHES:
            new, old = l1p_errors[f"{run}__{mesh}__mae_roll"], l1p[key][f"rollout_mae_{mesh}"][i]
            keep = (new < NS_SPAN) & (old < NS_SPAN)
            if keep.any():
                devs.append(np.abs(new[keep] / old[keep] - 1).max())
if devs:
    print(f"recomputed vs saved rollout MAE, steps below {NS_SPAN:.0f} K: "
          f"max |relative difference| = {max(devs):.1e}\n")

# Geometric mean ×/÷ geometric std over the seeds with a finite value; (n/m): n
# finite of the m seeds that have the metric, the rest overflowed. The p row
# compares the two configurations on that metric and mesh.
cols = [("step", "rmae"), ("step", "rmse"), ("roll", "rmae"), ("roll", "rmse")]


def l1p_cell(v):
    n, n_have = int(np.isfinite(v).sum()), int((~np.isnan(v)).sum())
    if n == 0:
        return f"{'—' if n_have == 0 else f'inf ({n}/{n_have})':>28}"
    mu, sd = seed_stats(v, log=True)
    return f"{f'{10**mu:.2e} ×/÷ {10**sd:.2g} ({n}/{n_have})':>28}"


print(f"{'mesh':<11}{'run':<15}" + "".join(
    f"{f'{m} {x.upper()}':>28}" for m, x in (("one-step", "rmae"), ("one-step", "rmse"),
                                           ("rollout", "rmae"), ("rollout", "rmse"))))
for j, mesh in enumerate(L1P_MESHES):
    for key in L1P_CASES:
        print(f"{mesh:<11}{key:<15}" + "".join(l1p_cell(l1p_metric(key, metric, mode)[:, j])
                                             for mode, metric in cols))
    print(f"{mesh:<11}{'p (MWU)':<15}" + "".join(
        f"{pos_p(l1p_metric('positions', metric, mode)[:, j], l1p_metric('no positions', metric, mode)[:, j]):>28.3f}"
        for mode, metric in cols))
    print(f"{mesh:<11}{'earlier (pm)':<15}" + "".join(l1p_cell(ns_metric(L1P_REF, metric, mode)[:, j])
                                                    for mode, metric in cols))

In [ ]:
from matplotlib.patches import Patch

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The table above as a figure, laid out as in the positions section. Rows:
# one-step and rollout; columns: RMAE and RMSE. Each bar is the geometric mean
# over the seeds, the whisker ×/÷ one geometric std, the dots the seeds
# themselves; an overflowed seed is drawn as a triangle at the top of its panel.
x = np.arange(len(L1P_MESHES))
width = 0.3

fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE, Positions vs None, l1 α = 0.1, No LR Scheduler "
             "(geometric mean ×/÷ std)", fontsize=22, color=AX_COLOR)

panels = [(mode, metric) for mode in ("step", "roll") for metric in ("rmae", "rmse")]
for ax, (mode, metric) in zip(axs.ravel(), panels):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_title(f"{'one-step' if mode == 'step' else 'rollout'}  ·  {metric.upper()}",
                 fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(L1P_MESHES, fontsize=13)

    vals = {key: l1p_metric(key, metric, mode) for key in L1P_CASES}
    finite = np.concatenate([v[np.isfinite(v)] for v in vals.values()])
    if finite.size == 0:
        ax.set_yticks([])
        ax.annotate("needs the seed checkpoints\n(<exp>/seed_<n>/checkpoints/model.pt)",
                    xy=(0.5, 0.5), xycoords="axes fraction", ha="center", va="center",
                    fontsize=15, color=AX_COLOR, alpha=0.8)
        continue

    # bars on a log axis grow from the bottom of the axis, not from 0, so the
    # floor has to be set before they are drawn
    floor, hi = finite.min() / 10, finite.max() * 10
    ax.set_yscale("log")
    ax.set_ylim(floor, hi)
    for i, key in enumerate(L1P_CASES):
        color = POS_STYLES[key][0]
        off = (i - (len(L1P_CASES) - 1) / 2) * (width + 0.02)
        mu, sd = seed_stats(vals[key], log=True)            # (M,) each
        has = np.isfinite(mu)
        ax.bar((x + off)[has], (10 ** mu - floor)[has], width, bottom=floor,
               color=color, label=key)
        ax.errorbar((x + off)[has], 10 ** mu[has],
                    yerr=[10 ** mu[has] - 10 ** (mu - sd)[has], 10 ** (mu + sd)[has] - 10 ** mu[has]],
                    fmt="none", ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5)
        for j in range(len(L1P_MESHES)):
            v = vals[key][:, j]
            ok = np.isfinite(v)
            ax.plot(np.full(ok.sum(), x[j] + off), v[ok], "o", color=AX_COLOR,
                    markersize=2.8, alpha=0.7)
            if np.isinf(v).any():
                ax.plot(x[j] + off, hi, "v", color=color, markersize=7,
                        markeredgecolor=AX_COLOR, clip_on=False)

    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure, built from the styles, since a configuration
# with no checkpoint here has no bars to take a handle from.
legend = fig.legend([Patch(color=POS_STYLES[k][0]) for k in L1P_CASES], list(L1P_CASES),
                    loc="outside lower center", ncols=len(L1P_CASES), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_nopos_l1_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- Positions vs none with l1: message vs FV flux, per seed ----------------------
# compute_model_flux_stats (residual cell above) on each seed checkpoint here, with
# the Cf graphs of POS_LOAD: the same meshes, timesteps and seeded faces as every
# read-out above. Cached and stamped as the errors are. Every column is an R^2, a
# correlation or an energy fraction, so none depends on how large a run writes
# its message. The earlier (pm) rows are the nosched sweep's (ns_flux_tab).
l1p_flux_cache = processed_data_dir / "parametric_message_flux_nopos_l1.npz"

l1p_flux = (dict(np.load(l1p_flux_cache, allow_pickle=False))
            if l1p_flux_cache.exists() else {})
for key, runs in l1p_runs.items():
    stale = [run for run, ok in runs.values()
             if ok and l1p_flux.get(f"{run}__stamp") != pos_stamp(run)]
    for run in stale:
        l1p_flux.update(compute_model_flux_stats(run, load=POS_LOAD[key]))
        l1p_flux[f"{run}__stamp"] = pos_stamp(run)
    if stale:
        np.savez_compressed(l1p_flux_cache, **l1p_flux)


def l1p_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(L1P_SEEDS), len(L1P_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, s in enumerate(L1P_SEEDS):
        run, ok = l1p_runs[key][int(s)]
        if not ok:
            continue
        for j, mesh in enumerate(L1P_MESHES):
            out[i, j] = [*l1p_flux[f"{run}__{mesh}__r2"], float(l1p_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(l1p_flux[f"{run}__{mesh}__best_r"])),
                         l1p_flux[f"{run}__{mesh}__anti"].mean()]
    return out


l1p_flux_tab = {key: l1p_flux_table(key) for key in L1P_CASES}
l1p_flux_n = {key: int((~np.isnan(t[:, 0, 0])).sum()) for key, t in l1p_flux_tab.items()}


def l1p_flux_row(label, t):
    """One table row: arithmetic mean ± std over the seeds that have a checkpoint;
    these are all bounded in [0, 1], unlike the errors."""
    t = t[~np.isnan(t[:, 0])]
    sd = t.std(0, ddof=1) if len(t) > 1 else np.zeros(t.shape[1])
    return f"{label:<15}{len(t):>3}" + "".join(f"{f'{m:.3f} ± {s:.3f}':>15}"
                                               for m, s in zip(t.mean(0), sd))


if not any(l1p_flux_n.values()):
    print("No seed checkpoint here: the flux read-outs need "
          "<exp>/seed_<n>/checkpoints/model.pt (see the list in the errors cell).")
else:
    print(f"Linear read-out from the message, {N_STEPS} probed timesteps, "
          f"mean ± std over seeds (n = seeds with a checkpoint)")
    print(f"{'mesh':<11}{'run':<15}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
    for j, mesh in enumerate(L1P_MESHES):
        for key in L1P_CASES:
            if l1p_flux_n[key]:
                print(f"{mesh:<11}" + l1p_flux_row(key, l1p_flux_tab[key][:, j]))
        print(f"{mesh:<11}{'p (MWU)':<15}{'':>3}" + "".join(
            f"{pos_p(l1p_flux_tab['positions'][:, j, c], l1p_flux_tab['no positions'][:, j, c]):>15.3f}"
            for c in range(len(NS_FLUX_COLS))))
        if ns_flux_n[L1P_REF]:
            print(f"{mesh:<11}" + l1p_flux_row("earlier (pm)", ns_flux_tab[L1P_REF][:, j]))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The three edge-level read-outs plus the node-level one, per held-out mesh; bar
# = mean over the seeds with a checkpoint, whisker = ± one std, dots = the seeds.
# Both models are 64 channels wide, so each bar is the best linear combination of
# the same number of channels.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.3
flux_keys = [k for k in L1P_CASES if l1p_flux_n[k] > 0]

if not flux_keys:
    print("Nothing to plot: no seed checkpoint here (see the errors cell).")
else:
    fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
    fig.patch.set_alpha(0)
    fig.suptitle("Message vs FV Flux, Positions vs None, l1 α = 0.1, No LR Scheduler "
                 "(mean ± std over seeds)", fontsize=24, color=AX_COLOR)

    for j, (ax, mesh) in enumerate(zip(axs.ravel(), L1P_MESHES)):
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=14)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for i, key in enumerate(flux_keys):
            color = POS_STYLES[key][0]
            t = l1p_flux_tab[key][:, j, :4]                      # (S, 4) R^2
            t = t[~np.isnan(t[:, 0])]
            off = (i - (len(flux_keys) - 1) / 2) * (width + 0.02)
            sd = t.std(0, ddof=1) if len(t) > 1 else None
            ax.bar(x + off, t.mean(0), width, color=color, label=key,
                   yerr=sd, error_kw=dict(ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5))
            for row in t:
                ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.8, alpha=0.7)

        ax.set_title(mesh, fontsize=19)
        ax.set_xticks(x)
        ax.set_xticklabels(targets, fontsize=15)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
        ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
        ax.set_axisbelow(True)

    legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                        loc="outside lower center", ncols=len(flux_keys), fontsize=13)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    # plt.savefig("../outputs/plots/parametric_history1_nopos_l1_flux_r2.png", dpi=300, transparent = True)

### What comes out

All 10 seed checkpoints are here, so every number below is over the full 5 seeds. The one-step
errors, the rollout RMSE and the flux read-outs come from `model.pt`, which holds the
**last-epoch** weights. The losses, the PR and the rollout MAE come from `seeds.npz`.

**Best validation loss: positions help, by a factor of 2.6.** Best validation MSE is
$4.20 \pm 0.26$ with positions and $10.9 \pm 0.3$ without them (units of $10^{-5}$), with every
seed of one configuration below every seed of the other ($p = 0.008$). With positions it is
reached earlier, at epoch $24 \pm 14$ against $64 \pm 24$ ($p = 0.03$). That is the opposite of
the scheduled section above, where the 500-epoch no-position runs reached $1.16\times10^{-5}$
against $3.29\times10^{-5}$. Here both configurations have the same budget, and both best
epochs come early, before the drift described next, so this compares early training.

**After the ramp, both get worse, and the difference goes away.** At a constant
$3\times10^{-3}$ neither run settles. The validation MSE averaged over epochs 30 – 60 is
$1.2\times10^{-4}$ with positions and $1.8\times10^{-4}$ without, and over the last 50 epochs it
is $2.3$ and $3.2\times10^{-4}$, with $p = 0.22$. The training MSE climbs from
$7\times10^{-4}$ to $1.9\times10^{-3}$ with positions and from $1.4$ to $2.7\times10^{-3}$
without, 6 – 12 times the validation loss over the last 50 epochs. The earlier point-mean sweep
with the same recipe and, for the positions seeds, the same initial weights did not drift. Its
validation MSE stays at $6 - 7.5\times10^{-5}$ from epoch 30 to the end, 3 – 4 times below these
runs, with a training/validation ratio of 3.5 – 4.9.

The regularizer log shows the penalty taking over. The ratio of the penalty gradient to the data
gradient, $|\nabla_\text{pen}| / |\nabla_\text{data}|$, is 0.28 during the ramp and 6.3 over the last
50 epochs with positions, and 0.26 to 3.1 without. In the earlier sweep it went only from 0.08 to
0.29. The PR follows. It is still falling at epoch 200, to $14.6 \pm 4.3$ with positions and
$12.7 \pm 3.1$ without ($p = 0.55$), where the earlier sweep held at 44 from epoch 30 on. What
makes the penalty win here is not isolated. These runs differ from the earlier sweep in the
boundary nodes (Cf against point mean) and in the batch size (2 against 8). At batch 2 an epoch
takes 4 times as many steps at the same constant learning rate. A `nosched` control at batch 2 with no penalty (or the `hoyer 0` one,
scale fix only) would separate the penalty from the batch size.

**Rollout: no difference, and stability is a property of the seed.** Both configurations keep
12 of the 20 seed–mesh pairs below the 300 K span, against 14 for the earlier sweep. Every seed
here is either bounded on all four meshes or on none. With positions, seeds 0 – 2 stay bounded,
ending at 12 – 58 K MAE. Seed 3 leaves the span at step 24 – 48 and seed 4 at step 7 – 27, which
then overflows to nan. Without positions, seeds 2 – 4 stay bounded, ending at 17 – 157 K. Seed 1
leaves at step 19 – 30 and seed 0 at step 16 – 57 (then nan). The rollout RMAE has one
overflowed seed on each side, and its $p$ is 0.92 on every mesh. The early horizon, where every
seed is still finite, does not separate them either. The MAE averaged over steps 1 – 10 is 25 K
with positions and 21 K without (geometric mean over seeds and meshes, $p \ge 0.31$ per mesh).
That is about 3 times the earlier sweep's 7.4 K, in line with its 3 – 4 times lower validation
loss.

**One-step error: slightly lower with positions, and well above the earlier sweep.** At the
last-epoch weights, the one-step RMAE is $1.27\times10^{-3}$ with positions and
$1.74\times10^{-3}$ without (geometric mean over seeds and meshes). It is lower with positions on
all four meshes, but the test separates the two only on `model_003` ($p = 0.008$, against
0.10 – 1.0 on the others). The RMSE is $9.6\times10^{-5}$ against $1.5\times10^{-4}$
($p \ge 0.06$). Both are worse than the earlier sweep, whose RMAE of $8.9\times10^{-4}$ and RMSE
of $2.2\times10^{-5}$ are 1.4 and 4.4 times below the positions runs, in line with its lower
last-epoch validation loss. The rollout RMSE has the same overflowed seed on each side as the
RMAE, and $p \ge 0.92$. The rollouts recomputed from the checkpoints match the ones saved at
training time to $10^{-3}$ relative, over the steps below the span.

**The message is less flux-like than in the earlier sweep, and less still with positions.** The
$R^2$ of the linear read-out of the FV flux $F$ from the message is 0.10 – 0.12 with positions
and 0.18 – 0.22 without ($p = 0.15 - 0.22$), against 0.21 – 0.22 for the earlier sweep.
$\Delta T$ and the node sum separate more clearly. For $\Delta T$ it is 0.12 – 0.15 against
0.26 – 0.35 ($p = 0.03 - 0.06$), and for $\sum_j F_{ij}$ 0.34 – 0.40 against 0.48 – 0.56
($p = 0.008 - 0.10$). The earlier sweep has 0.30 – 0.39 and 0.64 – 0.72. The direction is the
one of the scheduled section above, where taking the positions out raised the $R^2$ of $F$ from
0.70 – 0.76 to 0.87 – 0.90. The antisymmetric energy fraction is 0.17 – 0.20 with positions and
0.19 – 0.23 without ($p \ge 0.55$), half the earlier sweep's 0.38 – 0.40.

These read-outs needed a fix in `_r2` (flux cell above). The message channels of positions seed 1
span five decades in standard deviation, `lstsq` cut the rank of $[X, 1]$ to 30 of 65, and the
$R^2$ came out negative. `_r2` now standardises the columns first, which leaves $R^2$ unchanged
in exact arithmetic. Recomputed with it, the cached `nosched` read-outs match to $10^{-5}$ (all
five seeds of `l1`, $\alpha = 0.1$ and seed 0 of the other five configurations), so the sections
above are unaffected.

**What the pair shows.** Under the `l1`, $\alpha = 0.1$ recipe at a constant learning rate, the
node positions give a 2.6 times lower best validation loss and a slightly lower one-step error,
and a message that reads out less like the flux. Rollout stability does not change (12 of 20
pairs each), and neither does the drift after the ramp. The recipe itself did not carry over to
this setup. The training loss rises from the ramp on, the penalty gradient grows to several times
the data gradient, and the validation loss ends 3 – 4 times above the earlier sweep's. The earlier
sweep is not a control, so which change causes this cannot be read off here.

## Message penalties under the LR scheduler, with and without positions

The `l1` and `hoyer` penalties so far were trained either with point-mean boundary nodes (the
single-run sections) or without the LR scheduler (`nosched`). This section puts them on the
setup of the positions section: Cf boundary nodes, `ReduceLROnPlateau` from the end of the ramp,
`batch_size = 2`, at most **200 epochs**, with and without the node positions. Five configurations:

- **positions, no penalty** (`parametric_history1_msg_dim64_residual_cf`, commit `cf45013`):
  the positions runs of the positions section, `seed_<n>` in `seeds.npz`.
- **positions, `l1`, α = 0.1** (`parametric_history1_msg_dim64_residual_l1_a0.1_cf`, commit
  `b984402` + uncommitted changes): seeds 1 – 4 are `seed_<n>` in `seeds.npz`. Seed 0 is
  `seed_0_01` in `seeds_01.npz`: the first launcher (`logs/20261014_112918`) was stopped
  during it, and the unfinished `seed_0` it left (no `model.pt`) is not used.
- **no positions, no penalty** (`parametric_history1_msg_dim64_residual_cf_nopos`, commit
  `cf45013` + uncommitted changes): the **200-epoch** `seed_<n>` sweep in `seeds.npz`. The
  positions section set it aside because it had not converged. Here it has the same budget as
  the penalised runs.
- **no positions, `l1`, α = 0.1** (`parametric_history1_msg_dim64_residual_l1_a0.1_cf_nopos`,
  commit `b984402` + uncommitted changes), `seed_<n>` in `seeds.npz`.
- **no positions, `hoyer`, α = 0.01** (`parametric_history1_msg_dim64_residual_hoyer_a0.01_cf_nopos`,
  commit `b984402` + uncommitted changes): **seed 0 only**. The first launcher was stopped after
  it, and the second (`logs/20261014_122224`) ran only the two `l1` configurations.

The `l1` and `hoyer` runs come from `launcher.sh`. Both launchers used the same copy of
`train_parametric.py`, whose only uncommitted change is `min_lr = 3e-7` → `3e-9` in the
early-stopping rule of `b984402`. That rule ends a run 10 epochs after its LR reaches
$3\times10^{-9}$. The positions `l1` seeds stop at epoch 107 – 140, and of the no-position
runs only `l1` seed 0 stops early (epoch 191). The no-penalty runs predate the rule and train all 200 epochs, but the
positions ones are at $3\times10^{-9}$ by epoch 111 – 128 and do not change after it (positions
section). The two budgets match where it matters. Early-stopped seeds are NaN past their last
epoch, so every "last" value below is each seed's own last epoch.

**Paired design.** Every penalised seed starts from the weights of the no-penalty seed with the
same number and the same node features (equal $c_0$, asserted below), and `FVGNN_SEED` also
seeds the shuffle. Within a pair only the penalty and its scale fix differ. So each penalised
configuration is compared with its own control seed by seed. The tables give the number of seeds
below the control and the two-sided Wilcoxon signed-rank $p$. With 5 pairs that $p$ is at
least $0.0625$, reached when all 5 pairs go the same way, so no comparison here can reach 0.05.
Hoyer is one pair, and no test.

- **Loss decay and PR**: training and validation MSE and the channel participation ratio, mean ±
  std over the seeds still training.
- **Rollout MAE** (plotted per step as RMAE), its mean over steps 1 – 10, the step at which each seed leaves the span of $T$,
  and the rollout RMAE, as in the sections above.
- **One-step RMAE / RMSE and rollout RMSE**, from the seed checkpoints (last-epoch weights).
- **Message vs flux, $R^2$**: `compute_model_flux_stats` per seed checkpoint.

The cells reuse `POS_CASES`, `POS_MESHES`, `POS_LOAD`, `POS_NODE_FEATS`, `pos_absT`, `pos_stamp`,
`pos_errors` and `pos_flux` from the positions section, `l1p_cell` from the section above,
`seed_stats`, `NS_SPAN`, `ns_seed_run` and `NS_FLUX_COLS` from the `nosched` sweep, and
`compute_reg_errors` and `compute_model_flux_stats` from the sections before.

In [ ]:
# --- Penalties under the scheduler: load the seeds, check them, summarise ---------
# Five configurations, each from one or more seeds.npz (aggregate_seeds.py), as
# (experiment, rerun index): 0 is seed_<n> in seeds.npz, i is seed_<n>_<ii> in
# seeds_<ii>.npz. The l1 positions sweep is split across two: its seed 0 was
# rerun after the first launcher was stopped mid-run (logs/20261014_112918).
from scipy.stats import wilcoxon

SP_BASE = "parametric_history1_msg_dim64_residual"
SP_SRC = {"pos, none":              [(f"{SP_BASE}_cf", 0)],
          "pos, l1 α = 0.1":        [(f"{SP_BASE}_l1_a0.1_cf", 0), (f"{SP_BASE}_l1_a0.1_cf", 1)],
          "no pos, none":           [(f"{SP_BASE}_cf_nopos", 0)],
          "no pos, l1 α = 0.1":     [(f"{SP_BASE}_l1_a0.1_cf_nopos", 0)],
          "no pos, hoyer α = 0.01": [(f"{SP_BASE}_hoyer_a0.01_cf_nopos", 0)]}
SP_KINDS = {"pos, none": ("none", 0.0), "pos, l1 α = 0.1": ("l1", 0.1),
            "no pos, none": ("none", 0.0), "no pos, l1 α = 0.1": ("l1", 0.1),
            "no pos, hoyer α = 0.01": ("hoyer", 0.01)}
# each penalised configuration -> the control it is paired with, seed by seed
SP_PAIRS = {"pos, l1 α = 0.1": "pos, none", "no pos, l1 α = 0.1": "no pos, none",
            "no pos, hoyer α = 0.01": "no pos, none"}
SP_MESHES = POS_MESHES
SP_EPOCHS = 200
SP_POS = {key: "positions" if key.startswith("pos") else "no positions" for key in SP_SRC}
# per-epoch fields, NaN-padded to SP_EPOCHS past each seed's n_epochs
SP_EPOCH_KEYS = ["train_losses", "val_losses", "lrs", "msg_pr", "msg_mean_abs", "msg_sigma",
                 "msg_grad_pen", "msg_grad_data"]
SP_SEED_KEYS = ["seeds", "n_params", "msg_c0", *(f"rollout_mae_{m}" for m in SP_MESHES)]


def sp_load(sources):
    """The seeds of one configuration from its seeds*.npz, sorted by seed, with
    the run path of each seed (for compute_reg_errors and friends)."""
    parts, runs = [], []
    for exp, i in sources:
        z = dict(np.load(processed_data_dir / exp / ("seeds.npz" if i == 0 else f"seeds_{i:02d}.npz")))
        S, E = z["train_losses"].shape
        assert list(z["test_meshes"]) == SP_MESHES, f"{exp}:{i}: test meshes {z['test_meshes']}"
        # n_params is stored once when every seed shares it
        part = {k: np.broadcast_to(z[k], (S,)) if k == "n_params" else z[k] for k in SP_SEED_KEYS}
        # runs from before early_stop.json have no n_epochs: they all ran E epochs
        part["n_epochs"] = z.get("n_epochs", np.full(S, E))
        for k in SP_EPOCH_KEYS:
            a = z[k].astype(float)
            pad = np.full((S, SP_EPOCHS - E, *a.shape[2:]), np.nan)
            part[k] = np.concatenate([a, pad], 1)
        part["meta"] = {k: z[k].item() for k in ("msg_kind", "msg_alpha", "msg_warmup", "msg_ramp_len")}
        parts.append(part)
        runs += [ns_seed_run(exp, f"{int(s)}_{i:02d}" if i else int(s)) for s in z["seeds"]]
    for p in parts[1:]:
        assert p["meta"] == parts[0]["meta"], f"{sources}: penalty settings differ between sweeps"
    out = {k: np.concatenate([p[k] for p in parts]) for k in parts[0] if k != "meta"}
    order = np.argsort(out["seeds"])
    out = {k: v[order] for k, v in out.items()}
    out.update(parts[0]["meta"])
    return out, [runs[i] for i in order]


sp, sp_runs = {}, {}
for key, src in SP_SRC.items():
    sp[key], sp_runs[key] = sp_load(src)

# Each configuration is what its name says, on one schedule and model ...
ref = sp["pos, none"]
for key, z in sp.items():
    kind, alpha = SP_KINDS[key]
    assert str(z["msg_kind"]) == kind and float(z["msg_alpha"]) == alpha, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(ref[field]), f"{key}: {field} differs"
    assert len(np.unique(z["seeds"])) == len(z["seeds"]), f"{key}: a seed twice: {z['seeds']}"
    assert (np.nan_to_num(z["lrs"], nan=z["lrs"][0, 0]) != z["lrs"][:, :1]).any(), \
        f"{key}: the LR never moved, not a scheduled run"
    if kind != "none":
        last_pr = z["msg_pr"][np.arange(len(z["seeds"])), z["n_epochs"] - 1]
        assert np.all(np.isfinite(last_pr)), f"{key}: PR missing at the last epoch"
n_drop = int(sp["pos, none"]["n_params"][0]) - int(sp["no pos, none"]["n_params"][0])
assert n_drop == 3 * 64, f"positions and none differ by {n_drop} parameters, not 3 x 64"
for key, z in sp.items():
    assert np.all(z["n_params"] == sp["pos, none" if SP_POS[key] == "positions" else "no pos, none"]["n_params"][0]), \
        f"{key}: not the model of its control"

# ... and each penalised seed starts from the weights of the control's seed with
# the same number (equal c0): within a pair only the penalty differs.
for key, ctrl in SP_PAIRS.items():
    z, c = sp[key], sp[ctrl]
    idx = np.searchsorted(c["seeds"], z["seeds"])
    assert np.array_equal(c["seeds"][idx], z["seeds"]), f"{key}: seeds {z['seeds']} not all in {ctrl}"
    assert np.array_equal(z["msg_c0"], c["msg_c0"][idx]), f"{key}: initial weights differ from {ctrl}"

# Per seed checkpoint: Cf graphs and the node features its normalizer was fitted
# on, and the commit it was trained with (for the markdown above).
for key, runs in sp_runs.items():
    commits = {}
    for s, (run, ok) in zip(sp[key]["seeds"], runs):
        ck = processed_data_dir / run / "checkpoints"
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg.get("excluded_patches") == EXCLUDED, \
            f"{run}: graphs built with {cfg}"
        if ok:
            n = torch.load(ck / "normalizer.pt")["node_mean"].numel()
            assert n == POS_NODE_FEATS[SP_POS[key]], f"{run}: normalizer fitted on {n} node features"
        c = load_git_commit(ck)
        tag = ("not recorded" if c is None or c["commit"] is None
               else c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else ""))
        commits.setdefault(tag, []).append(f"{run.split('/')[-1]}")
    print(f"{key:24s}: " + "; ".join(f"commit {tag} ({', '.join(rs)})" for tag, rs in commits.items()))

SP_WARM = int(ref["msg_warmup"])
SP_RAMP_END = SP_WARM + int(ref["msg_ramp_len"]) - 1


def sp_last(z, field):
    """Each seed's value at its own last epoch: early-stopped seeds are NaN past it."""
    return z[field][np.arange(len(z["seeds"])), z["n_epochs"] - 1]


def sp_paired(key, a, b):
    """'k/n' seeds where the penalised value a is below its control's b, and the
    two-sided Wilcoxon signed-rank p over the pairs (in log10 if log). With 5 pairs
    the smallest possible p is 0.0625; with fewer than 5 no p is given."""
    c = sp[SP_PAIRS[key]]
    b = np.asarray(b, dtype=float)[np.searchsorted(c["seeds"], sp[key]["seeds"])]
    a = np.asarray(a, dtype=float)
    k = f"{int((a < b).sum())}/{len(a)}"
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 5 or np.all(a[ok] == b[ok]):   # no test, or nothing to test
        return k, np.nan
    return k, float(wilcoxon(a[ok], b[ok]).pvalue)


# Rollout RMAE from the saved per-step MAE, as in the sections above; pos_absT
# holds the Cf fields these runs were tested on (the same with or without
# positions: only the node features differ).
sp_rmae_roll = {}
for key, z in sp.items():
    cols = []
    for mesh in SP_MESHES:
        mae = z[f"rollout_mae_{mesh}"].astype(float)
        r = mae.sum(1) / pos_absT[mesh][:mae.shape[1]].sum()
        cols.append(np.where(np.isfinite(r), r, np.inf))
    sp_rmae_roll[key] = np.stack(cols, 1)

print(f"\nwarmup {SP_WARM}, ramp complete (scheduler on) at epoch {SP_RAMP_END}, "
      f"at most {SP_EPOCHS} epochs\n")
sp_stats = {"epochs":         (lambda z: z["n_epochs"],                        "{:.0f}", False),
            "train, last":    (lambda z: sp_last(z, "train_losses"),            "{:.2e}", True),
            "val, last":      (lambda z: sp_last(z, "val_losses"),              "{:.2e}", True),
            "val, best":      (lambda z: np.nanmin(z["val_losses"], 1),         "{:.2e}", True),
            "best @":         (lambda z: np.nanargmin(z["val_losses"], 1),      "{:.0f}", False),
            "LR, last":       (lambda z: sp_last(z, "lrs"),                     "{:.0e}", True),
            "PR, last":       (lambda z: sp_last(z, "msg_pr"),                  "{:.1f}", False),
            "mean |m|, last": (lambda z: sp_last(z, "msg_mean_abs"),            "{:.3g}", False)}
W = 20
print(f"{'':24s}{'n':>3}" + "".join(f"{h:>{W}}" for h in sp_stats))
for key, z in sp.items():
    row = f"{key:24s}{len(z['seeds']):>3}"
    for h, (f, fmt, _) in sp_stats.items():
        v = np.asarray(f(z), dtype=float)
        if len(v) == 1:
            row += f"{fmt.format(v[0]):>{W}}"
        elif h == "epochs":   # a range: most seeds either stop early or run to the end
            row += f"{f'{v.min():.0f} – {v.max():.0f}':>{W}}"
        else:
            row += f"{fmt.format(v.mean()) + ' ± ' + fmt.format(v.std(ddof=1)):>{W}}"
    print(row)
    if key in SP_PAIRS:   # the penalised seeds against their own control seeds
        cells = [sp_paired(key, f(z), f(sp[SP_PAIRS[key]])) for f, _, _ in sp_stats.values()]
        print(f"{'  lower than control':24s}{'':>3}" + "".join(
            f"{k + ('' if np.isnan(p) else f', p {p:.3f}'):>{W}}" for k, p in cells))

# Rollout: how many seeds stay below the span of T, the MAE over steps 1-10
# (where every rollout is still a prediction), and the RMAE, as geometric means.
print(f"\nrollout, {ref[f'rollout_mae_{SP_MESHES[0]}'].shape[1]} steps: seeds below {NS_SPAN:.0f} K "
      f"at the end / MAE over steps 1-10 [K] / RMAE, geometric mean ×/÷ geometric std")
print(f"{'':24s}" + "".join(f"{mesh:>36}" for mesh in SP_MESHES) + f"{'below':>8}")
sp_early = {key: np.stack([z[f"rollout_mae_{m}"][:, :10].mean(1) for m in SP_MESHES], 1)
            for key, z in sp.items()}
for key, z in sp.items():
    final = np.stack([z[f"rollout_mae_{m}"][:, -1] for m in SP_MESHES], 1)
    below = np.isfinite(final) & (final < NS_SPAN)
    row = f"{key:24s}"
    for j in range(len(SP_MESHES)):
        mu_e, sd_e = seed_stats(sp_early[key][:, j], log=True)
        mu_r, sd_r = seed_stats(sp_rmae_roll[key][:, j], log=True)
        n_fin = int(np.isfinite(sp_rmae_roll[key][:, j]).sum())
        cell = (f"{int(below[:, j].sum())}/{len(below)} / {10**mu_e:.3g} / {10**mu_r:.1e}"
                + (f" ×/÷ {10**sd_r:.2g}" if len(below) > 1 else "")
                + (f" ({n_fin})" if n_fin < len(below) else ""))
        row += f"{cell:>36}"
    print(row + f"{f'{int(below.sum())}/{below.size}':>8}")
    if key in SP_PAIRS:
        c = SP_PAIRS[key]
        cells = [sp_paired(key, np.log10(sp_early[key][:, j]), np.log10(sp_early[c][:, j]))
                 for j in range(len(SP_MESHES))]
        print(f"{'  steps 1-10 lower':24s}" + "".join(
            f"{k + ('' if np.isnan(p) else f', p {p:.3f}'):>36}" for k, p in cells))

# The first step whose MAE is above the span (nan counts as above), per seed;
# "-" when the rollout never gets there.
print(f"\nrollout, first step with MAE above {NS_SPAN:.0f} K, per seed (- never)")
print(f"{'':24s}{'seed':>5}" + "".join(f"{mesh:>11}" for mesh in SP_MESHES))
for key, z in sp.items():
    for i, s in enumerate(z["seeds"]):
        row = f"{key if i == 0 else '':24s}{int(s):>5}"
        for mesh in SP_MESHES:
            out = ~(z[f"rollout_mae_{mesh}"][i] < NS_SPAN)
            row += f"{int(out.argmax()) + 1 if out.any() else '-':>11}"
        print(row)

In [ ]:
from matplotlib.ticker import NullLocator

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# No penalty dashed in PLOT2 as everywhere above, l1 in the dark blue of the
# nosched sweep, Hoyer in PLOT1. Rows: with and without positions.
SP_STYLES = {"none":             (PLOT2_COLOR, "--"),
             "l1 α = 0.1":       ("#1f4e79", "-"),
             "hoyer α = 0.01":   (PLOT1_COLOR, "-")}
SP_ROWS = {"positions": ["pos, none", "pos, l1 α = 0.1"],
           "no positions": ["no pos, none", "no pos, l1 α = 0.1", "no pos, hoyer α = 0.01"]}
PR_TICKS = [1, 2, 4, 8, 16, 32, 64]

fig, axs = plt.subplots(2, 3, figsize=(1800*px, 1000*px), sharex=True, constrained_layout=True)
for col in (0, 1):
    axs[1, col].sharey(axs[0, col])
axs[0, 1].sharey(axs[0, 0])

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Training, Message Penalties with LR Scheduler, Cf, msg_dim = 64, h = 1\n"
             "(mean ± std over the seeds still training)", fontsize=24, color=AX_COLOR)

epochs = np.arange(SP_EPOCHS)
for row, (pos_label, keys) in enumerate(SP_ROWS.items()):
    for col, (title, field) in enumerate((("Training Loss", "train_losses"),
                                          ("Validation Loss", "val_losses"),
                                          (r"Participation Ratio of $\sigma_c$", "msg_pr"))):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=15)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key in keys:
            color, ls = SP_STYLES[key.split(", ", 1)[1]]
            a = sp[key][field]
            n = len(a)
            # early-stopped seeds are NaN past their last epoch: the curve is over
            # the seeds still training, and ends where the last of them stops
            mu = np.nanmean(a, 0)
            label = key.split(", ", 1)[1] + (f" ({n} seeds)" if n > 1 else " (seed 0)")
            ax.plot(epochs, mu, label=label, color=color, linewidth=2.2, linestyle=ls)
            if n > 1:
                sd = np.nanstd(a, 0, ddof=1)
                # the band's lower edge is clipped: on a log axis mean - std <= 0
                # has no place to go
                ax.fill_between(epochs, np.clip(mu - sd, mu / 10, None), mu + sd,
                                color=color, alpha=0.15, linewidth=0)

        # Warmup ends where the penalty starts to ramp in, the ramp where it is at
        # full strength and the LR scheduler is allowed to step.
        for e in (SP_WARM - 1, SP_RAMP_END):
            ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))

        ax.set_yscale("log")
        ax.set_title(f"{title}, {pos_label}", fontsize=19)
        if row == 1:
            ax.set_xlabel("Epoch", fontsize=17)
        if col == 0:
            ax.set_ylabel("MSE Loss", fontsize=17)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

        legend = ax.legend(fontsize=13, loc="lower left" if col == 2 else "upper right")
        legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
        legend.get_frame().set_edgecolor(AX_COLOR)        # border color
        legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
        for text in legend.get_texts():
            text.set_color(AX_COLOR)                      # label text color

# PR spans 1 to 64: a base-2 axis gives every halving of the channel count the
# same height.
for ax in axs[:, 2]:
    ax.set_yscale("log", base=2)
    ax.set_yticks(PR_TICKS, labels=[str(k) for k in PR_TICKS])
    ax.yaxis.set_minor_locator(NullLocator())
    ax.set_ylim(1, 64)
    ax.set_ylabel("PR  [channels of 64]", fontsize=17)

# plt.savefig("../outputs/plots/parametric_history1_sched_penalty_training.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Rollout RMAE, from the MAE saved at training time. Rows: with and without positions;
# columns: the held-out meshes. Every seed as a thin line, the median over the
# seeds bold: a mean would be the one seed that diverges.
fig, axs = plt.subplots(2, len(SP_MESHES), figsize=(1800*px, 900*px), sharex=True, sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Rollout RMAE, Message Penalties with LR Scheduler, Cf, msg_dim = 64, h = 1\n"
             "(thin: seeds, bold: median)", fontsize=24, color=AX_COLOR)

for row, (pos_label, keys) in enumerate(SP_ROWS.items()):
    for col, mesh in enumerate(SP_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key in keys:
            color, ls = SP_STYLES[key.split(", ", 1)[1]]
            rel = rmae_curve(sp[key][f"rollout_mae_{mesh}"], param_T_stats(mesh)[0])
            steps = np.arange(1, rel.shape[1] + 1)
            for m in rel:
                ax.plot(steps, m, color=color, linewidth=0.8, alpha=0.35, linestyle=ls)
            # an overflowed step (nan) counts as above every finite one
            med = np.median(np.where(np.isfinite(rel), rel, np.inf), 0)
            ax.plot(steps, med, color=color, linewidth=2.5, linestyle=ls,
                    label=key.split(", ", 1)[1])
        ax.set_yscale("log")
        ax.set_ylim(2e-3, 2e5)

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        # (an error as large as the span at every step, in the same relative units)
        span_rel = rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0])
        ax.plot(steps, span_rel, color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, span_rel[0]), xytext=(4, 4),
                        textcoords="offset points", ha="left", va="bottom", fontsize=12,
                        color=AX_COLOR)
            ax.set_ylabel(f"RMAE, {pos_label}", fontsize=15)
            legend = ax.legend(fontsize=12, loc="upper left")
            legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
            legend.get_frame().set_edgecolor(AX_COLOR)        # border color
            legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
            for text in legend.get_texts():
                text.set_color(AX_COLOR)                      # label text color
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_sched_penalty_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- Penalties under the scheduler: one-step RMAE / RMSE and rollout RMSE ----------
# compute_reg_errors on each seed checkpoint, with the Cf graphs of POS_LOAD for
# its node features, cached and stamped with the model.pt each entry came from
# (pos_stamp). The positions control is the positions section's, whose entries
# are taken from its cache when their stamp still matches.
sp_errors_cache = processed_data_dir / "parametric_errors_sched_penalty.npz"

sp_absent = [run for runs in sp_runs.values() for run, ok in runs if not ok]
if sp_absent:
    print(f"{len(sp_absent)} seed checkpoints are not in {processed_data_dir}, left out of "
          f"one-step RMAE/RMSE and rollout RMSE:\n" + "\n".join(f"    {r}/checkpoints/model.pt" for r in sp_absent))

sp_errors = (dict(np.load(sp_errors_cache, allow_pickle=False))
             if sp_errors_cache.exists() else {})
for key, runs in sp_runs.items():
    for run, ok in runs:
        if ok and sp_errors.get(f"{run}__stamp") != pos_stamp(run) \
                and pos_errors.get(f"{run}__stamp") == pos_stamp(run):
            sp_errors.update({k: v for k, v in pos_errors.items() if k.startswith(f"{run}__")})
    stale = [run for run, ok in runs if ok and sp_errors.get(f"{run}__stamp") != pos_stamp(run)]
    if stale:
        sp_errors.update(compute_reg_errors(stale, load=POS_LOAD[SP_POS[key]]))
        sp_errors.update({f"{run}__stamp": pos_stamp(run) for run in stale})
        np.savez_compressed(sp_errors_cache, **sp_errors)


def sp_metric(key, metric, mode):
    """(S, M) array of one metric, nan where the seed's checkpoint is missing and
    inf where its rollout overflowed. Rollout RMAE is the one from the saved MAE."""
    if (metric, mode) == ("rmae", "roll"):
        return sp_rmae_roll[key]
    out = np.full((len(sp[key]["seeds"]), len(SP_MESHES)), np.nan)
    for i, (run, ok) in enumerate(sp_runs[key]):
        if ok:
            v = np.array([float(sp_errors[f"{run}__{mesh}__{metric}_{mode}"]) for mesh in SP_MESHES])
            out[i] = np.where(np.isfinite(v), v, np.inf)
    return out


# The recomputed rollouts have to be the ones saved at training time, over the
# steps where both are below the span of T.
devs = []
for key, runs in sp_runs.items():
    for i, (run, ok) in enumerate(runs):
        for mesh in SP_MESHES if ok else []:
            new, old = sp_errors[f"{run}__{mesh}__mae_roll"], sp[key][f"rollout_mae_{mesh}"][i]
            keep = (new < NS_SPAN) & (old < NS_SPAN)
            if keep.any():
                devs.append(np.abs(new[keep] / old[keep] - 1).max())
print(f"recomputed vs saved rollout MAE, steps below {NS_SPAN:.0f} K: "
      f"max |relative difference| = {max(devs):.1e}\n")

# Geometric mean ×/÷ geometric std over the seeds with a finite value, (n/m) as in
# the sections above; under each penalised configuration, the seeds below their
# own control and the Wilcoxon p (one-step only: a rollout can be inf).
cols = [("step", "rmae"), ("step", "rmse"), ("roll", "rmse")]
print(f"{'mesh':<11}{'run':<24}" + "".join(
    f"{h:>28}" for h in ("one-step RMAE", "one-step RMSE", "rollout RMSE")))
for j, mesh in enumerate(SP_MESHES):
    for key in sp:
        print(f"{mesh:<11}{key:<24}" + "".join(l1p_cell(sp_metric(key, metric, mode)[:, j])
                                             for mode, metric in cols))
        if key in SP_PAIRS:
            cells = []
            for mode, metric in cols:
                a, b = sp_metric(key, metric, mode)[:, j], sp_metric(SP_PAIRS[key], metric, mode)[:, j]
                k, p = sp_paired(key, np.log10(a), np.log10(b))
                cells.append(k + ("" if np.isnan(p) or mode == "roll" else f", p {p:.3f}"))
            print(f"{mesh:<11}{'  lower than control':<24}" + "".join(f"{c:>28}" for c in cells))
    print()

In [ ]:
# --- Penalties under the scheduler: message vs FV flux, per seed --------------------
# compute_model_flux_stats on each seed checkpoint, with the graphs of POS_LOAD:
# the same meshes, timesteps and seeded faces as every read-out above. Cached and
# stamped as the errors are, reusing the positions section's entries for its
# control. Every column is an R^2, a correlation or an energy fraction, so none
# depends on how large a run writes its message.
sp_flux_cache = processed_data_dir / "parametric_message_flux_sched_penalty.npz"

sp_flux = (dict(np.load(sp_flux_cache, allow_pickle=False))
           if sp_flux_cache.exists() else {})
for key, runs in sp_runs.items():
    for run, ok in runs:
        if ok and sp_flux.get(f"{run}__stamp") != pos_stamp(run) \
                and pos_flux.get(f"{run}__stamp") == pos_stamp(run):
            sp_flux.update({k: v for k, v in pos_flux.items() if k.startswith(f"{run}__")})
    stale = [run for run, ok in runs if ok and sp_flux.get(f"{run}__stamp") != pos_stamp(run)]
    for run in stale:
        sp_flux.update(compute_model_flux_stats(run, load=POS_LOAD[SP_POS[key]]))
        sp_flux[f"{run}__stamp"] = pos_stamp(run)
        np.savez_compressed(sp_flux_cache, **sp_flux)   # after every run: these are slow


def sp_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(sp[key]["seeds"]), len(SP_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, (run, ok) in enumerate(sp_runs[key]):
        if not ok:
            continue
        for j, mesh in enumerate(SP_MESHES):
            out[i, j] = [*sp_flux[f"{run}__{mesh}__r2"], float(sp_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(sp_flux[f"{run}__{mesh}__best_r"])),
                         sp_flux[f"{run}__{mesh}__anti"].mean()]
    return out


sp_flux_tab = {key: sp_flux_table(key) for key in sp}

# Arithmetic mean ± std over the seeds: these are bounded in [0, 1].
print(f"Linear read-out from the message, {N_STEPS} probed timesteps, mean ± std over seeds")
print(f"{'mesh':<11}{'run':<24}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
for j, mesh in enumerate(SP_MESHES):
    for key in sp:
        t = sp_flux_tab[key][:, j]
        t = t[~np.isnan(t[:, 0])]
        if len(t) > 1:
            cells = [f"{m:.3f} ± {s:.3f}" for m, s in zip(t.mean(0), t.std(0, ddof=1))]
        else:
            cells = [f"{m:.3f}" for m in t.mean(0)]
        print(f"{mesh:<11}{key:<24}{len(t):>3}" + "".join(f"{c:>15}" for c in cells))
        if key in SP_PAIRS:
            cells = [sp_paired(key, sp_flux_tab[key][:, j, c], sp_flux_tab[SP_PAIRS[key]][:, j, c])
                     for c in range(len(NS_FLUX_COLS))]
            print(f"{mesh:<11}{'  lower than control':<24}{'':>3}" + "".join(
                f"{k + ('' if np.isnan(p) else f', p {p:.2f}'):>15}" for k, p in cells))
    print()

# The "anti" column averages all 64 channels; under a penalty most of them carry
# next to nothing. The channel with the largest sigma_c at each seed's last
# epoch, read out on its own.
print("antisymmetric fraction of the channel with the largest sigma_c (last epoch), per seed")
for key in sp:
    if SP_KINDS[key][0] == "none":
        continue
    z = sp[key]
    for i, (run, ok) in enumerate(sp_runs[key]):
        top = int(np.argmax(z["msg_sigma"][i, z["n_epochs"][i] - 1]))
        print(f"{key if i == 0 else '':24s}seed {int(z['seeds'][i])}, channel {top:>2}: " + ", ".join(
            f"{sp_flux[f'{run}__{mesh}__anti'][top]:.3f}" for mesh in SP_MESHES))

### What comes out

All 21 seed checkpoints are here. The losses, the PR and the rollout MAE come from `seeds.npz`, and
the one-step errors, the rollout RMSE and the flux read-outs from `model.pt`, at each seed's
last-epoch weights. The rollouts recomputed from the checkpoints match the saved ones to
$9\times10^{-3}$ relative, over the steps below the span.

**`l1` costs validation loss, more without positions.** Best validation MSE rises from
$3.29 \pm 0.59$ to $4.08 \pm 0.64$ with positions and from $1.53 \pm 0.43$ to $4.13 \pm 0.59$
without (units of $10^{-5}$). Every penalised seed is above its own control in both
($0/5$, $p = 0.06$). Without positions the penalty takes away the advantage that dropping the
positions gave, and both settings end at the same validation loss. The training MSE without
positions ends $2.8\times$ higher ($2.98$ against $1.05\times10^{-4}$). The PR falls from 59 to
$41.2 \pm 3.0$ with positions and to $34.2 \pm 3.1$ without. Mean $|m|$ falls about sixfold,
which is the term the penalty acts on.

**`l1` makes no consistent change to rollout stability.** With positions almost nothing stays
bounded: 1 of 20 seed–mesh pairs without the penalty and 3 of 20 with it. Over steps 1 – 10 the
penalised seeds are worse (MAE 2.1 – 10 K against 1.8 – 6.9 K, $0 - 2/5$ seeds lower per mesh).
Seeds 0, 1 and 4 leave the span earlier than their control. Seed 2 leaves it later, and seed 3
stays bounded on three meshes where its control does not. Without positions, 11 of 20 pairs stay bounded against 8 of 20, and
the early MAE is lower for 4 of 5 seeds on every mesh (9 – 16 K against 12 – 20 K, $p \ge 0.19$).
That is a lean, not a result. Without positions the early MAE is higher (12 – 20 K against
2 – 7 K with them), but fewer runs diverge, with or without the penalty.

**One-step error: RMAE about the same, RMSE worse with `l1`.** One-step RMAE rises by at most
12 % ($0 - 2/5$ seeds lower). RMSE is higher on every mesh: 7 – 50 % with positions and 1.4 – 2.3
times without, with 0 of 5 seeds lower on six of the eight mesh–setting pairs and 1 of 5 on the other two.

**`l1` makes the message less flux-like.** The $R^2$ of the linear read-out of $F$ drops from
0.70 – 0.76 to 0.38 – 0.41 with positions and from 0.84 – 0.88 to 0.36 – 0.42 without. $\Delta T$
drops less, to 0.52 – 0.66, and the geometric weight $w$ most, to 0.12 – 0.42. All are 5/5 seeds
lower on every mesh. The node sum $\sum_j F_{ij}$ hardly moves (0.68 – 0.78 against
0.69 – 0.79). The antisymmetric energy fraction rises: from 0.21 – 0.23 to 0.42 – 0.43 with
positions and from 0.31 – 0.34 to 0.37 – 0.42 without. Its largest channel ranges from 0.12 to
0.85 across seeds. So `l1` makes the message more antisymmetric without making it more
like the flux.

**`hoyer` (one seed): one channel, a mild loss cost, and an early divergence.** Against its paired
control (seed 0 without positions):

- **PR.** PR falls from 58.7 at the end of warmup to 1.50 when the ramp completes and ends at
  1.39. Channel 38 holds 99.9 % of $\sum_c\sigma_c^2$ ($\sigma = 622$ against 5.1 for the next).
  Unlike the point-mean run above, the direction is not written twice.
- **Losses.** Best validation MSE is $2.37$ against $1.85\times10^{-5}$ (+28 %, and below both
  `l1` means). The last training MSE is $1.37$ against $1.02\times10^{-4}$.
- **One-step error** is close to the control. The RMAE is lower than the paired seed on all
  four meshes and the RMSE on three of four.
- **Rollout.** It diverges on every mesh. The MAE leaves the span at step 17 – 30 against 49 – 54 for the
  control. It is already higher over steps 1 – 10 (6.3 – 13.8 K against 3.8 – 9.5 K), and it
  grows geometrically to $10^{19} - 10^{24}$ K. The point-mean Hoyer run above, with positions, diverged
  the same way, so in these two runs the divergence does not depend on the boundary nodes or on
  the positions.
- **Flux read-out.** The full message keeps much of the flux, $R^2$ 0.74 – 0.81 against
  0.84 – 0.88, far above `l1`'s 0.36 – 0.42. But the dominant channel does not carry it. The
  single channel most correlated with $F$ is channel 34, one of the near-silent ones, with
  $|r| = 0.65 - 0.73$. The same happened in the point-mean run. Channel 38's antisymmetric
  fraction is 0.499 on all four meshes, which is what an $m_{ij}$ uncorrelated with $m_{ji}$
  gives. One reading is a channel that depends on only one end of the edge. This is not checked
  here.

**What this shows.** Under the scheduler, with Cf nodes, `l1` at $\alpha = 0.1$ costs one-step
accuracy and lowers the flux $R^2$ in every pair. Its effect on rollout stability is within the
seed-to-seed spread. Hoyer at $\alpha = 0.01$ compresses the message to one channel, as before, at a
small loss cost, but its rollout diverges earlier than its control. With one seed this is still
an observation, and Hoyer seeds 1 – 4 would make it a comparison. Without positions, the
no-penalty runs stay bounded in 8 of 20 seed–mesh pairs, against 1 of 20 with them. In these
runs, removing the positions changes rollout stability more than either penalty does. That
comparison is unpaired.

## Input noise for rollout stability (`FVGNN_NOISE`)

Every model so far was trained on one-step pairs (true $T^n$ → true $T^{n+1}$) and diverges in
rollout from some step on: the error stays small, then grows by a nearly constant factor per step.
In rollout the model is fed its own slightly wrong $T^n$, an input it never saw in training.
Following Sanchez-Gonzalez et al. (2020), these runs add Gaussian noise to the $T$ history of
every training sample and keep the target at the true $T^{n+1}$. The residual model adds its
increment to the noisy $T^n$, so it must learn to undo the perturbation. The noise std is
`FVGNN_NOISE` × `delta_scale` in normalised units: at `FVGNN_NOISE = 1` the input moves as much as
a typical one-step change $T^{n+1} - T^n$. Validation and test inputs are clean.

- **no noise** (`parametric_history1_msg_dim64_residual_cf_nopos`, commit `cf45013` +
  uncommitted changes): the 200-epoch `seed_<n>` sweep of the penalties section, seeds 0 – 4.
- **noise 0.1, 0.5** (same names, `seed_<n>` in `seeds.npz`, seeds 0 – 2, commit `f8f9112` +
  uncommitted changes): a later sweep, launcher log in `logs/20261016_113759`. The code is
  the same except for the budget, at most 500 epochs instead of 200. All six seeds early-stop,
  after 238 – 286 epochs at 0.1 and 125 – 243 at 0.5. The same sweep reran noise 0.3 with this
  budget into `seed_<n>_01` (`seeds_01.npz` once aggregated). That rerun is not used here.
- **noise 0.3, 1, 3** (`parametric_history1_msg_dim64_residual_noise<x>_cf_nopos`,
  `seed_<n>` in `seeds.npz`, seeds 0 – 2, commit `f8f9112`): launcher log in
  `logs/20261015_175729`.

Everything else is shared: the 64-channel residual model at `history = 1` with no LayerNorm and no
message penalty, Cf boundary nodes, node features $[\,\text{internal}, \text{boundary}\,]$, SOAP
from $3\times10^{-3}$ with `ReduceLROnPlateau` from epoch 29, `batch_size = 2`, at most 200 epochs (500 for noise 0.1 and 0.5).
The noise runs have the early-stopping rule (`min_lr` $= 3\times10^{-9}$); the control predates it
and trains all 200 epochs.

**Paired design.** The noise is drawn from the CUDA generator, so the CPU generator, which sets
the initial weights and every epoch's shuffle order, makes the same draws as in a noise-free run.
Seed $n$ of every noise level starts from the weights of control seed $n$ (equal $c_0$, asserted
below) and sees the same batches. The tables give the seeds below their control. With 3 seeds per
level no Wilcoxon $p$ is given: with 3 pairs its smallest value is 0.25.

**Two checkpoints per noise run.** The noise runs also roll out on the two val meshes for 40 steps
every epoch, and keep the epoch with the lowest mean MAE as `model_best_rollout.pt` next to the
last-epoch `model.pt`. Both are evaluated on the test meshes. The control has the last-epoch
model only, and its 40-step val rollout is computed here from that checkpoint.

- **Loss decay, LR and val rollout RMAE** per epoch.
- **Test rollout MAE** (plotted per step as RMAE): seeds below the span of $T$, MAE over steps 1 – 10, first step above the
  span, rollout RMAE, for both checkpoints.
- **One-step and rollout RMAE / RMSE** on the test meshes, as a table and a bar plot, and the
  val rollout, from both checkpoints.
- **Message vs flux, $R^2$**: `compute_model_flux_stats` on the last-epoch checkpoints.

The cells reuse `sp`, `sp_runs`, `sp_flux`, `SP_MESHES` and `SP_EPOCHS` from the penalties
section, `POS_LOAD`, `POS_NODE_FEATS`, `pos_absT` and `pos_stamp` from the positions section,
`l1p_cell`, `seed_stats`, `NS_SPAN`, `ns_seed_run` and `NS_FLUX_COLS` from the sections before.

In [ ]:
# --- Input noise: load the seeds, check them, summarise ---------------------------
# One seeds.npz per noise level (aggregate_seeds.py), against the 200-epoch
# no-position sweep without noise as the penalties section loaded it (sp, sp_runs).
# 0.1 and 0.5 come from a later sweep launched for 500 epochs.
NZ_BASE   = "parametric_history1_msg_dim64_residual"
NZ_LEVELS = [0.1, 0.3, 0.5, 1, 3]
NZ_CASES  = {f"noise {x:g}": f"{NZ_BASE}_noise{x:g}_cf_nopos" for x in NZ_LEVELS}
NZ_CTRL   = "no pos, none"     # key of the control in sp
NZ_MESHES = SP_MESHES
NZ_BUDGET = {0.1: 500, 0.3: SP_EPOCHS, 0.5: 500, 1: SP_EPOCHS, 3: SP_EPOCHS}  # max_epochs
NZ_LOAD   = POS_LOAD["no positions"]
# per-epoch fields, NaN-padded to NZ_EPOCHS, the longest seed of any level or the
# control (aggregate_seeds.py pads only to the longest seed of its own run)
NZ_EPOCH_KEYS = ["train_losses", "val_losses", "lrs", "val_rollout_mae"]

nz_no_npz = [exp for exp in NZ_CASES.values()
             if not (processed_data_dir / exp / "seeds.npz").exists()]
if nz_no_npz:
    raise FileNotFoundError(
        f"no seeds.npz in {', '.join(nz_no_npz)} yet: run the noise sweep of launcher.sh, "
        f"which aggregates at the end, or python aggregate_seeds.py <exp_name>")


def nz_pad(z, keys):
    """A copy of z with the per-epoch fields in keys NaN-padded to NZ_EPOCHS."""
    z = dict(z)
    for k in keys:
        a = z[k].astype(float)
        pad = np.full((a.shape[0], NZ_EPOCHS - a.shape[1], *a.shape[2:]), np.nan)
        z[k] = np.concatenate([a, pad], 1)
    return z


nz = {key: dict(np.load(processed_data_dir / exp / "seeds.npz")) for key, exp in NZ_CASES.items()}
NZ_EPOCHS = max(SP_EPOCHS, *(z["train_losses"].shape[1] for z in nz.values()))
for key, z in nz.items():
    nz[key] = nz_pad(z, NZ_EPOCH_KEYS)
    # stored once when every seed shares it
    nz[key]["n_params"] = np.broadcast_to(z["n_params"], z["seeds"].shape)
nz_runs = {key: [ns_seed_run(exp, int(s)) for s in nz[key]["seeds"]]
           for key, exp in NZ_CASES.items()}
nz_ctrl = nz_pad(sp[NZ_CTRL], ["train_losses", "val_losses", "lrs"])
nz_ctrl_runs = sp_runs[NZ_CTRL]
NZ_VAL_MESHES = [str(m) for m in nz[f"noise {NZ_LEVELS[0]:g}"]["val_meshes"]]
NZ_VAL_STEPS = nz[f"noise {NZ_LEVELS[0]:g}"]["val_rollout_mae"].shape[-1]

# Each level is what its name says, on the control's model, schedule and budget ...
for key, z in nz.items():
    x = float(key.split()[1])
    assert float(z["input_noise"]) == x, f"{key}: run says input_noise = {z['input_noise']}"
    assert str(z["msg_kind"]) == "none" and float(z["msg_alpha"]) == 0.0, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    assert int(z["max_epochs"]) == NZ_BUDGET[x], f"{key}: launched for {z['max_epochs']} epochs"
    assert list(z["test_meshes"]) == NZ_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert [str(m) for m in z["val_meshes"]] == NZ_VAL_MESHES, f"{key}: val meshes {z['val_meshes']}"
    assert np.all(z["n_params"] == nz_ctrl["n_params"][0]), f"{key}: not the control's model"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(nz_ctrl[field]), f"{key}: {field} differs from the control"
# ... and each seed starts from the weights of the control seed with its number.
    idx = np.searchsorted(nz_ctrl["seeds"], z["seeds"])
    assert np.array_equal(nz_ctrl["seeds"][idx], z["seeds"]), \
        f"{key}: seeds {z['seeds']} not all in the control"
    assert np.array_equal(z["msg_c0"], nz_ctrl["msg_c0"][idx]), \
        f"{key}: initial weights differ from the control's"

# Per seed checkpoint: Cf graphs, the node features its normalizer was fitted
# on, and the commit it was trained with (for the markdown above).
for key, runs in {NZ_CTRL: nz_ctrl_runs, **nz_runs}.items():
    commits = {}
    for run, ok in runs:
        ck = processed_data_dir / run / "checkpoints"
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg.get("excluded_patches") == EXCLUDED, \
            f"{run}: graphs built with {cfg}"
        if ok:
            n = torch.load(ck / "normalizer.pt")["node_mean"].numel()
            assert n == POS_NODE_FEATS["no positions"], f"{run}: normalizer fitted on {n} node features"
        c = load_git_commit(ck)
        tag = ("not recorded" if c is None or c["commit"] is None
               else c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else ""))
        commits.setdefault(tag, []).append(run.split("/")[-1])
    print(f"{key:14s}: " + "; ".join(f"commit {tag} ({', '.join(rs)})" for tag, rs in commits.items()))


def nz_last(z, field):
    """Each seed's value at its own last epoch: early-stopped seeds are NaN past it."""
    return z[field][np.arange(len(z["seeds"])), z["n_epochs"] - 1]


def nz_paired(key, a, b):
    """'k/n' seeds of a noise level where a is below its control seed's b."""
    b = np.asarray(b, dtype=float)[np.searchsorted(nz_ctrl["seeds"], nz[key]["seeds"])]
    return f"{int((np.asarray(a, dtype=float) < b).sum())}/{len(b)}"


def nz_val_roll(z):
    """(S, E) val rollout MAE [K], mean over the val meshes and steps."""
    return z["val_rollout_mae"].mean((2, 3))


print(f"\nnoise levels {NZ_LEVELS}, seeds {nz[f'noise {NZ_LEVELS[0]:g}']['seeds'].tolist()} "
      f"(control: {nz_ctrl['seeds'].tolist()}), at most "
      f"{', '.join(f'{b} ({x:g})' for x, b in NZ_BUDGET.items())} epochs; val rollout: "
      f"{NZ_VAL_STEPS} steps on {', '.join(NZ_VAL_MESHES)}\n")
nz_stats = {"epochs":      (lambda z: z["n_epochs"],                    "{:.0f}"),
            "train, last": (lambda z: nz_last(z, "train_losses"),        "{:.2e}"),
            "val, last":   (lambda z: nz_last(z, "val_losses"),          "{:.2e}"),
            "val, best":   (lambda z: np.nanmin(z["val_losses"], 1),     "{:.2e}"),
            "best @":      (lambda z: np.nanargmin(z["val_losses"], 1),  "{:.0f}"),
            "LR, last":    (lambda z: nz_last(z, "lrs"),                 "{:.0e}")}
W = 20
print(f"{'':16s}{'n':>3}" + "".join(f"{h:>{W}}" for h in nz_stats))
for key, z in {"no noise": nz_ctrl, **nz}.items():
    row = f"{key:16s}{len(z['seeds']):>3}"
    for h, (f, fmt) in nz_stats.items():
        v = np.asarray(f(z), dtype=float)
        if h == "epochs":
            row += f"{f'{v.min():.0f} – {v.max():.0f}':>{W}}"
        else:
            row += f"{fmt.format(v.mean()) + ' ± ' + fmt.format(v.std(ddof=1)):>{W}}"
    print(row)
    if key != "no noise":
        print(f"{'  below control':16s}{'':>3}" + "".join(
            f"{nz_paired(key, f(z), f(nz_ctrl)):>{W}}" for f, _ in nz_stats.values()))

# The val rollout as logged in training: its best epoch (that of
# model_best_rollout.pt), its best value and its value at the last epoch.
print(f"\nval rollout MAE [K], {NZ_VAL_STEPS} steps, mean over val meshes and steps, per seed")
print(f"{'':16s}{'seed':>5}{'best @':>9}{'best':>11}{'last':>11}")
for key, z in nz.items():
    vr = nz_val_roll(z)
    for i, s in enumerate(z["seeds"]):
        print(f"{key if i == 0 else '':16s}{int(s):>5}{int(z['val_rollout_best_epoch'][i]):>9}"
              f"{np.nanmin(vr[i]):>11.3g}{vr[i, z['n_epochs'][i] - 1]:>11.3g}")

# Test rollout of each checkpoint, from the saved MAE: seeds below the span of T
# at the end, MAE over steps 1-10, and the RMAE, as geometric means.
NZ_CKPTS = {"last": "rollout_mae_", "best": "rollout_mae_best_"}


def nz_rmae(mae, mesh):
    r = mae.astype(float).sum(1) / pos_absT[mesh][:mae.shape[1]].sum()
    return np.where(np.isfinite(r), r, np.inf)


nz_rows = [("no noise, last", nz_ctrl, "rollout_mae_", None)] + [
    (f"{key}, {ck}", z, prefix, key) for key, z in nz.items() for ck, prefix in NZ_CKPTS.items()]
print(f"\ntest rollout, {nz_ctrl[f'rollout_mae_{NZ_MESHES[0]}'].shape[1]} steps: seeds below "
      f"{NS_SPAN:.0f} K at the end / MAE over steps 1-10 [K] / "
      f"RMAE, geometric means over seeds")
print(f"{'':22s}" + "".join(f"{mesh:>30}" for mesh in NZ_MESHES) + f"{'below':>8}")
for label, z, prefix, key in nz_rows:
    final = np.stack([z[f"{prefix}{m}"][:, -1] for m in NZ_MESHES], 1)
    below = np.isfinite(final) & (final < NS_SPAN)
    row = f"{label:22s}"
    for j, mesh in enumerate(NZ_MESHES):
        mae = z[f"{prefix}{mesh}"]
        mu_e, _ = seed_stats(mae[:, :10].mean(1), log=True)
        mu_r, _ = seed_stats(nz_rmae(mae, mesh), log=True)
        n_fin = int(np.isfinite(nz_rmae(mae, mesh)).sum())
        row += f"{f'{int(below[:, j].sum())}/{len(below)} / {10**mu_e:.3g} / {10**mu_r:.1e}' + (f' ({n_fin})' if n_fin < len(below) else ''):>30}"
    print(row + f"{f'{int(below.sum())}/{below.size}':>8}")
    if key is not None:
        print(f"{'  steps 1-10 below':22s}" + "".join(
            f"{nz_paired(key, z[f'{prefix}{m}'][:, :10].mean(1), nz_ctrl[f'rollout_mae_{m}'][:, :10].mean(1)):>30}"
            for m in NZ_MESHES))

# The first step whose MAE is above the span (nan counts as above), per seed;
# "-" when the rollout never gets there.
print(f"\ntest rollout, first step with MAE above {NS_SPAN:.0f} K, per seed (- never)")
print(f"{'':22s}{'seed':>5}" + "".join(f"{mesh:>11}" for mesh in NZ_MESHES))
for label, z, prefix, _ in nz_rows:
    for i, s in enumerate(z["seeds"]):
        row = f"{label if i == 0 else '':22s}{int(s):>5}"
        for mesh in NZ_MESHES:
            out = ~(z[f"{prefix}{mesh}"][i] < NS_SPAN)
            row += f"{int(out.argmax()) + 1 if out.any() else '-':>11}"
        print(row)

# Line styles / curves of this section, also used by the sections below.
# No noise dashed in PLOT2 as everywhere above; the noise levels one hue, darker
# with more noise.
NZ_STYLES = {"no noise":  (PLOT2_COLOR, "--"),
             "noise 0.1": ("#c4b2de", "-"),
             "noise 0.3": ("#a68dc7", "-"),
             "noise 0.5": ("#8a6cb5", "-"),
             "noise 1":   ("#6c4c9b", "-"),
             "noise 3":   ("#3f2560", "-")}


In [ ]:
# --- Input noise: one-step and rollout errors, and val rollout of each checkpoint -
# compute_reg_errors evaluates checkpoints/model.pt only; here both checkpoints of
# the noise runs are needed, so this is what it computes (one-step and test
# rollout RMAE / RMSE, summed over every step and node) plus a val rollout of the
# length the runs logged. Cached per checkpoint file and stamped with its mtime.
# The control is evaluated the same way, at its last epoch.
nz_eval_cache = processed_data_dir / "parametric_eval_noise.npz"
NZ_FILES = {"last": "model.pt", "best": "model_best_rollout.pt"}


def nz_stamp(run, fname):
    return np.int64((processed_data_dir / run / "checkpoints" / fname).stat().st_mtime_ns)


def nz_evaluate(run, fname):
    """One-step and rollout RMAE / RMSE on the test meshes and the val rollout
    MAE per step of one checkpoint file, keyed <run>__<fname>__<mesh>__<metric>."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / fname, map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in NZ_MESHES + NZ_VAL_MESHES:
        graph, T_seq = NZ_LOAD(mesh)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        if mesh in NZ_MESHES:
            sums = torch.zeros(4, dtype=torch.float64)   # |e|, |T|, e^2, T^2
            with torch.no_grad():
                for i in range(len(ds)):
                    d = ds[i].to(device)
                    T_p = norm.inverse_transform_T(model(d).cpu()).double()
                    T_t = norm.inverse_transform_T(d.y.cpu()).double()
                    e = T_p - T_t
                    sums += torch.stack([e.abs().sum(), T_t.abs().sum(),
                                         e.pow(2).sum(), T_t.pow(2).sum()])
            out[f"{run}__{fname}__{mesh}__rmae_step"] = float(sums[0] / sums[1])
            out[f"{run}__{fname}__{mesh}__rmse_step"] = float(sums[2] / sums[3])
            # the whole test rollout, as compute_reg_errors: overflowed is nan
            T_p, T_t = rollout(model, ds, device=device)
            e, T_t = (T_p - T_t).double(), T_t.double()
            out[f"{run}__{fname}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_t.abs().sum())
            out[f"{run}__{fname}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_t.pow(2).sum())
        if mesh in NZ_VAL_MESHES:
            T_p, T_t = rollout(model, ds, device=device, n_steps=NZ_VAL_STEPS)
            out[f"{run}__{fname}__{mesh}__val_roll"] = (T_p - T_t).abs().mean(1).numpy()
        del model
    return out


# (label, z, run list, checkpoint) for every row of the tables below
nz_ckpt_rows = [("no noise, last", NZ_CTRL, "last")] + [
    (f"{key}, {ck}", key, ck) for key in nz for ck in NZ_FILES]


def nz_runs_of(key):
    return nz_ctrl_runs if key == NZ_CTRL else nz_runs[key]


def nz_z(key):
    return nz_ctrl if key == NZ_CTRL else nz[key]


def nz_has(key, i, ck):
    """Whether seed i of key has checkpoint ck here (no best one if every val
    rollout overflowed)."""
    run, ok = nz_runs_of(key)[i]
    return ok and (ck == "last" or int(nz_z(key)["val_rollout_best_epoch"][i]) >= 0)


nz_eval = (dict(np.load(nz_eval_cache, allow_pickle=False))
           if nz_eval_cache.exists() else {})
todo = [(run, NZ_FILES[ck]) for _, key, ck in nz_ckpt_rows
        for i, (run, ok) in enumerate(nz_runs_of(key))
        if nz_has(key, i, ck)
        and (nz_eval.get(f"{run}__{NZ_FILES[ck]}__stamp") != nz_stamp(run, NZ_FILES[ck])
             # entries cached before the rollout RMAE / RMSE were added
             or f"{run}__{NZ_FILES[ck]}__{NZ_MESHES[0]}__rmse_roll" not in nz_eval)]
for run, fname in todo:
    nz_eval.update(nz_evaluate(run, fname))
    nz_eval[f"{run}__{fname}__stamp"] = nz_stamp(run, fname)
if todo:
    np.savez_compressed(nz_eval_cache, **nz_eval)


def nz_metric(key, ck, metric, meshes):
    """(S, M) one metric of checkpoint ck, nan where a seed lacks it; a val
    rollout is reduced to its mean over the steps."""
    out = np.full((len(nz_z(key)["seeds"]), len(meshes)), np.nan)
    for i, (run, ok) in enumerate(nz_runs_of(key)):
        if nz_has(key, i, ck):
            v = [np.mean(nz_eval[f"{run}__{NZ_FILES[ck]}__{m}__{metric}"]) for m in meshes]
            out[i] = np.where(np.isfinite(v), v, np.inf)
    return out


# The recomputed val rollout of each last-epoch noise checkpoint has to be the one
# logged at its last epoch in training, over the steps below the span.
devs = []
for key, z in nz.items():
    for i, (run, ok) in enumerate(nz_runs[key]):
        if not ok:
            continue
        for v, mesh in enumerate(NZ_VAL_MESHES):
            new = nz_eval[f"{run}__model.pt__{mesh}__val_roll"]
            old = z["val_rollout_mae"][i, z["n_epochs"][i] - 1, v]
            keep = (new < NS_SPAN) & (old < NS_SPAN)
            if keep.any():
                devs.append(np.abs(new[keep] / old[keep] - 1).max())
if devs:
    print(f"recomputed vs logged last-epoch val rollout MAE, steps below {NS_SPAN:.0f} K: "
          f"max |relative difference| = {max(devs):.1e}")

# ... and the recomputed test rollout RMAE the one from the MAE saved at training
# time (nz_rmae), where both are finite.
devs = []
for label, key, ck in nz_ckpt_rows:
    for j, mesh in enumerate(NZ_MESHES):
        new = nz_metric(key, ck, "rmae_roll", NZ_MESHES)[:, j]
        old = nz_rmae(nz_z(key)[f"{NZ_CKPTS[ck]}{mesh}"], mesh)
        ok = np.isfinite(new) & np.isfinite(old)
        if ok.any():
            devs.append(np.abs(new[ok] / old[ok] - 1).max())
print(f"recomputed vs saved test rollout RMAE: max |relative difference| = {max(devs):.1e}\n")

# Geometric mean ×/÷ geometric std over the seeds, (n/m) as in the sections above,
# and under each noise row the seeds below their own control seed.
NZ_ERR_COLS = {"rmae_step": "one-step RMAE", "rmse_step": "one-step RMSE",
               "rmae_roll": "rollout RMAE", "rmse_roll": "rollout RMSE"}
print(f"{'mesh':<11}{'run':<22}" + "".join(f"{h:>28}" for h in NZ_ERR_COLS.values()))
for j, mesh in enumerate(NZ_MESHES):
    for label, key, ck in nz_ckpt_rows:
        print(f"{mesh:<11}{label:<22}" + "".join(
            l1p_cell(nz_metric(key, ck, m, NZ_MESHES)[:, j]) for m in NZ_ERR_COLS))
        if key != NZ_CTRL:
            print(f"{mesh:<11}{'  below control':<22}" + "".join(
                f"{nz_paired(key, nz_metric(key, ck, m, NZ_MESHES)[:, j], nz_metric(NZ_CTRL, 'last', m, NZ_MESHES)[:, j]):>28}"
                for m in NZ_ERR_COLS))
    print()

print(f"val rollout MAE [K], {NZ_VAL_STEPS} steps, mean over the steps")
print(f"{'':22s}" + "".join(f"{m:>28}" for m in NZ_VAL_MESHES))
for label, key, ck in nz_ckpt_rows:
    vr = nz_metric(key, ck, "val_roll", NZ_VAL_MESHES)
    print(f"{label:22s}" + "".join(l1p_cell(vr[:, v]) for v in range(len(NZ_VAL_MESHES))))
    if key != NZ_CTRL:
        print(f"{'  below control':22s}" + "".join(
            f"{nz_paired(key, vr[:, v], nz_metric(NZ_CTRL, 'last', 'val_roll', NZ_VAL_MESHES)[:, v]):>28}"
            for v in range(len(NZ_VAL_MESHES))))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

NZ_ALL = {"no noise": nz_ctrl, **nz}

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 1000*px), sharex=True, constrained_layout=True)
axs[0, 1].sharey(axs[0, 0])

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Training with Input Noise, No Positions, Cf, msg_dim = 64, h = 1\n"
             "(losses: mean ± std over the seeds still training; LR, val rollout: every seed)",
             fontsize=22, color=AX_COLOR)

epochs = np.arange(NZ_EPOCHS)
for ax, (title, field) in zip(axs[0], (("Training Loss", "train_losses"),
                                       ("Validation Loss", "val_losses"))):
    for key, z in NZ_ALL.items():
        color, ls = NZ_STYLES[key]
        mu, sd = np.nanmean(z[field], 0), np.nanstd(z[field], 0, ddof=1)
        ax.plot(epochs, mu, label=f"{key} ({len(z['seeds'])} seeds)", color=color,
                linewidth=2.2, linestyle=ls)
        # the band's lower edge is clipped: on a log axis mean - std <= 0 has no
        # place to go
        ax.fill_between(epochs, np.clip(mu - sd, mu / 10, None), mu + sd,
                        color=color, alpha=0.15, linewidth=0)
    ax.set_title(title, fontsize=20)
    ax.set_ylabel("MSE Loss", fontsize=17)

ax = axs[1, 0]
for key, z in NZ_ALL.items():
    color, ls = NZ_STYLES[key]
    for i, lr in enumerate(z["lrs"]):
        ax.plot(epochs, lr, color=color, linewidth=1.4, linestyle=ls, alpha=0.7,
                label=key if i == 0 else None)
ax.set_title("Learning Rate", fontsize=20)
ax.set_ylabel("LR", fontsize=17)

# The val rollout every epoch as RMAE over its steps (sum of |error| over steps
# and nodes over the same sum of |T_true|) per val mesh, then the mean over the
# val meshes; noise runs only. The control has its last-epoch value alone
# (computed above), drawn as its geometric mean over the seeds. The marker is
# each seed's best epoch, model_best_rollout.pt, which training picked on the
# mean MAE.
nz_val_absT = np.stack([param_T_stats(m)[0][:NZ_VAL_STEPS] for m in NZ_VAL_MESHES])  # (V, K)
ax = axs[1, 1]
for key, z in nz.items():
    color, ls = NZ_STYLES[key]
    vr = (z["val_rollout_mae"].sum(-1) / nz_val_absT.sum(-1)).mean(-1)   # (S, E)
    for i, v in enumerate(vr):
        ax.plot(epochs, v, color=color, linewidth=1.2, linestyle=ls, alpha=0.8,
                label=key if i == 0 else None)
        b = int(z["val_rollout_best_epoch"][i])
        if b >= 0:
            ax.plot(b, v[b], marker="o", markersize=6, color=color)
# nz_metric gives the mean MAE over the steps: over the mean |T_true|, the RMAE
ctrl_vr = (nz_metric(NZ_CTRL, "last", "val_roll", NZ_VAL_MESHES) / nz_val_absT.mean(-1)).mean(1)
ax.axhline(10 ** seed_stats(ctrl_vr, log=True)[0], color=PLOT2_COLOR, linewidth=2,
           linestyle="--", label="no noise, last epoch")
ax.set_title(f"Val Rollout RMAE, {NZ_VAL_STEPS} Steps (o: best)", fontsize=20)
ax.set_ylabel("RMAE", fontsize=17)

for ax in axs.ravel():
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=15)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_yscale("log")
    # warmup and ramp end: the scheduler starts stepping at the second line
    for e in (SP_WARM - 1, SP_RAMP_END):
        ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    legend = ax.legend(fontsize=13, loc="lower left" if ax is axs[1, 0] else "upper right")
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color
for ax in axs[1]:
    ax.set_xlabel("Epoch", fontsize=17)

# plt.savefig("../outputs/plots/parametric_history1_noise_training.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Test rollout RMAE, from the MAE saved at training time. Rows: the last-epoch and the
# best-rollout checkpoint of the noise runs (the no-noise control hidden);
# columns: the held-out meshes. Every seed thin, the median bold: a mean
# would be the one seed that diverges.
fig, axs = plt.subplots(2, len(NZ_MESHES), figsize=(1800*px, 900*px), sharex=True, sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Test Rollout RMAE with Input Noise, No Positions, Cf, msg_dim = 64, h = 1\n"
             "(thin: seeds, bold: median)", fontsize=24, color=AX_COLOR)

for row, (ck, prefix) in enumerate(NZ_CKPTS.items()):
    for col, mesh in enumerate(NZ_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, z in NZ_ALL.items():
            if key == "no noise":
                continue
            color, ls = NZ_STYLES[key]
            rel = rmae_curve(z[f"{'rollout_mae_' if key == 'no noise' else prefix}{mesh}"],
                             param_T_stats(mesh)[0])
            steps = np.arange(1, rel.shape[1] + 1)
            for m in rel:
                ax.plot(steps, m, color=color, linewidth=0.8, alpha=0.35, linestyle=ls)
            # an overflowed step (nan) counts as above every finite one
            med = np.median(np.where(np.isfinite(rel), rel, np.inf), 0)
            ax.plot(steps, med, color=color, linewidth=2.5, linestyle=ls,
                    label=key + (", last epoch" if key == "no noise" else ""))
        ax.set_yscale("log")
        ax.set_ylim(2e-3, 2e0)

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        # (an error as large as the span at every step, in the same relative units)
        span_rel = rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0])
        ax.plot(steps, span_rel, color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, span_rel[0]), xytext=(4, 4),
                        textcoords="offset points", ha="left", va="bottom", fontsize=12,
                        color=AX_COLOR)
            ax.set_ylabel(f"RMAE, {ck} checkpoint", fontsize=15)
            legend = ax.legend(fontsize=12, loc="center right")
            legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
            legend.get_frame().set_edgecolor(AX_COLOR)        # border color
            legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
            for text in legend.get_texts():
                text.set_color(AX_COLOR)                      # label text color
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_noise_rollout.png", dpi=300, transparent = True)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The errors table above as a figure, laid out as in the sections before. Rows:
# one-step and rollout; columns: RMAE and RMSE. Per mesh, the control's last
# epoch, then each noise level's last-epoch (solid) and best-rollout (hatched)
# checkpoint. Each bar is the geometric mean over the seeds, the whisker ×/÷ one
# geometric std, the dots the seeds themselves. The rollout panels stop at
# NZ_ROLL_TOP: a relative error of 1 is already an error as large as the field,
# and the diverged control seeds would stretch the axis over twenty decades. A
# seed above the top of its panel, or overflowed, is drawn as a triangle there,
# and a bar that would pass the top is cut off by it.
NZ_ROLL_TOP = 10.0
nz_bars = [(label, key, ck, NZ_STYLES["no noise" if key == NZ_CTRL else key][0])
           for label, key, ck in nz_ckpt_rows]
x = np.arange(len(NZ_MESHES))
width = 0.8 / len(nz_bars)

fig, axs = plt.subplots(2, 2, figsize=(1600*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE with Input Noise, No Positions, Cf, msg_dim = 64, h = 1 "
             "(geometric mean ×/÷ std)", fontsize=22, color=AX_COLOR)

for ax, metric in zip(axs.ravel(), NZ_ERR_COLS):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_title(NZ_ERR_COLS[metric].replace(" ", "  ·  ", 1), fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(NZ_MESHES, fontsize=13)

    vals = [nz_metric(key, ck, metric, NZ_MESHES) for _, key, ck, _ in nz_bars]
    finite = np.concatenate([v[np.isfinite(v)] for v in vals])
    # bars on a log axis grow from the bottom of the axis, not from 0, so the
    # floor has to be set before they are drawn
    top = NZ_ROLL_TOP if metric.endswith("_roll") else np.inf
    shown = finite[finite <= top]
    floor, hi = shown.min() / 10, min(shown.max() * 10, top)
    ax.set_yscale("log")
    ax.set_ylim(floor, hi)
    for i, ((label, key, ck, color), v_all) in enumerate(zip(nz_bars, vals)):
        off = (i - (len(nz_bars) - 1) / 2) * width
        mu, sd = seed_stats(v_all, log=True)               # (M,) each
        has = np.isfinite(mu)
        ax.bar((x + off)[has], (10 ** mu - floor)[has], width * 0.92, bottom=floor,
               color=color, alpha=0.55 if ck == "best" else 1.0,
               hatch="//" if ck == "best" else None, edgecolor=AX_COLOR, linewidth=0.4)
        ax.errorbar((x + off)[has], 10 ** mu[has],
                    yerr=[10 ** mu[has] - 10 ** (mu - sd)[has], 10 ** (mu + sd)[has] - 10 ** mu[has]],
                    fmt="none", ecolor=AX_COLOR, elinewidth=1.1, capsize=2)
        for j in range(len(NZ_MESHES)):
            v = v_all[:, j]
            ok = np.isfinite(v) & (v <= hi)
            ax.plot(np.full(ok.sum(), x[j] + off), v[ok], "o", color=AX_COLOR,
                    markersize=2.6, alpha=0.7)
            if (~ok & ~np.isnan(v)).any():   # above the top, or overflowed (inf)
                ax.plot(x[j] + off, hi, "v", color=color, markersize=7,
                        markeredgecolor=AX_COLOR, clip_on=False)

    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure: a colour per configuration, the hatch for the
# best-rollout checkpoint, and the triangle.
handles = [Patch(color=NZ_STYLES[k][0]) for k in NZ_STYLES] + [
    Patch(facecolor="white", edgecolor=AX_COLOR, hatch="//"),
    Line2D([], [], marker="v", linestyle="none", color="white", markeredgecolor=AX_COLOR,
           markersize=8)]
labels = [k + (", last epoch" if k == "no noise" else "") for k in NZ_STYLES] + [
    "best-rollout checkpoint", "seed above the top or overflowed"]
legend = fig.legend(handles, labels, loc="outside lower center", ncols=(len(handles) + 1) // 2, fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_noise_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- Input noise: message vs FV flux, per seed -------------------------------------
# compute_model_flux_stats on the last-epoch checkpoints (it reads model.pt), with
# the graphs of NZ_LOAD; the control's entries come from the penalties section's
# cache when their stamp still matches. Every column is an R^2, a correlation or
# an energy fraction.
nz_flux_cache = processed_data_dir / "parametric_message_flux_noise.npz"

nz_flux = (dict(np.load(nz_flux_cache, allow_pickle=False))
           if nz_flux_cache.exists() else {})
for key in [NZ_CTRL, *nz]:
    for run, ok in nz_runs_of(key):
        if not ok or nz_flux.get(f"{run}__stamp") == pos_stamp(run):
            continue
        if sp_flux.get(f"{run}__stamp") == pos_stamp(run):
            nz_flux.update({k: v for k, v in sp_flux.items() if k.startswith(f"{run}__")})
        else:
            nz_flux.update(compute_model_flux_stats(run, load=NZ_LOAD))
            nz_flux[f"{run}__stamp"] = pos_stamp(run)
        np.savez_compressed(nz_flux_cache, **nz_flux)   # after every run: these are slow


def nz_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(nz_z(key)["seeds"]), len(NZ_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, (run, ok) in enumerate(nz_runs_of(key)):
        if not ok:
            continue
        for j, mesh in enumerate(NZ_MESHES):
            out[i, j] = [*nz_flux[f"{run}__{mesh}__r2"], float(nz_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(nz_flux[f"{run}__{mesh}__best_r"])),
                         nz_flux[f"{run}__{mesh}__anti"].mean()]
    return out


nz_flux_tab = {key: nz_flux_table(key) for key in [NZ_CTRL, *nz]}

# Arithmetic mean ± std over the seeds: these are bounded in [0, 1].
print(f"Linear read-out from the message, {N_STEPS} probed timesteps, last-epoch checkpoints, "
      f"mean ± std over seeds")
print(f"{'mesh':<11}{'run':<16}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
for j, mesh in enumerate(NZ_MESHES):
    for key in [NZ_CTRL, *nz]:
        t = nz_flux_tab[key][:, j]
        t = t[~np.isnan(t[:, 0])]
        cells = ([f"{m:.3f} ± {s:.3f}" for m, s in zip(t.mean(0), t.std(0, ddof=1))]
                 if len(t) > 1 else [f"{m:.3f}" for m in t.mean(0)])
        print(f"{mesh:<11}{'no noise' if key == NZ_CTRL else key:<16}{len(t):>3}"
              + "".join(f"{c:>15}" for c in cells))
        if key != NZ_CTRL:
            print(f"{mesh:<11}{'  below control':<16}{'':>3}" + "".join(
                f"{nz_paired(key, nz_flux_tab[key][:, j, c], nz_flux_tab[NZ_CTRL][:, j, c]):>15}"
                for c in range(len(NS_FLUX_COLS))))
    print()

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Message vs FV flux, no noise against noise 0.3, laid out as in the positions
# sections: the three edge-level read-outs plus the node-level one, per held-out
# mesh; bar = mean over the seeds with a checkpoint, whisker = ± one std, dots =
# the seeds (5 for the control, 3 for noise 0.3, paired with control seeds 0 – 2).
# Last-epoch checkpoints, from nz_flux_tab.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.3
nz_r2_rows = [(NZ_CTRL, "no noise"), ("noise 0.3", "noise 0.3")]

fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs FV Flux, No Noise vs Noise 0.3, No Positions, Cf "
             "(mean ± std over seeds)", fontsize=24, color=AX_COLOR)

for j, (ax, mesh) in enumerate(zip(axs.ravel(), NZ_MESHES)):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, (key, label) in enumerate(nz_r2_rows):
        color = NZ_STYLES[label][0]
        t = nz_flux_tab[key][:, j, :4]                       # (S, 4) R^2
        t = t[~np.isnan(t[:, 0])]
        off = (i - (len(nz_r2_rows) - 1) / 2) * (width + 0.02)
        sd = t.std(0, ddof=1) if len(t) > 1 else None
        ax.bar(x + off, t.mean(0), width, color=color,
               label=f"{label} ({len(t)} seeds)" + "",
               yerr=sd, error_kw=dict(ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5))
        for row in t:
            ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.8, alpha=0.7)

    ax.set_title(mesh, fontsize=19)
    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
    ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                    loc="outside lower center", ncols=len(nz_r2_rows), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_noise_flux_r2.png", dpi=300, transparent = True)

### What comes out

All 15 noise checkpoints and the 5 control seeds are here. The losses, the LR and the logged val
rollout come from `seeds.npz`. The one-step errors, the rollout RMSE and the recomputed val rollout
come from the checkpoints, and the flux read-outs from the last-epoch `model.pt`. The recomputed
val rollouts match the ones logged at the last epoch to $6\times10^{-7}$ relative, and the
recomputed test rollout RMAE matches the one from the saved MAE to $6\times10^{-3}$. Seed $n$ of
each level is paired with control seed $n$, so control seeds 3 and 4 have no noise partner.
Noise 0.1 and 0.5 had a 500-epoch budget, against 200 for the control and the other levels. Noise
0.1 used it: its best validation epoch is $219 \pm 31$ and it stops after 239 – 287 epochs.
Noise 0.5 stops after 126 – 244 epochs, one seed beyond 200.

**From noise 0.3 up, every run stays bounded. At 0.1 one seed does not.** At 0.3, 0.5, 1 and 3,
from both checkpoints, every seed stays below the 300 K span on all four test meshes for all 120
steps (12 of 12 seed–mesh pairs per row). At noise 0.1, seeds 0 and 1 stay bounded everywhere.
Seed 2 leaves the span on `model_003`, `model_007` and `model_010` at step 16 – 27 and overflows,
from both checkpoints (9 of 12). That is earlier than any control seed, whose earliest exit is at
step 28. The control stays bounded in 8 of 20 pairs. Its seed 3 is bounded on every mesh, seeds 0
and 2 leave the span on every mesh (at step 28 – 54), and seeds 1 and 4 on three meshes and one.
None of the bounded noise rollouts grows geometrically. The per-step RMAE starts at about
$5\times10^{-3}$ – $6\times10^{-2}$, falls or flattens over the first 5 – 20 steps and then rises
slowly. It ends at $3\times10^{-3}$ – $5\times10^{-2}$ at step 120, against a span of about
0.6 – 1 in the same units.

**The early rollout improves at every level.** The MAE over steps 1 – 10 is 12 – 19.5 K for the
control. At the last epoch it is 2.3 – 6.4 K at noise 0.1, 1.6 – 4.6 K at 0.3, 2.2 – 6.0 K at
0.5, 3.2 – 7.7 K at 1 and 4.5 – 10.9 K at 3. Every noise seed is below its control seed on every
mesh, noise 0.1 seed 2 included, except at noise 3 (2 of 3 on three meshes). On the val meshes
the 40-step mean MAE is 135 – 147 K for the control and 0.9 – 7 K for the noise runs, with every
seed lower. Among the bounded seeds the 120-step test rollout RMAE is $3.3\times10^{-3}$ –
$9.8\times10^{-3}$ at noise 0.1, $3.7\times10^{-3}$ – $1.3\times10^{-2}$ at 0.3,
$4.3\times10^{-3}$ – $1.5\times10^{-2}$ at 0.5, $5.1\times10^{-3}$ – $1.8\times10^{-2}$ at 1 and
$7.3\times10^{-3}$ – $2.6\times10^{-2}$ at 3. The control's geometric mean is 78 – 152, set by
its diverging seeds.

**The cost is one-step accuracy, and it grows with the noise.** Validation inputs are clean. Best
validation MSE is $1.53 \pm 0.43$ for the control, $1.82 \pm 0.82$ at noise 0.1,
$3.79 \pm 2.21$ at 0.3, $7.66 \pm 2.59$ at 0.5, $18.3 \pm 0.7$ at 1 and $94.5 \pm 3.7$ at 3
(units of $10^{-5}$). That is 1.2, 2.5, 5, 12 and 62 times the control. Only at noise 0.1 is any
seed below its control seed (1 of 3). On the test meshes the one-step RMAE is 1.1 – 1.6 times the
control's at noise 0.1, 1.9 – 2.6 times at 0.3, 2.8 – 4.2 times at 0.5, 5.3 – 7.7 times at 1 and
14 – 25 times at 3. No seed is below its control on any mesh, except 2 of 3 at noise 0.1 on
`model_010`. The one-step RMSE suffers less: at noise 0.1 it is within 7 % of the control's, and
at 0.3 and 0.5 it is 1.5 – 3 times.

At noise 1 and 3 the validation loss bottoms out early, at epoch $64 \pm 8$ and $31 \pm 10$. The
scheduler then takes the LR to its floor and early stopping ends these runs after 107 – 127 epochs.
At 0.1, 0.3 and 0.5 the loss keeps improving much longer (best at epoch $219 \pm 31$,
$175 \pm 27$ and $121 \pm 61$), with the largest seed spreads. The training loss is measured on
noisy inputs, so it is not comparable across levels.

**Noise 0.3 is the best level tested.** Above 0.3, every added level costs one-step accuracy and
rollout error on every mesh, from both checkpoints: 0.5 is above 0.3, 1 above 0.5 and 3 above 1.
The rollout RMAE curves meet only at the end of `model_010`. Below 0.3 the one-step cost almost
disappears, but so does the guarantee: at 0.1 one seed in three diverges, and the val rollout
spikes to $10^{2}$ – $10^{4}$ RMAE, past the span, on many epochs up to about epoch 170.
Its two bounded seeds are about as good as noise 0.3 over 120 steps: lower on `model_007` and
`model_011`, higher on `model_010`, about the same on `model_003`. But over steps 1 – 10 the noise 0.1
runs are behind 0.3 and 0.5 on every mesh, and so is their 40-step val rollout (1.6 – 1.7 K at the best
epoch against 0.86 – 0.99 at 0.3 and 1.15 – 1.4 at 0.5). Between 0.1 and 0.3 the stability is
seed-dependent. Three seeds cannot place the threshold more precisely than that.

**The best-rollout checkpoint does not carry over to the test meshes.** On the val meshes the
best checkpoint is better by construction (40-step mean MAE 0.88 – 0.98 against 1.11 – 1.19 K at
noise 0.3, and 4.3 – 4.7 against 6.7 – 7.1 K at noise 3). On the test meshes its 120-step rollout
RMAE is higher than the last epoch's on 16 of the 20 level–mesh pairs. It is tied at noise 0.3 on
`model_010` and lower on three pairs: noise 3 on `model_010`, and noise 0.1 on `model_003` and
`model_010`. Its one-step RMAE is higher on 17 of 20. It does not rescue noise 0.1 seed 2, which
leaves the span at step 17 – 20 from the best checkpoint too. At 0.1 and 0.3 the val rollout
spikes by one to several decades on many epochs while the LR is still high. So which epoch scores
best on two meshes over 40 steps is partly that noise. Picking on the val rollout costs nothing in
stability here, but it does not beat simply training to the end.

**The message becomes more symmetric, and less flux-like at node level.** The antisymmetric
energy fraction is 0.31 – 0.34 for the control and lower at every level, with every seed below its
control on every mesh. It falls with the noise, though not monotonically: 0.24 – 0.26 at noise
0.1, 0.19 – 0.21 at 0.3, 0.22 – 0.25 at 0.5, 0.17 – 0.19 at 1 and 0.12 – 0.15 at 3. The 0.5 runs
also trained longer. Up to noise 1 the per-edge read-outs hold up or improve. The $R^2$ for $F$
is 0.84 – 0.88 for the control and highest at noise 0.1, at 0.89 – 0.92, with every seed above
its control. It is 0.87 – 0.90 at 0.3, 0.85 – 0.88 at 0.5 and 0.82 – 0.86 at 1. $\Delta T$ goes
from 0.89 – 0.92 to 0.91 – 0.93 at 0.1 and 0.93 – 0.96 from 0.3 to 1. At noise 3 everything drops:
$F$ to 0.52 – 0.59, $\Delta T$ to 0.70 – 0.80 and $w$ to 0.66 – 0.78. The node-level $R^2$ of
$\sum_j m_{ij}$ on the net flux is 0.69 – 0.79 for the control and 0.73 – 0.84 at noise 0.1. Above
that it falls: 0.68 – 0.77 at 0.3, 0.66 – 0.77 at 0.5, 0.60 – 0.70 at 1 and 0.41 – 0.47 at 3. So
the most flux-like messages, at noise 0.1, belong to the level where one seed still diverges, and
the stable levels are less flux-like than the control. Stability here comes from training on
perturbed inputs, not from a message that is closer to a conservative flux.

**What this shows.** Input noise is the first change in this notebook that makes every seed's
rollout stable, on every held-out mesh, over the full 120 steps, from noise 0.3 up. At 0.3 the
price is about 2 – 2.6 times the one-step RMAE and 2.5 times the validation loss. At 0.1 the price
is nearly gone, and so is the guarantee. Despite the worse one-step error, the rollout is better
from its first steps on at every level. One-step error has again been a poor guide to the rollout,
now in the other direction. The comparison has 3 seeds per level against the control's 5. The
budgets also differ: the control trains all 200 epochs with no early stopping, noise 0.3, 1 and 3
have 200 epochs with early stopping, and noise 0.1 and 0.5 have 500.

## Log cell volume as a node feature (`_vol`)

The node features so far are the node type alone, $[\,\text{internal}, \text{boundary}\,]$: nothing
tells the model how large the cell behind a node is, although the FV update divides each cell's
flux balance by its volume, $T_i^{n+1} - T_i^n = \frac{\Delta t}{V_i}\sum_j F_{ij}$. These runs
add the cell volume as a third node feature, $[\,\text{internal}, \text{boundary}, \log V\,]$:

- $V_i$ from `getter_of.getV()` (OpenFOAM's `cellVolumes()`), one per cell in cell order.
- **Log**, because the snappyHexMesh refinement levels are $8\times$ apart: $V$ spans about
  $3500\times$ on `model_000`, and a z-score of $V$ itself would leave about 80 % of the cells (the
  two finest levels) within 0.15 of each other, with the few coarse cells dominating the std.
  $\log V$ puts the levels at equal steps of $\log 8$.
- **Boundary nodes** are face centres and have no volume: they take the mesh's mean internal
  $\log V$, so they stay out of the way of the normaliser's statistics (a 0 would sit about 20
  away from the cells and dominate the std). `node_type` already tells them apart.
- Then z-scored by `FeatureNormalizer` with every other node feature, fitted on the training
  meshes. The volume counts as an FV feature: `use_fv_features = False` drops it with the FV edge
  features.

Runs, both with input noise 0.3 (the best level of the input-noise section) and at most 500 epochs:

- **no volume** (`parametric_history1_msg_dim64_residual_noise0.3_cf_nopos`, `seed_<n>_01` in
  `seeds_01.npz`, seeds 0 – 2, commit `f8f9112` + uncommitted changes): the 500-epoch rerun of
  noise 0.3 from the sweep in `logs/20261016_113759`, which the input-noise section left out.
  It trains 159 – 240 epochs.
- **log volume** (`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos`, `seed_<n>` in
  `seeds.npz`, seeds 0 – 2, commit `48186db` + uncommitted changes): launcher log in
  `logs/20261019_151511`. It trains 181 – 240 epochs.

Between the two commits the training code differs only by the volume feature and the budget
(`epochs = 500`, which the control had as an uncommitted change). Everything else is shared: the
64-channel residual model at `history = 1` with no LayerNorm and no message penalty, Cf boundary
nodes, no positions, SOAP from $3\times10^{-3}$ with `ReduceLROnPlateau` from epoch 29,
`batch_size = 2`, the early-stopping rule (`min_lr` $= 3\times10^{-9}$). The volume adds
$1 \times 64$ weights to the node encoder (116 673 parameters instead of 116 609).

**Not paired.** The extra input column changes the node encoder's initialisation, so the CPU
generator's draws differ from there on: seed $n$ starts from other weights than control seed $n$
($c_0$ differs, printed below) and sees other batches. Every comparison is between the two samples:
Mann-Whitney U, two-sided. With 3 seeds a side its smallest $p$ is 0.1, so no difference can reach
0.05; $p = 0.1$ means the three seeds of one run are all below the three of the other.

**Two checkpoints per run**: the last-epoch `model.pt` and `model_best_rollout.pt`, the epoch with
the lowest 40-step val rollout MAE, both evaluated on the test meshes.

- **Loss decay, LR and val rollout RMAE** per epoch.
- **Test rollout** (plotted per step as RMAE): seeds below the span of $T$, MAE over steps 1 – 10,
  first step above the span, rollout RMAE, for both checkpoints.
- **One-step and rollout RMAE / RMSE** on the test meshes, as a table and a bar plot, and the val
  rollout, from both checkpoints.
- **Message vs flux, $R^2$**: `compute_model_flux_stats` on the last-epoch checkpoints.

The cells reuse `SP_MESHES`, `SP_WARM` and `SP_RAMP_END` from the penalties section, `nz_stamp`
from the input-noise section, `pos_absT`, `pos_p` and `pos_stamp` from the positions section,
`l1p_cell`, `seed_stats`, `NS_SPAN`, `ns_seed_run` and `NS_FLUX_COLS` from the sections before.

In [ ]:
# --- Log cell volume: load the seeds, check them, summarise ------------------------
# Noise 0.3 with and without the log cell volume as a node feature, both from
# 500-epoch sweeps: the control is the noise 0.3 rerun (seed_<n>_01, seeds_01.npz).
# NOT paired: the volume adds an input column to the node encoder, so the
# initialisation draws (c0, printed below) and every shuffle after them differ
# from the control's. Mann-Whitney U, two-sided: with 3 seeds a side its smallest
# p is 0.1.
VOL_CASES = {"no volume":  ("parametric_history1_msg_dim64_residual_noise0.3_cf_nopos", 1),
             "log volume": ("parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos", 0)}
VOL_MESHES = SP_MESHES
VOL_NOISE = 0.3
VOL_BUDGET = 500                               # max_epochs of both sweeps
VOL_NODE_FEATS = {"no volume": 2, "log volume": 3}   # node type (2), + log V
VOL_EPOCH_KEYS = ["train_losses", "val_losses", "lrs", "val_rollout_mae"]


def vol_npz(key):
    exp, i = VOL_CASES[key]
    return processed_data_dir / exp / ("seeds.npz" if i == 0 else f"seeds_{i:02d}.npz")


def vol_load(cell_volume):
    """load_mesh for these runs: Cf graphs from the mesh cache train_parametric.py
    fills, node type and, for the _vol runs, the log cell volume; no positions."""
    def load(mesh):
        g, T = load_mesh_cached(str(raw_param_dir / mesh), EXCLUDED, "cf",
                                processed_data_dir / "parametric_preproc_cache")
        g.node_attr = g.node_attr[:, node_feature_columns(cell_volume, False)]
        return g, T
    return load


VOL_LOAD = {"no volume": vol_load(False), "log volume": vol_load(True)}

vol_no_npz = [str(vol_npz(key)) for key in VOL_CASES if not vol_npz(key).exists()]
if vol_no_npz:
    raise FileNotFoundError(
        f"no {', '.join(vol_no_npz)} yet: python aggregate_seeds.py <exp_name>:<rerun index>")

vol = {key: dict(np.load(vol_npz(key))) for key in VOL_CASES}
VOL_EPOCHS = max(z["train_losses"].shape[1] for z in vol.values())
for key, z in vol.items():
    # per-epoch fields NaN-padded to VOL_EPOCHS, the longest seed of either run
    for k in VOL_EPOCH_KEYS:
        a = z[k].astype(float)
        z[k] = np.concatenate([a, np.full((a.shape[0], VOL_EPOCHS - a.shape[1], *a.shape[2:]),
                                          np.nan)], 1)
VOL_SEEDS = vol["no volume"]["seeds"]
VOL_VAL_MESHES = [str(m) for m in vol["no volume"]["val_meshes"]]
VOL_VAL_STEPS = vol["no volume"]["val_rollout_mae"].shape[-1]
# seed_<n>, with the rerun's _<ii> appended; (run, whether its weights are here)
vol_runs = {key: [ns_seed_run(exp, f"{int(s)}_{i:02d}" if i else int(s)) for s in VOL_SEEDS]
            for key, (exp, i) in VOL_CASES.items()}

# The same seeds, meshes, noise, budget, schedule and model, but for the node
# encoder's weights on log V (1 x hidden_dim = 64).
for key, z in vol.items():
    assert np.array_equal(z["seeds"], VOL_SEEDS), f"{key}: seeds {z['seeds']}"
    assert list(z["test_meshes"]) == VOL_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert [str(m) for m in z["val_meshes"]] == VOL_VAL_MESHES, f"{key}: val meshes {z['val_meshes']}"
    assert float(z["input_noise"]) == VOL_NOISE, f"{key}: input_noise = {z['input_noise']}"
    assert int(z["max_epochs"]) == VOL_BUDGET, f"{key}: launched for {z['max_epochs']} epochs"
    assert str(z["msg_kind"]) == "none" and float(z["msg_alpha"]) == 0.0, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(vol["no volume"][field]), f"{key}: {field} differs"
n_add = int(vol["log volume"]["n_params"]) - int(vol["no volume"]["n_params"])
assert n_add == 64, f"the models differ by {n_add} parameters, not the 1 x 64 of log V"

# Per seed checkpoint: Cf graphs, whether they carry the volume, the node features
# its normalizer was fitted on, and the commit it was trained with (for the
# markdown above).
for key, runs in vol_runs.items():
    commits = {}
    for run, ok in runs:
        ck = processed_data_dir / run / "checkpoints"
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg.get("excluded_patches") == EXCLUDED, \
            f"{run}: graphs built with {cfg}"
        assert cfg["cell_volume"] == (key == "log volume"), f"{run}: cell_volume = {cfg['cell_volume']}"
        if ok:
            n = torch.load(ck / "normalizer.pt")["node_mean"].numel()
            assert n == VOL_NODE_FEATS[key], f"{run}: normalizer fitted on {n} node features"
        c = load_git_commit(ck)
        tag = ("not recorded" if c is None or c["commit"] is None
               else c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else ""))
        commits.setdefault(tag, []).append(run.split("/")[-1])
    print(f"{VOL_CASES[key][0]}: " + "; ".join(f"commit {tag} ({', '.join(rs)})"
                                               for tag, rs in commits.items()))
print("c0 (mean message column norm at initialisation) per seed: "
      + "; ".join(f"{key} {np.round(z['msg_c0'], 4).tolist()}" for key, z in vol.items()))


def vol_last(z, field):
    """Each seed's value at its own last epoch: early-stopped seeds are NaN past it."""
    return z[field][np.arange(len(z["seeds"])), z["n_epochs"] - 1]


def vol_first_cut(lrs):
    """Epoch of each seed's first LR cut, nan if it never came; the NaN padding
    past an early stop is not a cut."""
    d = np.diff(lrs, axis=1)
    cut = np.isfinite(d) & (d != 0)
    return np.where(cut.any(1), cut.argmax(1) + 1, np.nan)


za, zb = vol["no volume"], vol["log volume"]
print(f"\nseeds {VOL_SEEDS.tolist()}, noise {VOL_NOISE:g}, at most {VOL_BUDGET} epochs, "
      f"{int(za['n_params'])} vs {int(zb['n_params'])} parameters; val rollout: "
      f"{VOL_VAL_STEPS} steps on {', '.join(VOL_VAL_MESHES)}\n")
vol_stats = {"epochs":      (lambda z: z["n_epochs"],                    "{:.0f}"),
             "train, last": (lambda z: vol_last(z, "train_losses"),        "{:.2e}"),
             "val, last":   (lambda z: vol_last(z, "val_losses"),          "{:.2e}"),
             "val, best":   (lambda z: np.nanmin(z["val_losses"], 1),     "{:.2e}"),
             "best @":      (lambda z: np.nanargmin(z["val_losses"], 1),  "{:.0f}"),
             "1st LR cut":  (lambda z: vol_first_cut(z["lrs"]), "{:.0f}"),
             "LR, last":    (lambda z: vol_last(z, "lrs"),                 "{:.0e}")}
W = 20
print(f"{'':14s}{'n':>3}" + "".join(f"{h:>{W}}" for h in vol_stats))
for key, z in vol.items():
    row = f"{key:14s}{len(z['seeds']):>3}"
    for h, (f, fmt) in vol_stats.items():
        v = np.asarray(f(z), dtype=float)
        if h == "epochs":
            row += f"{f'{v.min():.0f} – {v.max():.0f}':>{W}}"
        else:
            row += f"{fmt.format(np.nanmean(v)) + ' ± ' + fmt.format(np.nanstd(v, ddof=1)):>{W}}"
    print(row)
print(f"{'p (MWU)':14s}{'':>3}" + "".join(f"{pos_p(f(za), f(zb)):>{W}.3f}" for f, _ in vol_stats.values()))

# The val rollout as logged in training: its best epoch (that of
# model_best_rollout.pt), its best value and its value at the last epoch.
print(f"\nval rollout MAE [K], {VOL_VAL_STEPS} steps, mean over val meshes and steps, per seed")
print(f"{'':14s}{'seed':>5}{'best @':>9}{'best':>11}{'last':>11}")
for key, z in vol.items():
    vr = z["val_rollout_mae"].mean((2, 3))
    for i, s in enumerate(z["seeds"]):
        print(f"{key if i == 0 else '':14s}{int(s):>5}{int(z['val_rollout_best_epoch'][i]):>9}"
              f"{np.nanmin(vr[i]):>11.3g}{vr[i, z['n_epochs'][i] - 1]:>11.3g}")

# Test rollout of each checkpoint, from the MAE saved at training time: seeds below
# the span of T at the end, MAE over steps 1-10, and the RMAE, as geometric means.
VOL_CKPTS = {"last": "rollout_mae_", "best": "rollout_mae_best_"}
vol_rows = [(f"{key}, {ck}", key, ck) for ck in VOL_CKPTS for key in VOL_CASES]


def vol_rmae(mae, mesh):
    r = mae.astype(float).sum(1) / pos_absT[mesh][:mae.shape[1]].sum()
    return np.where(np.isfinite(r), r, np.inf)


print(f"\ntest rollout, {za[f'rollout_mae_{VOL_MESHES[0]}'].shape[1]} steps: seeds below "
      f"{NS_SPAN:.0f} K at the end / MAE over steps 1-10 [K] / RMAE, geometric means over seeds")
print(f"{'':20s}" + "".join(f"{mesh:>30}" for mesh in VOL_MESHES) + f"{'below':>8}")
for ck, prefix in VOL_CKPTS.items():
    for key, z in vol.items():
        final = np.stack([z[f"{prefix}{m}"][:, -1] for m in VOL_MESHES], 1)
        below = np.isfinite(final) & (final < NS_SPAN)
        row = f"{f'{key}, {ck}':20s}"
        for j, mesh in enumerate(VOL_MESHES):
            mae = z[f"{prefix}{mesh}"]
            mu_e, _ = seed_stats(mae[:, :10].mean(1), log=True)
            mu_r, _ = seed_stats(vol_rmae(mae, mesh), log=True)
            n_fin = int(np.isfinite(vol_rmae(mae, mesh)).sum())
            row += f"{f'{int(below[:, j].sum())}/{len(below)} / {10**mu_e:.3g} / {10**mu_r:.1e}' + (f' ({n_fin})' if n_fin < len(below) else ''):>30}"
        print(row + f"{f'{int(below.sum())}/{below.size}':>8}")
    print(f"{'  p, steps 1-10':20s}" + "".join(
        f"{pos_p(za[f'{prefix}{m}'][:, :10].mean(1), zb[f'{prefix}{m}'][:, :10].mean(1)):>30.3f}"
        for m in VOL_MESHES))
    print(f"{'  p, RMAE':20s}" + "".join(
        f"{pos_p(vol_rmae(za[f'{prefix}{m}'], m), vol_rmae(zb[f'{prefix}{m}'], m)):>30.3f}"
        for m in VOL_MESHES))

# The first step whose MAE is above the span (nan counts as above), per seed;
# "-" when the rollout never gets there.
print(f"\ntest rollout, first step with MAE above {NS_SPAN:.0f} K, per seed (- never)")
print(f"{'':20s}{'seed':>5}" + "".join(f"{mesh:>11}" for mesh in VOL_MESHES))
for label, key, ck in vol_rows:
    z = vol[key]
    for i, s in enumerate(z["seeds"]):
        row = f"{label if i == 0 else '':20s}{int(s):>5}"
        for mesh in VOL_MESHES:
            out = ~(z[f"{VOL_CKPTS[ck]}{mesh}"][i] < NS_SPAN)
            row += f"{int(out.argmax()) + 1 if out.any() else '-':>11}"
        print(row)

In [ ]:
# --- Log cell volume: one-step and rollout errors, and val rollout of each checkpoint
# As in the input-noise section (nz_evaluate), but each run evaluated on its own
# graphs (VOL_LOAD): one-step and test rollout RMAE / RMSE, summed over every step
# and node, plus a val rollout of the length the runs logged, for the last-epoch
# and the best-rollout checkpoint. Cached per checkpoint file, stamped with its mtime.
vol_eval_cache = processed_data_dir / "parametric_eval_volume.npz"
VOL_FILES = {"last": "model.pt", "best": "model_best_rollout.pt"}


def vol_evaluate(run, fname, load):
    """One-step and rollout RMAE / RMSE on the test meshes and the val rollout
    MAE per step of one checkpoint file, keyed <run>__<fname>__<mesh>__<metric>."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / fname, map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in VOL_MESHES + VOL_VAL_MESHES:
        graph, T_seq = load(mesh)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        if mesh in VOL_MESHES:
            sums = torch.zeros(4, dtype=torch.float64)   # |e|, |T|, e^2, T^2
            with torch.no_grad():
                for i in range(len(ds)):
                    d = ds[i].to(device)
                    T_p = norm.inverse_transform_T(model(d).cpu()).double()
                    T_t = norm.inverse_transform_T(d.y.cpu()).double()
                    e = T_p - T_t
                    sums += torch.stack([e.abs().sum(), T_t.abs().sum(),
                                         e.pow(2).sum(), T_t.pow(2).sum()])
            out[f"{run}__{fname}__{mesh}__rmae_step"] = float(sums[0] / sums[1])
            out[f"{run}__{fname}__{mesh}__rmse_step"] = float(sums[2] / sums[3])
            T_p, T_t = rollout(model, ds, device=device)
            e, T_t = (T_p - T_t).double(), T_t.double()
            out[f"{run}__{fname}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_t.abs().sum())
            out[f"{run}__{fname}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_t.pow(2).sum())
        if mesh in VOL_VAL_MESHES:
            T_p, T_t = rollout(model, ds, device=device, n_steps=VOL_VAL_STEPS)
            out[f"{run}__{fname}__{mesh}__val_roll"] = (T_p - T_t).abs().mean(1).numpy()
        del model
    return out


def vol_has(key, i, ck):
    """Whether seed i of key has checkpoint ck here (no best one if every val
    rollout overflowed)."""
    run, ok = vol_runs[key][i]
    return ok and (ck == "last" or int(vol[key]["val_rollout_best_epoch"][i]) >= 0)


vol_eval = (dict(np.load(vol_eval_cache, allow_pickle=False))
            if vol_eval_cache.exists() else {})
todo = [(key, run, VOL_FILES[ck]) for _, key, ck in vol_rows
        for i, (run, ok) in enumerate(vol_runs[key])
        if vol_has(key, i, ck)
        and vol_eval.get(f"{run}__{VOL_FILES[ck]}__stamp") != nz_stamp(run, VOL_FILES[ck])]
for key, run, fname in todo:
    vol_eval.update(vol_evaluate(run, fname, VOL_LOAD[key]))
    vol_eval[f"{run}__{fname}__stamp"] = nz_stamp(run, fname)
    np.savez_compressed(vol_eval_cache, **vol_eval)   # after every checkpoint: these are slow


def vol_metric(key, ck, metric, meshes):
    """(S, M) one metric of checkpoint ck, nan where a seed lacks it; a val
    rollout is reduced to its mean over the steps."""
    out = np.full((len(vol[key]["seeds"]), len(meshes)), np.nan)
    for i, (run, ok) in enumerate(vol_runs[key]):
        if vol_has(key, i, ck):
            v = [np.mean(vol_eval[f"{run}__{VOL_FILES[ck]}__{m}__{metric}"]) for m in meshes]
            out[i] = np.where(np.isfinite(v), v, np.inf)
    return out


# The recomputed val rollout of each last-epoch checkpoint has to be the one logged
# at its last epoch in training, over the steps below the span ...
devs = []
for key, z in vol.items():
    for i, (run, ok) in enumerate(vol_runs[key]):
        if not ok:
            continue
        for v, mesh in enumerate(VOL_VAL_MESHES):
            new = vol_eval[f"{run}__model.pt__{mesh}__val_roll"]
            old = z["val_rollout_mae"][i, z["n_epochs"][i] - 1, v]
            keep = (new < NS_SPAN) & (old < NS_SPAN)
            if keep.any():
                devs.append(np.abs(new[keep] / old[keep] - 1).max())
print(f"recomputed vs logged last-epoch val rollout MAE, steps below {NS_SPAN:.0f} K: "
      f"max |relative difference| = {max(devs):.1e}")

# ... and the recomputed test rollout RMAE the one from the MAE saved at training
# time (vol_rmae), where both are finite.
devs = []
for label, key, ck in vol_rows:
    for j, mesh in enumerate(VOL_MESHES):
        new = vol_metric(key, ck, "rmae_roll", VOL_MESHES)[:, j]
        old = vol_rmae(vol[key][f"{VOL_CKPTS[ck]}{mesh}"], mesh)
        ok = np.isfinite(new) & np.isfinite(old)
        if ok.any():
            devs.append(np.abs(new[ok] / old[ok] - 1).max())
print(f"recomputed vs saved test rollout RMAE: max |relative difference| = {max(devs):.1e}\n")

# Geometric mean ×/÷ geometric std over the seeds, (n/m) as in the sections above,
# and under each checkpoint the Mann-Whitney p of log volume against no volume.
VOL_ERR_COLS = {"rmae_step": "one-step RMAE", "rmse_step": "one-step RMSE",
                "rmae_roll": "rollout RMAE", "rmse_roll": "rollout RMSE"}
print(f"{'mesh':<11}{'run':<20}" + "".join(f"{h:>28}" for h in VOL_ERR_COLS.values()))
for j, mesh in enumerate(VOL_MESHES):
    for ck in VOL_FILES:
        for key in VOL_CASES:
            print(f"{mesh:<11}{f'{key}, {ck}':<20}" + "".join(
                l1p_cell(vol_metric(key, ck, m, VOL_MESHES)[:, j]) for m in VOL_ERR_COLS))
        print(f"{mesh:<11}{'  p (MWU)':<20}" + "".join(
            f"{pos_p(vol_metric('no volume', ck, m, VOL_MESHES)[:, j], vol_metric('log volume', ck, m, VOL_MESHES)[:, j]):>28.3f}"
            for m in VOL_ERR_COLS))
    print()

print(f"val rollout MAE [K], {VOL_VAL_STEPS} steps, mean over the steps")
print(f"{'':20s}" + "".join(f"{m:>28}" for m in VOL_VAL_MESHES))
for ck in VOL_FILES:
    for key in VOL_CASES:
        vr = vol_metric(key, ck, "val_roll", VOL_VAL_MESHES)
        print(f"{f'{key}, {ck}':20s}" + "".join(l1p_cell(vr[:, v]) for v in range(len(VOL_VAL_MESHES))))
    print(f"{'  p (MWU)':20s}" + "".join(
        f"{pos_p(vol_metric('no volume', ck, 'val_roll', VOL_VAL_MESHES)[:, v], vol_metric('log volume', ck, 'val_roll', VOL_VAL_MESHES)[:, v]):>28.3f}"
        for v in range(len(VOL_VAL_MESHES))))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The control dashed in PLOT2 as everywhere above, the volume run in PLOT3.
VOL_STYLES = {"no volume": (PLOT2_COLOR, "--"), "log volume": (PLOT3_COLOR, "-")}

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 1000*px), sharex=True, constrained_layout=True)
axs[0, 1].sharey(axs[0, 0])

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Training with the Log Cell Volume, Noise 0.3, No Positions, Cf, msg_dim = 64, h = 1\n"
             "(losses: mean ± std over the seeds still training; LR, val rollout: every seed)",
             fontsize=22, color=AX_COLOR)

epochs = np.arange(VOL_EPOCHS)
for ax, (title, field) in zip(axs[0], (("Training Loss", "train_losses"),
                                       ("Validation Loss", "val_losses"))):
    for key, z in vol.items():
        color, ls = VOL_STYLES[key]
        mu, sd = np.nanmean(z[field], 0), np.nanstd(z[field], 0, ddof=1)
        ax.plot(epochs, mu, label=f"{key} ({len(z['seeds'])} seeds)", color=color,
                linewidth=2.2, linestyle=ls)
        # the band's lower edge is clipped: on a log axis mean - std <= 0 has no
        # place to go
        ax.fill_between(epochs, np.clip(mu - sd, mu / 10, None), mu + sd,
                        color=color, alpha=0.15, linewidth=0)
    ax.set_title(title, fontsize=20)
    ax.set_ylabel("MSE Loss", fontsize=17)

ax = axs[1, 0]
for key, z in vol.items():
    color, ls = VOL_STYLES[key]
    for i, lr in enumerate(z["lrs"]):
        ax.plot(epochs, lr, color=color, linewidth=1.4, linestyle=ls, alpha=0.7,
                label=key if i == 0 else None)
ax.set_title("Learning Rate", fontsize=20)
ax.set_ylabel("LR", fontsize=17)

# The val rollout every epoch as RMAE over its steps (sum of |error| over steps
# and nodes over the same sum of |T_true|) per val mesh, then the mean over the
# val meshes. The marker is each seed's best epoch, model_best_rollout.pt, which
# training picked on the mean MAE.
vol_val_absT = np.stack([param_T_stats(m)[0][:VOL_VAL_STEPS] for m in VOL_VAL_MESHES])  # (V, K)
ax = axs[1, 1]
for key, z in vol.items():
    color, ls = VOL_STYLES[key]
    vr = (z["val_rollout_mae"].sum(-1) / vol_val_absT.sum(-1)).mean(-1)   # (S, E)
    for i, v in enumerate(vr):
        ax.plot(epochs, v, color=color, linewidth=1.2, linestyle=ls, alpha=0.8,
                label=key if i == 0 else None)
        b = int(z["val_rollout_best_epoch"][i])
        if b >= 0:
            ax.plot(b, v[b], marker="o", markersize=6, color=color)
ax.set_title(f"Val Rollout RMAE, {VOL_VAL_STEPS} Steps (o: best)", fontsize=20)
ax.set_ylabel("RMAE", fontsize=17)

for ax in axs.ravel():
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=15)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_yscale("log")
    # warmup and ramp end: the scheduler starts stepping at the second line
    for e in (SP_WARM - 1, SP_RAMP_END):
        ax.axvline(e, color=AX_COLOR, linewidth=1.3, alpha=0.85, linestyle=(0, (4, 3)))
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    legend = ax.legend(fontsize=13, loc="lower left" if ax is axs[1, 0] else "upper right")
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color
for ax in axs[1]:
    ax.set_xlabel("Epoch", fontsize=17)

# plt.savefig("../outputs/plots/parametric_history1_volume_training.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Test rollout RMAE per step, from the MAE saved at training time. Rows: the
# last-epoch and the best-rollout checkpoint; columns: the held-out meshes. Every
# seed thin, the median bold: a mean would be the one seed that diverges.
fig, axs = plt.subplots(2, len(VOL_MESHES), figsize=(1800*px, 900*px), sharex=True, sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Test Rollout RMAE with the Log Cell Volume, Noise 0.3, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(thin: seeds, bold: median)", fontsize=24, color=AX_COLOR)

for row, (ck, prefix) in enumerate(VOL_CKPTS.items()):
    for col, mesh in enumerate(VOL_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, z in vol.items():
            color, ls = VOL_STYLES[key]
            rel = rmae_curve(z[f"{prefix}{mesh}"], param_T_stats(mesh)[0])
            steps = np.arange(1, rel.shape[1] + 1)
            for m in rel:
                ax.plot(steps, m, color=color, linewidth=0.8, alpha=0.35, linestyle=ls)
            # an overflowed step (nan) counts as above every finite one
            med = np.median(np.where(np.isfinite(rel), rel, np.inf), 0)
            ax.plot(steps, med, color=color, linewidth=2.5, linestyle=ls, label=key)
        ax.set_yscale("log")
        ax.set_ylim(2e-3, 2e0)

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        # (an error as large as the span at every step, in the same relative units)
        span_rel = rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0])
        ax.plot(steps, span_rel, color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, span_rel[0]), xytext=(4, 4),
                        textcoords="offset points", ha="left", va="bottom", fontsize=12,
                        color=AX_COLOR)
            ax.set_ylabel(f"RMAE, {ck} checkpoint", fontsize=15)
            legend = ax.legend(fontsize=12, loc="center right")
            legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
            legend.get_frame().set_edgecolor(AX_COLOR)        # border color
            legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
            for text in legend.get_texts():
                text.set_color(AX_COLOR)                      # label text color
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_volume_rollout.png", dpi=300, transparent = True)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The errors table above as a figure, laid out as in the input-noise section. Rows:
# one-step and rollout; columns: RMAE and RMSE. Per mesh, each run's last-epoch
# (solid) and best-rollout (hatched) checkpoint. Each bar is the geometric mean
# over the seeds, the whisker ×/÷ one geometric std, the dots the seeds
# themselves. The rollout panels stop at VOL_ROLL_TOP, a relative error of 10; a
# seed above it, or overflowed, is drawn as a triangle there.
VOL_ROLL_TOP = 10.0
vol_bars = [(key, ck, VOL_STYLES[key][0]) for key in VOL_CASES for ck in VOL_FILES]
x = np.arange(len(VOL_MESHES))
width = 0.8 / len(vol_bars)

fig, axs = plt.subplots(2, 2, figsize=(1600*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("RMAE and RMSE with the Log Cell Volume, Noise 0.3, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(geometric mean ×/÷ std)", fontsize=22, color=AX_COLOR)

for ax, metric in zip(axs.ravel(), VOL_ERR_COLS):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.set_title(VOL_ERR_COLS[metric].replace(" ", "  ·  ", 1), fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(VOL_MESHES, fontsize=13)

    vals = [vol_metric(key, ck, metric, VOL_MESHES) for key, ck, _ in vol_bars]
    finite = np.concatenate([v[np.isfinite(v)] for v in vals])
    # bars on a log axis grow from the bottom of the axis, not from 0, so the
    # floor has to be set before they are drawn
    top = VOL_ROLL_TOP if metric.endswith("_roll") else np.inf
    shown = finite[finite <= top]
    floor, hi = shown.min() / 10, min(shown.max() * 10, top)
    ax.set_yscale("log")
    ax.set_ylim(floor, hi)
    for i, ((key, ck, color), v_all) in enumerate(zip(vol_bars, vals)):
        off = (i - (len(vol_bars) - 1) / 2) * width
        mu, sd = seed_stats(v_all, log=True)               # (M,) each
        has = np.isfinite(mu)
        ax.bar((x + off)[has], (10 ** mu - floor)[has], width * 0.92, bottom=floor,
               color=color, alpha=0.55 if ck == "best" else 1.0,
               hatch="//" if ck == "best" else None, edgecolor=AX_COLOR, linewidth=0.4)
        ax.errorbar((x + off)[has], 10 ** mu[has],
                    yerr=[10 ** mu[has] - 10 ** (mu - sd)[has], 10 ** (mu + sd)[has] - 10 ** mu[has]],
                    fmt="none", ecolor=AX_COLOR, elinewidth=1.1, capsize=2)
        for j in range(len(VOL_MESHES)):
            v = v_all[:, j]
            ok = np.isfinite(v) & (v <= hi)
            ax.plot(np.full(ok.sum(), x[j] + off), v[ok], "o", color=AX_COLOR,
                    markersize=2.6, alpha=0.7)
            if (~ok & ~np.isnan(v)).any():   # above the top, or overflowed (inf)
                ax.plot(x[j] + off, hi, "v", color=color, markersize=7,
                        markeredgecolor=AX_COLOR, clip_on=False)

    ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5,
            color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

# One legend for the whole figure: a colour per run, the hatch for the
# best-rollout checkpoint, and the triangle.
handles = [Patch(color=VOL_STYLES[k][0]) for k in VOL_STYLES] + [
    Patch(facecolor="white", edgecolor=AX_COLOR, hatch="//"),
    Line2D([], [], marker="v", linestyle="none", color="white", markeredgecolor=AX_COLOR,
           markersize=8)]
labels = list(VOL_STYLES) + ["best-rollout checkpoint", "seed above the top or overflowed"]
legend = fig.legend(handles, labels, loc="outside lower center", ncols=len(handles), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_volume_rmae_rmse.png", dpi=300, transparent = True)

In [ ]:
# --- Log cell volume: message vs FV flux, per seed ---------------------------------
# compute_model_flux_stats on the last-epoch checkpoints (it reads model.pt), each
# run on its own graphs (VOL_LOAD). Every column is an R^2, a correlation or an
# energy fraction.
vol_flux_cache = processed_data_dir / "parametric_message_flux_volume.npz"

vol_flux = (dict(np.load(vol_flux_cache, allow_pickle=False))
            if vol_flux_cache.exists() else {})
for key, runs in vol_runs.items():
    for run, ok in runs:
        if not ok or vol_flux.get(f"{run}__stamp") == pos_stamp(run):
            continue
        vol_flux.update(compute_model_flux_stats(run, load=VOL_LOAD[key]))
        vol_flux[f"{run}__stamp"] = pos_stamp(run)
        np.savez_compressed(vol_flux_cache, **vol_flux)   # after every run: these are slow


def vol_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(vol[key]["seeds"]), len(VOL_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, (run, ok) in enumerate(vol_runs[key]):
        if not ok:
            continue
        for j, mesh in enumerate(VOL_MESHES):
            out[i, j] = [*vol_flux[f"{run}__{mesh}__r2"], float(vol_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(vol_flux[f"{run}__{mesh}__best_r"])),
                         vol_flux[f"{run}__{mesh}__anti"].mean()]
    return out


vol_flux_tab = {key: vol_flux_table(key) for key in VOL_CASES}

# Arithmetic mean ± std over the seeds: these are bounded in [0, 1].
print(f"Linear read-out from the message, {N_STEPS} probed timesteps, last-epoch checkpoints, "
      f"mean ± std over seeds")
print(f"{'mesh':<11}{'run':<14}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
for j, mesh in enumerate(VOL_MESHES):
    for key in VOL_CASES:
        t = vol_flux_tab[key][:, j]
        t = t[~np.isnan(t[:, 0])]
        cells = ([f"{m:.3f} ± {s:.3f}" for m, s in zip(t.mean(0), t.std(0, ddof=1))]
                 if len(t) > 1 else [f"{m:.3f}" for m in t.mean(0)])
        print(f"{mesh:<11}{key:<14}{len(t):>3}" + "".join(f"{c:>15}" for c in cells))
    print(f"{mesh:<11}{'  p (MWU)':<14}{'':>3}" + "".join(
        f"{pos_p(vol_flux_tab['no volume'][:, j, c], vol_flux_tab['log volume'][:, j, c]):>15.3f}"
        for c in range(len(NS_FLUX_COLS))))
    print()

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Message vs FV flux, no volume against log volume, laid out as in the sections
# above: the three edge-level read-outs plus the node-level one, per held-out mesh;
# bar = mean over the seeds with a checkpoint, whisker = ± one std, dots = the
# seeds. Last-epoch checkpoints, from vol_flux_tab.
targets = [r"$F$", r"$\Delta T$", r"$w$", r"$\sum_j F_{ij}$ (node)"]
x = np.arange(len(targets))
width = 0.3

fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs FV Flux, No Volume vs Log Cell Volume, Noise 0.3, No Positions, Cf\n"
             "(mean ± std over seeds)", fontsize=24, color=AX_COLOR)

for j, (ax, mesh) in enumerate(zip(axs.ravel(), VOL_MESHES)):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, key in enumerate(VOL_CASES):
        color = VOL_STYLES[key][0]
        t = vol_flux_tab[key][:, j, :4]                       # (S, 4) R^2
        t = t[~np.isnan(t[:, 0])]
        off = (i - (len(VOL_CASES) - 1) / 2) * (width + 0.02)
        sd = t.std(0, ddof=1) if len(t) > 1 else None
        ax.bar(x + off, t.mean(0), width, color=color, label=f"{key} ({len(t)} seeds)",
               yerr=sd, error_kw=dict(ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5))
        for row in t:
            ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.8, alpha=0.7)

    ax.set_title(mesh, fontsize=19)
    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
    ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                    loc="outside lower center", ncols=len(VOL_CASES), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_volume_flux_r2.png", dpi=300, transparent = True)

### What comes out

All 6 seed checkpoints are here: **no volume** (`parametric_history1_msg_dim64_residual_noise0.3_cf_nopos`,
`seed_<n>_01`, commit `f8f9112` + uncommitted changes) and **log volume**
(`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos`, `seed_<n>`, commit `48186db` +
uncommitted changes). The recomputed val rollouts match the ones logged at the last epoch to
$5\times10^{-5}$ relative, and the recomputed test rollout RMAE matches the one from the saved MAE
to $6\times10^{-7}$. With 3 seeds a side the smallest Mann-Whitney $p$ is 0.1, reached whenever the
three seeds of one run are all below the three of the other; nothing below can reach 0.05.

**Both runs are stable.** From both checkpoints, every seed of both runs stays below the 300 K span
on all four test meshes for all 120 steps (12 of 12 pairs each). The volume does not change what
input noise 0.3 already gives.

**With the volume every error is lower, by a little in RMAE and by about a third in one-step RMSE.**
At the last epoch:

- One-step RMSE is 29 – 40 % lower on every mesh (e.g. $4.5$ to $2.7\times10^{-5}$ on `model_003`,
  $1.20$ to $0.74\times10^{-5}$ on `model_007`); $p = 0.1$ on `model_011`, 0.2 elsewhere. The
  one-step RMAE is 9 % lower on `model_003` ($p = 0.1$) and within 3 % on the other three.
- The 120-step rollout RMAE is 3 – 6 % lower on every mesh: $9.6$ to $9.1\times10^{-3}$, $3.8$ to
  $3.6\times10^{-3}$, $4.9$ to $4.6\times10^{-3}$, $13.5$ to $13.1\times10^{-3}$. $p = 0.1$ on
  `model_003`, `model_007` and `model_010` (every volume seed below every control seed), 0.4 on
  `model_011`. The rollout RMSE is 8 – 18 % lower.
- The MAE over steps 1 – 10 is 13 – 17 % lower: 5.1 to 4.3, 2.1 to 1.8, 1.7 to 1.5, 2.4 to 2.0 K.
- The 40-step val rollout MAE at the last epoch, mean over both val meshes, is 1.06 – 1.11 K per
  seed against 1.17 – 1.26 K, and every volume seed is below every control seed on each val mesh
  ($p = 0.1$).

The RMSE gains being several times the RMAE ones means the volume mostly removes the *largest*
one-step errors rather than shifting all of them. Where those errors sit is in the follow-ups below: not in the smallest cells.

**Training.** The best validation loss is $3.7 \pm 1.2$ against $5.7 \pm 1.8\times10^{-5}$ (35 %
lower, $p = 0.2$: the seed ranges overlap, 2.5 – 4.9 against 3.9 – 7.5). The first LR cut comes
earlier with the volume, at epoch $48 \pm 4$ against $61 \pm 7$ ($p = 0.1$), yet the volume runs
train as long or longer (181 – 240 against 159 – 240 epochs) and reach their best validation
epoch later ($170 \pm 20$ against $151 \pm 41$). The training loss, on noisy inputs, ends at
$2.4$ against $3.8\times10^{-4}$ ($p = 0.2$).

**The best-rollout checkpoint again does not carry over.** On the test meshes its 120-step rollout RMAE
and RMSE are higher than the last epoch's for both runs on every mesh, as in the input-noise
section (its MAE over steps 1 – 10 is lower only for the volume run on `model_010`, 1.43 against
1.48 K).
Between the two best-rollout checkpoints the volume is again ahead on every rollout metric, but by
less consistently ($p \geq 0.1$ everywhere, 1.0 for the RMAE on `model_010`), and on the val meshes,
where they were picked, the two are the same (0.99 against 1.02 K and 0.90 against 0.93 K, $p = 1$).

**The message is slightly less flux-like per edge.** The $R^2$ of the linear read-out of $F$ falls
from 0.86 – 0.89 to 0.82 – 0.86, by 0.03 – 0.04 on every mesh ($p = 0.1$ on `model_003` and
`model_007`). $\Delta T$ falls by 0.01 – 0.02 and the node-level $R^2$ of $\sum_j m_{ij}$ on
$\sum_j F_{ij}$ by 0.01 – 0.03; the geometric weight $w$ rises by 0.01 – 0.02. None of these but
$F$ separates the seeds. The antisymmetric fraction is unchanged on average (0.22 – 0.25 against
0.24 – 0.26), but its spread over the seeds is 7 – 13 times smaller with the volume (± 0.006 – 0.010
against ± 0.073 – 0.080). The read-out targets are the flux $F_{ij}$ itself, not the
increment $F_{ij}/V_j$ it makes in the receiving cell: with $V$ in the node update the network no longer needs the
message to carry the $1/V$ scaling, so a lower $R^2$ on $F$ is not by itself a sign of a worse
message. The follow-ups below read out $F_{ij}/V_j$ as well.

**What this shows.** The log cell volume is a cheap feature (64 parameters) that lowers every test
error of the noise 0.3 model, on every held-out mesh: the one-step RMSE by about a third, the
120-step rollout RMAE by 3 – 6 %, the early rollout by 13 – 17 %. The direction is the same on every
mesh and metric at the last epoch, and the rollout RMAE separates the seeds completely on three of
four meshes. The sizes are another matter: with 3 unpaired seeds a side, a 3 – 6 % rollout gain
is not distinguishable from a lucky draw of initial weights, and the validation loss, the one
quantity with a large gain, overlaps between the runs. Both runs used a 500-epoch budget with
early stopping and input noise 0.3; whether the volume helps the same way without noise, or with
positions, is not tested.

### Follow-ups: errors per refinement level, and the read-out of $F/V$

Two questions the results above leave open, on the same last-epoch checkpoints (the 3 seeds of
**no volume**, commit `f8f9112` + uncommitted changes, and of **log volume**, commit `48186db` +
uncommitted changes).

**Where the volume removes error.** The one-step RMSE fell by 29 – 40 % and the RMAE by 0 – 9 %,
so the gain is in the largest errors. The cells are grouped by refinement level,
$\ell = \operatorname{round}(\log_8 (V_0/V))$ with $V_0$ the background blockMesh cell
($6.84\times10^{-8}$ m³, the largest cell of every mesh), so level 0 is the coarsest and each
level is $8\times$ smaller. Per level, over that level's cells only, the one-step RMAE and RMSE
(summed over every step) and the RMAE / RMSE over rollout steps 1 – 10, plus each level's share
of the mesh's one-step squared error. Boundary nodes are left out, and so is any level with fewer
than 100 cells on a mesh.

**What the message carries.** `compute_model_flux_stats` reads out the flux $F_{ij}$. A cell's
temperature responds to $F_{ij}/V_j$, the increment the flux makes in the receiving cell $j$.
Without $V$ the model has no direct way to apply that $1/V_j$ at the node update, so it could only
build it into the message from the edge geometry; with $V$ it can scale at the node. The read-out
is repeated on the same faces and timesteps for both targets: per edge $m_{ij}$ against $F_{ij}$
and $F_{ij}/V_j$, per cell $\sum_j m_{ij}$ against $\sum_j F_{ij}$ and $\sum_j F_{ij}/V_i$. The $F$
columns reproduce the read-out above, which is checked.

In [ ]:
# --- Log cell volume: errors per refinement level ------------------------------------
# The one-step error and the rollout error over steps 1-10, summed per refinement
# level of the internal cells, for the last-epoch checkpoints. The snappyHexMesh
# levels are 8x apart in volume, so a cell's level is log8(V0 / V) rounded, with
# V0 the largest cell of the meshes here (the background blockMesh cell, the same
# on every mesh): level 0 the coarsest, 5 the finest. Boundary nodes are left out.
# Per level the RMAE / RMSE are taken over that level's cells alone, as everywhere
# else (sum of |e| over sum of |T|, sum of e^2 over sum of T^2).
from mesh2graph.utils import load_cell_volumes

vol_lvl_cache = processed_data_dir / "parametric_eval_volume_levels.npz"
VOL_ROLL_LVL_STEPS = 10
VOL_MIN_CELLS = 100           # a level with fewer cells on a mesh is not shown

vol_cell_V = {m: load_cell_volumes(str(raw_param_dir / m)).numpy().ravel() for m in VOL_MESHES}
VOL_V0 = max(v.max() for v in vol_cell_V.values())
vol_level = {m: np.floor(np.log(VOL_V0 / v) / np.log(8) + 0.5).astype(int)
             for m, v in vol_cell_V.items()}
VOL_LEVELS = np.arange(max(l.max() for l in vol_level.values()) + 1)
vol_lvl_n = np.stack([np.bincount(vol_level[m], minlength=len(VOL_LEVELS)) for m in VOL_MESHES])  # (M, L)


def vol_level_sums(e, T_t, lvl):
    """(L, 4) sums of |e|, |T|, e^2, T^2 per level over the internal cells; e and
    T_t are (steps, N) with the cells first, in cell order."""
    n = len(lvl)
    idx = torch.from_numpy(lvl)
    out = torch.zeros(len(VOL_LEVELS), 4, dtype=torch.float64)
    for c, x in enumerate((e.abs(), T_t.abs(), e.pow(2), T_t.pow(2))):
        out[:, c].index_add_(0, idx, x[:, :n].double().sum(0))
    return out.numpy()


def vol_level_evaluate(run, load):
    """Per-level one-step and steps 1-10 rollout sums of model.pt on the test
    meshes, keyed <run>__<mesh>__lvl_step / __lvl_roll."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / "model.pt", map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in VOL_MESHES:
        graph, T_seq = load(mesh)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        assert n_int == len(vol_level[mesh]), f"{mesh}: {n_int} internal nodes, {len(vol_level[mesh])} cells"
        sums = np.zeros((len(VOL_LEVELS), 4))
        with torch.no_grad():
            for i in range(len(ds)):
                d = ds[i].to(device)
                T_p = norm.inverse_transform_T(model(d).cpu()).double()
                T_t = norm.inverse_transform_T(d.y.cpu()).double()
                sums += vol_level_sums((T_p - T_t)[None], T_t[None], vol_level[mesh])
        out[f"{run}__{mesh}__lvl_step"] = sums
        T_p, T_t = rollout(model, ds, device=device, n_steps=VOL_ROLL_LVL_STEPS)
        out[f"{run}__{mesh}__lvl_roll"] = vol_level_sums((T_p - T_t).double(), T_t.double(),
                                                         vol_level[mesh])
        del model
    return out


vol_lvl = (dict(np.load(vol_lvl_cache, allow_pickle=False))
           if vol_lvl_cache.exists() else {})
for key, runs in vol_runs.items():
    for run, ok in runs:
        if not ok or vol_lvl.get(f"{run}__stamp") == pos_stamp(run):
            continue
        vol_lvl.update(vol_level_evaluate(run, VOL_LOAD[key]))
        vol_lvl[f"{run}__stamp"] = pos_stamp(run)
        np.savez_compressed(vol_lvl_cache, **vol_lvl)


def vol_lvl_metric(key, mode, metric):
    """(S, M, L) per-level RMAE ("rmae"), RMSE ("rmse") or share of the mesh's
    squared error ("e2_share"); nan where a seed lacks the checkpoint or a level
    has fewer than VOL_MIN_CELLS cells."""
    out = np.full((len(vol[key]["seeds"]), len(VOL_MESHES), len(VOL_LEVELS)), np.nan)
    for i, (run, ok) in enumerate(vol_runs[key]):
        if not ok:
            continue
        for j, mesh in enumerate(VOL_MESHES):
            s = vol_lvl[f"{run}__{mesh}__lvl_{mode}"]
            with np.errstate(invalid="ignore"):          # 0 / 0 on a level with no cells
                out[i, j] = {"rmae": s[:, 0] / s[:, 1], "rmse": s[:, 2] / s[:, 3],
                             "e2_share": s[:, 2] / s[:, 2].sum()}[metric]
    return np.where(vol_lvl_n[None] >= VOL_MIN_CELLS, out, np.nan)


print(f"cells per level (level 0: V0 = {VOL_V0:.3g} m^3, each level 8x smaller); "
      f"levels with fewer than {VOL_MIN_CELLS} cells left out")
print(f"{'':11s}" + "".join(f"{f'level {l}':>10}" for l in VOL_LEVELS))
for j, mesh in enumerate(VOL_MESHES):
    print(f"{mesh:<11s}" + "".join(f"{n:>10d}" for n in vol_lvl_n[j]))

# Per mesh and level: each run's geometric mean over seeds, the ratio log volume /
# no volume of those means, and the Mann-Whitney p.
VOL_LVL_COLS = {("step", "rmae"): "one-step RMAE", ("step", "rmse"): "one-step RMSE",
                ("roll", "rmae"): f"steps 1-{VOL_ROLL_LVL_STEPS} RMAE",
                ("roll", "rmse"): f"steps 1-{VOL_ROLL_LVL_STEPS} RMSE"}
for (mode, metric), title in VOL_LVL_COLS.items():
    a, b = (vol_lvl_metric(k, mode, metric) for k in VOL_CASES)
    print(f"\n{title}: no volume / log volume (geometric means over seeds), ratio, p (MWU)")
    print(f"{'':11s}" + "".join(f"{f'level {l}':>33}" for l in VOL_LEVELS))
    for j, mesh in enumerate(VOL_MESHES):
        row = f"{mesh:<11s}"
        for l in VOL_LEVELS:
            if np.isnan(a[:, j, l]).all():
                row += f"{'—':>33}"
                continue
            ma, mb = (10 ** seed_stats(x[:, j, l], log=True)[0] for x in (a, b))
            row += f"{f'{ma:.2e} / {mb:.2e}  {mb / ma:.2f}  {pos_p(a[:, j, l], b[:, j, l]):.1f}':>33}"
        print(row)

# Where the squared error is: each level's share of the mesh's one-step e^2
# (arithmetic mean over seeds), next to its share of the cells.
print(f"\none-step squared error, share per level (mean over seeds): "
      f"cells / no volume / log volume")
print(f"{'':11s}" + "".join(f"{f'level {l}':>22}" for l in VOL_LEVELS))
sh = {k: vol_lvl_metric(k, "step", "e2_share") for k in VOL_CASES}
for j, mesh in enumerate(VOL_MESHES):
    row = f"{mesh:<11s}"
    for l in VOL_LEVELS:
        if vol_lvl_n[j, l] < VOL_MIN_CELLS:
            row += f"{'—':>22}"
            continue
        cell = " / ".join(f"{x:.2f}" for x in (vol_lvl_n[j, l] / vol_lvl_n[j].sum(),
                                                np.nanmean(sh["no volume"][:, j, l]),
                                                np.nanmean(sh["log volume"][:, j, l])))
        row += f"{cell:>22}"
    print(row)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The per-level errors above as a figure. Rows: one-step RMAE, one-step RMSE and
# the RMSE over rollout steps 1-10; columns: the held-out meshes; x: the refinement
# level, coarsest left. Each run's geometric mean over the seeds (line) with ×/÷
# one geometric std (whisker) and the seeds (dots), slightly apart so both show.
vol_lvl_rows = [("step", "rmae"), ("step", "rmse"), ("roll", "rmse")]

fig, axs = plt.subplots(len(vol_lvl_rows), len(VOL_MESHES), figsize=(1800*px, 1250*px),
                        sharex=True, sharey="row", constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Errors per Refinement Level, No Volume vs Log Cell Volume, Noise 0.3, No Positions, Cf\n"
             "(last-epoch checkpoints; geometric mean ×/÷ std over seeds, dots: seeds)",
             fontsize=22, color=AX_COLOR)

for r, (mode, metric) in enumerate(vol_lvl_rows):
    for col, mesh in enumerate(VOL_MESHES):
        ax = axs[r, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for k, key in enumerate(VOL_CASES):
            color, ls = VOL_STYLES[key]
            lv = vol_lvl_n[col] >= VOL_MIN_CELLS                     # the levels shown
            v = vol_lvl_metric(key, mode, metric)[:, col, lv]        # (S, L shown)
            mu, sd = seed_stats(v, log=True)
            xs = VOL_LEVELS[lv] + (k - 0.5) * 0.12
            ax.errorbar(xs, 10 ** mu, yerr=[10 ** mu - 10 ** (mu - sd), 10 ** (mu + sd) - 10 ** mu],
                        color=color, linestyle=ls, linewidth=2, marker="s", markersize=5,
                        capsize=3, label=key)
            for row in v:
                ax.plot(xs, row, "o", color=AX_COLOR, markersize=2.6, alpha=0.6)
        ax.set_yscale("log")
        ax.set_xticks(VOL_LEVELS)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
        if r == 0:
            ax.set_title(mesh, fontsize=19)
        if r == len(vol_lvl_rows) - 1:
            ax.set_xlabel("refinement level (0: coarsest)", fontsize=15)
        if col == 0:
            ax.set_ylabel(VOL_LVL_COLS[(mode, metric)], fontsize=15)

legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                    loc="outside lower center", ncols=len(VOL_CASES), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_volume_levels.png", dpi=300, transparent = True)

In [ ]:
# --- Log cell volume: the message against F and against F / V -----------------------
# compute_model_flux_stats reads out the flux F_ij. What a cell's temperature
# responds to is F_ij / V_j, the increment the flux makes in the receiving cell j
# (the edge's target, where the message is aggregated). This repeats its read-out
# on the same faces and timesteps (same seed) for both targets: per edge, the
# message m_ij against F_ij and F_ij / V_j; per cell, sum_j m_ij against
# sum_j F_ij and sum_j F_ij / V_i. The F columns must reproduce vol_flux_tab.
vol_fv_cache = processed_data_dir / "parametric_message_flux_volume_fv.npz"


def vol_flux_over_volume(run, load, layer=-1):
    # layer as in compute_model_flux_stats: the last by default, another one
    # keyed <run>__L<layer>__...
    name = run if layer == -1 else f"{run}__L{layer}"
    """R^2 of the linear read-out of F and F / V from the message (per edge) and
    from the aggregate (per cell), keyed <run>__<mesh>__r2_fv_edge / __r2_fv_node."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / "model.pt", map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in VOL_MESHES:
        graph, T_seq = load(mesh)
        case_dir = str(raw_param_dir / mesh)
        times = filter_of_time_directories(case_dir)

        ei, ea = graph.edge_index, graph.edge_attr   # RAW geometry
        N, E = graph.num_nodes, ei.shape[1]
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)

        # the same whole faces and timesteps as compute_model_flux_stats
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()

        V = torch.from_numpy(vol_cell_V[mesh]).double()      # (n_int,), cells in node order
        assert len(V) == n_int, f"{mesh}: {n_int} internal nodes, {len(V)} cells"
        gradT = load_cell_gradient(case_dir, [times[i] for i in steps])
        T_at = T_seq[steps].double()
        F = fv_flux_corrected(ei, ea, T_at, gradT, DT=DT_PARAM,
                              n_internal_nodes=n_int)            # (S, E)
        F_node = torch.zeros(N_STEPS, N, dtype=torch.double)
        F_node.index_add_(1, ei[1], F)                           # (S, N)
        edge_tgts = [F[:, kept].reshape(-1), (F[:, kept] / V[ei[1, kept]]).reshape(-1)]
        node_tgts = [F_node[:, :n_int].reshape(-1), (F_node[:, :n_int] / V).reshape(-1)]

        ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)
        model = FVSurrogate(
            in_node_feat=HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()

        msg_kept, aggr_all = [], []
        probe = MessageProbe(model)
        with probe, torch.no_grad():
            for i in steps:
                model(ds[i].to(device))
                msg_kept.append(probe.messages[layer][kept])
                aggr_all.append(probe.aggregated[layer])
                probe.clear()
        msg = torch.stack(msg_kept)                 # (S, kept, C)
        aggr = torch.stack(aggr_all)                # (S, N, C)
        m_flat = msg.reshape(-1, msg.shape[-1]).double()
        a_flat = aggr[:, :n_int].reshape(-1, msg.shape[-1]).double()
        out[f"{name}__{mesh}__r2_fv_edge"] = np.array([_r2(m_flat, t) for t in edge_tgts])
        out[f"{name}__{mesh}__r2_fv_node"] = np.array([_r2(a_flat, t) for t in node_tgts])
        del msg, aggr, m_flat, a_flat, probe, model
    return out


vol_fv = (dict(np.load(vol_fv_cache, allow_pickle=False))
          if vol_fv_cache.exists() else {})
for key, runs in vol_runs.items():
    for run, ok in runs:
        if not ok or vol_fv.get(f"{run}__stamp") == pos_stamp(run):
            continue
        vol_fv.update(vol_flux_over_volume(run, VOL_LOAD[key]))
        vol_fv[f"{run}__stamp"] = pos_stamp(run)
        np.savez_compressed(vol_fv_cache, **vol_fv)   # after every run: these are slow


def vol_fv_table(key):
    """(S, M, 4) R^2 of F and F / V per edge, then of sum F and sum F / V per cell."""
    out = np.full((len(vol[key]["seeds"]), len(VOL_MESHES), 4), np.nan)
    for i, (run, ok) in enumerate(vol_runs[key]):
        if ok:
            for j, mesh in enumerate(VOL_MESHES):
                out[i, j] = [*vol_fv[f"{run}__{mesh}__r2_fv_edge"], *vol_fv[f"{run}__{mesh}__r2_fv_node"]]
    return out


vol_fv_tab = {key: vol_fv_table(key) for key in VOL_CASES}

# The F columns are compute_model_flux_stats' own R^2 F and R2 node.
dev = max(np.nanmax(np.abs(vol_fv_tab[k][..., [0, 2]] - vol_flux_tab[k][..., [0, 3]]))
          for k in VOL_CASES)
print(f"R^2 of F and sum F against compute_model_flux_stats: max |difference| = {dev:.1e}\n")

VOL_FV_COLS = ("edge F", "edge F/V", "cell ΣF", "cell ΣF/V")
print(f"Linear read-out of F and F / V, {N_STEPS} probed timesteps, last-epoch checkpoints, "
      f"mean ± std over seeds")
print(f"{'mesh':<11}{'run':<14}" + "".join(f"{c:>16}" for c in VOL_FV_COLS))
for j, mesh in enumerate(VOL_MESHES):
    for key in VOL_CASES:
        t = vol_fv_tab[key][:, j]
        t = t[~np.isnan(t[:, 0])]
        print(f"{mesh:<11}{key:<14}" + "".join(
            f"{f'{m:.3f} ± {s:.3f}':>16}" for m, s in zip(t.mean(0), t.std(0, ddof=1))))
    print(f"{mesh:<11}{'  p (MWU)':<14}" + "".join(
        f"{pos_p(vol_fv_tab['no volume'][:, j, c], vol_fv_tab['log volume'][:, j, c]):>16.3f}"
        for c in range(len(VOL_FV_COLS))))
    print()

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The read-out of F against F / V, laid out as the flux figure above: per edge F_ij
# and F_ij / V_j, per cell sum_j F_ij and sum_j F_ij / V_i, per held-out mesh;
# bar = mean over the seeds, whisker = ± one std, dots = the seeds. Last-epoch
# checkpoints, from vol_fv_tab.
targets = [r"$F_{ij}$", r"$F_{ij}/V_j$", r"$\sum_j F_{ij}$", r"$\sum_j F_{ij}/V_i$"]
x = np.arange(len(targets))
width = 0.3

fig, axs = plt.subplots(2, 2, figsize=(1500*px, 950*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs F and F / V, No Volume vs Log Cell Volume, Noise 0.3, No Positions, Cf\n"
             "(edge: m_ij, cell: Σ_j m_ij; mean ± std over seeds)", fontsize=24, color=AX_COLOR)

for j, (ax, mesh) in enumerate(zip(axs.ravel(), VOL_MESHES)):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, key in enumerate(VOL_CASES):
        color = VOL_STYLES[key][0]
        t = vol_fv_tab[key][:, j]                             # (S, 4) R^2
        t = t[~np.isnan(t[:, 0])]
        off = (i - (len(VOL_CASES) - 1) / 2) * (width + 0.02)
        sd = t.std(0, ddof=1) if len(t) > 1 else None
        ax.bar(x + off, t.mean(0), width, color=color, label=f"{key} ({len(t)} seeds)",
               yerr=sd, error_kw=dict(ecolor=AX_COLOR, elinewidth=1.1, capsize=2.5))
        for row in t:
            ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.8, alpha=0.7)

    ax.set_title(mesh, fontsize=19)
    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("$R^2$ of the linear read-out", fontsize=15)
    ax.grid(True, axis="y", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)
    ax.set_axisbelow(True)

legend = fig.legend(*axs[0, 0].get_legend_handles_labels(),
                    loc="outside lower center", ncols=len(VOL_CASES), fontsize=13)
legend.get_frame().set_facecolor(LEGEND_BG)
legend.get_frame().set_edgecolor(AX_COLOR)
legend.get_frame().set_alpha(1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)

# plt.savefig("../outputs/plots/parametric_history1_volume_flux_per_volume_r2.png", dpi=300, transparent = True)

### What the follow-ups show

Same checkpoints as above: **no volume** (`parametric_history1_msg_dim64_residual_noise0.3_cf_nopos`,
`seed_<n>_01`, commit `f8f9112` + uncommitted changes) and **log volume**
(`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos`, `seed_<n>`, commit `48186db` +
uncommitted changes), last epoch. The $F$ read-outs reproduce `compute_model_flux_stats` to
$1\times10^{-9}$.

**The large errors are not in the smallest cells.** Most of the one-step squared error sits at
level 3: 63 – 81 % of it on `model_003`, `model_007` and `model_010`, which have 21 – 40 % of
their cells there. On `model_011` it is level 2, 78 – 82 % of the squared error with 51 % of the
cells. The finest levels have the *smallest* errors: on `model_007` level 5 holds 36 % of the
cells and 2 – 3 % of the squared error, and levels 4 and 5 have the lowest one-step RMAE of every
mesh that has them. Both runs share this distribution; the volume barely moves it (level 3 of `model_003`: 81 to
78 %).

**The volume lowers the error at levels 0 – 3, most at the coarsest; in the finest it is mixed.**
Log volume over no volume, ratio of the geometric means over seeds, range over the meshes:

| level | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| one-step RMSE | 0.64 – 0.73 | 0.74 – 0.90 | 0.63 – 0.69 | 0.56 – 0.82 | 0.87 – 1.01 | 0.84 |
| one-step RMAE | 0.80 – 0.84 | 0.91 – 0.96 | 0.91 – 0.97 | 0.89 – 0.96 | 0.99 – 1.01 | 0.77 |
| steps 1 – 10 RMSE | 0.49 – 0.65 | 0.64 – 0.74 | 0.67 – 0.71 | 0.59 – 0.83 | 0.83 – 0.92 | 0.86 |

Level 5 is on `model_007` only, level 4 on `model_003` and `model_007`. At level 0 every volume
seed is below every control seed in one-step RMAE on all four meshes ($p = 0.1$), and in one-step
RMSE too; at levels 1 – 3 the seeds separate on some meshes and not others ($p$ 0.1 – 0.7). The
largest absolute gain is at level 3, where the error is (one-step RMSE ×0.57 and ×0.56 on
`model_003` and `model_007`, $p = 0.4$: one seed of each run crosses). At levels 4 and 5 the ratios
range from 0.77 to 1.01 and none of the three metrics separates the seeds ($p \geq 0.4$). Within each of levels
0 – 3, on every mesh, the RMSE ratio is below the RMAE ratio, as over the whole mesh: the volume
shrinks the large errors inside each level, not one level's errors against another's.

**The message tracks $F$, not $F/V$, in both runs.** Per edge the read-out of $F_{ij}$ reaches
$R^2 = 0.82$ – $0.89$, of $F_{ij}/V_j$ only 0.18 – 0.65. Without the volume the message is *less*
$F/V$-like, not more: 0.32, 0.18, 0.46, 0.61 on the four meshes, against 0.41, 0.23, 0.49, 0.65 with
it (every seed above on `model_003` and `model_007`, $p = 0.1$). So the model without $V$ does not
build the $1/V_j$ scaling into the message from the edge geometry. With $V$ the message moves a
little from $F$ toward $F/V$: $F$ down 0.03 – 0.04, $F/V$ up 0.03 – 0.09. Per cell the aggregate
$\sum_j m_{ij}$ explains the net flux $\sum_j F_{ij}$ (0.67 – 0.78) and the rate $\sum_j F_{ij}/V_i$
(0.58 – 0.82) to a similar degree: the net flux better on `model_007` and `model_010`, the rate
better on `model_011`. Neither changes with the volume ($p \geq 0.2$), though with it the seed
spread of the rate read-out is smaller on three meshes (± 0.009 – 0.024 against ± 0.026 – 0.054).

**What this adds.** The guess that the volume would mostly help the smallest cells is wrong: they
are where the errors are smallest, and the volume's effect there is mixed. The gain is broad, over the
coarse and mid levels, and largest in relative terms at the coarsest level. The volume reaches the
message (it is a node feature, and the message function sees both end nodes), which shifts slightly
toward the per-volume increment; the aggregate's relation to the cell's flux balance does not
change.

## FV features vs none, with input noise (`_nofv`)

The best recipe of the sections above against the same model without its finite-volume
inputs: no face geometry on the edges and no cell volume on the nodes, with input noise 0.3 as
in the best recipe, and without it.

- **FV + log volume** (`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos`, `seed_<n>`
  in `seeds.npz`, seeds 0 – 2, commit `48186db` + uncommitted changes): the log-volume runs of
  the section above. Edge features $\mathbf{d}_{ij}$, $|\mathbf{d}_{ij}|$, $\mathbf{S}_f$,
  $|\mathbf{S}_f|$, skewness and non-orthogonality (10); node features
  $[\,\text{internal}, \text{boundary}, \log V\,]$.
- **no FV** (`parametric_history1_msg_dim64_residual_noise0.3_cf_nopos_nofv`, seeds 0 – 2, commit
  `df7f45d` + uncommitted changes as recorded by seed 0): only the first 4 edge features,
  $\mathbf{d}_{ij}$ and $|\mathbf{d}_{ij}|$, the usual mesh-GNN ones, and node features
  $[\,\text{internal}, \text{boundary}\,]$. Seeds 1 – 2 are `seed_<n>` in `seeds.npz`. Seed 0 is
  `seed_0_01` in `seeds_01.npz`: a first launcher (`logs/20261020_121944`) was stopped during it,
  and the unfinished `seed_0` it left is not used.
- **no FV, no noise** (`parametric_history1_msg_dim64_residual_cf_nopos_nofv`, `seed_<n>` in
  `seeds.npz`, seeds 0 – 2): the no-FV model trained on clean inputs. Its runs had not started
  when this was written, so their commit is not given here; the cell below prints every seed's.

Both `_nofv` configurations come from one sweep, `logs/20261020_122553`, with
`FVGNN_FV_FEATURES = 0`. Its copy of `train_parametric.py` differs from the one of the FV runs
only by the multi-layer and skip options, which change nothing with one layer, and by that
switch. Everything else is shared: the 64-channel residual model at `history = 1` with one MP
layer, no LayerNorm and no message penalty, Cf boundary nodes, no positions, SOAP from
$3\times10^{-3}$ with `ReduceLROnPlateau` from epoch 29, `batch_size = 2`, at most 500 epochs
with the early-stopping rule (`min_lr` $= 3\times10^{-9}$). Without the FV inputs the encoders
lose $6 \times 64$ weights on the edges and $1 \times 64$ on the nodes (116 225 parameters
instead of 116 673).

**Not paired.** The narrower encoders change the initialisation, so the CPU generator's draws
differ from there on: seed $n$ starts from other weights than FV seed $n$ ($c_0$ differs,
printed below) and sees other batches. Each no-FV configuration is compared with the FV run as
two samples: Mann-Whitney U, two-sided. With 3 seeds a side its smallest $p$ is 0.1.

**Two checkpoints per run**: the last-epoch `model.pt` and `model_best_rollout.pt`, both rolled
out on the test meshes at training time.

- **Losses and LR**, and the val rollout each run logged.
- **Test rollout** (plotted per step as RMAE): seeds below the span of $T$, MAE over steps
  1 – 10, first step above the span, rollout RMAE, for both checkpoints, from the MAE saved at
  training time.
- **Message vs FV flux**: the linear read-out of $F_{ij}$, $\Delta T_{ij}$, $w_{ij}$ and
  $\sum_j F_{ij}$ from the message, as in the sections above, last-epoch checkpoints. The no-FV
  models are fed only the first 4 edge features, but the reference flux is computed from the
  whole face geometry, as for every other run; $w_{ij}$ depends on $|\mathbf{S}_f|$ and the
  non-orthogonality, which they never see.

The cells reuse `vol_stats`, `vol_last`, `vol_rmae` and `VOL_MESHES` from the section above,
`pos_p` and `pos_absT` from the positions section, and `seed_stats`, `NS_SPAN` and
`ns_seed_run` from the sections before.

In [ ]:
# --- FV features vs none: load the seeds, check them, analyse the test rollouts ----
# Three configurations, each from one or more seeds*.npz (aggregate_seeds.py), as
# (experiment, rerun index): 0 is seed_<n> in seeds.npz, i is seed_<n>_<ii> in
# seeds_<ii>.npz. The no-FV noise-0.3 sweep is split across two: its seed 0 ran as
# seed_0_01 after the first launcher was stopped mid-run (logs/20261020_121944), and
# aggregate_seeds.py leaves the unfinished seed_0 out.
# NOT paired: without the FV inputs the encoders are narrower, so the initialisation
# draws (c0, printed below) and every shuffle after them differ. Mann-Whitney U,
# two-sided, against the FV run: with 3 seeds a side its smallest p is 0.1.
NF_BASE = "parametric_history1_msg_dim64_residual"
NF_SRC = {"FV + log volume": [(f"{NF_BASE}_noise0.3_cf_vol_nopos", 0)],
          "no FV":           [(f"{NF_BASE}_noise0.3_cf_nopos_nofv", 0),
                              (f"{NF_BASE}_noise0.3_cf_nopos_nofv", 1)],
          "no FV, no noise": [(f"{NF_BASE}_cf_nopos_nofv", 0)]}
NF_REF = "FV + log volume"       # what the other two are tested against
NF_NOISE = {"FV + log volume": 0.3, "no FV": 0.3, "no FV, no noise": 0.0}
NF_FV = {"FV + log volume": True, "no FV": False, "no FV, no noise": False}
NF_SEEDS = [0, 1, 2]
NF_MESHES = VOL_MESHES
NF_BUDGET = 500                  # max_epochs of every sweep
# (edge, node) features: with FV the 10 edge features and node type + log V,
# without it the first 4 and the node type
NF_FEATS = {True: (10, 3), False: (4, 2)}
NF_EPOCH_KEYS = ["train_losses", "val_losses", "lrs", "val_rollout_mae"]
NF_SEED_KEYS = ["seeds", "n_params", "n_epochs", "msg_c0", "val_rollout_best_epoch",
                *(f"rollout_mae_{m}" for m in NF_MESHES),
                *(f"rollout_mae_best_{m}" for m in NF_MESHES)]
NF_META = ("input_noise", "max_epochs", "msg_kind", "msg_alpha", "msg_warmup", "msg_ramp_len")


def nf_npz(exp, i):
    return processed_data_dir / exp / ("seeds.npz" if i == 0 else f"seeds_{i:02d}.npz")


nf_no_npz = [str(nf_npz(exp, i)) for src in NF_SRC.values() for exp, i in src
             if not nf_npz(exp, i).exists()]
if nf_no_npz:
    raise FileNotFoundError(
        f"no {', '.join(nf_no_npz)} yet: the launcher.sh sweep aggregates when it ends, "
        f"or python aggregate_seeds.py <exp_name>:<rerun index>")


def nf_load(sources):
    """The seeds of one configuration from its seeds*.npz, sorted by seed, with every
    per-epoch field NaN-padded to the longest seed of any of the files (each file is
    padded only to its own), and the run path of each seed."""
    zs = [dict(np.load(nf_npz(exp, i))) for exp, i in sources]
    E = max(z["train_losses"].shape[1] for z in zs)
    parts, runs = [], []
    for (exp, i), z in zip(sources, zs):
        S = len(z["seeds"])
        # n_params is stored once when every seed shares it
        part = {k: np.broadcast_to(z[k], (S,)) if k == "n_params" else z[k] for k in NF_SEED_KEYS}
        for k in NF_EPOCH_KEYS:
            a = z[k].astype(float)
            part[k] = np.concatenate([a, np.full((S, E - a.shape[1], *a.shape[2:]), np.nan)], 1)
        part["meta"] = {k: z[k].item() for k in NF_META}
        part["meshes"] = (list(z["test_meshes"]), [str(m) for m in z["val_meshes"]])
        parts.append(part)
        runs += [ns_seed_run(exp, f"{int(s)}_{i:02d}" if i else int(s)) for s in z["seeds"]]
    for p in parts[1:]:
        assert p["meta"] == parts[0]["meta"] and p["meshes"] == parts[0]["meshes"], \
            f"{sources}: the sweeps differ, {p['meta']} against {parts[0]['meta']}"
    out = {k: np.concatenate([p[k] for p in parts]) for k in NF_SEED_KEYS + NF_EPOCH_KEYS}
    order = np.argsort(out["seeds"])
    out = {k: v[order] for k, v in out.items()}
    out.update(parts[0]["meta"])
    out["test_meshes"], out["val_meshes"] = parts[0]["meshes"]
    return out, [runs[j] for j in order]


nf, nf_runs = {}, {}
for key, src in NF_SRC.items():
    nf[key], nf_runs[key] = nf_load(src)
ref = nf[NF_REF]
NF_VAL_MESHES = ref["val_meshes"]
NF_VAL_STEPS = ref["val_rollout_mae"].shape[-1]

# Each configuration is what its name says, on the same seeds, meshes, budget and
# schedule, and the same model but for the encoder weights on the FV inputs.
for key, z in nf.items():
    assert z["seeds"].tolist() == NF_SEEDS, f"{key}: seeds {z['seeds']}"
    assert z["test_meshes"] == NF_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert z["val_meshes"] == NF_VAL_MESHES, f"{key}: val meshes {z['val_meshes']}"
    assert float(z["input_noise"]) == NF_NOISE[key], f"{key}: input_noise = {z['input_noise']}"
    assert int(z["max_epochs"]) == NF_BUDGET, f"{key}: launched for {z['max_epochs']} epochs"
    assert str(z["msg_kind"]) == "none" and float(z["msg_alpha"]) == 0.0, \
        f"{key}: log says kind={z['msg_kind']}, alpha={z['msg_alpha']}"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(ref[field]), f"{key}: {field} differs from {NF_REF}"
    # input columns the encoders lose, times hidden_dim = 64
    n_less = 64 * sum(np.subtract(NF_FEATS[NF_FV[NF_REF]], NF_FEATS[NF_FV[key]]))
    assert np.all(int(ref["n_params"][0]) - z["n_params"] == n_less), \
        f"{key}: {z['n_params']} parameters, not {int(ref['n_params'][0])} - {n_less}"

# Per seed checkpoint: Cf graphs with or without the cell volume, the edge and node
# features its normalizer was fitted on, and the commit it was trained with (for
# the markdown above).
for key, runs in nf_runs.items():
    commits = {}
    for run, ok in runs:
        ck = processed_data_dir / run / "checkpoints"
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg.get("excluded_patches") == EXCLUDED, \
            f"{run}: graphs built with {cfg}"
        assert cfg["cell_volume"] == NF_FV[key], f"{run}: cell_volume = {cfg['cell_volume']}"
        if ok:
            norm = torch.load(ck / "normalizer.pt")
            n = (norm["edge_mean"].numel(), norm["node_mean"].numel())
            assert n == NF_FEATS[NF_FV[key]], f"{run}: normalizer fitted on {n} edge, node features"
        c = load_git_commit(ck)
        tag = ("not recorded" if c is None or c["commit"] is None
               else c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else ""))
        commits.setdefault(tag, []).append(run.split("/")[-1])
    print(f"{key:16s}: {NF_SRC[key][0][0]}: "
          + "; ".join(f"commit {tag} ({', '.join(rs)})" for tag, rs in commits.items()))
print("c0 (mean message column norm at initialisation) per seed: "
      + "; ".join(f"{key} {np.round(z['msg_c0'], 4).tolist()}" for key, z in nf.items()))

print(f"\nseeds {NF_SEEDS}, at most {NF_BUDGET} epochs, "
      + ", ".join(f"{key} {int(z['n_params'][0])}" for key, z in nf.items())
      + f" parameters; val rollout: {NF_VAL_STEPS} steps on {', '.join(NF_VAL_MESHES)}\n")
W = 20
print(f"{'':18s}{'n':>3}" + "".join(f"{h:>{W}}" for h in vol_stats))
for key, z in nf.items():
    row = f"{key:18s}{len(z['seeds']):>3}"
    for h, (f, fmt) in vol_stats.items():
        v = np.asarray(f(z), dtype=float)
        if h == "epochs":
            row += f"{f'{v.min():.0f} – {v.max():.0f}':>{W}}"
        else:
            row += f"{fmt.format(np.nanmean(v)) + ' ± ' + fmt.format(np.nanstd(v, ddof=1)):>{W}}"
    print(row)
    if key != NF_REF:
        print(f"{'  p (MWU)':18s}{'':>3}" + "".join(
            f"{pos_p(f(ref), f(z)):>{W}.3f}" for f, _ in vol_stats.values()))

# The val rollout as logged in training: its best epoch (that of
# model_best_rollout.pt), its best value and its value at the last epoch.
print(f"\nval rollout MAE [K], {NF_VAL_STEPS} steps, mean over val meshes and steps, per seed")
print(f"{'':18s}{'seed':>5}{'best @':>9}{'best':>11}{'last':>11}")
for key, z in nf.items():
    vr = z["val_rollout_mae"].mean((2, 3))
    for i, s in enumerate(z["seeds"]):
        print(f"{key if i == 0 else '':18s}{int(s):>5}{int(z['val_rollout_best_epoch'][i]):>9}"
              f"{np.nanmin(vr[i]):>11.3g}{vr[i, z['n_epochs'][i] - 1]:>11.3g}")

# Test rollout of each checkpoint, from the MAE saved at training time: seeds below
# the span of T at the end, MAE over steps 1-10, and the RMAE, as geometric means;
# under each no-FV row the Mann-Whitney p against the FV run.
NF_CKPTS = {"last": "rollout_mae_", "best": "rollout_mae_best_"}
print(f"\ntest rollout, {ref[f'rollout_mae_{NF_MESHES[0]}'].shape[1]} steps: seeds below "
      f"{NS_SPAN:.0f} K at the end / MAE over steps 1-10 [K] / RMAE, geometric means over seeds")
print(f"{'':24s}" + "".join(f"{mesh:>30}" for mesh in NF_MESHES) + f"{'below':>8}")
for ck, prefix in NF_CKPTS.items():
    for key, z in nf.items():
        final = np.stack([z[f"{prefix}{m}"][:, -1] for m in NF_MESHES], 1)
        below = np.isfinite(final) & (final < NS_SPAN)
        row = f"{f'{key}, {ck}':24s}"
        for j, mesh in enumerate(NF_MESHES):
            mae = z[f"{prefix}{mesh}"]
            mu_e, _ = seed_stats(mae[:, :10].mean(1), log=True)
            mu_r, _ = seed_stats(vol_rmae(mae, mesh), log=True)
            n_fin = int(np.isfinite(vol_rmae(mae, mesh)).sum())
            row += f"{f'{int(below[:, j].sum())}/{len(below)} / {10**mu_e:.3g} / {10**mu_r:.1e}' + (f' ({n_fin})' if n_fin < len(below) else ''):>30}"
        print(row + f"{f'{int(below.sum())}/{below.size}':>8}")
        if key != NF_REF:
            print(f"{'  p, steps 1-10':24s}" + "".join(
                f"{pos_p(ref[f'{prefix}{m}'][:, :10].mean(1), z[f'{prefix}{m}'][:, :10].mean(1)):>30.3f}"
                for m in NF_MESHES))
            print(f"{'  p, RMAE':24s}" + "".join(
                f"{pos_p(vol_rmae(ref[f'{prefix}{m}'], m), vol_rmae(z[f'{prefix}{m}'], m)):>30.3f}"
                for m in NF_MESHES))

# The first step whose MAE is above the span (nan counts as above), per seed;
# "-" when the rollout never gets there.
print(f"\ntest rollout, first step with MAE above {NS_SPAN:.0f} K, per seed (- never)")
print(f"{'':24s}{'seed':>5}" + "".join(f"{mesh:>11}" for mesh in NF_MESHES))
for ck, prefix in NF_CKPTS.items():
    for key, z in nf.items():
        for i, s in enumerate(z["seeds"]):
            row = f"{f'{key}, {ck}' if i == 0 else '':24s}{int(s):>5}"
            for mesh in NF_MESHES:
                out = ~(z[f"{prefix}{mesh}"][i] < NS_SPAN)
                row += f"{int(out.argmax()) + 1 if out.any() else '-':>11}"
            print(row)

# Line styles / curves of this section, also used by the sections below.
# FV in PLOT1, as the FV runs above; no FV in PLOT3, lighter and dashed without the
# noise. Green, the no-noise and no-volume controls of the sections above, stays out,
# so that each colour keeps one meaning on the poster as well.
NF_STYLES = {"FV + log volume": (PLOT1_COLOR, "-"),
             "no FV":           (PLOT3_COLOR, "-"),
             "no FV, no noise": ("#d9a07a", "--")}


In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT3_COLOR = "#b4622a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Test rollout RMAE per step, from the MAE saved at training time. Rows: the
# last-epoch and the best-rollout checkpoint; columns: the held-out meshes. Every
# seed thin, the median bold: a mean would be the one seed that diverges.
fig, axs = plt.subplots(2, len(NF_MESHES), figsize=(1800*px, 900*px), sharex=True, sharey=True,
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)
fig.suptitle("Test Rollout RMAE with and without the FV Features, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(thin: seeds, bold: median)", fontsize=24, color=AX_COLOR)

for row, (ck, prefix) in enumerate(NF_CKPTS.items()):
    for col, mesh in enumerate(NF_MESHES):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        for key, z in nf.items():
            color, ls = NF_STYLES[key]
            rel = rmae_curve(z[f"{prefix}{mesh}"], param_T_stats(mesh)[0])
            steps = np.arange(1, rel.shape[1] + 1)
            for m in rel:
                ax.plot(steps, m, color=color, linewidth=0.8, alpha=0.35, linestyle=ls)
            # an overflowed step (nan) counts as above every finite one
            med = np.median(np.where(np.isfinite(rel), rel, np.inf), 0)
            ax.plot(steps, med, color=color, linewidth=2.5, linestyle=ls, label=key)
        ax.set_yscale("log")
        ax.set_ylim(2e-3, 2e0)

        # The span of the field itself: an error above this line is no longer a
        # prediction of anything.
        # (an error as large as the span at every step, in the same relative units)
        span_rel = rmae_curve(np.full(len(steps), NS_SPAN), param_T_stats(mesh)[0])
        ax.plot(steps, span_rel, color=AX_COLOR, linewidth=1.5, linestyle=":")
        if col == 0:
            ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(1, span_rel[0]), xytext=(4, 4),
                        textcoords="offset points", ha="left", va="bottom", fontsize=12,
                        color=AX_COLOR)
            ax.set_ylabel(f"RMAE, {ck} checkpoint", fontsize=15)
            legend = ax.legend(fontsize=12, loc="center right")
            legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
            legend.get_frame().set_edgecolor(AX_COLOR)        # border color
            legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
            for text in legend.get_texts():
                text.set_color(AX_COLOR)                      # label text color
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig("../outputs/plots/parametric_history1_nofv_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- FV features vs none: message vs FV flux, per seed -------------------------------
# compute_model_flux_stats on the last-epoch checkpoints (it reads model.pt), each run
# on its own graphs (VOL_LOAD: Cf, node type and, for the FV run, log V). The no-FV
# models are fed the first 4 edge columns, as in training; the reference flux is
# computed from all 10. The FV run's entries come from the log-volume section's cache
# when their stamp still matches. Every column is an R^2, a correlation or an energy
# fraction.
nf_flux_cache = processed_data_dir / "parametric_message_flux_nofv.npz"

nf_flux = (dict(np.load(nf_flux_cache, allow_pickle=False))
           if nf_flux_cache.exists() else {})
for key, runs in nf_runs.items():
    for run, ok in runs:
        if not ok or nf_flux.get(f"{run}__stamp") == pos_stamp(run):
            continue
        if vol_flux.get(f"{run}__stamp") == pos_stamp(run):
            nf_flux.update({k: v for k, v in vol_flux.items() if k.startswith(f"{run}__")})
        else:
            nf_flux.update(compute_model_flux_stats(
                run, load=VOL_LOAD["log volume" if NF_FV[key] else "no volume"],
                n_edge_feats=NF_FEATS[NF_FV[key]][0]))
            nf_flux[f"{run}__stamp"] = pos_stamp(run)
        np.savez_compressed(nf_flux_cache, **nf_flux)   # after every run: these are slow


def nf_flux_table(key):
    """(S, M, 6) read-outs in NS_FLUX_COLS order, nan where a checkpoint is missing."""
    out = np.full((len(nf[key]["seeds"]), len(NF_MESHES), len(NS_FLUX_COLS)), np.nan)
    for i, (run, ok) in enumerate(nf_runs[key]):
        if not ok:
            continue
        for j, mesh in enumerate(NF_MESHES):
            out[i, j] = [*nf_flux[f"{run}__{mesh}__r2"], float(nf_flux[f"{run}__{mesh}__r2_node"]),
                         abs(float(nf_flux[f"{run}__{mesh}__best_r"])),
                         nf_flux[f"{run}__{mesh}__anti"].mean()]
    return out


nf_flux_tab = {key: nf_flux_table(key) for key in NF_SRC}

# Arithmetic mean ± std over the seeds: these are bounded in [0, 1]. Under each no-FV
# row the Mann-Whitney p against the FV run.
print(f"Linear read-out from the message, {N_STEPS} probed timesteps, last-epoch checkpoints, "
      f"mean ± std over seeds")
print(f"{'mesh':<11}{'run':<18}{'n':>3}" + "".join(f"{c:>15}" for c in NS_FLUX_COLS))
for j, mesh in enumerate(NF_MESHES):
    for key in NF_SRC:
        t = nf_flux_tab[key][:, j]
        t = t[~np.isnan(t[:, 0])]
        cells = ([f"{m:.3f} ± {s:.3f}" for m, s in zip(t.mean(0), t.std(0, ddof=1))]
                 if len(t) > 1 else [f"{m:.3f}" for m in t.mean(0)])
        print(f"{mesh:<11}{key:<18}{len(t):>3}" + "".join(f"{c:>15}" for c in cells))
        if key != NF_REF:
            print(f"{mesh:<11}{'  p (MWU)':<18}{'':>3}" + "".join(
                f"{pos_p(nf_flux_tab[NF_REF][:, j, c], nf_flux_tab[key][:, j, c]):>15.3f}"
                for c in range(len(NS_FLUX_COLS))))
    print()


## Cell volume as $V^{1/3}$ instead of $\log V$ (`_cbrt`)

The volume feature of the `_vol` runs is $\log V$, with the boundary nodes, which have no volume,
set to the mesh's mean $\log V$. The alternative here is $h = V^{1/3}$, the cell's length
scale. It gives the boundary nodes a natural value: $h = 0$, a face centre being a cell of zero
thickness. Two things to keep in mind:

- After the z-score, $\log V$ and $\log h = \tfrac13 \log V$ are *the same feature*. What changes
  is the scale: $h$ is linear in length, so the refinement levels ($h$ halves at each one) are no
  longer equally spaced. The coarse levels spread out and the fine ones bunch together.
- The boundary nodes move from the mesh's mean $\log V$ (among the mid-size cells, levels 2 – 4, at a
  different value on every mesh) to $h = 0$, a single value below every cell.

Runs, both with input noise 0.3, at most 500 epochs, on the **dt = 0.025 data**. That data has been
in `raw_data/parametric_old` since `raw_data/parametric` was rewritten at dt = 0.005 on
2026-11-09, so every graph and $T$ sequence in this section is loaded from `parametric_old`:

- **log V** (`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos`, `seed_<n>`, seeds 0 – 2,
  commit `48186db` + uncommitted changes): the `_vol` run of the section above, launcher log in
  `logs/20261019_151511`.
- **V^(1/3)** (`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_old_nopos`, `seed_<n>`,
  seeds 0 – 2, commit `378f1d6` + uncommitted changes): `FVGNN_VOL_TRANSFORM=cbrt`,
  `FVGNN_DATASET=parametric_old`, launcher log in `logs/20261116_101528`.

The two launched copies of `train_parametric.py` differ only by the switches that select the
transform and the dataset: same model (116 673 parameters in both), schedule, budget, early stopping
and mesh split.

**The best model of each** is the seed whose last-epoch checkpoint has the lowest 40-step val
rollout MAE (mean over both val meshes and the steps). The pick is on the val meshes, so the test
meshes stay held out, and it uses the last epoch because the volume section found that
`model_best_rollout.pt` does not carry over to the test meshes. The lowest best val loss picks the
same seeds. A single pair has no statistics, so every comparison of the two best models is held
against the other seeds: all six checkpoints are evaluated, and the seed comparisons use
Mann-Whitney U, two-sided, 3 against 3, unpaired (the transform changes the input values, so seed
$n$ of the two runs trains differently from the first step). Its smallest possible $p$ is 0.1.

- **Per seed**: epochs, best val loss, val rollout, and the test rollout RMAE from the MAE saved in
  training.
- **The two best models** on the test meshes: one-step and 120-step rollout RMAE / RMSE (sums over
  every node and step); the volume column as each model's normalizer presents it, per refinement
  level and on the boundary nodes; the errors per refinement level and for the cells next to a
  boundary node against the rest.
- **Every seed**, one-step RMSE next to the boundary, inside, and over the whole mesh.

The section is self-contained: it needs only the first two cells of the notebook.

In [ ]:
# --- log V vs V^(1/3): load the seeds, check them, pick each one's best model -------
# Self-contained: needs only the imports and processed_data_dir of the first cells.
# Both runs trained on the dt = 0.025 data, now in raw_data/parametric_old
# (raw_data/parametric has held the dt = 0.005 data since 2026-11-09), so every
# graph and T sequence here comes from parametric_old, with its own mesh cache.
from datetime import datetime

from scipy.stats import mannwhitneyu

from data_preparation.mesh_cache import load_mesh_cached
from data_preparation.mesh_dataset import SingleMeshDataset
from data_preparation.normalization import FeatureNormalizer
from data_preparation.static_graph import load_graph_config, node_feature_columns
from git_commit import load_git_commit
from mesh2graph.utils import load_cell_volumes
from models.autoregressive_training import rollout
from models.fvgnn import FVSurrogate, widths_from_state_dict

CB_CASES = {"log V":   "parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos",
            "V^(1/3)": "parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_old_nopos"}
CB_TRANSFORM = {"log V": "log", "V^(1/3)": "cbrt"}
CB_RAW = Path("../raw_data/parametric_old")
CB_CACHE = processed_data_dir / "parametric_old_preproc_cache"
CB_EXCLUDED = ["top", "bottom", "cbores"]
CB_HISTORY = 1
CB_NOISE = 0.3
CB_BUDGET = 500
CB_SPAN = 300.0                       # K, span of T on every held-out mesh
CB_NEW_DATA = datetime(2026, 11, 9)   # raw_data/parametric rewritten at dt = 0.005


def cb_load(key):
    """load_mesh for one run: Cf graphs of parametric_old with that run's volume
    transform, node type + volume, no positions; all 10 edge features."""
    def load(mesh):
        g, T = load_mesh_cached(str(CB_RAW / mesh), CB_EXCLUDED, "cf", CB_CACHE,
                                cell_volume_transform=CB_TRANSFORM[key])
        g.node_attr = g.node_attr[:, node_feature_columns(True, False)]
        return g, T
    return load


CB_LOAD = {key: cb_load(key) for key in CB_CASES}


def cb_p(a, b):
    """Two-sided Mann-Whitney U p of two seed samples; inf ranks above every value."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue) if min(len(a), len(b)) > 1 else np.nan


cb = {key: dict(np.load(processed_data_dir / exp / "seeds.npz")) for key, exp in CB_CASES.items()}
CB_SEEDS = cb["log V"]["seeds"]
CB_MESHES = [str(m) for m in cb["log V"]["test_meshes"]]
CB_VAL_MESHES = [str(m) for m in cb["log V"]["val_meshes"]]
CB_VAL_STEPS = cb["log V"]["val_rollout_mae"].shape[-1]

# The same seeds, meshes, noise, budget, schedule and model size: the transform
# changes the values of the volume column, not the number of inputs.
for key, z in cb.items():
    assert np.array_equal(z["seeds"], CB_SEEDS), f"{key}: seeds {z['seeds']}"
    assert [str(m) for m in z["test_meshes"]] == CB_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert [str(m) for m in z["val_meshes"]] == CB_VAL_MESHES, f"{key}: val meshes {z['val_meshes']}"
    assert float(z["input_noise"]) == CB_NOISE, f"{key}: input_noise = {z['input_noise']}"
    assert int(z["max_epochs"]) == CB_BUDGET, f"{key}: launched for {z['max_epochs']} epochs"
    assert str(z["msg_kind"]) == "none" and float(z["msg_alpha"]) == 0.0, f"{key}: a message penalty"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(cb["log V"][field]), f"{key}: {field} differs"
assert int(cb["log V"]["n_params"]) == int(cb["V^(1/3)"]["n_params"]), "the models differ in size"

# Per seed: the graphs it was built with, the data it saw, the commit.
cb_runs = {key: [f"{exp}/seed_{int(s)}" for s in CB_SEEDS] for key, exp in CB_CASES.items()}
for key, runs in cb_runs.items():
    commits = {}
    for run in runs:
        ck = processed_data_dir / run / "checkpoints"
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg["excluded_patches"] == CB_EXCLUDED, f"{run}: {cfg}"
        assert cfg["cell_volume"] and cfg["cell_volume_transform"] == CB_TRANSFORM[key], f"{run}: {cfg}"
        if key == "V^(1/3)":
            assert cfg["dataset"] == "parametric_old", f"{run}: trained on {cfg['dataset']}"
        else:
            # older than the dataset option: what raw_data/parametric was then
            t = datetime.fromtimestamp((ck / "model.pt").stat().st_mtime)
            assert t < CB_NEW_DATA, f"{run}: trained {t}, after the data changed"
        assert torch.load(ck / "normalizer.pt")["node_mean"].numel() == 3, f"{run}: node features"
        c = load_git_commit(ck)
        tag = c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else "")
        commits.setdefault(tag, []).append(run.split("/")[-1])
    print(f"{CB_CASES[key]}: " + "; ".join(f"commit {t} ({', '.join(r)})" for t, r in commits.items()))


def cb_last(z, field):
    """Each seed's value at its own last epoch (early-stopped seeds are NaN past it)."""
    return z[field][np.arange(len(z["seeds"])), z["n_epochs"] - 1]


# |T_true| per rollout step on the old data, for the RMAE from the saved MAE.
cb_absT = {m: CB_LOAD["log V"](m)[1][CB_HISTORY:].double().abs().mean(1).numpy() for m in CB_MESHES}


def cb_rmae(mae, mesh):
    """Rollout RMAE over all steps from the per-step MAE [K]; inf if it overflowed."""
    r = mae.astype(float).sum(-1) / cb_absT[mesh][:mae.shape[-1]].sum()
    return np.where(np.isfinite(r), r, np.inf)


# The best model of each: the seed whose last-epoch checkpoint has the lowest val
# rollout MAE (40 steps, mean over both val meshes and the steps), picked on the
# val meshes so the test meshes stay held out. The last epoch, not
# model_best_rollout.pt: the volume section found that the best-rollout pick does
# not carry over to the test meshes. Also shown: the seed with the lowest best
# val loss.
cb_val_roll = {key: cb_last(z, "val_rollout_mae").mean((1, 2)) for key, z in cb.items()}
CB_BEST = {key: int(np.argmin(v)) for key, v in cb_val_roll.items()}
for key, z in cb.items():
    by_loss = int(np.nanargmin(np.nanmin(z["val_losses"], 1)))
    print(f"{key}: best seed {int(CB_SEEDS[CB_BEST[key]])} by last-epoch val rollout, "
          f"{int(CB_SEEDS[by_loss])} by best val loss")
cb_best_run = {key: cb_runs[key][i] for key, i in CB_BEST.items()}

print(f"\nper seed (* the best model): epochs, best val loss, last-epoch val rollout MAE [K], "
      f"and the test rollout RMAE over {cb['log V'][f'rollout_mae_{CB_MESHES[0]}'].shape[1]} steps "
      f"from the MAE saved at training time")
print(f"{'':10s}{'seed':>6}{'epochs':>8}{'val best':>11}{'val roll':>10}"
      + "".join(f"{m:>12}" for m in CB_MESHES))
for key, z in cb.items():
    for i, s in enumerate(CB_SEEDS):
        star = "*" if i == CB_BEST[key] else " "
        print(f"{key if i == 0 else '':10s}{f'{int(s)}{star}':>6}{int(z['n_epochs'][i]):>8}"
              f"{np.nanmin(z['val_losses'][i]):>11.2e}{cb_val_roll[key][i]:>10.3f}"
              + "".join(f"{cb_rmae(z[f'rollout_mae_{m}'][i], m):>12.3e}" for m in CB_MESHES))
print(f"{'p (MWU)':10s}{'':>6}{cb_p(cb['log V']['n_epochs'], cb['V^(1/3)']['n_epochs']):>8.2f}"
      f"{cb_p(*(np.nanmin(z['val_losses'], 1) for z in cb.values())):>11.2f}"
      f"{cb_p(*cb_val_roll.values()):>10.2f}"
      + "".join(f"{cb_p(*(cb_rmae(z[f'rollout_mae_{m}'], m) for z in cb.values())):>12.2f}"
                for m in CB_MESHES))

In [ ]:
# --- log V vs V^(1/3): evaluate the two best models ----------------------------------
# On the old data, each on its own graphs: one-step and full-rollout RMAE / RMSE
# over every node and step, the rollout MAE per step, and the same sums per group
# of internal cells, one-step and over rollout steps 1-10. Groups: the refinement
# level, round(log8(V0 / V)) with V0 the largest cell of the test meshes (the
# background blockMesh cell), 0 the coarsest; and whether the cell has an edge to a
# boundary node, where V^(1/3) differs from log V the most (0 against the mean log
# V). Plus the val rollout, to check against the one logged in training. Every
# seed is evaluated, so the best-vs-best differences can be held against the
# spread over seeds. Cached per checkpoint, stamped with the mtime of its model.pt.
cb_eval_cache = processed_data_dir / "parametric_eval_volume_cbrt.npz"
CB_LVL_STEPS = 10
CB_MIN_CELLS = 100    # a group with fewer cells on a mesh is not shown

cb_V = {m: load_cell_volumes(str(CB_RAW / m)).numpy().ravel() for m in CB_MESHES}
CB_V0 = max(v.max() for v in cb_V.values())
cb_level = {m: np.floor(np.log(CB_V0 / v) / np.log(8) + 0.5).astype(int) for m, v in cb_V.items()}
CB_LEVELS = np.arange(max(l.max() for l in cb_level.values()) + 1)


def cb_adjacent(graph, n_int):
    """(n_int,) 1 where an internal cell has an edge to a boundary node, else 0."""
    ei = graph.edge_index
    adj = np.zeros(n_int, dtype=int)
    for a, b in ((ei[0], ei[1]), (ei[1], ei[0])):
        hit = (a < n_int) & (b >= n_int)
        adj[a[hit].numpy()] = 1
    return adj


def cb_group_sums(e, T_t, labels, n_groups):
    """(n_groups, 4) sums of |e|, |T|, e^2, T^2 per label over the internal cells
    (the first len(labels) nodes); e, T_t are (steps, N)."""
    idx = torch.from_numpy(labels)
    out = torch.zeros(n_groups, 4, dtype=torch.float64)
    for c, x in enumerate((e.abs(), T_t.abs(), e.pow(2), T_t.pow(2))):
        out[:, c].index_add_(0, idx, x[:, :len(labels)].double().sum(0))
    return out.numpy()


def cb_evaluate(run, load):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / "model.pt", map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in CB_MESHES + CB_VAL_MESHES:
        graph, T_seq = load(mesh)
        model = FVSurrogate(
            in_node_feat=CB_HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=CB_HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        ds = SingleMeshDataset(T_seq, graph, norm, CB_HISTORY)
        if mesh in CB_VAL_MESHES:
            T_p, T_t = rollout(model, ds, device=device, n_steps=CB_VAL_STEPS)
            out[f"{run}__{mesh}__val_roll"] = (T_p - T_t).abs().mean(1).numpy()
            continue
        n_int = int((graph.node_attr[:, 0] == 1.0).sum())
        assert n_int == len(cb_level[mesh]), f"{mesh}: {n_int} internal nodes, {len(cb_level[mesh])} cells"
        groups = {"lvl": (cb_level[mesh], len(CB_LEVELS)), "adj": (cb_adjacent(graph, n_int), 2)}
        tot = np.zeros(4)
        step = {g: np.zeros((n, 4)) for g, (_, n) in groups.items()}
        with torch.no_grad():
            for i in range(len(ds)):
                d = ds[i].to(device)
                T_p = norm.inverse_transform_T(model(d).cpu()).double()[None]
                T_t = norm.inverse_transform_T(d.y.cpu()).double()[None]
                e = T_p - T_t
                tot += [e.abs().sum(), T_t.abs().sum(), e.pow(2).sum(), T_t.pow(2).sum()]
                for g, (lab, n) in groups.items():
                    step[g] += cb_group_sums(e, T_t, lab, n)
        out[f"{run}__{mesh}__rmae_step"] = tot[0] / tot[1]
        out[f"{run}__{mesh}__rmse_step"] = tot[2] / tot[3]
        T_p, T_t = rollout(model, ds, device=device)
        e, T_t = (T_p - T_t).double(), T_t.double()
        out[f"{run}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_t.abs().sum())
        out[f"{run}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_t.pow(2).sum())
        out[f"{run}__{mesh}__mae_roll"] = e.abs().mean(1).numpy()
        for g, (lab, n) in groups.items():
            out[f"{run}__{mesh}__{g}_step"] = step[g]
            out[f"{run}__{mesh}__{g}_roll"] = cb_group_sums(e[:CB_LVL_STEPS], T_t[:CB_LVL_STEPS], lab, n)
            out[f"{run}__{mesh}__{g}_n"] = np.bincount(lab, minlength=n)
        # what the model sees: the volume column after its own normalizer, mean
        # per level and on the boundary nodes
        zv = norm.transform_node_attr(graph.node_attr)[:, 2].double().numpy()
        out[f"{run}__{mesh}__z_lvl"] = np.array([zv[:n_int][cb_level[mesh] == l].mean()
                                                 if (cb_level[mesh] == l).any() else np.nan
                                                 for l in CB_LEVELS])
        out[f"{run}__{mesh}__z_bnd"] = zv[n_int:].mean()
        del model
    return out


def cb_stamp(run):
    return np.int64((processed_data_dir / run / "checkpoints" / "model.pt").stat().st_mtime_ns)


cb_eval = dict(np.load(cb_eval_cache, allow_pickle=False)) if cb_eval_cache.exists() else {}
for key, runs in cb_runs.items():
    for run in runs:
        if cb_eval.get(f"{run}__stamp") != cb_stamp(run):
            cb_eval.update(cb_evaluate(run, CB_LOAD[key]))
            cb_eval[f"{run}__stamp"] = cb_stamp(run)
            np.savez_compressed(cb_eval_cache, **cb_eval)   # after every checkpoint: these are slow


def cb_get(key, mesh, name, i=None):
    """One evaluated quantity of seed index i of key, the best model by default."""
    run = cb_best_run[key] if i is None else cb_runs[key][i]
    return cb_eval[f"{run}__{mesh}__{name}"]


# The data are the ones they trained on: the val rollout recomputed on
# parametric_old has to be the one logged at the last epoch, and the test
# rollout MAE the one saved at the end of training, over the steps below the span.
dv, dt = [], []
for key, z in cb.items():
    for i in range(len(CB_SEEDS)):
        dv += [np.abs(cb_get(key, m, "val_roll", i) / z["val_rollout_mae"][i, z["n_epochs"][i] - 1, v] - 1).max()
               for v, m in enumerate(CB_VAL_MESHES)]
        dt += [np.abs(cb_get(key, m, "mae_roll", i) / z[f"rollout_mae_{m}"][i] - 1).max() for m in CB_MESHES]
print(f"recomputed on parametric_old vs logged in training, every seed, max |relative difference|: "
      f"last-epoch val rollout MAE {max(dv):.1e}, test rollout MAE {max(dt):.1e}")

print(f"\nthe two best models on the test meshes (ratio: V^(1/3) / log V)")
CB_ERR_COLS = {"rmae_step": "one-step RMAE", "rmse_step": "one-step RMSE",
               "rmae_roll": "rollout RMAE", "rmse_roll": "rollout RMSE"}
print(f"{'mesh':<11}" + "".join(f"{h:>30}" for h in CB_ERR_COLS.values()))
for m in CB_MESHES:
    a, b = ([float(cb_get(k, m, c)) for c in CB_ERR_COLS] for k in CB_CASES)
    print(f"{m:<11}" + "".join(f"{f'{x:.3e} / {y:.3e}  {y / x:.2f}':>30}" for x, y in zip(a, b)))


def cb_group_metric(key, mesh, group, mode, metric, i=None):
    """Per-group RMAE / RMSE over that group's cells, or its share of the mesh's
    squared error; nan for a group with fewer than CB_MIN_CELLS cells."""
    s = cb_get(key, mesh, f"{group}_{mode}", i)
    with np.errstate(invalid="ignore", divide="ignore"):
        v = {"rmae": s[:, 0] / s[:, 1], "rmse": s[:, 2] / s[:, 3],
             "e2_share": s[:, 2] / s[:, 2].sum()}[metric]
    return np.where(cb_get(key, mesh, f"{group}_n") >= CB_MIN_CELLS, v, np.nan)


print(f"\nthe volume feature after each model's normalizer: mean z per refinement level, "
      f"and on the boundary nodes")
print(f"{'mesh':<11}{'':<10}" + "".join(f"{f'level {l}':>9}" for l in CB_LEVELS) + f"{'boundary':>10}")
for m in CB_MESHES:
    for k in CB_CASES:
        print(f"{m if k == 'log V' else '':<11}{k:<10}"
              + "".join(f"{x:>9.2f}" for x in cb_get(k, m, "z_lvl")) + f"{float(cb_get(k, m, 'z_bnd')):>10.2f}")

CB_GRP_COLS = {("step", "rmae"): "one-step RMAE", ("step", "rmse"): "one-step RMSE",
               ("roll", "rmse"): f"steps 1-{CB_LVL_STEPS} RMSE"}
for group, labels in (("lvl", [f"level {l}" for l in CB_LEVELS]), ("adj", ["inner", "next to bnd"])):
    print(f"\n{'per refinement level' if group == 'lvl' else 'cells next to a boundary node or not'}: "
          f"cells, then log V / V^(1/3) and the ratio; e^2 share: of the mesh's one-step squared error")
    print(f"{'mesh':<11}{'':<20}" + "".join(f"{l:>26}" for l in labels))
    for m in CB_MESHES:
        n = cb_get("log V", m, f"{group}_n")
        print(f"{m:<11}{'cells':<20}" + "".join(f"{int(x):>26d}" for x in n))
        for (mode, metric), title in {**CB_GRP_COLS, ("step", "e2_share"): "one-step e^2 share"}.items():
            a, b = (cb_group_metric(k, m, group, mode, metric) for k in CB_CASES)
            print(f"{'':<11}{title:<20}" + "".join(
                f"{'—':>26}" if np.isnan(x) else f"{f'{x:.2e} / {y:.2e}  {y / x:.2f}':>26}"
                for x, y in zip(a, b)))

# Is the gap next to the boundary a property of the transform or of the two seeds
# picked? Every seed's one-step RMSE over the cells next to a boundary node, over
# the rest and over the whole mesh, with the Mann-Whitney p of the 3 against 3
# (unpaired: the transform changes the input values, so seed n of each run trains
# differently from the first step; the smallest p possible is 0.1).
CB_SEED_COLS = ("next to bnd", "inner", "whole mesh")


def cb_seed_rmse(key, mesh):
    """(S, 3) one-step RMSE of every seed: next to a boundary node, inner, whole mesh."""
    return np.array([[*cb_group_metric(key, mesh, "adj", "step", "rmse", i)[::-1],
                      float(cb_get(key, mesh, "rmse_step", i))] for i in range(len(CB_SEEDS))])


print(f"\nevery seed, one-step RMSE (* the best model): " + " / ".join(CB_SEED_COLS))
print(f"{'mesh':<11}{'':<10}" + "".join(f"{f'seed {int(s)}':>32}" for s in CB_SEEDS)
      + f"{'p (MWU)':>20}")
for m in CB_MESHES:
    t = {k: cb_seed_rmse(k, m) for k in CB_CASES}
    for k in CB_CASES:
        cells = [" / ".join(f"{x:.2e}" for x in r) + ("*" if i == CB_BEST[k] else " ")
                 for i, r in enumerate(t[k])]
        ps = (" / ".join(f"{cb_p(t['log V'][:, c], t['V^(1/3)'][:, c]):.1f}" for c in range(3))
              if k == "V^(1/3)" else "")
        print(f"{m if k == 'log V' else '':<11}{k:<10}" + "".join(f"{c:>32}" for c in cells) + f"{ps:>20}")

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT3_COLOR = "#b4622a"
PLOT4_COLOR = "#2b5d8a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# log V in the colour of the volume section, V^(1/3) in a new one. Every seed thin,
# the best model of each bold.
CB_STYLES = {"log V": (PLOT3_COLOR, "-"), "V^(1/3)": (PLOT4_COLOR, "--")}


def cb_style_ax(ax, labelsize=14):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=labelsize)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


def cb_legend(legend):
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)


# Training: validation loss and the val rollout RMAE (sum over its steps and nodes
# of |error| over the same sum of |T_true|, mean over the val meshes) per epoch.
CB_EPOCHS = max(z["val_losses"].shape[1] for z in cb.values())
cb_val_absT = np.stack([CB_LOAD["log V"](m)[1][CB_HISTORY:CB_HISTORY + CB_VAL_STEPS].double()
                        .abs().mean(1).numpy() for m in CB_VAL_MESHES])          # (V, K)

fig, axs = plt.subplots(1, 2, figsize=(1500*px, 560*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Training with log V and with V^(1/3), Noise 0.3, No Positions, Cf, msg_dim = 64, h = 1\n"
             "(thin: every seed, bold: the best model of each)", fontsize=20, color=AX_COLOR)
for key, z in cb.items():
    color, ls = CB_STYLES[key]
    vr = (z["val_rollout_mae"].sum(-1) / cb_val_absT.sum(-1)).mean(-1)          # (S, E)
    for ax, curves in zip(axs, (z["val_losses"], vr)):
        for i, c in enumerate(curves):
            best = i == CB_BEST[key]
            ax.plot(np.arange(len(c)), c, color=color, linestyle=ls,
                    linewidth=2.4 if best else 0.9, alpha=1 if best else 0.45,
                    label=f"{key}, seed {int(CB_SEEDS[i])} (best)" if best else None)
for ax, title, ylabel in zip(axs, ("Validation Loss", f"Val Rollout RMAE, {CB_VAL_STEPS} Steps"),
                             ("MSE Loss", "RMAE")):
    cb_style_ax(ax)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=18)
    ax.set_xlabel("Epoch", fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    cb_legend(ax.legend(fontsize=12, loc="upper right"))

# plt.savefig("../outputs/plots/parametric_history1_volume_cbrt_training.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Test rollout RMAE per step (sum_i |e| / sum_i |T_true| at each step), from the
# MAE saved at training time: every seed thin, the best model of each bold. Below,
# the ratio V^(1/3) / log V of the two best models' per-step RMAE.
fig, axs = plt.subplots(2, len(CB_MESHES), figsize=(1800*px, 850*px), sharex=True, sharey="row",
                        gridspec_kw={"height_ratios": [2, 1]}, constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Test Rollout RMAE, log V vs V^(1/3), Noise 0.3, No Positions, Cf, msg_dim = 64, h = 1\n"
             "(thin: every seed, bold: the best model of each; below: ratio of the best models)",
             fontsize=21, color=AX_COLOR)

for col, mesh in enumerate(CB_MESHES):
    ax = axs[0, col]
    absT = cb_absT[mesh]
    best_curves = {}
    for key, z in cb.items():
        color, ls = CB_STYLES[key]
        rel = z[f"rollout_mae_{mesh}"].astype(float) / absT[:z[f"rollout_mae_{mesh}"].shape[1]]
        steps = np.arange(1, rel.shape[1] + 1)
        for i, r in enumerate(rel):
            best = i == CB_BEST[key]
            ax.plot(steps, r, color=color, linestyle=ls, linewidth=2.4 if best else 0.9,
                    alpha=1 if best else 0.45,
                    label=f"{key}, seed {int(CB_SEEDS[i])}" if best else None)
        best_curves[key] = rel[CB_BEST[key]]
    cb_style_ax(ax, 13)
    ax.set_yscale("log")
    ax.set_title(mesh, fontsize=19)
    if col == 0:
        ax.set_ylabel("RMAE", fontsize=15)
        cb_legend(ax.legend(fontsize=12, loc="lower right"))

    ax = axs[1, col]
    ax.plot(steps, best_curves["V^(1/3)"] / best_curves["log V"], color=AX_COLOR, linewidth=1.8)
    ax.axhline(1, color=AX_COLOR, linewidth=1.0, linestyle=":")
    cb_style_ax(ax, 13)
    ax.set_xlabel("Time step", fontsize=15)
    if col == 0:
        ax.set_ylabel("V^(1/3) / log V", fontsize=15)

# plt.savefig("../outputs/plots/parametric_history1_volume_cbrt_rollout.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Where the two differ. Top: one-step RMSE per refinement level of the two best
# models (0 the coarsest; levels with fewer than CB_MIN_CELLS cells left out).
# Bottom: every seed's one-step RMSE over the cells next to a boundary node and
# over the rest (filled marker: the best model), the split where the inputs differ
# most: V^(1/3) puts the boundary nodes at 0, log V at the mesh's mean log V.
fig, axs = plt.subplots(2, len(CB_MESHES), figsize=(1800*px, 900*px), sharey="row",
                        constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("One-step RMSE per Refinement Level and next to the Boundary, log V vs V^(1/3)\n"
             "(top: the best models; bottom: every seed, filled: the best model)",
             fontsize=21, color=AX_COLOR)

for col, mesh in enumerate(CB_MESHES):
    ax = axs[0, col]
    for k, key in enumerate(CB_CASES):
        color, ls = CB_STYLES[key]
        v = cb_group_metric(key, mesh, "lvl", "step", "rmse")
        ok = np.isfinite(v)
        ax.plot(CB_LEVELS[ok] + (k - 0.5) * 0.1, v[ok], color=color, linestyle=ls, linewidth=2,
                marker="s", markersize=6, label=f"{key}, seed {int(CB_SEEDS[CB_BEST[key]])}")
    cb_style_ax(ax, 13)
    ax.set_yscale("log")
    ax.set_xticks(CB_LEVELS)
    ax.set_xlabel("refinement level (0: coarsest)", fontsize=14)
    ax.set_title(mesh, fontsize=19)
    if col == 0:
        ax.set_ylabel("one-step RMSE", fontsize=15)
        cb_legend(ax.legend(fontsize=12, loc="lower left"))

    ax = axs[1, col]
    for k, key in enumerate(CB_CASES):
        color, _ = CB_STYLES[key]
        t = cb_seed_rmse(key, mesh)[:, :2]                       # (S, 2): next to bnd, inner
        for g in range(2):
            x = g + (k - 0.5) * 0.25
            for i, v in enumerate(t[:, g]):
                best = i == CB_BEST[key]
                ax.plot(x, v, "o", markersize=9 if best else 7, color=color,
                        markerfacecolor=color if best else "none", markeredgewidth=1.6,
                        label=key if (g == 0 and i == 0) else None)
    cb_style_ax(ax, 13)
    ax.set_yscale("log")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["next to a boundary node", "inner cells"], fontsize=13)
    ax.set_xlim(-0.6, 1.6)
    if col == 0:
        ax.set_ylabel("one-step RMSE, every seed", fontsize=15)
        cb_legend(ax.legend(fontsize=12, loc="upper left"))

# plt.savefig("../outputs/plots/parametric_history1_volume_cbrt_levels_boundary.png", dpi=300, transparent = True)

### What comes out

Best models: **log V seed 2** (commit `48186db` + uncommitted changes) and **V^(1/3) seed 0**
(commit `378f1d6` + uncommitted changes), the same by the last-epoch val rollout (1.062 and
1.049 K) and by the best val loss ($2.52$ and $2.62\times10^{-5}$). Recomputed on `parametric_old`,
the val and test rollouts of all six checkpoints match the ones logged in training to
$4\times10^{-5}$ relative, which confirms the data are the ones both runs trained on.

**Over the seeds, the two transforms cannot be told apart.** Best val loss, val rollout and epochs
overlap ($p = 0.4$ – $0.7$). The 120-step test rollout RMAE gives $p = 1$ on `model_003`,
`model_007` and `model_010`, and $p = 0.1$ only on `model_011` (1.279 – 1.284 against
1.292 – 1.322 $\times10^{-2}$). The spread is wider with $V^{1/3}$: the best val loss runs from
$2.6$ to $5.3\times10^{-5}$ against $2.5$ – $4.9\times10^{-5}$, and two of its three seeds stopped
early (166 and 172 epochs).

**Best against best, $V^{1/3}$ is a little ahead over the full rollout, and behind at the start on
two meshes.** Ratio $V^{1/3}$ / $\log V$:

| | `model_003` | `model_007` | `model_010` | `model_011` |
|---|---|---|---|---|
| one-step RMAE | 1.03 | 0.88 | 0.89 | 0.88 |
| one-step RMSE | 1.08 | 1.03 | 0.86 | 0.88 |
| rollout RMAE | 0.99 | 0.97 | 0.98 | 0.97 |
| rollout RMSE | 0.97 | 0.96 | 0.95 | 0.94 |

Per step, the ratio of the RMAE peaks at 1.10 around step 3 on `model_003` and `model_007` (and 1.04
on `model_011`; on `model_010` it starts below 1) and ends at 0.94 – 0.96 at step 120 on three
meshes, 0.99 on `model_010`. These gaps are the same
size as the gaps between seeds of a single run. The best log V seed has the *highest* rollout RMAE
of its three seeds on `model_003`, `model_007` and `model_011`, so a different pick would shrink or flip them.

**The input is what the reasoning above expects.** After each model's normalizer, the refinement
levels (0: the coarsest) sit at

- $\log V$: $z$ = 2.20, 1.49, 0.78, 0.09, about −0.3 to −0.6, about −1.0 to −1.3, equal steps;
- $V^{1/3}$: $z$ = 5.6, 2.3, 0.67, −0.12, about −0.4 to −0.5, about −0.7, with the two finest
  levels 0.1 – 0.2 apart.

The boundary nodes sit at −0.93 with $V^{1/3}$ on every mesh, below every cell. With $\log V$
they sit at −0.24 to 0.69 depending on the mesh, among the cells of levels 2 – 4.

**The one consistent difference is next to the boundary.** For the cells with an edge to a boundary
node, the one-step RMSE of the best models drops by a factor 0.51 on `model_007`, 0.81 on
`model_010` and 0.77 on `model_011`, and is unchanged on `model_003` (1.01). Over the seeds, every
$V^{1/3}$ seed is below every $\log V$ seed there on `model_007` and `model_010` ($p = 0.1$), all but
one pair on `model_011` ($p = 0.2$), and the seeds mix on `model_003` ($p = 1$). On the inner cells nothing
separates ($p = 0.7$ – $1$): their error varies 2 – 3× between seeds of one run, and the best-model
ratios there (1.09, 1.11, 0.89, 0.96) follow the seeds picked. The cells next to the boundary carry
only 5 – 13 % of the one-step squared error, so a gain there hardly reaches the whole-mesh numbers.
Per refinement level, the best models differ most at level 3, which holds 56 – 71 % of the squared
error on the three meshes where it is the largest. The ratio there is 1.10, 1.22, 0.86 on those
meshes: the seed, again.

**What this shows.** As expected, $V^{1/3}$ changes little. The only effect that holds across
seeds is the one its motivation predicts: a lower one-step error in the cells next to the
boundary, where the boundary nodes now carry a distinct, mesh-independent value instead of a
mid-level cell size. On three of four meshes it is 20 – 50 % for the best models. Everywhere else
the two transforms are within the spread between seeds. That includes the 120-step rollout, where
the best $V^{1/3}$ model is 1 – 3 % ahead in RMAE but the three-seed samples overlap on three of
four meshes. With 3 seeds a side, none of this can reach $p < 0.1$. A paired design (the same
initial weights for both transforms, which the equal input width allows) would test the boundary
effect more sharply than more unpaired seeds.

## Laplacian eigenmodes as node features (`_eig<k>`)

The first $k$ eigenmodes of each mesh's own diffusion operator, appended to the node features:
$K\phi = \lambda\, \mathrm{diag}(V)\, \phi$, with $K = -\nabla\cdot(D_T \nabla\,\cdot)$ discretised
by OpenFOAM with the BCs of $T$ (read at time 0), and $V$ the cell volumes. The modes are
$V$-orthonormal and sign-canonical. The boundary nodes take the face value of the homogeneous BC:
0 on a fixedValue patch, the owner's value on a zeroGradient one. They come after node type and
$V^{1/3}$, and are normalized by scale only (RMS over the training meshes, no mean subtracted), so a
mode's sign stays meaningful. There are no random sign flips (`FVGNN_EIG_FLIP=0`): train, val and
test all see the canonical signs. The modes are low-frequency patterns of the mesh, a
geometry-aware positional encoding. They tell a node where it sits with respect to the BCs,
which the network otherwise has to infer through message passing.

Runs, all with input noise 0.3, $V^{1/3}$ as the volume feature, at most 500 epochs, on the
**dt = 0.025 data** in `raw_data/parametric_old` (so every graph and $T$ sequence here is loaded
from there):

- **V^(1/3)**, the baseline (`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_old_nopos`,
  `seed_<n>`, seeds 0 – 2, commit `378f1d6` + uncommitted changes): the $V^{1/3}$ run of the section
  above, launcher log in `logs/20261116_101528`.
- **eig 4, eig 8, eig 16** (`parametric_history1_msg_dim64_residual_noise0.3_eig<k>_cf_vol_cbrt_old_nopos`,
  `seed_<n>`, seeds 0 – 2, commit `f9709a1`): `FVGNN_EIG=4`, `8`, `16`, launcher log in
  `logs/20261116_171027`.

The modes add $k$ input columns to the node encoder and nothing else (116 673 parameters in the
baseline, 116 929 / 117 185 / 117 697 with 4 / 8 / 16 modes). Same schedule, budget, early stopping
and mesh split. The extra columns change the initialisation draws, so seed $n$ of two configurations
trains differently from the first step. The comparisons are therefore unpaired: Mann-Whitney U,
two-sided, 3 against 3, smallest possible $p$ = 0.1.

All on the **last-epoch checkpoints** (`model.pt`), every seed:

- **Per seed**: epochs, best val loss, val rollout, and the test rollout RMAE from the MAE saved in
  training.
- **Errors**: one-step and 120-step rollout RMAE / RMSE on the test meshes, as sums over every node and
  step. Geometric mean ×/÷ geometric std over the seeds.
- **Error accumulation**: $\mathrm{RMAE}_t$ and $\mathrm{RMSE}_t$ at each rollout step $t$.
- **Correlations**: the linear read-out of the FV flux from the message, as in the sections above
  ($R^2$ of $F$, $\Delta T$, $w$, of $\sum_j F$ from the aggregated message, the best single channel's
  $|r|$, and the antisymmetric energy fraction).

The section is self-contained apart from the flux read-out, which uses the helpers of the flux
section: it needs only the Setup section.

In [ ]:
# --- Laplacian eigenmodes: load the seeds, check them, summarise ---------------------
# Self-contained: needs only the Setup section. Every run here trained on the
# dt = 0.025 data, now in raw_data/parametric_old, with V^(1/3) as the volume
# feature, so every graph and T sequence comes from parametric_old, with its own
# mesh cache. The cached graphs hold EIG_K_MAX eigenmodes; a run takes the first
# n_eig of them, appended after node type and volume (static_graph.node_features).
from scipy.stats import mannwhitneyu

from data_preparation.mesh_cache import load_mesh_cached
from data_preparation.mesh_dataset import SingleMeshDataset
from data_preparation.normalization import FeatureNormalizer
from data_preparation.static_graph import load_graph_config, node_features
from git_commit import load_git_commit
from models.autoregressive_training import rollout
from models.fvgnn import FVSurrogate, widths_from_state_dict

LP_BASE = "parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_old_nopos"
LP_EIGS = {"V^(1/3)": 0, "eig 4": 4, "eig 8": 8, "eig 16": 16}   # baseline first
LP_CASES = {key: LP_BASE if k == 0 else LP_BASE.replace("_noise0.3_", f"_noise0.3_eig{k}_")
            for key, k in LP_EIGS.items()}
LP_REF = "V^(1/3)"
LP_RAW = Path("../raw_data/parametric_old")
LP_CACHE = processed_data_dir / "parametric_old_preproc_cache"
LP_EXCLUDED = ["top", "bottom", "cbores"]
LP_HISTORY = 1
LP_NOISE = 0.3
LP_BUDGET = 500

_lp_graphs = {}


def lp_load(n_eig):
    """load for one run: Cf graphs of parametric_old with V^(1/3), node type +
    volume + the first n_eig eigenmodes, no positions; all 10 edge features. Each
    mesh is read from the cache once; every call returns its own copy of the graph."""
    def load(mesh):
        if mesh not in _lp_graphs:
            _lp_graphs[mesh] = load_mesh_cached(str(LP_RAW / mesh), LP_EXCLUDED, "cf", LP_CACHE,
                                                cell_volume_transform="cbrt")
        g, T = _lp_graphs[mesh]
        g = g.clone()
        g.node_attr = node_features(g, True, False, n_eig)
        return g, T
    return load


LP_LOAD = {key: lp_load(k) for key, k in LP_EIGS.items()}


def lp_p(a, b):
    """Two-sided Mann-Whitney U p of two seed samples; inf ranks above every value."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue) if min(len(a), len(b)) > 1 else np.nan


lp = {key: dict(np.load(processed_data_dir / exp / "seeds.npz")) for key, exp in LP_CASES.items()}
LP_SEEDS = lp[LP_REF]["seeds"]
LP_MESHES = [str(m) for m in lp[LP_REF]["test_meshes"]]
LP_VAL_MESHES = [str(m) for m in lp[LP_REF]["val_meshes"]]
LP_VAL_STEPS = lp[LP_REF]["val_rollout_mae"].shape[-1]

# The same seeds, meshes, noise, budget and schedule: the modes add n_eig input
# columns to the node encoder, nothing else.
for key, z in lp.items():
    assert np.array_equal(z["seeds"], LP_SEEDS), f"{key}: seeds {z['seeds']}"
    assert [str(m) for m in z["test_meshes"]] == LP_MESHES, f"{key}: test meshes {z['test_meshes']}"
    assert [str(m) for m in z["val_meshes"]] == LP_VAL_MESHES, f"{key}: val meshes {z['val_meshes']}"
    assert float(z["input_noise"]) == LP_NOISE, f"{key}: input_noise = {z['input_noise']}"
    assert int(z["max_epochs"]) == LP_BUDGET, f"{key}: launched for {z['max_epochs']} epochs"
    assert str(z["msg_kind"]) == "none" and float(z["msg_alpha"]) == 0.0, f"{key}: a message penalty"
    for field in ("msg_warmup", "msg_ramp_len"):
        assert int(z[field]) == int(lp[LP_REF][field]), f"{key}: {field} differs"

# Per seed: the graphs it was built with, the data it saw, the commit.
lp_runs = {key: [f"{exp}/seed_{int(s)}" for s in LP_SEEDS] for key, exp in LP_CASES.items()}
for key, runs in lp_runs.items():
    k, commits = LP_EIGS[key], {}
    for run in runs:
        ck = processed_data_dir / run / "checkpoints"
        cfg = load_graph_config(ck)
        assert cfg["boundary_pos"] == "cf" and cfg["excluded_patches"] == LP_EXCLUDED, f"{run}: {cfg}"
        assert cfg["cell_volume"] and cfg["cell_volume_transform"] == "cbrt", f"{run}: {cfg}"
        assert cfg["dataset"] == "parametric_old", f"{run}: trained on {cfg['dataset']}"
        assert cfg["n_eig"] == k, f"{run}: n_eig = {cfg['n_eig']}"
        if k:
            assert (cfg["eig_field"], cfg["eig_coeff"], cfg["eig_time"]) == ("T", "DT", "0"), f"{run}: {cfg}"
        node_mean = torch.load(ck / "normalizer.pt")["node_mean"]
        assert node_mean.numel() == 3 + k, f"{run}: {node_mean.numel()} node features"
        assert torch.all(node_mean[3:] == 0), f"{run}: the eigenmode columns are not scale-only"
        c = load_git_commit(ck)
        tag = c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else "")
        commits.setdefault(tag, []).append(run.split("/")[-1])
    print(f"{LP_CASES[key]} ({int(lp[key]['n_params'])} parameters): "
          + "; ".join(f"commit {t} ({', '.join(r)})" for t, r in commits.items()))


def lp_last(z, field):
    """Each seed's value at its own last epoch (early-stopped seeds are NaN past it)."""
    return z[field][np.arange(len(z["seeds"])), z["n_epochs"] - 1]


# |T_true| per rollout step on the old data, for the RMAE from the saved MAE.
lp_absT = {m: LP_LOAD[LP_REF](m)[1][LP_HISTORY:].double().abs().mean(1).numpy() for m in LP_MESHES}


def lp_rmae(mae, mesh):
    """Rollout RMAE over all steps from the per-step MAE [K]; inf if it overflowed."""
    r = mae.astype(float).sum(-1) / lp_absT[mesh][:mae.shape[-1]].sum()
    return np.where(np.isfinite(r), r, np.inf)


lp_val_roll = {key: lp_last(z, "val_rollout_mae").mean((1, 2)) for key, z in lp.items()}

print(f"\nper seed: epochs, best val loss, last-epoch val rollout MAE [K] ({LP_VAL_STEPS} steps), "
      f"and the test rollout RMAE over {lp[LP_REF][f'rollout_mae_{LP_MESHES[0]}'].shape[1]} steps "
      f"from the MAE saved at training time; p: Mann-Whitney U against {LP_REF}")
print(f"{'':10s}{'seed':>6}{'epochs':>8}{'val best':>11}{'val roll':>10}"
      + "".join(f"{m:>12}" for m in LP_MESHES))
for key, z in lp.items():
    for i, s in enumerate(LP_SEEDS):
        print(f"{key if i == 0 else '':10s}{int(s):>6}{int(z['n_epochs'][i]):>8}"
              f"{np.nanmin(z['val_losses'][i]):>11.2e}{lp_val_roll[key][i]:>10.3f}"
              + "".join(f"{lp_rmae(z[f'rollout_mae_{m}'][i], m):>12.3e}" for m in LP_MESHES))
    if key != LP_REF:
        r = lp[LP_REF]
        print(f"{'  p (MWU)':10s}{'':>6}{lp_p(r['n_epochs'], z['n_epochs']):>8.2f}"
              f"{lp_p(np.nanmin(r['val_losses'], 1), np.nanmin(z['val_losses'], 1)):>11.2f}"
              f"{lp_p(lp_val_roll[LP_REF], lp_val_roll[key]):>10.2f}"
              + "".join(f"{lp_p(lp_rmae(r[f'rollout_mae_{m}'], m), lp_rmae(z[f'rollout_mae_{m}'], m)):>12.2f}"
                        for m in LP_MESHES))

In [ ]:
# --- Laplacian eigenmodes: one-step and rollout errors of every seed -------------------
# Every seed's last-epoch checkpoint (model.pt) on the test meshes, each on its own
# graphs: the one-step RMAE / RMSE summed over every sample and node, the full
# rollout RMAE / RMSE summed over every step and node, and per rollout step t
# RMAE_t and RMSE_t (sums over the nodes), for the accumulation plot below. RMSE is
# the relative SQUARED error (no square root), as everywhere in this notebook.
# Cached per checkpoint, stamped with the mtime of its model.pt.
lp_eval_cache = processed_data_dir / "parametric_eval_laplacian.npz"
LP_ERR_COLS = {"rmae_step": "one-step RMAE", "rmse_step": "one-step RMSE",
               "rmae_roll": "rollout RMAE", "rmse_roll": "rollout RMSE"}


def lp_evaluate(run, load):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / "model.pt", map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in LP_MESHES:
        graph, T_seq = load(mesh)
        model = FVSurrogate(
            in_node_feat=LP_HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=LP_HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        ds = SingleMeshDataset(T_seq, graph, norm, LP_HISTORY)   # no sign flips
        tot = np.zeros(4)
        with torch.no_grad():
            for i in range(len(ds)):
                d = ds[i].to(device)
                e = (norm.inverse_transform_T(model(d).cpu()) - norm.inverse_transform_T(d.y.cpu())).double()
                T_t = norm.inverse_transform_T(d.y.cpu()).double()
                tot += [e.abs().sum().item(), T_t.abs().sum().item(),
                        e.pow(2).sum().item(), T_t.pow(2).sum().item()]
        out[f"{run}__{mesh}__rmae_step"] = tot[0] / tot[1]
        out[f"{run}__{mesh}__rmse_step"] = tot[2] / tot[3]
        T_p, T_t = rollout(model, ds, device=device)
        e, T_t = (T_p - T_t).double(), T_t.double()
        out[f"{run}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_t.abs().sum())
        out[f"{run}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_t.pow(2).sum())
        out[f"{run}__{mesh}__rmae_curve"] = (e.abs().sum(1) / T_t.abs().sum(1)).numpy()
        out[f"{run}__{mesh}__rmse_curve"] = (e.pow(2).sum(1) / T_t.pow(2).sum(1)).numpy()
        out[f"{run}__{mesh}__mae_roll"] = e.abs().mean(1).numpy()
        del model
    return out


def lp_stamp(run):
    return np.int64((processed_data_dir / run / "checkpoints" / "model.pt").stat().st_mtime_ns)


lp_eval = dict(np.load(lp_eval_cache, allow_pickle=False)) if lp_eval_cache.exists() else {}
for key, runs in lp_runs.items():
    for run in runs:
        if lp_eval.get(f"{run}__stamp") != lp_stamp(run):
            lp_eval.update(lp_evaluate(run, LP_LOAD[key]))
            lp_eval[f"{run}__stamp"] = lp_stamp(run)
            np.savez_compressed(lp_eval_cache, **lp_eval)   # after every checkpoint


def lp_err(key, name):
    """(S, M) one error of every seed of key on every test mesh; inf if it overflowed."""
    a = np.array([[float(lp_eval[f"{run}__{m}__{name}"]) for m in LP_MESHES] for run in lp_runs[key]])
    return np.where(np.isfinite(a), a, np.inf)


def lp_gstat(a):
    """Geometric mean and geometric std over the seeds (axis 0): the errors span
    decades between seeds, and one diverging seed would be the whole arithmetic mean."""
    with np.errstate(invalid="ignore"):
        l = np.log10(a)
        return 10 ** l.mean(0), 10 ** l.std(0, ddof=1)


# The data are the ones they trained on: the test rollout MAE recomputed here has
# to be the one saved at the end of training.
dt = [np.abs(lp_eval[f"{run}__{m}__mae_roll"] / z[f"rollout_mae_{m}"][i] - 1).max()
      for key, z in lp.items() for i, run in enumerate(lp_runs[key]) for m in LP_MESHES]
print(f"recomputed on parametric_old vs saved in training, every seed and test mesh: "
      f"max |relative difference| of the rollout MAE {max(dt):.1e}")

print(f"\nlast-epoch checkpoints on the test meshes: geometric mean ×/÷ geometric std over the "
      f"{len(LP_SEEDS)} seeds; p: Mann-Whitney U against {LP_REF}, 3 against 3 (smallest possible 0.1)")
print(f"{'mesh':<11}{'run':<11}" + "".join(f"{h:>22}" for h in LP_ERR_COLS.values()))
lp_tab = {key: {c: lp_err(key, c) for c in LP_ERR_COLS} for key in LP_CASES}
for j, m in enumerate(LP_MESHES + ["all meshes"]):
    for key in LP_CASES:
        # all meshes: each seed's geometric mean over the meshes
        cols = {c: (a[:, j] if j < len(LP_MESHES) else 10 ** np.log10(a).mean(1))
                for c, a in lp_tab[key].items()}
        g = {c: lp_gstat(v) for c, v in cols.items()}
        print(f"{m if key == LP_REF else '':<11}{key:<11}"
              + "".join(f"{f'{mu:.3e} ×/÷ {sd:.2f}':>22}" for mu, sd in g.values()))
        if key != LP_REF:
            ref = {c: (a[:, j] if j < len(LP_MESHES) else 10 ** np.log10(a).mean(1))
                   for c, a in lp_tab[LP_REF].items()}
            print(f"{'':<11}{'  p (MWU)':<11}" + "".join(f"{lp_p(ref[c], cols[c]):>22.2f}" for c in LP_ERR_COLS))
    print()

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT4_COLOR = "#2b5d8a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# V^(1/3) in its colour of the cbrt section, dashed; the eigenmode runs one hue,
# darker with more modes.
LP_STYLES = {"V^(1/3)": (PLOT4_COLOR, "--"),
             "eig 4":   ("#e09aa6", "-"),
             "eig 8":   ("#c0566c", "-"),
             "eig 16":  ("#7f1d3b", "-")}


def lp_style_ax(ax, labelsize=14):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=labelsize)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


def lp_legend(legend):
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)


# One-step and rollout RMAE / RMSE of every last-epoch checkpoint: bar = geometric
# mean over the seeds, dots = the seeds. A rollout that overflowed has no dot and
# is counted in the printout instead.
fig, axs = plt.subplots(1, 4, figsize=(2000*px, 600*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Test Errors with Laplacian Eigenmodes, V^(1/3), Noise 0.3, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(bar: geometric mean over the seeds, dots: the seeds)",
             fontsize=21, color=AX_COLOR)
width = 0.8 / len(LP_CASES)
overflow = 0
for ax, (col, title) in zip(axs, LP_ERR_COLS.items()):
    for k, key in enumerate(LP_CASES):
        color, ls = LP_STYLES[key]
        a = lp_tab[key][col]                                 # (S, M)
        x = np.arange(len(LP_MESHES)) + (k - (len(LP_CASES) - 1) / 2) * width
        ax.bar(x, lp_gstat(a)[0], width=width * 0.9, color=color, alpha=0.35, edgecolor=color,
               linewidth=1.4, linestyle=ls, label=key)
        for j in range(len(LP_MESHES)):
            ok = np.isfinite(a[:, j])
            overflow += int((~ok).sum()) if col == "rmae_roll" else 0
            ax.plot(np.full(ok.sum(), x[j]), a[ok, j], "o", color=color, markersize=5)
    lp_style_ax(ax, 13)
    ax.set_yscale("log")
    ax.set_xticks(np.arange(len(LP_MESHES)))
    ax.set_xticklabels(LP_MESHES, rotation=20, fontsize=13)
    ax.set_title(title, fontsize=18)
lp_legend(axs[0].legend(fontsize=12, loc="upper left"))
if overflow:
    print(f"{overflow} seed-mesh rollouts overflowed (no dot in the rollout panels)")

# plt.savefig("../outputs/plots/parametric_history1_laplacian_errors.png", dpi=300, transparent = True)

In [ ]:
AX_COLOR = "#1e2a2c"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Error accumulation over the rollout: RMAE_t and RMSE_t at each step t (sums over
# the nodes), every seed thin, the median over the seeds bold (a mean would be the
# one seed that diverges). Line styles of the error plot above.
fig, axs = plt.subplots(2, len(LP_MESHES), figsize=(1900*px, 950*px), sharex=True, sharey="row",
                        constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation with Laplacian Eigenmodes, V^(1/3), Noise 0.3, No Positions, "
             "Cf, msg_dim = 64, h = 1\n(thin: every seed, bold: median over the seeds)",
             fontsize=21, color=AX_COLOR)
for row, (name, ylabel) in enumerate((("rmae_curve", "RMAE$_t$"), ("rmse_curve", "RMSE$_t$"))):
    for col, mesh in enumerate(LP_MESHES):
        ax = axs[row, col]
        for key, runs in lp_runs.items():
            color, ls = LP_STYLES[key]
            curves = np.stack([lp_eval[f"{run}__{mesh}__{name}"] for run in runs])   # (S, steps)
            steps = np.arange(1, curves.shape[1] + 1)
            for c in curves:
                ax.plot(steps, c, color=color, linestyle=ls, linewidth=0.8, alpha=0.4)
            ax.plot(steps, np.median(curves, 0), color=color, linestyle=ls, linewidth=2.4, label=key)
        lp_style_ax(ax, 13)
        ax.set_yscale("log")
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step", fontsize=15)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=16)
lp_legend(axs[0, 0].legend(fontsize=12, loc="lower right"))

# plt.savefig("../outputs/plots/parametric_history1_laplacian_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- Laplacian eigenmodes: message vs FV flux, per seed ----------------------------------
# compute_model_flux_stats (residual cell of the flux section) on every seed's
# last-epoch checkpoint, each on its own graphs (LP_LOAD), with the time directories
# and grad(T) of parametric_old (raw_dir: the T sequences are the old ones), on the
# same seeded faces and N_STEPS probed timesteps as every read-out above. Every
# column is an R^2, a correlation or an energy fraction, so none depends on how
# large a run writes its message.
lp_flux_cache = processed_data_dir / "parametric_message_flux_laplacian.npz"
LP_FLUX_COLS = ("R2 F", "R2 dT", "R2 w", "R2 node", "|r| best", "anti")
assert list(FLUX_MESHES) == LP_MESHES, "the read-out meshes are not the test meshes"

lp_flux = dict(np.load(lp_flux_cache, allow_pickle=False)) if lp_flux_cache.exists() else {}
for key, runs in lp_runs.items():
    for run in runs:
        if lp_flux.get(f"{run}__stamp") != lp_stamp(run):
            lp_flux.update(compute_model_flux_stats(run, load=LP_LOAD[key], raw_dir=LP_RAW))
            lp_flux[f"{run}__stamp"] = lp_stamp(run)
            np.savez_compressed(lp_flux_cache, **lp_flux)   # after every run: these are slow


def lp_flux_table(key):
    """(S, M, 6) read-outs in LP_FLUX_COLS order."""
    return np.array([[[*lp_flux[f"{run}__{m}__r2"], float(lp_flux[f"{run}__{m}__r2_node"]),
                       abs(float(lp_flux[f"{run}__{m}__best_r"])), lp_flux[f"{run}__{m}__anti"].mean()]
                      for m in LP_MESHES] for run in lp_runs[key]])


lp_flux_tab = {key: lp_flux_table(key) for key in LP_CASES}

# Arithmetic mean ± std over the seeds: these are bounded in [0, 1]. Under each
# eigenmode row the Mann-Whitney p against V^(1/3).
print(f"Linear read-out from the message, {N_STEPS} probed timesteps, last-epoch checkpoints, "
      f"mean ± std over the seeds")
print(f"{'mesh':<11}{'run':<11}" + "".join(f"{c:>15}" for c in LP_FLUX_COLS))
for j, m in enumerate(LP_MESHES + ["all meshes"]):
    for key in LP_CASES:
        t = lp_flux_tab[key][:, j] if j < len(LP_MESHES) else lp_flux_tab[key].mean(1)   # (S, 6)
        print(f"{m if key == LP_REF else '':<11}{key:<11}"
              + "".join(f"{f'{mu:.3f} ± {sd:.3f}':>15}" for mu, sd in zip(t.mean(0), t.std(0, ddof=1))))
        if key != LP_REF:
            r = lp_flux_tab[LP_REF][:, j] if j < len(LP_MESHES) else lp_flux_tab[LP_REF].mean(1)
            print(f"{'':<11}{'  p (MWU)':<11}" + "".join(f"{lp_p(r[:, c], t[:, c]):>15.2f}"
                                                        for c in range(len(LP_FLUX_COLS))))
    print()

In [ ]:
AX_COLOR = "#1e2a2c"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The read-outs of the cell above, each seed averaged over the four test meshes:
# bar = mean over the seeds, whisker = one std, dots = the seeds.
fig, ax = plt.subplots(figsize=(1500*px, 600*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs FV Flux with Laplacian Eigenmodes, V^(1/3), Noise 0.3, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(mean over the test meshes; bar: mean over the seeds, "
             "whisker: std, dots: the seeds)", fontsize=19, color=AX_COLOR)
width = 0.8 / len(LP_CASES)
for k, key in enumerate(LP_CASES):
    color, ls = LP_STYLES[key]
    t = lp_flux_tab[key].mean(1)                             # (S, 6)
    x = np.arange(len(LP_FLUX_COLS)) + (k - (len(LP_CASES) - 1) / 2) * width
    ax.bar(x, t.mean(0), width=width * 0.9, yerr=t.std(0, ddof=1), color=color, alpha=0.35,
           edgecolor=color, linewidth=1.4, linestyle=ls, label=key,
           error_kw={"ecolor": AX_COLOR, "elinewidth": 1.2, "capsize": 3})
    for c in range(len(LP_FLUX_COLS)):
        ax.plot(np.full(len(t), x[c]), t[:, c], "o", color=color, markersize=5)
lp_style_ax(ax, 14)
ax.set_xticks(np.arange(len(LP_FLUX_COLS)))
ax.set_xticklabels(["$R^2$ flux $F$", "$R^2$ $\\Delta T$", "$R^2$ weight $w$", "$R^2$ node $\\sum_j F$",
                    "$|r|$ best channel", "antisymmetric\nenergy"], fontsize=14)
ax.set_ylim(0, 1.05)
lp_legend(ax.legend(fontsize=12, loc="upper right", ncols=len(LP_CASES)))

# plt.savefig("../outputs/plots/parametric_history1_laplacian_flux.png", dpi=300, transparent = True)

### What comes out

**The eigenmodes make every model much worse, and more modes make it worse still.** Every eigenmode
seed is worse than every baseline seed on every test mesh and every metric ($p = 0.1$
throughout, the smallest possible with 3 against 3). Geometric mean over the seeds of each seed's
geometric mean over the four test meshes:

| | V^(1/3) | eig 4 | eig 8 | eig 16 |
|---|---|---|---|---|
| one-step RMAE | 1.01e-3 | 4.49e-3 (×4.4) | 9.49e-3 (×9.4) | 1.42e-2 (×14) |
| one-step RMSE | 1.15e-5 | 2.40e-4 (×21) | 9.02e-4 (×78) | 1.66e-3 (×144) |
| rollout RMAE | 6.66e-3 | 0.127 (×19) | 0.201 (×30) | 0.200 (×30) |
| rollout RMSE | 1.68e-4 | 5.94e-2 (×354) | 0.105 (×628) | 0.104 (×621) |

Recomputed on `parametric_old`, the test rollout MAE of all twelve checkpoints matches the one saved
in training to $7\times10^{-6}$ relative, so these are the models as trained, on their own data.

**The failure is in generalisation, and it is already there in training.** The eigenmode runs fit
the training meshes *better* than the baseline: their lowest training loss is
$6.8$ – $7.3\times10^{-5}$ in all nine seeds, against $1.4$ – $3.1\times10^{-4}$ (both measured with
noise 0.3 on the input). Their val loss goes the other way. Its minimum is $1.1$ – $1.7\times10^{-4}$ with 4
modes (reached at epochs 11 – 40), and $1.0\times10^{-3}$ and $2.6$ – $3.3\times10^{-3}$ with 8 and 16
modes, reached at **epoch 0**: from the first epoch on, the better they fit the training meshes the worse they do
on the val meshes. The baseline's minimum is $2.6$ – $5.3\times10^{-5}$. Early stopping, which
follows the val loss, ends them at 116 – 126 epochs against 166 – 227. The 40-step val rollout MAE at
the last epoch is 46 – 66 K, 61 – 84 K and 114 – 135 K, against 1.05 – 1.14 K.

**Over the rollout, the error grows from the first step and saturates.** The baseline's
$\mathrm{RMAE}_t$ falls over the first ~10 steps (from $0.7$ – $2.6\times10^{-2}$ to $0.3$ –
$0.7\times10^{-2}$) and then grows slowly, to $0.5$ – $1.8\times10^{-2}$ at step 120. Every eigenmode
model starts at about the baseline's step-1 level ($0.7$ – $4\times10^{-2}$) but never recovers.
$\mathrm{RMAE}_t$ rises from step 1, passes 0.1 within 4 – 10 steps on `model_010` and
`model_011` with 8 or 16 modes, and flattens at 0.1 – 0.5. With 4 modes the growth is slower
(above 0.1 at step 12 on `model_010`, 27 on `model_011`, 62 on `model_003`, never on `model_007`).
`model_010` is the worst mesh for every eigenmode run, whereas the baseline does well on it.

**The message still carries the flux.** Averaged over the test meshes, the linear read-outs
lose 0.01 – 0.06 in $R^2$. $F$: 0.837 → 0.823 / 0.780 / 0.804; $\Delta T$: 0.935 → 0.903 / 0.874 /
0.887; $\sum_j F$ from the aggregated message: 0.666 → 0.644 / 0.623 / 0.638. The drop is below the
noise with 4 modes ($p = 0.2$ – $0.4$) and consistent with 8 and 16 ($p = 0.1$ for $F$ and
$\Delta T$). With 4 modes the message gets more antisymmetric (0.256 → 0.329, $p = 0.1$). A
message whose flux read-out is 93 – 98 % of the baseline's cannot account for errors 20 – 600× larger: what breaks
is not the edge function but what the node update does with the extra columns.

**What this suggests (not tested here).** The modes are computed on each mesh separately. On a
geometry the network has not seen, mode $j$ need not resemble mode $j$ of the training meshes:
near-degenerate modes swap order or mix, and the sign convention is a convention, not geometry. With
six training meshes, the network can use the modes to tell the meshes apart and fit each one, which
is what the falling training loss and the rising val loss from epoch 0 look like. More modes
give it more to memorise, and more high-frequency, less transferable patterns. Random sign
flips (`FVGNN_EIG_FLIP=1`) would remove the sign shortcut only. Order and mixing would remain.
Features that do not depend on mode identity (the eigenvalue-weighted heat kernel or a diffusion
distance to the boundary, for instance) would be the natural next test.

## $V^{1/3}$ on the dt = 0.005 data (120 steps)

The first run on `raw_data/parametric` as it has been since 2026-11-17: written at every solver step
(dt = 0.005 s) up to endTime 0.6 s, so **120 steps per mesh**, with one model step covering 0.005 s.

- **Run**: `parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_nopos`, `seed_<n>`, seeds 0 – 2,
  commit `f9709a1` + uncommitted changes (the copy of `train_parametric.py` the launcher saved
  is identical to the one in `f9709a1`). Launcher log in `logs/20261117_150510`.
- **Configuration**: the residual model, `msg_dim = 64`, history 1, Cf boundary nodes, node type +
  $V^{1/3}$, no positions, all 10 edge features, input noise 0.3, the LR scheduler, no message
  penalty, at most 500 epochs with early stopping (116 673 parameters). Same mesh split as every
  parametric run: training `model_001`, `002`, `004`, `005`, `006`, `008`; validation `model_000`, `009`;
  test `model_003`, `007`, `010`, `011`.

The 120-step rollout covers 0.6 s, against 3 s for the 120-step rollouts of the dt = 0.025 runs in
the sections above, so the numbers here are **not comparable step for step** with those.

All on the **last-epoch checkpoints** (`model.pt`), every seed, on the test meshes:

- **Errors**: one-step and 120-step rollout RMAE / RMSE, as sums over every node and sample (one-step)
  or step (rollout). Every seed, and the geometric mean ×/÷ geometric std over them.
- **Error accumulation**: $\mathrm{RMAE}_t$ and $\mathrm{RMSE}_t$ at each rollout step $t$, against the
  one-step error at the same step, teacher-forced (the model fed the true field $T_{t-1}$ instead of
  its own prediction). The gap between the two is what the rollout accumulates.
- **Message vs FV flux**: the linear read-out of the FV flux from the message, as in the sections
  above ($R^2$ of $F$, $\Delta T$, $w$, of $\sum_j F$ from the aggregated message, the best single
  channel's $|r|$, and the antisymmetric energy fraction), on 12 timesteps spread over the trajectory.

Needs the Setup section: the flux read-out uses the helpers of the flux section.

In [ ]:
# --- dt = 0.005 data: load the seeds, check them, evaluate the test rollouts --------
# Needs the Setup section. The run trained on raw_data/parametric as it is since
# 2026-11-17: written at every solver step (dt = 0.005 s) up to endTime 0.6, so 120
# steps per mesh, with V^(1/3) as the volume feature. Every seed's last-epoch
# checkpoint (model.pt) on the test meshes, on its own graphs: the one-step RMAE /
# RMSE summed over every sample and node, the full rollout RMAE / RMSE summed over
# every step and node, and per rollout step t RMAE_t and RMSE_t (sums over the
# nodes), for the accumulation plot below. RMSE is the relative SQUARED error (no
# square root), as everywhere in this notebook. Cached per checkpoint, stamped with
# the mtime of its model.pt.
from data_preparation.mesh_cache import load_mesh_cached
from data_preparation.mesh_dataset import SingleMeshDataset
from data_preparation.normalization import FeatureNormalizer
from data_preparation.static_graph import load_graph_config, node_features
from git_commit import load_git_commit
from models.autoregressive_training import rollout
from models.fvgnn import FVSurrogate, widths_from_state_dict

FT_EXP = "parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_nopos"
FT_RAW = Path("../raw_data/parametric")
FT_CACHE = processed_data_dir / "parametric_preproc_cache"
FT_EXCLUDED = ["top", "bottom", "cbores"]
FT_HISTORY = 1
FT_N_STEPS = 120                      # steps per trajectory: endTime 0.6 / dt 0.005
FT_DT = 0.005                         # s, one model step
ft_eval_cache = processed_data_dir / "parametric_eval_fine.npz"
FT_ERR_COLS = {"rmae_step": "one-step RMAE", "rmse_step": "one-step RMSE",
               "rmae_roll": "rollout RMAE", "rmse_roll": "rollout RMSE"}

_ft_graphs = {}


def ft_load(mesh):
    """(graph, T_seq) of one mesh as this run built them: Cf graphs of
    raw_data/parametric with V^(1/3), node type + volume, no positions, all 10 edge
    features. Each mesh is read from the cache once; every call returns its own
    copy of the graph."""
    if mesh not in _ft_graphs:
        _ft_graphs[mesh] = load_mesh_cached(str(FT_RAW / mesh), FT_EXCLUDED, "cf", FT_CACHE,
                                            cell_volume_transform="cbrt")
    g, T = _ft_graphs[mesh]
    g = g.clone()
    g.node_attr = node_features(g, True, False)
    return g, T


ft = dict(np.load(processed_data_dir / FT_EXP / "seeds.npz"))
FT_SEEDS = ft["seeds"]
FT_MESHES = [str(m) for m in ft["test_meshes"]]
FT_RUNS = [f"{FT_EXP}/seed_{int(s)}" for s in FT_SEEDS]
assert str(ft["msg_kind"]) == "none" and float(ft["msg_alpha"]) == 0.0, "a message penalty"
assert float(ft["input_noise"]) == 0.3, f"input_noise = {ft['input_noise']}"

# Per seed: the graphs it was built with, the data it saw, the commit.
commits = {}
for run in FT_RUNS:
    ck = processed_data_dir / run / "checkpoints"
    cfg = load_graph_config(ck)
    assert cfg["dataset"] == "parametric" and cfg["boundary_pos"] == "cf", f"{run}: {cfg}"
    assert cfg["excluded_patches"] == FT_EXCLUDED and cfg["n_eig"] == 0, f"{run}: {cfg}"
    assert cfg["cell_volume"] and cfg["cell_volume_transform"] == "cbrt", f"{run}: {cfg}"
    assert torch.load(ck / "normalizer.pt")["node_mean"].numel() == 3, f"{run}: node features"
    c = load_git_commit(ck)
    tag = c["commit"][:7] + (" + uncommitted changes" if c["dirty"] else "")
    commits.setdefault(tag, []).append(run.split("/")[-1])
print(f"{FT_EXP} ({int(ft['n_params'])} parameters): "
      + "; ".join(f"commit {t} ({', '.join(r)})" for t, r in commits.items()))
for m in FT_MESHES:
    n = len(ft_load(m)[1])
    assert n == FT_N_STEPS + 1, f"{m}: {n} fields, not {FT_N_STEPS + 1}: the cache is not the 120-step data"


def ft_last(field):
    """Each seed's value at its own last epoch (early-stopped seeds are NaN past it)."""
    return ft[field][np.arange(len(FT_SEEDS)), ft["n_epochs"] - 1]


ft_val_roll = ft["val_rollout_mae"].mean((2, 3))                  # (S, epochs) [K]
print(f"\n{'seed':>4}{'epochs':>8}{'best val loss':>15}{'(epoch)':>8}"
      f"{'val rollout MAE [K], 40 steps: last':>38}{'best':>8}{'(epoch)':>8}")
for i, s in enumerate(FT_SEEDS):
    print(f"{int(s):>4}{int(ft['n_epochs'][i]):>8}{np.nanmin(ft['val_losses'][i]):>15.2e}"
          f"{int(np.nanargmin(ft['val_losses'][i])):>8}{ft_last('val_rollout_mae')[i].mean():>38.3f}"
          f"{np.nanmin(ft_val_roll[i]):>8.3f}{int(ft['val_rollout_best_epoch'][i]):>8}")


def ft_evaluate(run):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = processed_data_dir / run / "checkpoints"
    state = torch.load(ck / "model.pt", map_location=device)
    norm = FeatureNormalizer()
    norm.load(ck / "normalizer.pt")
    out = {}
    for mesh in FT_MESHES:
        graph, T_seq = ft_load(mesh)
        model = FVSurrogate(
            in_node_feat=FT_HISTORY + graph.node_attr.shape[1],
            in_edge_feat=graph.edge_attr.shape[1],
            **widths_from_state_dict(state), out_dim=1,
            residual="delta_scale" in state, history=FT_HISTORY,
            layer_norm="node_encoder_norm.weight" in state,
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        ds = SingleMeshDataset(T_seq, graph, norm, FT_HISTORY)   # no input noise
        tot = np.zeros(4)
        step_rmae, step_rmse = [], []
        with torch.no_grad():
            for i in range(len(ds)):
                d = ds[i].to(device)
                T_t = norm.inverse_transform_T(d.y.cpu()).double()
                e = norm.inverse_transform_T(model(d).cpu()).double() - T_t
                tot += [e.abs().sum().item(), T_t.abs().sum().item(),
                        e.pow(2).sum().item(), T_t.pow(2).sum().item()]
                step_rmae.append(e.abs().sum().item() / T_t.abs().sum().item())
                step_rmse.append(e.pow(2).sum().item() / T_t.pow(2).sum().item())
        out[f"{run}__{mesh}__rmae_step"] = tot[0] / tot[1]
        out[f"{run}__{mesh}__rmse_step"] = tot[2] / tot[3]
        # the one-step errors of each sample: teacher-forced, from the true field
        out[f"{run}__{mesh}__rmae_step_curve"] = np.array(step_rmae)
        out[f"{run}__{mesh}__rmse_step_curve"] = np.array(step_rmse)
        T_p, T_t = rollout(model, ds, device=device)
        e, T_t = (T_p - T_t).double(), T_t.double()
        out[f"{run}__{mesh}__rmae_roll"] = float(e.abs().sum() / T_t.abs().sum())
        out[f"{run}__{mesh}__rmse_roll"] = float(e.pow(2).sum() / T_t.pow(2).sum())
        out[f"{run}__{mesh}__rmae_curve"] = (e.abs().sum(1) / T_t.abs().sum(1)).numpy()
        out[f"{run}__{mesh}__rmse_curve"] = (e.pow(2).sum(1) / T_t.pow(2).sum(1)).numpy()
        out[f"{run}__{mesh}__mae_roll"] = e.abs().mean(1).numpy()
        del model
    return out


def ft_stamp(run):
    return np.int64((processed_data_dir / run / "checkpoints" / "model.pt").stat().st_mtime_ns)


ft_eval = dict(np.load(ft_eval_cache, allow_pickle=False)) if ft_eval_cache.exists() else {}
for run in FT_RUNS:
    if ft_eval.get(f"{run}__stamp") != ft_stamp(run):
        ft_eval.update(ft_evaluate(run))
        ft_eval[f"{run}__stamp"] = ft_stamp(run)
        np.savez_compressed(ft_eval_cache, **ft_eval)   # after every checkpoint


def ft_err(name):
    """(S, M) one error of every seed on every test mesh; inf if it overflowed."""
    a = np.array([[float(ft_eval[f"{run}__{m}__{name}"]) for m in FT_MESHES] for run in FT_RUNS])
    return np.where(np.isfinite(a), a, np.inf)


def ft_curves(name, mesh):
    """(S, steps) per-step curve of every seed on one mesh."""
    return np.stack([ft_eval[f"{run}__{mesh}__{name}"] for run in FT_RUNS])


def ft_gstat(a):
    """Geometric mean and geometric std over the seeds (axis 0)."""
    with np.errstate(invalid="ignore"):
        l = np.log10(a)
        return 10 ** l.mean(0), 10 ** l.std(0, ddof=1)


# The data are the ones the run trained on: the test rollout MAE recomputed here
# has to be the one saved at the end of training.
d = [np.abs(ft_eval[f"{run}__{m}__mae_roll"] / ft[f"rollout_mae_{m}"][i] - 1).max()
     for i, run in enumerate(FT_RUNS) for m in FT_MESHES]
print(f"\nrecomputed vs saved in training, every seed and test mesh: "
      f"max |relative difference| of the rollout MAE {max(d):.1e}")

ft_tab = {c: ft_err(c) for c in FT_ERR_COLS}
print(f"\nlast-epoch checkpoints on the test meshes ({FT_N_STEPS}-step rollouts = "
      f"{FT_N_STEPS * FT_DT:.1f} s): every seed, and the geometric mean ×/÷ geometric std over them")
print(f"{'mesh':<12}{'seed':<8}" + "".join(f"{h:>22}" for h in FT_ERR_COLS.values())
      + f"{'rollout / one-step':>20}")
for j, m in enumerate(FT_MESHES + ["all meshes"]):
    # all meshes: each seed's geometric mean over the meshes
    cols = {c: (a[:, j] if j < len(FT_MESHES) else 10 ** np.log10(a).mean(1)) for c, a in ft_tab.items()}
    for i, s in enumerate(FT_SEEDS):
        print(f"{m if i == 0 else '':<12}{int(s):<8}" + "".join(f"{v[i]:>22.3e}" for v in cols.values())
              + f"{cols['rmae_roll'][i] / cols['rmae_step'][i]:>20.1f}")
    g = {c: ft_gstat(v) for c, v in cols.items()}
    print(f"{'':<12}{'geo':<8}" + "".join(f"{f'{mu:.3e} ×/÷ {sd:.2f}':>22}" for mu, sd in g.values()))
    print()

# The accumulation in numbers: the median over the seeds of RMAE_t at a few steps,
# the step of its minimum, and its last value over the one-step RMAE.
FT_PROBE = [1, 2, 5, 10, 20, 40, 80, 120]
print("rollout RMAE_t, median over the seeds, at step t")
print(f"{'mesh':<12}" + "".join(f"{f't={t}':>10}" for t in FT_PROBE) + f"{'argmin t':>10}{'t=120 / 1-step':>16}")
for j, m in enumerate(FT_MESHES):
    c = np.median(ft_curves("rmae_curve", m), 0)
    print(f"{m:<12}" + "".join(f"{c[t - 1]:>10.2e}" for t in FT_PROBE)
          + f"{int(np.argmin(c)) + 1:>10}{c[-1] / np.median(ft_tab['rmae_step'][:, j]):>16.1f}")

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT3_COLOR = "#b4622a"
PLOT4_COLOR = "#2b5d8a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches


def ft_style_ax(ax, labelsize=14):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=labelsize)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


def ft_legend(legend):
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)


# Error accumulation over the rollout: RMAE_t and RMSE_t at each step t (sums over
# the nodes), every seed thin, the median over the seeds bold. Against it, the
# one-step error of the same step, teacher-forced: the model fed the true field
# T[t-1] instead of its own prediction. The gap between the two is what the rollout
# accumulates; the dotted line is the one-step error over the whole trajectory.
fig, axs = plt.subplots(2, len(FT_MESHES), figsize=(1900*px, 950*px), sharex=True, sharey="row",
                        constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Rollout Error Accumulation, dt = 0.005 Data, V^(1/3), Noise 0.3, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(thin: every seed, bold: median over the seeds)",
             fontsize=21, color=AX_COLOR)
for row, (name, ylabel) in enumerate((("rmae", "RMAE$_t$"), ("rmse", "RMSE$_t$"))):
    for col, mesh in enumerate(FT_MESHES):
        ax = axs[row, col]
        for kind, color, label in (("curve", PLOT4_COLOR, "rollout"),
                                   ("step_curve", PLOT3_COLOR, "one-step (teacher-forced)")):
            curves = ft_curves(f"{name}_{kind}", mesh)               # (S, steps)
            steps = np.arange(1, curves.shape[1] + 1)
            for c in curves:
                ax.plot(steps, c, color=color, linewidth=0.8, alpha=0.4)
            ax.plot(steps, np.median(curves, 0), color=color, linewidth=2.4, label=label)
        ax.axhline(np.median(ft_tab[f"{name}_step"][:, col]), color=PLOT3_COLOR, linestyle=":",
                   linewidth=1.6, label="one-step, whole trajectory")
        ft_style_ax(ax, 13)
        ax.set_yscale("log")
        if row == 0:
            ax.set_title(mesh, fontsize=19)
        else:
            ax.set_xlabel("Time step (dt = 0.005 s)", fontsize=15)
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=16)
ft_legend(axs[0, 0].legend(fontsize=12, loc="upper left"))

# plt.savefig("../outputs/plots/parametric_history1_fine_rollout.png", dpi=300, transparent = True)

In [ ]:
# --- dt = 0.005 data: message vs FV flux, per seed -----------------------------------
# compute_model_flux_stats (residual cell of the flux section) on every seed's
# last-epoch checkpoint, on its own graphs (ft_load), with the time directories and
# grad(T) of raw_data/parametric (the 120-step data), on the same seeded faces and
# N_STEPS probed timesteps, spread over the trajectory, as every read-out above.
# Every column is an R^2, a correlation or an energy fraction, so none depends on
# how large a run writes its message.
ft_flux_cache = processed_data_dir / "parametric_message_flux_fine.npz"
FT_FLUX_COLS = ("R2 F", "R2 dT", "R2 w", "R2 node", "|r| best", "anti")
assert list(FLUX_MESHES) == FT_MESHES, "the read-out meshes are not the test meshes"

ft_flux = dict(np.load(ft_flux_cache, allow_pickle=False)) if ft_flux_cache.exists() else {}
for run in FT_RUNS:
    if ft_flux.get(f"{run}__stamp") != ft_stamp(run):
        ft_flux.update(compute_model_flux_stats(run, load=ft_load, raw_dir=FT_RAW))
        ft_flux[f"{run}__stamp"] = ft_stamp(run)
        np.savez_compressed(ft_flux_cache, **ft_flux)   # after every run: these are slow

# (S, M, 6) read-outs in FT_FLUX_COLS order
ft_flux_tab = np.array([[[*ft_flux[f"{run}__{m}__r2"], float(ft_flux[f"{run}__{m}__r2_node"]),
                          abs(float(ft_flux[f"{run}__{m}__best_r"])), ft_flux[f"{run}__{m}__anti"].mean()]
                         for m in FT_MESHES] for run in FT_RUNS])

# Every seed, and the arithmetic mean ± std over them: these are bounded in [0, 1].
print(f"Linear read-out from the message, {N_STEPS} probed timesteps, last-epoch checkpoints")
print(f"{'mesh':<12}{'seed':<8}" + "".join(f"{c:>15}" for c in FT_FLUX_COLS))
for j, m in enumerate(FT_MESHES + ["all meshes"]):
    t = ft_flux_tab[:, j] if j < len(FT_MESHES) else ft_flux_tab.mean(1)   # (S, 6)
    for i, s in enumerate(FT_SEEDS):
        print(f"{m if i == 0 else '':<12}{int(s):<8}" + "".join(f"{v:>15.3f}" for v in t[i]))
    print(f"{'':<12}{'mean':<8}"
          + "".join(f"{f'{mu:.3f} ± {sd:.3f}':>15}" for mu, sd in zip(t.mean(0), t.std(0, ddof=1))))
    print()

In [ ]:
AX_COLOR = "#1e2a2c"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"
PLOT3_COLOR = "#b4622a"
PLOT4_COLOR = "#2b5d8a"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# The read-outs of the cell above, one bar per test mesh: bar = mean over the
# seeds, whisker = one std, dots = the seeds. Style helpers from the accumulation
# plot above.
FT_MESH_COLORS = dict(zip(FT_MESHES, (PLOT1_COLOR, PLOT2_COLOR, PLOT3_COLOR, PLOT4_COLOR)))
fig, ax = plt.subplots(figsize=(1500*px, 600*px), constrained_layout=True)
fig.patch.set_alpha(0)
fig.suptitle("Message vs FV Flux, dt = 0.005 Data, V^(1/3), Noise 0.3, No Positions, Cf, "
             "msg_dim = 64, h = 1\n(bar: mean over the seeds, whisker: std, dots: the seeds)",
             fontsize=19, color=AX_COLOR)
width = 0.8 / len(FT_MESHES)
for k, mesh in enumerate(FT_MESHES):
    color = FT_MESH_COLORS[mesh]
    t = ft_flux_tab[:, k]                                    # (S, 6)
    x = np.arange(len(FT_FLUX_COLS)) + (k - (len(FT_MESHES) - 1) / 2) * width
    ax.bar(x, t.mean(0), width=width * 0.9, yerr=t.std(0, ddof=1), color=color, alpha=0.35,
           edgecolor=color, linewidth=1.4, label=mesh,
           error_kw={"ecolor": AX_COLOR, "elinewidth": 1.2, "capsize": 3})
    for c in range(len(FT_FLUX_COLS)):
        ax.plot(np.full(len(t), x[c]), t[:, c], "o", color=color, markersize=5)
ft_style_ax(ax, 14)
ax.set_xticks(np.arange(len(FT_FLUX_COLS)))
ax.set_xticklabels(["$R^2$ flux $F$", "$R^2$ $\\Delta T$", "$R^2$ weight $w$", "$R^2$ node $\\sum_j F$",
                    "$|r|$ best channel", "antisymmetric\nenergy"], fontsize=14)
ax.set_ylim(0, 1.05)
ft_legend(ax.legend(fontsize=12, loc="upper right", ncols=len(FT_MESHES)))

# plt.savefig("../outputs/plots/parametric_history1_fine_flux.png", dpi=300, transparent = True)

### What comes out

Run: `parametric_history1_msg_dim64_residual_noise0.3_cf_vol_cbrt_nopos` (commit `f9709a1` +
uncommitted changes). Recomputed here, the test rollout MAE of all three checkpoints matches the one
saved in training to $4\times10^{-7}$ relative, so these are the models as trained, on their own data.
All three seeds early-stopped (162, 142, 142 epochs), with best val losses within 2 % of each other
($1.03$ – $1.05\times10^{-5}$).

**The three seeds agree closely.** On every mesh and metric the geometric std over the seeds is
×/÷ 1.02 – 1.07 (1.12 for the one-step RMSE on `model_010`). The differences between meshes
are real, not seed noise. Geometric mean over the seeds:

| | `model_003` | `model_007` | `model_010` | `model_011` | all meshes |
|---|---|---|---|---|---|
| one-step RMAE | 8.60e-4 | 6.23e-4 | 5.40e-4 | 8.69e-4 | 7.08e-4 |
| one-step RMSE | 9.36e-6 | 8.01e-6 | 2.16e-6 | 2.60e-6 | 4.53e-6 |
| rollout RMAE | 3.32e-3 | 1.53e-3 | 1.69e-3 | 4.57e-3 | 2.51e-3 |
| rollout RMSE | 5.58e-5 | 2.21e-5 | 2.10e-5 | 1.28e-4 | 4.26e-5 |
| rollout / one-step, RMAE | 3.9 | 2.5 | 3.1 | 5.3 | 3.5 |
| rollout / one-step, RMSE | 6.0 | 2.8 | 9.8 | 50 | 9.4 |

("all meshes": each seed's geometric mean over the four meshes. The ratios are the medians over the seeds.)

**One-step.** The relative error of a single step is below $10^{-3}$ on every mesh (RMAE), with an
RMSE of $2$ – $9\times10^{-6}$. The RMSE splits the meshes into two pairs: `model_003` and `model_007`
are 3 – 4× higher than `model_010` and `model_011`, while their RMAE is similar. Their one-step error
is therefore concentrated on fewer nodes with larger errors. The per-step curves show where:
teacher-forced, the error is largest at the start, at $1.3$ and $1.6\times10^{-2}$ (RMAE) and
$\sim10^{-3}$ (RMSE) at step 1 on `model_003` and `model_007`, against $2.6$ – $2.9\times10^{-3}$ and
$6\times10^{-5}$ – $1.3\times10^{-4}$ on the other two. It then falls over the first ~20 steps (by more
than an order of magnitude in RMAE on `model_003` and `model_007`, 4 – 5× on the other two), to a minimum of $3.4$ – $5.9\times10^{-4}$ around steps 40 – 70, and rises
slowly again up to step 120 (to $1.0\times10^{-3}$ on `model_003` and $1.4\times10^{-3}$ on `model_011`).
The early transient, where $T$ changes fastest, is the hardest part of the trajectory for the
model, and on `model_003` and `model_007` it dominates the squared error.

**Rollout.** Over 120 steps, no seed diverges on any mesh. The rollout RMAE is 2.5 – 5.3× the one-step
one, the RMSE 3 – 50×. The accumulation has three phases:

1. *Steps 1 – ~20: the transient error decays.* $\mathrm{RMAE}_t$ follows the teacher-forced curve
   down from its step-1 value, staying 2 – 4× above it by step 10. The large initial error is not
   carried forward: on `model_003` it falls from $1.3\times10^{-2}$ to $4\times10^{-3}$ by step 10,
   on `model_007` from $1.6\times10^{-2}$ to $1.9\times10^{-3}$. Diffusion damps the error injected
   early, as it damps everything else.
2. *Steps ~20 – 100: a plateau about 3 – 5× above the one-step error.* On `model_003`, `model_007` and
   `model_010`, $\mathrm{RMAE}_t$ stays flat at $1.2$ – $3.4\times10^{-3}$. The ratio to the teacher-forced
   error grows from 2.4 – 4.6 at step 20 to 3.8 – 5.2 at step 60, as the rollout keeps a residual error that the
   one-step model, restarted from the true field, does not have.
3. *Steps ~100 – 120: a slight rise* on every mesh ($\mathrm{RMAE}_t$ +20 – 30 % on `model_007` and
   `model_010`), following the rise of the teacher-forced error.

**`model_011` is the exception: there the error keeps growing.** After a minimum of $1.8\times10^{-3}$
at step 6, $\mathrm{RMAE}_t$ rises steadily to $4.2\times10^{-3}$ at step 40 and $6.3\times10^{-3}$ at
step 120, 3.6× its minimum, while the teacher-forced error at the same steps stays at
$0.6$ – $1.4\times10^{-3}$. $\mathrm{RMSE}_t$ grows from $1.5\times10^{-5}$ to $2\times10^{-4}$ at step
69, up to 155× the teacher-forced one at step 60, and only then falls back to $1.2\times10^{-4}$. This
is a real drift: the model's own predictions take it to states where it does worse than from the
true field, and the per-step error compounds instead of being damped. All three seeds do it the
same way, so it belongs to the mesh (or its trajectory), not to the initialisation. `model_011` has the best one-step RMSE
of the four meshes and the worst rollout, so the one-step error alone does not predict how a mesh
will roll out.

**The message encodes the FV flux, consistently across seeds and meshes.** Averaged over the four
test meshes (mean ± std over the seeds):

- $R^2$ of $F$ = $0.886 \pm 0.017$, $\Delta T$ = $0.951 \pm 0.004$, $w$ = $0.903 \pm 0.005$. The message
  holds $\Delta T$ and the geometric weight $w$ separately, both better than the flux $F$ itself.
- Node level: $R^2$ of $\sum_j F_{ij}$ from the aggregated message = $0.746 \pm 0.026$, lower than the
  edge level, as in the earlier sections. The best single channel reaches $|r| = 0.79 \pm 0.04$ with $F$.
- Antisymmetric energy fraction: $0.261 \pm 0.030$. The message stays mostly symmetric across a face,
  far from the 1.0 of a conservative flux.

Per mesh the read-outs vary less than the errors: $R^2$ of $F$ runs from 0.867 (`model_007`) to 0.902
(`model_011`). `model_011`, the mesh that drifts in the rollout, has one of the best read-outs, so its
drift is not a failure to represent the flux on that geometry. The read-out is done from the true
field at each probed step, so it says nothing about the states the rollout reaches.

## Poster figures

Three compact figures for one row of the poster: the same size, so they line up, with the text
sized for print, and each with the four held-out meshes in one plot. Every one shows last-epoch
checkpoints and keeps the colours of the sections it comes from: green for the controls without
input noise, purple for the noise runs and the FV runs, orange for the no-FV runs.

1. **Rollout error with and without the FV features**
   (`parametric_history1_poster_rollout_fv.pdf`), from the section above
   (`parametric_history1_msg_dim64_residual_noise0.3_cf_vol_nopos`, commit `48186db` +
   uncommitted changes; `parametric_history1_msg_dim64_residual_noise0.3_cf_nopos_nofv` and
   `parametric_history1_msg_dim64_residual_cf_nopos_nofv`, commits as printed there): the test
   rollout RMAE per step, every seed on every mesh thin, the median over all of them bold.
2. **One-step against rollout error with input noise**
   (`parametric_history1_poster_noise_rmae_rmse.pdf`), the bar plot of the input-noise section
   (no noise: `parametric_history1_msg_dim64_residual_cf_nopos`, commit `cf45013` + uncommitted
   changes; noise 0.1 and 0.5: commit `f8f9112` + uncommitted changes; noise 0.3, 1 and 3: commit
   `f8f9112`). Each bar is the geometric mean over every seed and mesh. There are no whiskers and
   no seed dots: control seed 2 diverged, which puts its rollout RMAE at $10^{10}$ –
   $10^{12}$ and its RMSE up to $10^{29}$, and an axis that holds those cannot also show the
   one-step errors. The legend gives the stability instead: the seed–mesh rollouts still below
   the span of $T$ at the last step, as in the tables of that section. The three noise 0.1
   rollouts that overflowed are out of its bars, as there.
3. **Message vs FV flux with and without the FV features and input noise**
   (`parametric_history1_poster_noise_flux_r2.pdf`), the $R^2$ bar plot of the input-noise section
   for no noise against noise 0.3 (5 and 3 seeds), and the same two without the FV features from
   the section above (3 seeds each), with each seed's read-out averaged over the meshes: bar =
   mean over the seeds, whisker = one std, dots = the seeds. The FV runs are the input-noise
   ones, without $\log V$, so the two sides differ only by the 6 FV edge features.

Figure 2 needs only the input-noise section; figures 1 and 3 also need the section above.

In [ ]:
# --- Poster figures: shared style ---------------------------------------------------
# Three figures for one row of the poster, all POSTER_SIZE so that they line up, and
# drawn at the size they are printed. The poster is a portrait A1, 23.39 in (2245 px at
# the poster's 96 px/in) wide: three columns with two gaps of 0.375 in (36 px) are
# 7.55 in (724 px) each, so each figure is a little less, 7.5 x 5.4 in, placed at
# 720 x 518 px. They are saved as vector pdfs. The fonts, lines and markers
# were laid out on a 5 in wide figure and are scaled with it by POSTER_K, so the
# figures look the same, only bigger: the ticks print at 16.5 pt.
#
# The text is in Montserrat, the poster's font, maths included: set in rcParams, so it
# holds for every figure after this cell. The ttf files are in ~/.local/share/fonts;
# addfont registers them without a kernel restart. pdf.fonttype 42 embeds them as
# TrueType, so the text stays text in the poster.
from pathlib import Path
from matplotlib import font_manager
from matplotlib.lines import Line2D

for ttf in sorted(Path.home().glob(".local/share/fonts/Montserrat/*.ttf")):
    font_manager.fontManager.addfont(str(ttf))
plt.rcParams.update({"font.family": "Montserrat",
                     "mathtext.fontset": "custom",
                     "mathtext.rm": "Montserrat",
                     "mathtext.it": "Montserrat:italic",
                     "mathtext.bf": "Montserrat:bold",
                     "mathtext.cal": "Montserrat",     # unused, but else not found
                     "pdf.fonttype": 42})

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
POSTER_SIZE = (7.5, 5.4)                              # inches, every poster figure
POSTER_K = POSTER_SIZE[0] / 5.0                       # scale of the sizes below, 1.5
POSTER_FS = {k: v * POSTER_K for k, v in
             {"tick": 11, "label": 12, "legend": 9.5}.items()}  # pt
# The axes box, the same on every poster figure so that the plots line up: margins
# in inches, each the largest any of the three needs, plus 0.04 in (top: figure 2's
# legend with its title; left: its y tick and axis labels; bottom: figure 1's x
# label; right: its last x tick label).
POSTER_MARGINS = {"left": 0.90, "right": 0.22, "bottom": 0.66, "top": 1.08}


def poster_fig():
    """A transparent POSTER_SIZE figure with one axes in the notebook's colours, at
    POSTER_MARGINS."""
    (W, H), m = POSTER_SIZE, POSTER_MARGINS
    fig = plt.figure(figsize=POSTER_SIZE)
    ax = fig.add_axes([m["left"] / W, m["bottom"] / H, 1 - (m["left"] + m["right"]) / W,
                       1 - (m["bottom"] + m["top"]) / H])
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)
    ax.tick_params(which="both", colors=AX_COLOR, labelsize=POSTER_FS["tick"],
                   width=0.8 * POSTER_K)
    ax.tick_params(which="major", length=3.5 * POSTER_K)
    ax.tick_params(which="minor", length=2.0 * POSTER_K, width=0.6 * POSTER_K)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)
        spine.set_linewidth(0.8 * POSTER_K)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    return fig, ax


def poster_legend(ax, *args, **kwargs):
    """ax.legend in the notebook's legend style, at the poster's font size."""
    legend = ax.legend(*args, fontsize=POSTER_FS["legend"],
                       title_fontsize=POSTER_FS["legend"], **kwargs)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_linewidth(1.0 * POSTER_K)
    legend.get_frame().set_alpha(1)
    for text in [*legend.get_texts(), legend.get_title()]:
        text.set_color(AX_COLOR)
    return legend

In [ ]:
# --- Poster figure 1: rollout error with and without the FV features ---------------
# The test rollout RMAE per step of the section above, last-epoch checkpoints, all
# four held-out meshes in one plot: thin, each seed on each mesh; bold, the median
# over all of them. Leave "no FV, no noise" out of PST_ROLL_KEYS for the two-way
# comparison.
PST_ROLL_KEYS = ["FV + log volume", "no FV", "no FV, no noise"]
PST_LABELS = {"FV + log volume": "FV features + log V",
              "no FV":           "no FV features",
              "no FV, no noise": "no FV features, no noise"}

fig, ax = poster_fig()
n_steps = min(nf[NF_REF][f"rollout_mae_{m}"].shape[1] for m in NF_MESHES)
steps = np.arange(1, n_steps + 1)
lowest = np.inf
for key in PST_ROLL_KEYS:
    color, ls = NF_STYLES[key]
    rel = np.concatenate([rmae_curve(nf[key][f"rollout_mae_{m}"][:, :n_steps], param_T_stats(m)[0])
                          for m in NF_MESHES])           # (seeds x meshes, steps)
    for r in rel:
        ax.plot(steps, r, color=color, linewidth=0.6 * POSTER_K, alpha=0.3, linestyle=ls)
    # an overflowed step (nan) counts as above every finite one
    med = np.median(np.where(np.isfinite(rel), rel, np.inf), 0)
    ax.plot(steps, med, color=color, linewidth=2.4 * POSTER_K, linestyle=ls, label=PST_LABELS[key])
    lowest = min(lowest, np.nanmin(rel))

# The span of T, 300 K, is an RMAE of 0.6 - 1 on the four meshes: drawn at their mean.
span = np.mean([rmae_curve(np.full(n_steps, NS_SPAN), param_T_stats(m)[0]) for m in NF_MESHES], 0)
ax.plot(steps, span, color=AX_COLOR, linewidth=1.2 * POSTER_K, linestyle=":")
ax.annotate(f"span of T ({NS_SPAN:.0f} K)", xy=(steps[-1], span[-1]), xytext=(0, 3 * POSTER_K),
            textcoords="offset points", ha="right", va="bottom",
            fontsize=POSTER_FS["legend"], color=AX_COLOR)
ax.set_yscale("log")
ax.set_ylim(10 ** np.floor(np.log10(lowest)), 2.0)    # a diverging rollout leaves at the top
ax.set_xlim(0, steps[-1])
ax.set_xlabel("time step", fontsize=POSTER_FS["label"])
ax.set_ylabel("rollout RMAE", fontsize=POSTER_FS["label"])
ax.grid(True, which="major", ls="--", linewidth=0.5 * POSTER_K, color=AX_COLOR, alpha=0.5)
handles, labels = ax.get_legend_handles_labels()
handles.append(Line2D([], [], color=AX_COLOR, linewidth=0.6 * POSTER_K, alpha=0.5))
labels.append("one seed on one mesh")
# above the axes, in two columns: the plot has no empty corner that holds it
poster_legend(ax, handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0),
              ncols=2, frameon=False)

# plt.savefig("../outputs/plots/parametric_history1_poster_rollout_fv.pdf", transparent=True)

In [ ]:
# --- Poster figure 2: one-step against rollout error, with input noise ---------------
# The RMAE / RMSE bar plot of the input-noise section, last-epoch checkpoints, with
# the four held-out meshes pooled: each bar is the geometric mean over every seed and
# mesh, of the finite values (seed_stats), as there. No whiskers or seed dots: the
# diverged control seed reaches a rollout RMSE of 1e29, which no axis that also shows
# the one-step errors can hold. The legend gives the stability instead: the
# seed-mesh rollouts still below the span of T at the last step.
pst_bars = [("none", NZ_CTRL, nz_ctrl)] + [(key.split()[1], key, z) for key, z in nz.items()]
pst_metrics = {"rmae_step": "one-step\nRMAE", "rmse_step": "one-step\nRMSE",
               "rmae_roll": "rollout\nRMAE", "rmse_roll": "rollout\nRMSE"}
# (configurations, metrics) geometric means
heights = np.array([[10 ** seed_stats(nz_metric(key, "last", m, NZ_MESHES).ravel(), log=True)[0]
                     for m in pst_metrics] for _, key, _ in pst_bars])

fig, ax = poster_fig()
x = np.arange(len(pst_metrics))
width = 0.84 / len(pst_bars)
# bars on a log axis grow from the bottom of the axis, so the limits come first
floor, top = heights.min() / 4, heights.max() * 4
ax.set_yscale("log")
ax.set_ylim(floor, top)
ax.set_ylabel("relative error", fontsize=POSTER_FS["label"])
for i, (label, key, z) in enumerate(pst_bars):
    final = np.stack([z[f"rollout_mae_{m}"][:, -1] for m in NZ_MESHES], 1)
    below = np.isfinite(final) & (final < NS_SPAN)
    off = (i - (len(pst_bars) - 1) / 2) * width
    ax.bar(x + off, heights[i] - floor, width * 0.92, bottom=floor,
           color=NZ_STYLES["no noise" if key == NZ_CTRL else key][0],
           edgecolor=AX_COLOR, linewidth=0.4 * POSTER_K,
           label=f"{label}  ·  {int(below.sum())}/{below.size}")
ax.axvline(1.5, color=AX_COLOR, linewidth=0.8 * POSTER_K, alpha=0.6)    # one-step | rollout
ax.set_xticks(x)
ax.set_xticklabels(pst_metrics.values(), fontsize=POSTER_FS["tick"])
ax.tick_params(axis="x", length=0)
ax.grid(True, axis="y", which="major", ls="--", linewidth=0.5 * POSTER_K, color=AX_COLOR, alpha=0.5)
ax.set_axisbelow(True)
# above the axes, as in figures 1 and 3: two rows of three, read along the rows
# (legend columns fill top to bottom)
handles, labels = ax.get_legend_handles_labels()
order = [i + j for j in range(3) for i in (0, 3)]
poster_legend(ax, [handles[i] for i in order], [labels[i] for i in order],
              loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, frameon=False,
              title="input noise  ·  bounded rollouts")

# plt.savefig("../outputs/plots/parametric_history1_poster_noise_rmae_rmse.pdf", transparent=True)

In [ ]:
# --- Poster figure 3: message vs FV flux, with and without the FV features and noise -
# The R^2 bar plot of the input-noise section, no noise against noise 0.3, and the
# same two without the FV features (the section above), last-epoch checkpoints, with
# each seed's read-out averaged over the four held-out meshes: bar = mean over the
# seeds, whisker = one std, dots = the seeds. The FV runs are the input-noise ones,
# without log V: they differ from the no-FV runs only by the 6 FV edge features.
PST_R2_TARGETS = [r"$F_{ij}$", r"$\Delta T_{ij}$", r"$w_{ij}$", r"$\sum_j F_{ij}$"]
pst_r2_rows = [(nz_flux_tab[NZ_CTRL],           "FV, no noise",     NZ_STYLES["no noise"][0]),
               (nz_flux_tab["noise 0.3"],       "FV, noise 0.3",    NZ_STYLES["noise 0.3"][0]),
               (nf_flux_tab["no FV, no noise"], "no FV, no noise",  NF_STYLES["no FV, no noise"][0]),
               (nf_flux_tab["no FV"],           "no FV, noise 0.3", NF_STYLES["no FV"][0])]

fig, ax = poster_fig()
x = np.arange(len(PST_R2_TARGETS))
width = 0.84 / len(pst_r2_rows)
for i, (tab, label, color) in enumerate(pst_r2_rows):
    t = tab[:, :, :4]                                    # (S, M, 4) R^2
    t = t[~np.isnan(t).any((1, 2))].mean(1)              # (S, 4), seeds with every mesh
    off = (i - (len(pst_r2_rows) - 1) / 2) * width
    ax.bar(x + off, t.mean(0), width * 0.92, color=color,
           label=f"{label} ({len(t)} seeds)",
           yerr=t.std(0, ddof=1), error_kw=dict(ecolor=AX_COLOR, elinewidth=1.0 * POSTER_K,
                         capsize=2.5 * POSTER_K, capthick=1.0 * POSTER_K))
    for row in t:
        ax.plot(x + off, row, "o", color=AX_COLOR, markersize=2.6 * POSTER_K, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(PST_R2_TARGETS, fontsize=POSTER_FS["label"])
ax.tick_params(axis="x", length=0)
ax.set_ylim(0, 1.05)
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_ylabel(r"$R^2$ of the linear read-out", fontsize=POSTER_FS["label"])
ax.grid(True, axis="y", ls="--", linewidth=0.5 * POSTER_K, color=AX_COLOR, alpha=0.5)
ax.set_axisbelow(True)
# above the axes, in two columns (FV | no FV), as in figure 1
poster_legend(ax, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, frameon=False)

# plt.savefig("../outputs/plots/parametric_history1_poster_noise_flux_r2.pdf", transparent=True)

In [ ]:
# --- The test rollouts' window: how the temperature evolves -------------------------
# Not a poster figure. The true field over the 120 steps of the test rollouts on each
# held-out mesh, T_seq[0 .. 120] (rollout step t is compared with T_seq[t], history
# = 1; step 0 is the initial condition). Left: the volume-weighted mean T over the
# internal cells. Right: the volume-weighted mean |T_t - T_t-1|, how much the field
# moves in step t. Cell volumes from the log-volume section (vol_cell_V).
TW_STEPS = 120
TW_EARLY = 20                    # the window of the FV advantage

fig, axs = plt.subplots(1, 2, figsize=(1500*px, 550*px), constrained_layout=True)
print(f"share of the {TW_STEPS} steps' change in the first {TW_EARLY}:")
for mesh in NF_MESHES:
    graph, T_seq = VOL_LOAD["no volume"](mesh)
    n_int = int((graph.node_attr[:, 0] == 1.0).sum())
    V = vol_cell_V[mesh]
    assert len(V) == n_int, f"{mesh}: {n_int} internal nodes, {len(V)} cells"
    T = T_seq[:TW_STEPS + 1, :n_int].double().numpy()        # (steps + 1, cells)
    T_mean = T @ V / V.sum()
    dT = np.abs(np.diff(T, axis=0)) @ V / V.sum()            # (steps,), step t = dT[t - 1]
    axs[0].plot(np.arange(TW_STEPS + 1), T_mean, linewidth=2, label=mesh)
    axs[1].plot(np.arange(1, TW_STEPS + 1), dT, linewidth=2, label=mesh)
    print(f"  {mesh}: mean T {T_mean[0]:.1f} -> {T_mean[TW_EARLY]:.1f} -> {T_mean[-1]:.1f} K, "
          f"{(T_mean[TW_EARLY] - T_mean[0]) / (T_mean[-1] - T_mean[0]):.0%} of its change, "
          f"{dT[:TW_EARLY].sum() / dT.sum():.0%} of the summed |dT|")

axs[0].set_ylabel("mean T [K]")
axs[1].set_ylabel("mean |T$_t$ - T$_{t-1}$| [K]")
# axs[1].set_yscale("log")
for ax in axs:
    ax.axvspan(0, TW_EARLY, color=AX_COLOR, alpha=0.08, linewidth=0)
    ax.set_xlim(0, TW_STEPS)
    ax.set_xlabel("time step")
    ax.grid(True, which="major", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.5)
axs[0].legend()
plt.show()
